# Condition-gated spiking neural P forecasters: the complete experiment

This notebook reproduces every experiment, table and figure of the paper, from the raw series to the final LaTeX tables: the learned member **ALSTM-SNP**, the closed-form members **TRACE-SNP v1**, **TRACE-SNP v2** and **TRACE-SNP-AF**, and every baseline, on 11 univariate series with one walk-forward protocol.

**How to run:** run all cells, top to bottom, on a CUDA GPU. Everything is written to the local folder `results/` (set in cell 0). The long cells are resumable: if the kernel restarts or the time budget in cell 0 is reached, run all cells again and finished work is reused.

**Smoke test (minutes, CPU is fine):** set the environment variable `SMOKE=1` before starting the kernel. Then 3 seeds, 3 epochs, 2 lags and the last 400 points of each series are used, and every output carries the suffix `_smoke`.

| cells | what they do |
|---|---|
| 0–6 | configuration, setup and GPU energy meter, the results folder, the 11 datasets, the 9 networks, helpers, the walk-forward training harness |
| 7 | trains ALSTM-SNP and the 8 baseline networks on every dataset (60 seeds × 5 folds × 6 lags) |
| 8 | TRACE-SNP v1, its binary-spike version, six ablations, NG-RC, DLinear, ridge AR and persistence |
| 9 | the networks at horizons 6 and 24 from their saved checkpoints |
| 10 | TRACE-SNP v2 and its binary-spike version |
| 11 | every model on identical test targets: leaderboards, Diebold–Mariano / DeLong tests with Holm correction, noise robustness, energy |
| 12 | the 11 datasets together, inventory and the parameters actually selected |
| 13 | firing-condition ablation: gated receptive fields against a dense, non-gated RBF basis |
| 14 | ranking over the N = 11 datasets: Friedman, Iman–Davenport, Nemenyi, Wilcoxon–Holm, Bayesian signed-rank |
| 15 | energy per ensemble member and per ensemble forecast |
| 16 | the analysis figures |
| 17 | receptive-field shape × gating study, accuracy against the number of active neurons |
| 18 | TRACE-SNP-AF, TRACE-SNP v2 with an adaptive firing condition |
| 19 | the numbers the paper quotes: variance over seeds and blocks, uncertainty bands of the delay functions |
| 20–25 | control experiments: tuned networks, directly trained networks, ALSTM-SNP diagnostics, echo-type SNP reservoir, common-protocol energy |
| 26–31 | extended comparison: classical statistical forecasters (ARIMA, simple and damped exponential smoothing, seasonal naive, training median) on the same folds, constant forecasts, Lyapunov times of the chaotic generators, operation-count check, data manifest and stored-forecast checks |
| 32 | every figure and table of the paper and its supplementary material, including the extended comparison |

See `README.md` for the requirements, the data sources, run times and the map from cells to the tables and figures of the paper.

## Part A — Setup, data, models and the walk-forward harness (cells 0–6)
Run these cells first in every session. Nothing here trains a model.

In [ ]:
# ============================================================
# 0. CONFIG — everything you might want to change is in this cell
# ============================================================
import os
SMOKE = os.environ.get('SMOKE', '0') == '1'            # tiny end-to-end test: 3 seeds, 3 epochs, lags 1 and 5, short series

# ── storage (everything is local; nothing is uploaded anywhere) ──
LOCAL_ROOT     = os.environ.get('RESULTS_DIR', 'results')   # every checkpoint, prediction, table and figure goes here
DATA_DIR       = os.environ.get('DATA_DIR', 'data')         # optional local copies of the series (see the README)
FRESH_START    = False       # True deletes the checkpoints and results of the runs this session starts, then retrains them

# ── datasets ──
DATASETS_TO_RUN     = None   # None = all 11; or a list of names / slugs, e.g. ['MackeyGlass', 'lorenz']
EXPECTED_N_DATASETS = 11
DATASET_OVERRIDES   = {}     # per-dataset settings keyed by name: 'block_len', 'val_len', 'max_points'
SMOKE_MAX_POINTS    = 400

# ── evaluation protocol (identical for every model) ──
WF_N_FOLDS    = 5
BLOCK_LEN     = 60               # test block length (Mackey-Glass: t782-841 … t1022-1081)
VAL_LEN       = 60
TEST_FRAC_MAX = 0.30             # short series: blocks shrink so the 5 test blocks are ≤ 30 % of the series
MIN_TRAIN_ROWS = 40              # a lag is skipped on a dataset if the first fold has fewer training windows
TARGET_MODE   = 'level'
RIDGE_ALPHA   = 1.0
CLIP_NORM     = 1.0
NOISE_LEVELS  = [0.005, 0.05, 0.10]
N_NOISE_DRAWS = 2 if SMOKE else 10   # input-noise draws per noise level (robustness study)
RATE_TARGET   = 0.2
# Reference pooled baselines of the Mackey-Glass series (data check; mismatch = warning, not an error)
REF_BASELINES = {'*mackey*': {'len': 1082, 'persist': 0.0344, 'naive': 0.2305},
                 'mg*':      {'len': 1082, 'persist': 0.0344, 'naive': 0.2305}}

# ── models — every model runs every lag in COMMON_LAGS ──
COMMON_LAGS = [1, 5, 10, 20, 30, 60]      # window lengths L; every model runs every lag
ALSTM_CFG = dict(name='ALSTM-SNP', folder='alstm_snp', setup='A',
                 batch=8, lr=3e-3, max_epochs=300, patience=40, hidden=8, n_runs=60)
ALSTM_CELL = dict(layers=1, frac_adaptive=0.4, tau_a_range=(2.0, 50.0), tau_m=5.0, gamma_sg=0.3,
                  spiking=True, adaptive=True, strict=True, rate_reg=0.0)
BASELINES = [   # (name, folder) — trained with BASELINE_TRAIN on COMMON_LAGS
    ('LSTM [34]',       'lstm_34'),
    ('GRU [39]',        'gru_39'),
    ('LSTM-SNP [25]',   'lstm_snp_25'),
    ('BiLSTM-SNP [26]', 'bilstm_snp_26'),
    ('Attention',       'attention'),
    ('Informer [42]',   'informer_42'),
    ('Autoformer [40]', 'autoformer_40'),
    ('FEDformer [41]',  'fedformer_41'),
]
BASELINE_TRAIN = dict(batch=1, lr=1e-3, max_epochs=100, patience=10, hidden=8, n_runs=60)
MODELS_TO_RUN  = None            # None = all 9; or a list of names / folders, e.g. ['ALSTM-SNP', 'gru_39']
RUN_TRAINING   = True            # False = no network is trained (cells 7 and 21–23 then only read finished results)

# ── GPU ──
USE_GPU       = True
FOLD_PARALLEL = True             # the 5 folds × 60 seeds of a lag train as one vmapped ensemble of 300 models
RUN_BATCH     = None             # models per ensemble; None = all of a lag (lower only if VRAM is short)
CUDA_GRAPHS   = True             # capture the training step as a CUDA graph (checked against the eager step first)
AMP_DTYPE     = None             # 'bf16' changes results; None = fp32 as in the reported runs
ALLOW_TF32    = False
EVAL_CHUNK    = 2048
N_THREADS     = 4
CKPT_VERSION  = 'v2'
SAVE_PREDS    = True
SHOW_LAG_FIGURES = False         # per (dataset, model, lag) forecast figures are always saved; True also shows them

# ── energy ──
ENERGY_ENABLED   = True
ENERGY_SAMPLE_HZ = 10            # power sampling rate (peak power; the integral if the energy counter is missing)
IDLE_PROBE_S     = 3.0           # idle power measured once at start → 'energy above idle'
INFER_BENCH_S    = 0.5 if SMOKE else 3.0   # inference benchmark length per (dataset, model, lag)

# ── resource budgets / session ──
RAM_CAP_GB    = 30.0
VRAM_CAP_GB   = 70.0
TIME_BUDGET_H = 11.0             # stop cleanly (everything saved) after this long; rerun to continue
print(("SMOKE TEST | " if SMOKE else "") + f"results → {os.path.abspath(LOCAL_ROOT)} | data → {os.path.abspath(DATA_DIR)} | "
      f"training {'on' if RUN_TRAINING else 'off'}")

In [ ]:
# ============================================================
# 1. SETUP — torch, device, memory helpers, time budget, figure display, GPU energy meter
# ============================================================
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # before torch is imported
import re, json, shutil, glob, time, math, random, hashlib, fnmatch, gc, ctypes, threading, subprocess, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from scipy import stats
from math import sqrt
import matplotlib.pyplot as plt
from torch.func import stack_module_state, functional_call, vmap

T_START = time.time()
device = torch.device('cpu')
if USE_GPU and torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability(0)
    if f"sm_{_cap[0]}{_cap[1]}" in torch.cuda.get_arch_list():
        device = torch.device('cuda')
        _total_gb = torch.cuda.mem_get_info()[1] / 2**30
        torch.cuda.set_per_process_memory_fraction(min(1.0, min(VRAM_CAP_GB, 0.95 * _total_gb) / _total_gb))
        torch.backends.cuda.matmul.allow_tf32 = ALLOW_TF32
        torch.backends.cudnn.allow_tf32 = ALLOW_TF32
        print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{_cap[0]}{_cap[1]}) | "
              f"VRAM {_total_gb:.1f} GB (allocator capped at {min(VRAM_CAP_GB, 0.95 * _total_gb):.1f} GB)")
    else:
        print(f"WARNING: {torch.cuda.get_device_name(0)} (sm_{_cap[0]}{_cap[1]}) is not supported by this PyTorch build "
              f"({torch.cuda.get_arch_list()}) — running on CPU.")
elif USE_GPU:
    print("WARNING: no GPU — running on CPU (energy is not measured on CPU).")
if device.type == 'cpu':
    torch.set_num_threads(N_THREADS)
print(f"PyTorch {torch.__version__} | device: {device}")


def rss_gb():
    try:
        with open('/proc/self/status') as fh:
            for line in fh:
                if line.startswith('VmRSS:'):
                    return int(line.split()[1]) / 2**20
    except OSError:
        pass
    return float('nan')


def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except OSError:
        pass
    return rss_gb()


def vram_gb():
    return torch.cuda.max_memory_allocated() / 2**30 if device.type == 'cuda' else 0.0


class TimeBudgetReached(Exception):
    pass


def check_time_budget():
    if time.time() - T_START > TIME_BUDGET_H * 3600:
        raise TimeBudgetReached


# figures: shown inline in marimo or Jupyter, always closed afterwards (they are saved to disk by the caller)
try:
    import marimo as MO
    IN_MARIMO = MO.running_in_notebook()
except Exception:
    MO, IN_MARIMO = None, False


def show(fig):
    if IN_MARIMO:
        MO.output.append(MO.as_html(fig))
    else:
        try:
            from IPython import get_ipython
            if get_ipython() is not None:
                from IPython.display import display
                display(fig)
        except Exception:
            pass
    plt.close(fig)


class GPUEnergyMeter:
    """Board energy of the GPU used here, read through NVML.
    'counter': the cumulative energy counter (nvmlDeviceGetTotalEnergyConsumption, mJ; Volta and newer) — exact.
    'power-integral': fallback, power samples at ENERGY_SAMPLE_HZ integrated with the trapezoid rule.
    A daemon thread samples power in both cases (peak power, and the fallback integral). On CPU / without NVML every
    energy value is NaN, but durations are still recorded."""

    def __init__(self, enabled, hz, idle_probe_s):
        self.ok, self.method, self.name, self.reason = False, 'none', '', ''
        self.idle_w, self.int_j, self.peak_w = float('nan'), 0.0, 0.0
        self.lock = threading.Lock()
        if not enabled:
            self.reason = 'disabled in config'; return
        if device.type != 'cuda':
            self.reason = 'no GPU'; return
        try:
            import pynvml
        except ImportError:
            for cmd in ([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-ml-py'],
                        ['uv', 'pip', 'install', '-q', '--python', sys.executable, 'nvidia-ml-py']):   # some hosted notebooks use uv
                try:
                    subprocess.run(cmd, check=True, timeout=600, capture_output=True)
                    break
                except Exception:
                    pass
            try:
                import pynvml
            except ImportError:
                self.reason = 'pynvml not available (pip install nvidia-ml-py)'; return
        try:
            pynvml.nvmlInit()
            self.nv = pynvml
            self.h = self._handle()
            nm = pynvml.nvmlDeviceGetName(self.h)
            self.name = nm.decode() if isinstance(nm, bytes) else str(nm)
        except Exception as e:
            self.reason = f'NVML init failed ({type(e).__name__}: {e})'; return
        try:
            pynvml.nvmlDeviceGetTotalEnergyConsumption(self.h)
            self.method = 'counter'
        except Exception:
            self.method = 'power-integral'
        try:
            self._power_w()
        except Exception as e:
            if self.method != 'counter':
                self.reason = f'no power reading ({type(e).__name__})'; return
        self.ok = True
        self.stop_evt = threading.Event()
        self.thread = threading.Thread(target=self._loop, args=(1.0 / hz,), daemon=True)
        self.thread.start()
        t0, e0 = self.snapshot()
        time.sleep(idle_probe_s)
        t1, e1 = self.snapshot()
        self.idle_w = (e1 - e0) / (t1 - t0)

    def _handle(self):
        nv = self.nv
        try:   # match the CUDA device to its NVML handle by PCI address (the two orderings can differ)
            p = torch.cuda.get_device_properties(device)
            return nv.nvmlDeviceGetHandleByPciBusId(f"{p.pci_domain_id:08X}:{p.pci_bus_id:02X}:{p.pci_device_id:02X}.0")
        except Exception:
            return nv.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())

    def _power_w(self):
        return self.nv.nvmlDeviceGetPowerUsage(self.h) / 1000.0

    def _loop(self, dt):
        try:
            t_prev, w_prev = time.perf_counter(), self._power_w()
        except Exception:
            t_prev, w_prev = time.perf_counter(), 0.0
        while not self.stop_evt.wait(dt):
            try:
                w = self._power_w()
            except Exception:
                continue
            t = time.perf_counter()
            with self.lock:
                self.int_j += 0.5 * (w + w_prev) * (t - t_prev)
                self.peak_w = max(self.peak_w, w)
            t_prev, w_prev = t, w

    def snapshot(self):
        # (time s, cumulative energy J) after all queued GPU work has finished
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t = time.perf_counter()
        if not self.ok:
            return t, float('nan')
        if self.method == 'counter':
            return t, self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h) / 1000.0
        with self.lock:
            return t, self.int_j

    def take_peak(self):
        with self.lock:
            p, self.peak_w = self.peak_w, 0.0
        return p if self.ok else float('nan')


class EnergySegment:
    """Accumulates energy and time over resume()/pause() spans (so checkpoint writes are left out).
    prev: totals carried over from an interrupted session (stored in the group checkpoint)."""

    def __init__(self, meter, prev=None):
        p = prev or {}
        self.m, self.j, self.s, self.peak, self.t0 = meter, p.get('j', 0.0), p.get('s', 0.0), p.get('peak_w', 0.0), None

    def resume(self):
        self.m.take_peak()
        self.t0, self.e0 = self.m.snapshot()

    def pause(self):
        if self.t0 is None:
            return
        t, e = self.m.snapshot()
        self.j += e - self.e0
        self.s += t - self.t0
        self.peak = max(self.peak, self.m.take_peak()) if self.m.ok else float('nan')
        self.t0 = None

    def state(self):
        return {'j': float(self.j), 's': float(self.s), 'peak_w': float(self.peak)}


METER = GPUEnergyMeter(ENERGY_ENABLED, ENERGY_SAMPLE_HZ, IDLE_PROBE_S)
print(f"Energy: {METER.method} on {METER.name} | idle {METER.idle_w:.1f} W" if METER.ok
      else f"Energy: not measured ({METER.reason}) — energy columns will be NaN")
print(f"RAM in use {rss_gb():.2f} GB | disk free {shutil.disk_usage('.').free / 2**30:.0f} GB")

In [ ]:
# ============================================================
# 2. Result store — a local folder (LOCAL_ROOT)
#    Every cell saves through this object: add() marks a written file, remove() deletes one, fetch() returns the local
#    path of a stored file (or None), matching() lists stored files by pattern. Nothing leaves the machine, so push()
#    only applies pending deletions. Rerunning any cell reuses what is already in LOCAL_ROOT, which is what makes the
#    long cells resumable after a kernel restart or the time budget.
# ============================================================
class LocalStore:
    def __init__(self, root):
        self.root, self.enabled = root, False
        self.dirty, self.gone, self.remote = set(), set(), set()
        os.makedirs(root, exist_ok=True)
        self.refresh()
        print(f"Results folder: {os.path.abspath(root)} ({len(self.remote):,} files)")

    def refresh(self):
        self.remote = {self.rel(p) for p in glob.glob(os.path.join(self.root, '**', '*'), recursive=True) if os.path.isfile(p)}

    def rel(self, path):
        return os.path.relpath(path, self.root).replace(os.sep, '/')

    def matching(self, pattern):
        self.refresh()
        return sorted(f for f in self.remote if fnmatch.fnmatch(f, pattern))

    def pull(self, pattern):
        return []                       # everything is already local

    def fetch(self, rel_path):
        local = os.path.join(self.root, rel_path)
        return local if os.path.exists(local) else None

    def add(self, path):
        r = self.rel(path); self.dirty.add(r); self.gone.discard(r)

    def remove(self, path):
        if os.path.exists(path):
            os.remove(path)
        r = self.rel(path); self.dirty.discard(r); self.gone.add(r)

    def push(self, msg='update', force=False):
        for r in self.gone:
            p = os.path.join(self.root, r)
            if os.path.isfile(p):
                os.remove(p)
        self.dirty.clear(); self.gone.clear()
        return True


store = LocalStore(LOCAL_ROOT)

In [ ]:
# ============================================================
# 3. Datasets — the 11 univariate series, each → a 1-D chronological series + its 5 walk-forward folds
#    Real:     EPL (English Premier League, per-match series of 3,800 consecutive matches; local file, see the README),
#              Sunspots (monthly mean total sunspot number, WDC-SILSO, Jan 1749 – Jan 2021, 3,265 months),
#              ETTh1, ETTh2 (oil temperature 'OT', hourly, last 4,000 points), Melbourne daily minimum temperature,
#              Mauna Loa CO2 weekly, El Niño monthly sea-surface temperature (the last two via statsmodels).
#    Chaotic:  Mackey–Glass (τ = 17, 1,082 points), Lorenz-63 x(t), Lorenz-63 with 1 % observation noise, Rössler x(t).
#    A file DATA_DIR/<name>.csv (columns t, value) always takes precedence over the download / generator, so an exact
#    copy of a series can be dropped in. The series actually used are written once to LOCAL_ROOT/data_main/ (EPL,
#    Sunspots, Mackey–Glass) and LOCAL_ROOT/extra_data/ (the other eight) and reused from there on every rerun.
#    Folds: the last 5 blocks of BLOCK_LEN points are the test blocks (shorter for short series: ≤ 30 % of the series);
#    each is preceded by a validation block of VAL_LEN points, and every earlier point is training data.
#    Interior gaps are filled linearly, leading / trailing gaps dropped.
# ============================================================
INDEX_NAMES = {'t', 'time', 'index', 'idx', 'id', 'date', 'datetime', 'timestamp', 'year', 'month', 'day', 'hour',
               'minute', 'step', 'unnamed: 0'}
VALUE_NAMES = ('value', 'values', 'y', 'target')


def slugify(s):
    return re.sub(r'[^A-Za-z0-9]+', '_', str(s)).strip('_').lower()


def _is_num(s):
    try:
        float(s); return True
    except (TypeError, ValueError):
        return False


def _delimiter(path):
    import csv
    with open(path, newline='', errors='replace') as fh:
        sample = fh.read(20000)
    try:
        return csv.Sniffer().sniff(sample, delimiters=',;\t|').delimiter
    except csv.Error:
        return ','


def read_series(path, column=None):
    sep = _delimiter(path)
    df = pd.read_csv(path, sep=sep)
    if len(df.columns) and all(_is_num(c) for c in df.columns):              # no header row
        df = pd.read_csv(path, sep=sep, header=None)
        df.columns = [f'col{i}' for i in range(df.shape[1])]
    num = df.apply(lambda c: pd.to_numeric(c, errors='coerce'))
    numeric = [c for c in df.columns if num[c].notna().mean() >= 0.9]

    def index_like(c):
        if str(c).strip().lower() in INDEX_NAMES:
            return True
        v = num[c].dropna().to_numpy()
        if len(v) < 3:
            return False
        d = np.diff(v)
        return bool(np.all(v == np.round(v)) and np.all(d == d[0]) and d[0] != 0)

    lower = {str(c).strip().lower(): c for c in df.columns}
    mg_sheet = column is None and 't' in lower and ('t-taw' in lower or 't-tau' in lower)   # the Mackey-Glass xlsx layout
    if mg_sheet:
        column = lower['t']
    if column is not None:
        if column not in df.columns:
            raise KeyError(f"column {column!r} not in {list(df.columns)}")
        col = column
    else:
        named = [c for c in numeric if str(c).strip().lower() in VALUE_NAMES]
        rest = [c for c in numeric if not index_like(c)]
        col = named[0] if named else (rest[-1] if rest else (numeric[-1] if numeric else None))
        if col is None:
            raise ValueError(f"no numeric column in {list(df.columns)}")
    s = num[col]
    if mg_sheet:   # that sheet pads the series with zeros: keep from the first to the last non-zero value
        nz = np.flatnonzero(s.fillna(0).to_numpy())
        s = s.iloc[nz[0]:nz[-1] + 1]
    first, last = s.first_valid_index(), s.last_valid_index()
    s = s.loc[first:last]
    n_gap = int(s.isna().sum())
    s = s.interpolate(method='linear') if n_gap else s
    return s.to_numpy(dtype=np.float64), col, list(df.columns), n_gap


def make_folds(n, block, val):
    out = []
    for k in range(WF_N_FOLDS):
        te_end = n - (WF_N_FOLDS - 1 - k) * block
        te_start = te_end - block
        lbl = f"t{te_start}-{te_end - 1}"
        out.append({'label': lbl, 'tag': f"test{lbl}", 'va_start': te_start - val, 'te_start': te_start, 'te_end': te_end})
    return out


def pooled_reference(values, folds):
    y = np.concatenate([values[f['te_start']:f['te_end']] for f in folds])
    p = np.concatenate([values[f['te_start'] - 1:f['te_end'] - 1] for f in folds])
    nv = np.concatenate([np.full(f['te_end'] - f['te_start'], values[:f['te_start']].mean()) for f in folds])
    return float(np.sqrt(np.mean((y - p) ** 2))), float(np.sqrt(np.mean((y - nv) ** 2)))




# ── how each series is obtained ──
SERIES_SOURCES = {   # name: (source, column, keep the last N points, kind)
    'EPL':            ('local', None, None, 'real'),
    'MackeyGlass':    ('generate:mackey_glass', None, None, 'chaotic'),
    'Sunspots':       ('silso', None, None, 'real'),
    'etth1':          ('https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv', 'OT', 4000, 'real'),
    'etth2':          ('https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv', 'OT', 4000, 'real'),
    'melbourne_temp': ('https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv', 'Temp', None, 'real'),
    'co2_weekly':     ('statsmodels:co2', None, None, 'real'),
    'elnino_monthly': ('statsmodels:elnino', None, None, 'real'),
    'lorenz':         ('generate:lorenz', None, 3000, 'chaotic'),
    'lorenz_noisy':   ('generate:lorenz_noisy', None, 3000, 'chaotic'),
    'rossler':        ('generate:rossler', None, 3000, 'chaotic'),
}
MAIN_SERIES   = ('EPL', 'MackeyGlass', 'Sunspots')     # stored under data_main/, the rest under extra_data/
SILSO_URL     = 'https://www.sidc.be/SILSO/INFO/snmtotcsv.php'
SILSO_LAST    = (2021, 1)                               # last month of the sunspot series used


def mackey_glass(n=1082, tau=17, beta=0.2, gamma=0.1, power=10, x0=(1.2, 1.0)):
    # unit-step Euler recursion x(t+1) = x(t) + β x(t−τ) / (1 + x(t−τ)^n) − γ x(t), with x(t) = 0 for t < 0
    x = list(x0)
    while len(x) < n:
        k = len(x) - 1
        xd = x[k - tau] if k - tau >= 0 else 0.0
        x.append(x[k] + beta * xd / (1 + xd ** power) - gamma * x[k])
    return np.array(x[:n])


def lorenz(n, dt=0.01, sub=3, burn=500):
    v = np.array([1.0, 1.0, 1.0]); out = []
    f = lambda v: np.array([10 * (v[1] - v[0]), v[0] * (28 - v[2]) - v[1], v[0] * v[1] - 8 / 3 * v[2]])
    for i in range((n + burn) * sub):
        k1 = f(v); k2 = f(v + dt / 2 * k1); k3 = f(v + dt / 2 * k2); k4 = f(v + dt * k3); v = v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        if i % sub == 0:
            out.append(v[0])
    return np.array(out[burn:burn + n])


def rossler(n, dt=0.05, sub=2, burn=1000):
    v = np.array([1.0, 1.0, 1.0]); out = []
    f = lambda v: np.array([-v[1] - v[2], v[0] + 0.2 * v[1], 0.2 + v[2] * (v[0] - 5.7)])
    for i in range((n + burn) * sub):
        k1 = f(v); k2 = f(v + dt / 2 * k1); k3 = f(v + dt / 2 * k2); k4 = f(v + dt * k3); v = v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        if i % sub == 0:
            out.append(v[0])
    return np.array(out[burn:burn + n])


def statsmodels_series(which):
    try:
        import statsmodels.api as sm
    except ImportError:
        for cmd in ([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'],
                    ['uv', 'pip', 'install', '-q', '--python', sys.executable, 'statsmodels']):
            try:
                subprocess.run(cmd, check=True, timeout=900, capture_output=True); break
            except Exception:
                pass
        import statsmodels.api as sm
    if which == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].interpolate().to_numpy(float)
    if which == 'elnino':
        return sm.datasets.elnino.load_pandas().data.iloc[:, 1:].to_numpy(float).ravel()
    raise ValueError(which)


def silso_sunspots():
    import urllib.request, io
    with urllib.request.urlopen(SILSO_URL, timeout=120) as r:
        df = pd.read_csv(io.BytesIO(r.read()), sep=';', header=None)
    df = df[(df[0] < SILSO_LAST[0]) | ((df[0] == SILSO_LAST[0]) & (df[1] <= SILSO_LAST[1]))]
    return df[3].to_numpy(float)


def obtain_series(name, src, col):
    local = os.path.join(DATA_DIR, f"{name}.csv")
    if os.path.exists(local):                                     # an exact copy supplied by the user wins
        print(f"{name}: using {local}")
        return read_series(local, None)[0]
    if src == 'local':
        raise FileNotFoundError(f"{local} not found — this series has no public download (see the README)")
    if src == 'silso':
        return silso_sunspots()
    if src.startswith('generate:'):
        kind = src.split(':')[1]
        if kind == 'mackey_glass':
            return mackey_glass()
        if kind == 'lorenz':
            return lorenz(4000)
        if kind == 'lorenz_noisy':
            x = lorenz(4000)
            return x + 0.01 * x.std() * np.random.default_rng(1).normal(size=len(x))
        if kind == 'rossler':
            return rossler(4000)
    if src.startswith('statsmodels:'):
        return statsmodels_series(src.split(':')[1])
    import urllib.request, io
    with urllib.request.urlopen(src, timeout=120) as r:
        df = pd.read_csv(io.BytesIO(r.read()))
    return pd.to_numeric(df[col], errors='coerce').interpolate().to_numpy(float)


def load_datasets():
    out = []
    for name, (src, col, keep, kind) in SERIES_SOURCES.items():
        slug = slugify(name)
        sub = 'data_main' if name in MAIN_SERIES else 'extra_data'
        path = os.path.join(LOCAL_ROOT, sub, f"{name if name in MAIN_SERIES else slug}.csv")
        if not os.path.exists(path):
            try:
                x = obtain_series(name, src, col)
            except Exception as e:
                print(f"SKIP {name}: could not obtain it ({type(e).__name__}: {str(e)[:200]})"); continue
            if keep:
                x = x[-keep:]
            os.makedirs(os.path.dirname(path), exist_ok=True)
            pd.DataFrame({'t': np.arange(len(x)), 'value': x}).to_csv(path, index=False)
            store.add(path)
        values = read_series(path, 'value')[0]
        n_raw = len(values)
        ov = DATASET_OVERRIDES.get(name, DATASET_OVERRIDES.get(slug, {}))
        max_points = ov.get('max_points')
        if SMOKE:
            max_points = min(max_points or SMOKE_MAX_POINTS, SMOKE_MAX_POINTS)
        if max_points and n_raw > max_points:
            values = values[-max_points:]
        n = len(values)
        block = int(ov.get('block_len', min(BLOCK_LEN, max(5, int(TEST_FRAC_MAX * n // WF_N_FOLDS)))))
        val = int(ov.get('val_len', min(VAL_LEN, block)))
        folds = make_folds(n, block, val)
        persist, naive = pooled_reference(values, folds)
        check = ''
        for pat, ref in REF_BASELINES.items():
            if fnmatch.fnmatch(slug, pat) and n == ref['len']:
                ok = abs(persist - ref['persist']) <= 5e-4 and abs(naive - ref['naive']) <= 5e-4
                check = ('reproduces' if ok else 'DOES NOT reproduce') + \
                        f" the reference Mackey-Glass baselines (persist {ref['persist']}, naive {ref['naive']})"
                break
        out.append({'name': name if name in MAIN_SERIES else slug, 'slug': slug, 'file': store.rel(path), 'column': 'value',
                    'columns': ['t', 'value'], 'values': values, 'n': n, 'n_raw': n_raw, 'n_gap': 0, 'block': block,
                    'val': val, 'folds': folds, 'persist_rmse': persist, 'naive_rmse': naive, 'check': check,
                    'extra': name not in MAIN_SERIES, 'kind': kind})
    return out


ALL_DATASETS = load_datasets()
DATASETS = [d for d in ALL_DATASETS if DATASETS_TO_RUN is None or d['slug'] in DATASETS_TO_RUN or d['name'] in DATASETS_TO_RUN]
assert DATASETS, "no dataset selected"
if len(DATASETS) != EXPECTED_N_DATASETS and DATASETS_TO_RUN is None:
    print(f"WARNING: expected {EXPECTED_N_DATASETS} datasets, have {len(DATASETS)}: every dataset-level test uses N = {len(DATASETS)}")
for _d in DATASETS:
    print(f"\n{_d['name']} ({_d['kind']}, {_d['file']}): {_d['n']} points" + (f" (last {_d['n']} of {_d['n_raw']})" if _d['n'] != _d['n_raw'] else '')
          + f"\n   blocks: test {_d['block']} × {WF_N_FOLDS} ({', '.join(f['label'] for f in _d['folds'])}), val {_d['val']}"
          + f"\n   pooled test RMSE: persistence {_d['persist_rmse']:.4g} | naive {_d['naive_rmse']:.4g}"
          + (f"\n   data check: {_d['check']}" if _d['check'] else ''))

_fig, _axes = plt.subplots(len(DATASETS), 1, figsize=(14, 1.7 * len(DATASETS)), squeeze=False)
for _ax, _d in zip(_axes[:, 0], DATASETS):
    _ax.plot(_d['values'], color='#2a78d6' if _d['kind'] == 'real' else '#d55e00', lw=0.7)
    _ax.axvspan(_d['folds'][0]['va_start'], _d['folds'][0]['te_start'], color='#9a9890', alpha=0.15, lw=0)
    _ax.axvspan(_d['folds'][0]['te_start'], _d['n'], color='#9a9890', alpha=0.35, lw=0)
    _ax.set_title(f"{_d['name']} ({_d['kind']}, {_d['n']} points; shaded: first validation block, then the 5 test blocks)",
                  fontsize=9, loc='left')
    _ax.grid(True, alpha=0.25); _ax.spines[['top', 'right']].set_visible(False)
_fig.tight_layout()
os.makedirs(os.path.join(LOCAL_ROOT, 'comparison'), exist_ok=True)
_fig.savefig(os.path.join(LOCAL_ROOT, 'comparison', f"datasets{'_smoke' if SMOKE else ''}.png"), dpi=130, bbox_inches='tight')
store.add(os.path.join(LOCAL_ROOT, 'comparison', f"datasets{'_smoke' if SMOKE else ''}.png"))
show(_fig)

In [ ]:
# ============================================================
# 4. Models — ALSTM-SNP (as in alstm_snp.py) and the 8 baselines, exactly as in the reference ALSTM-SNP implementation
#    SpikeFn is in the torch.func form (setup_context + generate_vmap_rule) so all seeds train in parallel;
#    its forward (Heaviside) and surrogate backward γ·max(0, 1−|v|) are unchanged.
# ============================================================
class SpikeFn(torch.autograd.Function):
    generate_vmap_rule = True

    @staticmethod
    def forward(v, gamma):
        return (v > 0).to(v.dtype)

    @staticmethod
    def setup_context(ctx, inputs, output):
        v, gamma = inputs
        ctx.save_for_backward(v)
        ctx.gamma = gamma

    @staticmethod
    def backward(ctx, grad_out):
        (v,) = ctx.saved_tensors
        sg = ctx.gamma * torch.clamp(1.0 - v.abs(), min=0.0)
        return grad_out * sg, None


spike = SpikeFn.apply


def _inv_softplus(y: torch.Tensor) -> torch.Tensor:
    return y + torch.log(-torch.expm1(-y))


class ALSTMSNPCell(nn.Module):
    def __init__(self, input_size, hidden_size, frac_adaptive=0.4, tau_a_range=(2.0, 50.0), tau_m=5.0, beta_init=1.7,
                 b0_init=0.1, gamma=0.3, spiking=True, adaptive=True, learn_tau=True, strict=True, no_self_syn=None,
                 local_v=None):
        super().__init__()
        H = hidden_size
        self.input_size, self.hidden_size = input_size, H
        self.gamma, self.spiking, self.adaptive = gamma, spiking, adaptive
        self.no_self_syn = strict if no_self_syn is None else no_self_syn
        self.local_v = strict if local_v is None else local_v

        self.W = nn.Linear(input_size, 4 * H)
        self.U = nn.Linear(H, 4 * H, bias=False)
        self.register_buffer("syn_mask", (1.0 - torch.eye(H)).repeat(4, 1) if self.no_self_syn else torch.ones(4 * H, H))
        if self.local_v:
            self.v = nn.Parameter(torch.zeros(3 * H))
        else:
            self.V = nn.Linear(H, 3 * H, bias=False)

        n_a = int(round(frac_adaptive * H)) if adaptive else 0
        mask = torch.zeros(H)
        mask[:n_a] = 1.0
        self.register_buffer("adapt_mask", mask)

        tau_a = torch.empty(H).uniform_(*tau_a_range)
        rho = torch.exp(-1.0 / tau_a)
        rho_logit = torch.log(rho / (1 - rho))
        alpha = torch.full((H,), math.exp(-1.0 / tau_m))
        alpha_logit = torch.log(alpha / (1 - alpha))
        if learn_tau:
            self.rho_logit = nn.Parameter(rho_logit)
            self.alpha_logit = nn.Parameter(alpha_logit)
        else:
            self.register_buffer("rho_logit", rho_logit)
            self.register_buffer("alpha_logit", alpha_logit)

        self.beta_raw = nn.Parameter(_inv_softplus(torch.full((H,), beta_init)))
        self.b0_raw = nn.Parameter(_inv_softplus(torch.full((H,), b0_init)))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.zeros_(self.W.bias)
        H = self.hidden_size
        with torch.no_grad():
            self.W.bias[:H].fill_(1.0)
        for w in self.U.weight.chunk(4, 0):
            nn.init.orthogonal_(w)
        with torch.no_grad():
            self.U.weight.mul_(self.syn_mask)
        if self.local_v:
            with torch.no_grad():
                self.v.uniform_(-0.5, 0.5)
        else:
            for w in self.V.weight.chunk(3, 0):
                nn.init.orthogonal_(w)

    @property
    def rho(self):
        return torch.sigmoid(self.rho_logit)

    @property
    def alpha(self):
        return torch.sigmoid(self.alpha_logit)

    @property
    def beta(self):
        return F.softplus(self.beta_raw) * self.adapt_mask

    @property
    def b0(self):
        return F.softplus(self.b0_raw) + 1e-3

    def init_state(self, batch, device):
        z = torch.zeros(batch, self.hidden_size, device=device)
        return (z, z.clone(), z.clone())

    def forward(self, x, state):
        h, u, b = state
        H = self.hidden_size
        wx = self.W(x)
        uh = F.linear(h, self.U.weight * self.syn_mask)
        vu = self.v * u.repeat(1, 3) if self.local_v else self.V(u)

        r = torch.sigmoid(wx[:, :H] + uh[:, :H] + vu[:, :H])
        c = torch.sigmoid(wx[:, H:2 * H] + uh[:, H:2 * H] + vu[:, H:2 * H])
        o = torch.sigmoid(wx[:, 2 * H:3 * H] + uh[:, 2 * H:3 * H] + vu[:, 2 * H:])
        n = torch.tanh(wx[:, 3 * H:] + uh[:, 3 * H:])

        u_tilde = self.alpha * r * u + n
        B = self.b0 + self.beta * b

        if self.spiking:
            v = (u_tilde - B) / B
            z = spike(v, self.gamma)
        else:
            z = torch.ones_like(u_tilde)

        u_new = u_tilde - z * c * B
        b_new = self.rho * b + (1.0 - self.rho) * z
        h_new = o * z * F.softplus(u_tilde)
        return h_new, (h_new, u_new, b_new), z


class ALSTMSNP(nn.Module):
    """Sequence (B, T, input_size) -> prediction (B, output_size)."""

    def __init__(self, input_size=1, hidden_size=32, output_size=1, num_layers=1, dropout=0.0, **cell_kw):
        super().__init__()
        self.cells = nn.ModuleList(
            [ALSTMSNPCell(input_size if i == 0 else hidden_size, hidden_size, **cell_kw) for i in range(num_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Linear(4 * hidden_size, output_size)

    def forward(self, x, return_spikes=False):
        Bsz, T, _ = x.shape
        seq = x
        all_z = []
        for cell in self.cells:
            state = cell.init_state(Bsz, x.device)
            outs, zs = [], []
            for t in range(T):
                h, state, z = cell(seq[:, t], state)
                outs.append(h)
                zs.append(z)
            seq = self.drop(torch.stack(outs, 1))
            all_z.append(torch.stack(zs, 1))
        h_T, u_T, b_T = state
        feat = torch.cat([h_T, u_T, b_T, seq.mean(1)], dim=-1)
        y = self.readout(feat)
        if return_spikes:
            return y, torch.cat(all_z, -1)
        return y


# ── LSTM [34] ──
class LSTM34(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        # nn.LSTM's equations on its own parameters (gate order i, f, g, o), zero initial state
        L = self.lstm
        h = x.new_zeros(x.shape[0], L.hidden_size); c = h
        for t in range(x.shape[1]):
            i, f, g, o = (x[:, t] @ L.weight_ih_l0.T + L.bias_ih_l0 + h @ L.weight_hh_l0.T + L.bias_hh_l0).chunk(4, -1)
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
        return self.fc(h)


# ── GRU [39] ──
class GRU39(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        # nn.GRU's equations on its own parameters (gate order r, z, n), zero initial state
        G = self.gru
        h = x.new_zeros(x.shape[0], G.hidden_size)
        for t in range(x.shape[1]):
            xr, xz, xn = (x[:, t] @ G.weight_ih_l0.T + G.bias_ih_l0).chunk(3, -1)
            hr, hz, hn = (h @ G.weight_hh_l0.T + G.bias_hh_l0).chunk(3, -1)
            r, z = torch.sigmoid(xr + hr), torch.sigmoid(xz + hz)
            n = torch.tanh(xn + r * hn)
            h = (1 - z) * n + z * h
        return self.fc(h)


# ── LSTM-SNP cell (hard-sigmoid reset / consumption / generation gates, u = r·u − c·a, h = o·a) — [25] and [26] ──
class LSTMSNPCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.kernel = nn.Linear(input_size, hidden_size * 4, bias=False)
        self.recurrent_kernel = nn.Linear(hidden_size, hidden_size * 4, bias=False)
        self.bias = nn.Parameter(torch.zeros(hidden_size * 4))
        nn.init.xavier_uniform_(self.kernel.weight)
        nn.init.orthogonal_(self.recurrent_kernel.weight)
        with torch.no_grad():
            self.bias.data[hidden_size:2 * hidden_size] = 1.0

    def forward(self, x, u_tm1):
        z0, z1, z2, z3 = (self.kernel(x) + self.recurrent_kernel(u_tm1) + self.bias).chunk(4, dim=-1)
        hs = lambda v: torch.clamp(0.2 * v + 0.5, 0.0, 1.0)
        r, c, o, a = hs(z0), hs(z1), hs(z2), torch.tanh(z3)
        return o * a, r * u_tm1 - c * a          # h, u


class LSTMSNP25(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.cell = LSTMSNPCell(1, hidden)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        u = x.new_zeros(x.shape[0], self.cell.hidden_size)
        for t in range(x.shape[1]):
            h, u = self.cell(x[:, t], u)
        return self.fc(h)


class BiLSTMSNP26(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.cell_fw = LSTMSNPCell(1, hidden)
        self.cell_bw = LSTMSNPCell(1, hidden)
        self.fc = nn.Linear(hidden * 2, 1)

    def forward(self, x):
        u_fw = x.new_zeros(x.shape[0], self.cell_fw.hidden_size); u_bw = u_fw
        for t in range(x.shape[1]):
            h_fw, u_fw = self.cell_fw(x[:, t], u_fw)
        for t in range(x.shape[1] - 1, -1, -1):
            h_bw, u_bw = self.cell_bw(x[:, t], u_bw)
        return self.fc(torch.cat([h_fw, h_bw], dim=-1))


# ── Attention: LSTM + softmax attention over the time steps ──
class LSTMAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.attn_w = nn.Linear(hidden, 1)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        L = self.lstm
        h = x.new_zeros(x.shape[0], L.hidden_size); c = h
        outs = []
        for t in range(x.shape[1]):
            i, f, g, o = (x[:, t] @ L.weight_ih_l0.T + L.bias_ih_l0 + h @ L.weight_hh_l0.T + L.bias_hh_l0).chunk(4, -1)
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
            outs.append(h)
        lstm_out = torch.stack(outs, 1)                                   # (N, T, H)
        attn_weights = torch.softmax(self.attn_w(lstm_out).squeeze(-1), dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)
        return self.fc(context)


# ── Informer [42] ──
class ProbSparseAttention(nn.Module):
    def __init__(self, d_model=8, n_heads=2):
        super().__init__()
        self.d_k, self.n_heads = d_model // n_heads, n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, _ = x.size()
        Q = self.q_proj(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        attn = torch.softmax(torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5), dim=-1)
        return self.out_proj(torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, L, -1))


class Informer42(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden)
        self.attn = ProbSparseAttention(hidden, n_heads=2)
        self.norm1 = nn.LayerNorm(hidden)
        self.ff = nn.Sequential(nn.Linear(hidden, hidden * 2), nn.GELU(), nn.Linear(hidden * 2, hidden))
        self.norm2 = nn.LayerNorm(hidden)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.input_proj(x)
        h = self.norm1(h + self.attn(h))
        h = self.norm2(h + self.ff(h))
        return self.fc(h[:, -1, :])


# ── Autoformer [40] ──
class AutoCorrelation40(nn.Module):
    def __init__(self, d_model=8):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q, K, V = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        attn = torch.softmax(torch.bmm(Q, K.transpose(1, 2)) / (Q.size(-1) ** 0.5), dim=-1)
        return torch.bmm(attn, V)


class Autoformer40(nn.Module):
    def __init__(self, hidden, kernel_size=3):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden)
        self.decomp = nn.AvgPool1d(kernel_size, stride=1, padding=kernel_size // 2)
        self.auto_corr = AutoCorrelation40(hidden)
        self.ff = nn.Sequential(nn.Linear(hidden, hidden * 2), nn.GELU(), nn.Linear(hidden * 2, hidden))
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.input_proj(x)
        trend = self.decomp(h.transpose(1, 2)).transpose(1, 2)
        h = self.auto_corr(h - trend) + trend
        h = h + self.ff(h)
        return self.fc(h[:, -1, :])


# ── FEDformer [41] ──
class FreqBlock41(nn.Module):
    def __init__(self, d_model=8):
        super().__init__()
        self.fc_real = nn.Linear(d_model, d_model)
        self.fc_imag = nn.Linear(d_model, d_model)

    def forward(self, x):
        x_ft = torch.fft.rfft(x, dim=1)
        return torch.fft.irfft(torch.complex(self.fc_real(x_ft.real), self.fc_imag(x_ft.imag)), n=x.size(1), dim=1)


class FEDformer41(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden)
        self.freq = FreqBlock41(hidden)
        self.norm = nn.LayerNorm(hidden)
        self.ff = nn.Sequential(nn.Linear(hidden, hidden * 2), nn.GELU(), nn.Linear(hidden * 2, hidden))
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.input_proj(x)
        h = self.norm(h + self.freq(h))
        h = self.norm(h + self.ff(h))
        return self.fc(h[:, -1, :])


BASELINE_CLASSES = {'lstm_34': LSTM34, 'gru_39': GRU39, 'lstm_snp_25': LSTMSNP25, 'bilstm_snp_26': BiLSTMSNP26,
                    'attention': LSTMAttention, 'informer_42': Informer42, 'autoformer_40': Autoformer40,
                    'fedformer_41': FEDformer41}


def make_model_specs():
    # everything model-specific; the dataset part of the run tag is added per dataset by run_tag_for()
    def smoke(d):
        return {**d, 'n_runs': 3, 'max_epochs': 3, 'patience': 2, 'lags': d['lags'][:2]} if SMOKE else d

    a, c = ALSTM_CFG, ALSTM_CELL
    variant = (f"_set{a['setup']}" + ('' if c['spiking'] else '_nospike') + ('' if c['adaptive'] else '_noadapt')
               + ('' if c['strict'] else '_loose') + (f'_{AMP_DTYPE}' if AMP_DTYPE else ''))
    alstm_build = lambda: ALSTMSNP(input_size=1, hidden_size=a['hidden'], num_layers=c['layers'],
                                   frac_adaptive=c['frac_adaptive'], tau_a_range=c['tau_a_range'], tau_m=c['tau_m'],
                                   gamma=c['gamma_sg'], spiking=c['spiking'], adaptive=c['adaptive'], strict=c['strict'])
    specs = [smoke({'name': a['name'], 'folder': a['folder'], 'build': alstm_build, 'lags': list(COMMON_LAGS),
                    'batch': a['batch'], 'lr': a['lr'], 'max_epochs': a['max_epochs'], 'patience': a['patience'],
                    'n_runs': a['n_runs'], 'spikes': True, 'rate_reg': c['rate_reg'], 'variant': variant})]
    b = BASELINE_TRAIN
    for name, folder in BASELINES:
        specs.append(smoke({'name': name, 'folder': folder, 'build': (lambda cls=BASELINE_CLASSES[folder]: cls(b['hidden'])),
                            'lags': list(COMMON_LAGS), 'batch': b['batch'], 'lr': b['lr'], 'max_epochs': b['max_epochs'],
                            'patience': b['patience'], 'n_runs': b['n_runs'], 'spikes': False, 'rate_reg': 0.0,
                            'variant': ''}))
    return [s for s in specs if MODELS_TO_RUN is None or s['name'] in MODELS_TO_RUN or s['folder'] in MODELS_TO_RUN]


MODEL_SPECS = make_model_specs()
ALSTM_NAME = ALSTM_CFG['name']
for _s in MODEL_SPECS:
    print(f"{_s['name']:<16} {sum(p.numel() for p in _s['build']().parameters()):5d} parameters | lags {_s['lags']} | "
          f"batch {_s['batch']} | lr {_s['lr']} | ≤{_s['max_epochs']} epochs | patience {_s['patience']} | seeds {_s['n_runs']}")

In [ ]:
# ============================================================
# 5. Helpers: windows, parallel ensemble, Adam, metrics, tests (identical to the reference ALSTM-SNP implementation)
# ============================================================
def make_supervised(series, lag, h=1):
    # Row k predicts series[t], t = lag + h - 1 + k, from series[t-h], ..., series[t-h-lag+1] (column 0 = most recent).
    # h = 1 is the one-step windowing of every published run; h > 1 trains a network directly for step h (cell 22).
    t_idx = np.arange(lag + h - 1, len(series))
    X = np.stack([series[t_idx - h + 1 - i] for i in range(1, lag + 1)], axis=1)
    return np.column_stack([X, series[t_idx]]), t_idx


def to_seq(X_scaled):
    # (N, L) lag columns [most recent … oldest] → (N, L, 1) sequence [oldest … most recent]
    return torch.tensor(X_scaled[:, ::-1].copy()[..., None], dtype=torch.float32, device=device)


def add_gaussian_noise(X, noise_level, ref_std, rng):
    return X + rng.normal(0.0, noise_level * ref_std, size=X.shape)


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)


def make_ensemble(seeds, spec):
    # Each member is built exactly as a sequential run with that seed would build it (spec['build']()).
    # One vmapped call runs all K members, each on its own inputs: x is (K, N, T, 1). Returns (y, z): z are the
    # spikes of a spiking model, or a zero placeholder (firing rate 0) for the others.
    build = spec['build']
    members = []
    for s in seeds:
        seed_all(s)
        members.append(build())
    params, buffers = stack_module_state(members)
    params = {k: v.detach().to(device).requires_grad_(True) for k, v in params.items()}
    buffers = {k: v.to(device) for k, v in buffers.items()}
    base = build().to('meta')
    if spec['spikes']:
        def one(p, b, x):
            return functional_call(base, (p, b), (x,), {'return_spikes': True})
    else:
        def one(p, b, x):
            y = functional_call(base, (p, b), (x,))
            return y, y.new_zeros(x.shape[0], 1, 1)
    ens_fp32 = vmap(one, in_dims=(0, 0, 0))
    if AMP_DTYPE is None or device.type != 'cuda':
        return params, buffers, ens_fp32

    def ens(p, b, x):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            y, z = ens_fp32(p, b, x)
        return y.float(), z.float()
    return params, buffers, ens


def clip_grad_norm_per_model(grads, max_norm):
    # grads: {name: (K, …)} → same, each model's gradient clipped to total norm max_norm
    gs = list(grads.values())
    K = gs[0].shape[0]
    total = torch.sqrt(sum(g.reshape(K, -1).pow(2).sum(1) for g in gs))
    coef = (max_norm / (total + 1e-6)).clamp(max=1.0)
    return {k: g * coef.view(-1, *([1] * (g.dim() - 1))) for k, g in grads.items()}


def seed_lr(lr, seeds, dev):
    # one learning rate for the whole ensemble, or a list: seed s trains with lr[s % len(lr)] (the grid of cells 21–22,
    # trained side by side in one vmapped ensemble). MaskedAdam takes either: lr / bias correction broadcasts per model.
    if isinstance(lr, (list, tuple)):
        return torch.tensor([float(lr[s % len(lr)]) for s in seeds], dtype=torch.float64, device=dev)
    return lr


class MaskedAdam:
    """torch.optim.Adam (betas 0.9/0.999, eps 1e-8, no weight decay) on K stacked models, where model k only
    takes a step when mask[k] = 1 — folds have different numbers of mini-batches per epoch, and a model whose
    fold has run out of batches must not move (Adam would, even with a zero gradient). Same update as torch's
    single-tensor / foreach Adam; bias corrections in float64 as torch computes them on the host."""

    def __init__(self, params, lr, betas=(0.9, 0.999), eps=1e-8):
        self.params, self.lr, (self.b1, self.b2), self.eps = params, lr, betas, eps
        p0 = next(iter(params.values()))
        self.m = {k: torch.zeros_like(v) for k, v in params.items()}
        self.v = {k: torch.zeros_like(v) for k, v in params.items()}
        self.step = torch.zeros(p0.shape[0], dtype=torch.float64, device=p0.device)

    @torch.no_grad()
    def update(self, grads, mask):
        self.step.add_(mask.to(torch.float64))
        st = self.step.clamp(min=1.0)
        step_size = (self.lr / (1.0 - self.b1 ** st)).float()
        bc2_sqrt = (1.0 - self.b2 ** st).sqrt().float()
        for k, p in self.params.items():
            g, m, v = grads[k], self.m[k], self.v[k]
            shp = (-1,) + (1,) * (p.dim() - 1)
            on = mask.view(shp) > 0
            m.copy_(torch.where(on, torch.lerp(m, g, 1.0 - self.b1), m))
            v.copy_(torch.where(on, torch.addcmul(v * self.b2, g, g, value=1.0 - self.b2), v))
            denom = (v.sqrt() / bc2_sqrt.view(shp)).add_(self.eps)
            ss = step_size.view(shp)
            # same rounding as torch's addcdiv_: a + s·(b/c) on CUDA, a + (s·b)/c on CPU
            p.copy_(torch.where(on, p - ((m / denom) * ss if p.is_cuda else (ss * m) / denom), p))

    def state_dict(self):
        return {'m': {k: t.detach().cpu() for k, t in self.m.items()},
                'v': {k: t.detach().cpu() for k, t in self.v.items()}, 'step': self.step.cpu()}

    def load_state_dict(self, sd):
        # in place: the CUDA graph holds these tensors' addresses
        for k in self.m:
            self.m[k].copy_(sd['m'][k]); self.v[k].copy_(sd['v'][k])
        self.step.copy_(sd['step'])


@torch.no_grad()
def ens_predict(ens, params, buffers, X_seq):
    # X_seq (K, N, T, 1), one input set per model. Returns scaled predictions (K, N) and mean firing rate per model (K,)
    preds, rates = [], 0.0
    for s in range(0, X_seq.size(1), EVAL_CHUNK):
        xb = X_seq[:, s:s + EVAL_CHUNK]
        y, z = ens(params, buffers, xb)
        preds.append(y[..., 0])
        rates = rates + z.mean(dim=(1, 2, 3)) * xb.size(1)
    return torch.cat(preds, 1).cpu().numpy(), (rates / X_seq.size(1)).cpu().numpy()


def pad_stack(tensors):
    # list of (N_i, …) → (len, max N_i, …), zero-padded
    n = max(t.shape[0] for t in tensors)
    out = tensors[0].new_zeros((len(tensors), n) + tuple(tensors[0].shape[1:]))
    for i, t in enumerate(tensors):
        out[i, :t.shape[0]] = t
    return out


def inv_target(y_scaled, scaler):
    return (np.asarray(y_scaled) - scaler.min_[-1]) / scaler.scale_[-1]


def reconstruct(y_scaled, scaler, anchor):
    return anchor + inv_target(y_scaled, scaler)


def safe_auc(y, s):
    y = np.asarray(y)
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else float('nan')


def metrics(pred_raw, actual_raw, prev_raw, tau):
    pred_raw, actual_raw, prev_raw = map(np.asarray, (pred_raw, actual_raw, prev_raw))
    rmse = sqrt(np.mean((actual_raw - pred_raw) ** 2))
    true_dir = ((actual_raw - prev_raw) > tau).astype(int)
    score = (pred_raw - prev_raw) - tau
    pred_dir = (score > 0).astype(int)
    TP = np.sum((true_dir == 1) & (pred_dir == 1)); TN = np.sum((true_dir == 0) & (pred_dir == 0))
    FP = np.sum((true_dir == 0) & (pred_dir == 1)); FN = np.sum((true_dir == 1) & (pred_dir == 0))
    acc  = (TP + TN) / len(true_dir)
    spec = TN / (TN + FP) if (TN + FP) else 0.0
    prec = TP / (TP + FP) if (TP + FP) else 0.0
    rec  = TP / (TP + FN) if (TP + FN) else 0.0
    auc  = safe_auc(true_dir, score)
    return {'rmse': rmse, 'acc': acc, 'spec': spec, 'prec': prec, 'rec': rec, 'auc': auc}


def dm_test(e_model, e_base, h=1):
    # Diebold–Mariano (HLN), squared-error loss. stat < 0 → model has lower squared error.
    d = e_model ** 2 - e_base ** 2
    n = len(d)
    gamma = [np.mean((d[k:] - d.mean()) * (d[:n - k] - d.mean())) for k in range(h)]
    var = (gamma[0] + 2 * sum(gamma[1:])) / n
    if var <= 0:
        return 0.0, 1.0
    stat = d.mean() / np.sqrt(var) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return float(stat), float(2 * stats.t.sf(abs(stat), df=n - 1))


def midrank(x):
    order = np.argsort(x); z = x[order]; n = len(x)
    t = np.zeros(n); i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n); out[order] = t
    return out


def delong_test(y_true, s1, s2):
    y_true = np.asarray(y_true).astype(bool)
    m, n = int(y_true.sum()), int((~y_true).sum())
    if m < 2 or n < 2:      # AUC undefined (one class) or its variance undefined
        nan = float('nan')
        return nan, nan, nan, nan
    aucs, v10, v01 = [], [], []
    for s in (np.asarray(s1, float), np.asarray(s2, float)):
        pos, neg = s[y_true], s[~y_true]
        tx, ty, tz = midrank(pos), midrank(neg), midrank(np.concatenate([pos, neg]))
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        v10.append((tz[:m] - tx) / n)
        v01.append(1.0 - (tz[m:] - ty) / m)
    S = np.cov(np.vstack(v10)) / m + np.cov(np.vstack(v01)) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
    return aucs[0], aucs[1], float(z), float(2 * stats.norm.sf(abs(z)))


def holm(p):
    p = np.asarray(p, float); adj = np.full(len(p), np.nan)
    ok = np.flatnonzero(np.isfinite(p)); m = len(ok); run = 0.0
    for rank, i in enumerate(ok[np.argsort(p[ok])]):
        run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
    return adj


def safe_save(obj, path, retries=10):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    for k in range(retries):
        try:
            os.replace(tmp, path); return
        except PermissionError:
            time.sleep(0.5 * (k + 1))
    torch.save(obj, path)
    if os.path.exists(tmp): os.remove(tmp)


def save_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(obj, fh, indent=1, default=float)
    os.replace(tmp, path)


def load_json(path):
    if path and os.path.exists(path):
        with open(path) as fh:
            return json.load(fh)
    return None


METRIC_KEYS = ['acc', 'spec', 'prec', 'rec', 'auc']

In [ ]:
# ============================================================
# 6. Training harness: one (dataset, lag, fold)'s data, and a group of (fold, seed) models trained together
#    A group trains all its models as one vmapped ensemble. Model k sees only its own fold's data, its own seed's
#    batch order and initialisation, and steps only while its fold has batches left, so its result is the one a
#    per-fold run would give; the folds just share the kernel launches. Batches are padded to BATCH rows with zero
#    weight, so every step has the same shapes and the whole step (forward, backward, per-model clipping, Adam) is
#    captured as one CUDA graph and replayed.
#    Energy: each group's training (forward/backward/Adam + per-epoch validation) is one EnergySegment, paused while
#    checkpoints are written; its totals are stored in the group checkpoint, so they survive a restart.
#    After training, an inference benchmark replays the test-set forward pass for INFER_BENCH_S → J per forecast.
#    Layout on LOCAL_ROOT: <dataset>/<model folder>/{checkpoints_*, runs_*.csv, walkforward_*.csv, preds_*.csv,
#    energy_*_lag*.json, energy_*.csv, figures/}.
# ============================================================
def run_tag_for(spec, D):
    slug = re.sub(r'[^A-Za-z0-9]+', '_', spec['name']).strip('_')
    return f"{slug}_{D['slug']}_level{spec['variant']}{'_smoke' if SMOKE else ''}_wf"


def work_dir_for(spec, D):
    return os.path.join(LOCAL_ROOT, D['slug'], spec['folder'])


def ckpt_dir_for(spec, D, lag, fold_tag):
    # finished runs of one lag × fold
    return os.path.join(work_dir_for(spec, D), f"checkpoints_{run_tag_for(spec, D)}_lag{lag}_{fold_tag}_{CKPT_VERSION}")


def group_dir_for(spec, D, lag):
    # in-progress training groups of one lag (a group may span several folds)
    return os.path.join(work_dir_for(spec, D), f"checkpoints_{run_tag_for(spec, D)}_lag{lag}_groups_{CKPT_VERSION}")


def energy_path_for(spec, D, lag):
    return os.path.join(work_dir_for(spec, D), f"energy_{run_tag_for(spec, D)}_lag{lag}.json")


def train_rows_first_fold(D, lag):
    return int(np.sum(np.arange(lag, D['n']) < D['folds'][0]['va_start']))


def min_train_rows():
    # one threshold for every model, so a lag is kept or skipped for all models alike
    return max([MIN_TRAIN_ROWS, ALSTM_CFG['batch'] + 1, BASELINE_TRAIN['batch'] + 1])


def lag_ok(spec, D, lag):
    return train_rows_first_fold(D, lag) >= min_train_rows() and D['folds'][0]['va_start'] > lag


def prepare_fold(D, LAG, fi, supervised, t_idx, batch):
    raw_values, fold = D['values'], D['folds'][fi]
    tt = t_idx                                         # raw index of each row's target
    n_train = int(np.sum(tt < fold['va_start']))
    n_trval = int(np.sum(tt < fold['te_start']))
    n = int(np.sum(tt < fold['te_end']))
    assert n_train > batch, f"too few training rows at lag {LAG}"
    sl_tr, sl_va, sl_te = slice(0, n_train), slice(n_train, n_trval), slice(n_trval, n)
    sup = supervised[:n].copy()

    actual_raw = raw_values[tt[:n]]
    base_raw = raw_values[tt[:n] - 1]
    if TARGET_MODE == 'resid':
        sup[:, -1] = sup[:, -1] - sup[:, 0]
        anchor = base_raw
    else:
        anchor = np.zeros_like(base_raw)

    pre_test = raw_values[:fold['te_start']]
    tau = round(float(np.median(np.diff(pre_test))), 6)
    naive_level = float(pre_test.mean())

    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(sup[sl_tr])
    sup_s = scaler.transform(sup)
    X_all, y_all = sup_s[:, :-1], sup_s[:, -1]
    X_tr, X_te, y_tr = X_all[sl_tr], X_all[sl_te], y_all[sl_tr]
    y_te, prev_te = actual_raw[sl_te], base_raw[sl_te]
    true_te = ((y_te - prev_te) > tau).astype(int)
    test_size = n - n_trval

    print(f"\n── {D['name']} | LAG={LAG} | fold test {fold['label']} | train {n_train} | val {n_trval - n_train} | test {test_size} | "
          f"τ={tau} | naive {naive_level:.4g} | class-1 rate {true_te.mean()*100:.1f}%")

    naive_pred = np.full(test_size, naive_level)
    ridge = Ridge(alpha=RIDGE_ALPHA).fit(X_tr, y_tr)
    ridge_pred = reconstruct(ridge.predict(X_te), scaler, anchor[sl_te])
    persist_pred = prev_te.copy()

    ref_std = X_tr.std()
    noisy = {nl: [to_seq(add_gaussian_noise(X_te, nl, ref_std, np.random.default_rng([LAG, fi, li, d])))
                  for d in range(N_NOISE_DRAWS)] for li, nl in enumerate(NOISE_LEVELS)}
    return {'fold': fold, 'tt': tt, 'n_train': n_train, 'n_val': n_trval - n_train, 'n_test': test_size,
            'sl_va': sl_va, 'sl_te': sl_te, 'scaler': scaler, 'anchor': anchor, 'actual_raw': actual_raw,
            'tau': tau, 'y_te': y_te, 'prev_te': prev_te, 'true_te': true_te,
            'naive_pred': naive_pred, 'ridge_pred': ridge_pred, 'persist_pred': persist_pred,
            'Xtr': to_seq(X_tr), 'ytr': torch.tensor(y_tr, dtype=torch.float32, device=device),
            'Xva': to_seq(X_all[sl_va]), 'Xte': to_seq(X_te), 'noisy': noisy}


def build_step(spec, ens, params, buffers, opt, Xtr, ytr, fold_id):
    # Static tensors: the step reads the batch from s_idx / s_w and accumulates into ep_loss / n_b, so it can be
    # replayed from a CUDA graph. Rows with weight 0 are padding; a model with an all-zero row does not step.
    K, BATCH, rate_reg = fold_id.numel(), spec['batch'], spec['rate_reg']
    keys = list(params)
    s_idx = torch.zeros(K, BATCH, dtype=torch.long, device=device)
    s_w = torch.zeros(K, BATCH, device=device)
    ep_loss = torch.zeros(K, device=device)
    n_b = torch.zeros(K, device=device)
    fid = fold_id[:, None]

    def step():
        upd = (s_w.sum(1) > 0).float()
        wsum = s_w.sum(1).clamp(min=1.0)
        pred, z = ens(params, buffers, Xtr[fid, s_idx])
        loss_k = (((pred[..., 0] - ytr[fid, s_idx]) ** 2) * s_w).sum(1) / wsum
        if rate_reg > 0:
            rate = (z * s_w[:, :, None, None]).sum(dim=(1, 2)) / (wsum[:, None] * z.shape[2])
            loss_k = loss_k + rate_reg * ((rate - RATE_TARGET) ** 2).mean(-1)
        grads = torch.autograd.grad(loss_k.sum(), [params[k] for k in keys])
        opt.update(clip_grad_norm_per_model(dict(zip(keys, grads)), CLIP_NORM), upd)
        ep_loss.add_(loss_k.detach() * upd)
        n_b.add_(upd)

    return step, s_idx, s_w, ep_loss, n_b


def capture_graph(step, s_w):
    # warm-up on a side stream with all-zero weights (no model steps, so nothing changes), then capture
    s_w.zero_()
    side = torch.cuda.Stream()
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):
        for _ in range(3):
            step()
    torch.cuda.current_stream().wait_stream(side)
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        step()
    return graph


def bench_inference(ens, params, buffers, X):
    # replay the test-set forward pass of the whole ensemble for ≥ INFER_BENCH_S seconds; per-forecast energy and time
    seg = EnergySegment(METER)
    reps, t_end = 0, time.perf_counter() + INFER_BENCH_S
    seg.resume()
    while reps < 1 or time.perf_counter() < t_end:
        ens_predict(ens, params, buffers, X)
        reps += 1
    seg.pause()
    st, n_pred = seg.state(), reps * X.shape[0] * X.shape[1]
    return {'infer_j_per_forecast': st['j'] / n_pred, 'infer_s_per_forecast': st['s'] / n_pred,
            'infer_mean_w': st['j'] / st['s'] if st['s'] > 0 else float('nan'), 'infer_forecasts': int(n_pred)}


def train_group(spec, D, LAG, preps, jobs, group_path):
    # jobs: list of (fold index, seed). Returns per-job results and the group's energy record.
    K, BATCH, MAX_EPOCHS, PATIENCE = len(jobs), spec['batch'], spec['max_epochs'], spec['patience']
    fis = sorted({fi for fi, _ in jobs})
    slot = {fi: i for i, fi in enumerate(fis)}
    fold_id = torch.tensor([slot[fi] for fi, _ in jobs], device=device)
    seeds = [s for _, s in jobs]
    Xtr = pad_stack([preps[fi]['Xtr'] for fi in fis]); ytr = pad_stack([preps[fi]['ytr'] for fi in fis])
    n_tr = np.array([preps[fi]['n_train'] for fi, _ in jobs])
    NB = int(-(-n_tr.max() // BATCH))
    rows = {fi: np.array([j for j, (f, _) in enumerate(jobs) if f == fi]) for fi in fis}
    per_model = lambda key: pad_stack([preps[fi][key] for fi in fis])[fold_id]
    tag = run_tag_for(spec, D)

    params, buffers, ens = make_ensemble(seeds, spec)
    opt = MaskedAdam(params, seed_lr(spec['lr'], seeds, device))
    best = {k: v.detach().clone() for k, v in params.items()}
    best_val = np.full(K, np.inf); best_epoch = np.zeros(K, int)
    patience_ctr = np.zeros(K, int); active = np.ones(K, bool)
    losses = [[] for _ in range(K)]; start_epoch = 0; energy_prev = None

    if os.path.exists(group_path):
        ck = torch.load(group_path, map_location=device, weights_only=False)
        assert ck['jobs'] == jobs, "group checkpoint does not match pending runs"
        with torch.no_grad():
            for k in params: params[k].copy_(ck['params'][k])
        opt.load_state_dict(ck['optimizer'])
        best = {k: v.to(device) for k, v in ck['best'].items()}
        best_val, best_epoch = ck['best_val'], ck['best_epoch']
        patience_ctr, active, losses, start_epoch = ck['patience'], ck['active'], ck['losses'], ck['epoch']
        energy_prev = ck.get('energy')
        print(f"   resuming {K} runs from epoch {start_epoch + 1}")
    seg = EnergySegment(METER, energy_prev)

    seg.resume()
    step, s_idx, s_w, ep_loss_t, n_b_t = build_step(spec, ens, params, buffers, opt, Xtr, ytr, fold_id)
    graph, graph_checked = None, False
    if CUDA_GRAPHS and device.type == 'cuda':
        try:
            graph = capture_graph(step, s_w)
        except Exception as e:
            print(f"   CUDA graph capture failed ({type(e).__name__}: {e}) — eager steps")
    state = lambda: [*params.values(), *opt.m.values(), *opt.v.values(), opt.step, ep_loss_t, n_b_t]
    Xva, Xte = per_model('Xva'), per_model('Xte')
    seg.pause()

    for epoch in range(start_epoch, MAX_EPOCHS):
        if not active.any():
            break
        seg.resume()
        t0 = time.time()
        # per-seed shuffled batch order (as in the per-fold runs), reproducible on resume; padded to NB batches
        idx = torch.zeros(K, NB * BATCH, dtype=torch.long); w = torch.zeros(K, NB * BATCH)
        for j, s in enumerate(seeds):
            idx[j, :n_tr[j]] = torch.randperm(int(n_tr[j]), generator=torch.Generator().manual_seed(1_000_003 * (s + 1) + epoch))
            w[j, :n_tr[j]] = 1.0
        idx = idx.view(K, NB, BATCH).to(device); w = w.view(K, NB, BATCH).to(device)
        ep_loss_t.zero_(); n_b_t.zero_()
        for i in range(NB):
            s_idx.copy_(idx[:, i]); s_w.copy_(w[:, i])
            if graph is None:
                step()
            elif graph_checked:
                graph.replay()
            else:                                      # first step: the graph must reproduce the eager step
                snap = [t.detach().clone() for t in state()]
                step()
                eager = [t.detach().clone() for t in state()]
                with torch.no_grad():
                    for t, v in zip(state(), snap): t.copy_(v)
                graph.replay()
                ok = all(torch.allclose(a, b, rtol=1e-5, atol=1e-7) for a, b in zip(state(), eager))
                if not ok:
                    print("   CUDA graph result differs from the eager step — eager steps")
                    with torch.no_grad():
                        for t, v in zip(state(), eager): t.copy_(v)
                    graph = None
                graph_checked = True
        ep_loss = (ep_loss_t / n_b_t).cpu().numpy()

        pv, _ = ens_predict(ens, params, buffers, Xva)
        val_rmse = np.empty(K)
        for fi, r in rows.items():
            P = preps[fi]
            val_rmse[r] = np.sqrt(np.mean((P['actual_raw'][P['sl_va']]
                                           - reconstruct(pv[r, :P['n_val']], P['scaler'], P['anchor'][P['sl_va']])) ** 2, axis=1))

        improved = active & (val_rmse < best_val)
        idx_i = torch.as_tensor(np.where(improved)[0], device=device)
        if len(idx_i):
            with torch.no_grad():
                for k in params:
                    best[k][idx_i] = params[k].detach()[idx_i]
        seg.pause()
        for j in range(K):
            if not active[j]:
                continue
            losses[j].append(float(ep_loss[j]))
            if improved[j]:
                best_val[j], best_epoch[j], patience_ctr[j] = val_rmse[j], epoch + 1, 0
            else:
                patience_ctr[j] += 1
                if patience_ctr[j] >= PATIENCE:
                    active[j] = False

        st = seg.state()
        print(f"     epoch {epoch+1:03d} | {time.time()-t0:5.1f}s | active {active.sum():3d}/{K} | "
              f"train loss {ep_loss[active].mean() if active.any() else ep_loss.mean():.5f} | "
              f"best val RMSE (median) {np.median(best_val):.4g}"
              + (f" | GPU {st['j'] / 1e3:.2f} kJ so far, {st['j'] / max(st['s'], 1e-9):.0f} W" if METER.ok else ''))
        safe_save({'jobs': jobs, 'epoch': epoch + 1,
                   'params': {k: v.detach().cpu() for k, v in params.items()},
                   'optimizer': opt.state_dict(),
                   'best': {k: v.cpu() for k, v in best.items()},
                   'best_val': best_val, 'best_epoch': best_epoch, 'patience': patience_ctr,
                   'active': active, 'losses': losses, 'energy': st}, group_path)
        store.add(group_path)
        store.push(f"{tag} lag {LAG}: epoch {epoch + 1}")
        check_time_budget()

    seg.resume()
    pt_s, rate_te = ens_predict(ens, best, buffers, Xte)
    noisy_s = {nl: [ens_predict(ens, best, buffers, pad_stack([preps[fi]['noisy'][nl][d] for fi in fis])[fold_id])[0]
                    for d in range(N_NOISE_DRAWS)] for nl in NOISE_LEVELS}
    seg.pause()
    energy = {**seg.state(), 'n_models': K, 'model_epochs': int(sum(len(l) for l in losses)),
              'epochs_max': int(max(len(l) for l in losses)), **bench_inference(ens, best, buffers, Xte)}
    out = []
    for j, (fi, s) in enumerate(jobs):
        P = preps[fi]; a = P['anchor'][P['sl_te']]; nt = P['n_test']
        out.append({'run': s + 1,
                    'state_dict': {**{k: v[j].detach().cpu().clone() for k, v in best.items()},
                                   **{k: v[j].detach().cpu().clone() for k, v in buffers.items()}},
                    'losses': losses[j], 'best_val_rmse': float(best_val[j]), 'best_epoch': int(best_epoch[j]),
                    'epochs_run': len(losses[j]), 'test_pred': reconstruct(pt_s[j, :nt], P['scaler'], a),
                    'fire_rate': float(rate_te[j]),
                    'noisy_preds': {nl: [reconstruct(p[j, :nt], P['scaler'], a) for p in noisy_s[nl]] for nl in NOISE_LEVELS}})
    del params, opt, best, graph
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return out, energy


def finished_run(P, r):
    # metrics of one trained run on its fold (what the reference implementation stores per seed in runs.pt)
    y_te, prev_te, tau = P['y_te'], P['prev_te'], P['tau']
    m_noise = {}
    for nl in NOISE_LEVELS:
        draws = [metrics(p, y_te, prev_te, tau) for p in r['noisy_preds'][nl]]
        m_noise[nl] = {k: float(np.nanmean([dd[k] for dd in draws])) for k in ['rmse'] + METRIC_KEYS}
    rec = {k: v for k, v in r.items() if k != 'noisy_preds'}
    return {**rec, 'clean': metrics(r['test_pred'], y_te, prev_te, tau), 'noise': m_noise}


def summarize_fold(spec, D, LAG, P, done):
    fold = P['fold']
    y_te, prev_te, tau, true_te = P['y_te'], P['prev_te'], P['tau'], P['true_te']
    naive_pred, ridge_pred, persist_pred = P['naive_pred'], P['ridge_pred'], P['persist_pred']
    m_naive, m_ridge, m_persist = (metrics(p, y_te, prev_te, tau) for p in (naive_pred, ridge_pred, persist_pred))
    test_size, sl_te = P['n_test'], P['sl_te']

    runs = [done[r] for r in range(spec['n_runs'])]
    all_preds = np.array([r['test_pred'] for r in runs])
    ens_pred = all_preds.mean(0)
    m_ens = metrics(ens_pred, y_te, prev_te, tau)
    clean = {k: [r['clean'][k] for r in runs] for k in ['rmse'] + METRIC_KEYS}
    noise = {nl: {k: float(np.nanmean([r['noise'][nl][k] for r in runs])) for k in ['rmse'] + METRIC_KEYS}
             for nl in NOISE_LEVELS}
    fire = float(np.mean([r['fire_rate'] for r in runs]))

    sig = {}
    for bname, bpred in [('naive', naive_pred), ('ridge', ridge_pred), ('persist', persist_pred)]:
        dm, p_dm = dm_test(y_te - ens_pred, y_te - bpred)
        _, _, z, p_auc = delong_test(true_te, ens_pred - prev_te - tau, bpred - prev_te - tau)
        b_rmse = float(np.sqrt(np.mean((y_te - bpred) ** 2)))
        sig[bname] = {'dm': dm, 'dm_p': p_dm, 'delong_z': z, 'delong_p': p_auc,
                      'pct_seeds_lower_rmse': float(np.mean([r['clean']['rmse'] < b_rmse for r in runs]) * 100)}
    print(f"   {fold['label']}: RMSE  ens {m_ens['rmse']:.4g} | seed-mean {np.mean(clean['rmse']):.4g} | ridge {m_ridge['rmse']:.4g} | "
          f"persist {m_persist['rmse']:.4g} | naive {m_naive['rmse']:.4g}")
    print(f"   {' ' * len(fold['label'])}  AUC   ens {m_ens['auc']:.4f} | ridge {m_ridge['auc']:.4f} | naive {m_naive['auc']:.4f} | "
          f"firing rate (test) {fire:.3f}")

    runs_csv = os.path.join(work_dir_for(spec, D), f"runs_{run_tag_for(spec, D)}_lag{LAG}_{fold['tag']}.csv")
    pd.DataFrame([{'run': r['run'], 'best_epoch': r['best_epoch'], 'val_rmse': r['best_val_rmse'],
                   'fire_rate': r['fire_rate'],
                   **{f"clean_{k}": r['clean'][k] for k in ['rmse'] + METRIC_KEYS},
                   **{f"noise{nl}_{k}": r['noise'][nl][k] for nl in NOISE_LEVELS for k in ['rmse'] + METRIC_KEYS}}
                  for r in runs]).to_csv(runs_csv, index=False)
    store.add(runs_csv)

    return {'lag': LAG, 'label': fold['label'], 'n_train': P['n_train'], 'n_val': P['n_val'], 'n_test': test_size,
            'tau': tau, 'base_rate': float(true_te.mean()), 'y_te': y_te, 'prev_te': prev_te, 'true_te': true_te,
            'tau_vec': np.full(test_size, tau), 'tt_te': P['tt'][sl_te], 'fire': fire,
            'ens_pred': ens_pred, 'naive_pred': naive_pred, 'ridge_pred': ridge_pred, 'persist_pred': persist_pred,
            'all_preds': all_preds, 'm_ens': m_ens, 'm_naive': m_naive, 'm_ridge': m_ridge, 'm_persist': m_persist,
            'clean': clean, 'noise': noise, 'sig': sig,
            'best_epochs': np.array([r['best_epoch'] for r in runs]),
            'hit_cap': np.array([r['epochs_run'] >= spec['max_epochs'] for r in runs])}


def energy_total(rec):
    # one (dataset, model, lag): sum over its training groups (a lag is usually one group of 5 folds × 60 seeds)
    g = list((rec or {}).get('groups', {}).values())
    nan = float('nan')
    if not g:
        return {'energy_j': nan, 'seconds': nan, 'mean_w': nan, 'peak_w': nan, 'idle_w': nan, 'net_energy_j': nan,
                'n_models': 0, 'model_epochs': 0, 'j_per_model_epoch': nan, 'infer_j_per_forecast': nan,
                'infer_s_per_forecast': nan, 'infer_mean_w': nan, 'method': 'none', 'gpu': ''}
    j, s = sum(x['j'] for x in g), sum(x['s'] for x in g)
    me, nm = sum(x['model_epochs'] for x in g), sum(x['n_models'] for x in g)
    idle = g[-1].get('idle_w', nan)
    wavg = lambda key: float(sum(x[key] * x['n_models'] for x in g) / nm) if nm else nan
    return {'energy_j': j, 'seconds': s, 'mean_w': j / s if s > 0 else nan, 'peak_w': max(x['peak_w'] for x in g),
            'idle_w': idle, 'net_energy_j': j - idle * s, 'n_models': nm, 'model_epochs': me,
            'j_per_model_epoch': j / me if me else nan, 'infer_j_per_forecast': wavg('infer_j_per_forecast'),
            'infer_s_per_forecast': wavg('infer_s_per_forecast'), 'infer_mean_w': wavg('infer_mean_w'),
            'method': g[-1].get('method', ''), 'gpu': g[-1].get('gpu', '')}


def run_lag(spec, D, LAG, supervised, t_idx):
    N_RUNS, tag = spec['n_runs'], run_tag_for(spec, D)
    preps = [prepare_fold(D, LAG, fi, supervised, t_idx, spec['batch']) for fi in range(len(D['folds']))]
    done, runs_paths = [], []
    for P in preps:
        d = ckpt_dir_for(spec, D, LAG, P['fold']['tag']); os.makedirs(d, exist_ok=True)
        rp = os.path.join(d, 'runs.pt'); runs_paths.append(rp)
        done.append(torch.load(rp, map_location='cpu', weights_only=False) if os.path.exists(rp) else {})
    pending = [(fi, r) for fi in range(len(preps)) for r in range(N_RUNS) if r not in done[fi]]
    print(f"   runs done: {len(preps) * N_RUNS - len(pending)} | to train: {len(pending)}")
    if FOLD_PARALLEL:
        chunks = [pending]
    else:
        chunks = [[j for j in pending if j[0] == fi] for fi in range(len(preps))]
    groups = [c[g0:g0 + (RUN_BATCH or len(c))] for c in chunks if c for g0 in range(0, len(c), RUN_BATCH or len(c))]
    os.makedirs(group_dir_for(spec, D, LAG), exist_ok=True)
    e_path = energy_path_for(spec, D, LAG)
    e_rec = load_json(e_path) or {'groups': {}}
    for jobs in groups:
        gid = hashlib.sha1(repr(jobs).encode()).hexdigest()[:10]
        group_path = os.path.join(group_dir_for(spec, D, LAG), f"group_{len(jobs)}runs_{gid}.pt")
        print(f"   training {len(jobs)} runs (folds {sorted({fi + 1 for fi, _ in jobs})}) in parallel")
        out, energy = train_group(spec, D, LAG, preps, jobs, group_path)
        # energy first: if the session dies before runs.pt is written, the group resumes from its checkpoint and
        # rewrites the same entry (keyed by group id) with the carried-over totals, so nothing is counted twice
        e_rec['groups'][gid] = {**energy, 'idle_w': METER.idle_w, 'method': METER.method, 'gpu': METER.name}
        save_json(e_rec, e_path); store.add(e_path)
        for (fi, r), o in zip(jobs, out):
            done[fi][r] = finished_run(preps[fi], o)
        for fi in sorted({fi for fi, _ in jobs}):
            safe_save(done[fi], runs_paths[fi]); store.add(runs_paths[fi])
        store.remove(group_path)
        et = energy_total(e_rec)
        print(f"   lag {LAG} trained: {et['seconds'] / 60:.1f} GPU-min"
              + (f" | {et['energy_j'] / 1e3:.2f} kJ | mean {et['mean_w']:.0f} W | peak {et['peak_w']:.0f} W | "
                 f"{et['j_per_model_epoch']:.3g} J per model-epoch | {et['infer_j_per_forecast'] * 1e3:.4g} mJ per forecast"
                 if METER.ok else ''))
    summaries = [summarize_fold(spec, D, LAG, P, done[fi]) for fi, P in enumerate(preps)]
    if not groups and not e_rec['groups']:
        print(f"   (no energy record for lag {LAG}: trained before energy logging)")
    return summaries, energy_total(e_rec)


def resume_from_store(spec, D):
    # this run's files on LOCAL_ROOT → local copy (or, with FRESH_START, delete them on both sides)
    tag, wd = run_tag_for(spec, D), work_dir_for(spec, D)
    pattern = store.rel(os.path.join(wd, f"*{tag}*"))         # fnmatch '*' also matches '/': covers sub-folders
    if FRESH_START:
        for p in glob.glob(os.path.join(wd, '**', f"*{tag}*"), recursive=True):
            if os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)
            elif os.path.exists(p):
                os.remove(p)
        for r in store.matching(pattern):
            store.gone.add(r)
        store.push(f"fresh start {tag}", force=True)
    else:
        pulled = store.pull(pattern)
        if pulled:
            print(f"Resumed {len(pulled)} files of {tag} from {LOCAL_ROOT}")


def walk_forward(spec, D):
    name, N_RUNS, tag = spec['name'], spec['n_runs'], run_tag_for(spec, D)
    wd = work_dir_for(spec, D); os.makedirs(os.path.join(wd, 'figures'), exist_ok=True)
    print(f"\n{'#' * 110}\n{name} on {D['name']}: run tag {tag} | lags {spec['lags']} | batch {spec['batch']} | lr {spec['lr']} | "
          f"≤{spec['max_epochs']} epochs | patience {spec['patience']} | seeds {N_RUNS}\n{'#' * 110}")
    resume_from_store(spec, D)
    fold_rows, pooled_rows, pred_frames, energy_rows, results, stopped = [], [], [], [], {}, False
    for LAG in spec['lags']:
        if not lag_ok(spec, D, LAG):
            print(f"\nSKIP lag {LAG} on {D['name']}: only {train_rows_first_fold(D, LAG)} training windows in the first fold "
                  f"(need ≥ {min_train_rows()}) — skipped for every model")
            continue
        print(f"\n{'='*100}\n{name} — {D['name']} — LAG={LAG} — walk-forward\n{'='*100}")
        supervised, t_idx = make_supervised(D['values'], LAG, spec.get('h', 1))
        try:
            results[LAG], et = run_lag(spec, D, LAG, supervised, t_idx)
            print(f"   memory after lag {LAG}: RAM {trim_ram():.2f} / {RAM_CAP_GB:.0f} GB | peak VRAM {vram_gb():.2f} / {VRAM_CAP_GB:.0f} GB")
        except TimeBudgetReached:
            stopped = True
            store.push(f"{tag}: time budget reached at lag {LAG}", force=True)
            print(f"\n*** TIME BUDGET ({TIME_BUDGET_H} h) REACHED at {name} / {D['name']} / LAG={LAG} — checkpoints saved in "
                  f"{LOCAL_ROOT}. ***\nTo continue: run the notebook again; it resumes from the last saved epoch.")
            break
        energy_rows.append({'dataset': D['name'], 'model': name, 'lag': LAG, **et})

        for f in results[LAG]:
            fold_rows.append({'dataset': D['name'], 'model': name, 'lag': LAG, 'test_fold': f['label'], 'n_train': f['n_train'],
                              'n_val': f['n_val'], 'n_test': f['n_test'], 'base_rate': f['base_rate'], 'fire_rate': f['fire'],
                              **{f"ens_{k}": f['m_ens'][k] for k in ['rmse'] + METRIC_KEYS},
                              **{f"seedmean_{k}": float(np.nanmean(f['clean'][k])) for k in ['rmse'] + METRIC_KEYS},
                              **{f"seedstd_{k}": float(np.nanstd(f['clean'][k])) for k in ['rmse'] + METRIC_KEYS},
                              **{f"{b}_{k}": f[f'm_{b}'][k] for b in ('naive', 'ridge', 'persist') for k in ['rmse'] + METRIC_KEYS},
                              **{f"{b}_{s}": f['sig'][b][s] for b in ('naive', 'ridge', 'persist') for s in f['sig'][b]},
                              **{f"noise{nl}_{k}": f['noise'][nl][k] for nl in NOISE_LEVELS for k in ['rmse', 'acc', 'auc']}})

        cat = lambda key: np.concatenate([f[key] for f in results[LAG]])
        y_all, prev_all, lab_all, tau_all = cat('y_te'), cat('prev_te'), cat('true_te'), cat('tau_vec')
        preds = {m: cat(f'{m}_pred') for m in ('ens', 'naive', 'ridge', 'persist')}
        rmse = lambda e: float(np.sqrt(np.mean(e ** 2)))
        score = lambda p: p - prev_all - tau_all
        row = {'dataset': D['name'], 'model': name, 'lag': LAG, 'n_test_total': len(y_all),
               'fire_rate': float(np.mean([f['fire'] for f in results[LAG]])),
               'seedmean_rmse': float(np.mean([np.mean(f['clean']['rmse']) for f in results[LAG]])),
               'best_epoch_median': float(np.median(cat('best_epochs'))),
               'pct_seeds_hit_max_epochs': float(np.mean(cat('hit_cap')) * 100)}
        for m, p in preds.items():
            row[f'{m}_rmse'] = rmse(y_all - p)
            row[f'{m}_auc'] = safe_auc(lab_all, score(p))
            row[f'{m}_acc'] = float(np.mean((score(p) > 0) == lab_all))
        for b in ('naive', 'ridge', 'persist'):
            row[f'dm_vs_{b}'], row[f'dm_p_vs_{b}'] = dm_test(y_all - preds['ens'], y_all - preds[b])
            _, _, row[f'delong_z_vs_{b}'], row[f'delong_p_vs_{b}'] = delong_test(lab_all, score(preds['ens']), score(preds[b]))
        row.update({f'gpu_{k}': et[k] for k in ('energy_j', 'seconds', 'mean_w', 'peak_w', 'net_energy_j',
                                                 'j_per_model_epoch', 'infer_j_per_forecast')})
        pooled_rows.append(row)
        if SAVE_PREDS:
            for f in results[LAG]:
                pred_frames.append(pd.DataFrame({
                    'dataset': D['name'], 'model': name, 'run_tag': tag, 'lag': LAG, 'test_fold': f['label'],
                    'target_idx': f['tt_te'], 'y': f['y_te'], 'prev': f['prev_te'], 'tau': f['tau_vec'], 'label': f['true_te'],
                    'ens_pred': f['ens_pred'], 'naive_pred': f['naive_pred'], 'ridge_pred': f['ridge_pred'],
                    'persist_pred': f['persist_pred']}))

        print(f"\n{'─'*110}\n{D['name']} | LAG={LAG} — per test fold (ensemble of {N_RUNS} seeds)\n{'─'*110}")
        print(f"{'Test fold':<14}| {'Ens RMSE':>9} | {'Ridge':>9} | {'Persist':>9} | {'Naive':>9} || {'Ens AUC':>7} | {'Ridge':>6} | "
              f"{'Naive':>6} || {'fire':>5}")
        for f in results[LAG]:
            print(f"{f['label']:<14}| {f['m_ens']['rmse']:9.4g} | {f['m_ridge']['rmse']:9.4g} | {f['m_persist']['rmse']:9.4g} | "
                  f"{f['m_naive']['rmse']:9.4g} || {f['m_ens']['auc']:7.4f} | {f['m_ridge']['auc']:6.4f} | "
                  f"{f['m_naive']['auc']:6.4f} || {f['fire']:5.3f}")

        fr = results[LAG]
        xs = np.arange(len(fr)); lbl = [f['label'] for f in fr]
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'{name} — {D["name"]} (LAG={LAG}) — walk-forward', fontsize=14, fontweight='bold')
        for ax, key, title in [(axes[0], 'rmse', 'Test RMSE by fold (lower is better)'),
                               (axes[1], 'auc', 'Test AUC by fold (higher is better)')]:
            ax.boxplot([f['clean'][key] for f in fr], positions=xs, widths=0.35, showfliers=False)
            ax.plot(xs, [f['m_ens'][key] for f in fr], 'o-', color='#eb6834', label='Model (ensemble)')
            ax.plot(xs, [f['m_ridge'][key] for f in fr], 's--', color='#1baf7a', label='Ridge AR')
            ax.plot(xs, [f['m_persist'][key] for f in fr], 'd-.', color='#2a78d6', label='Persistence')
            ax.plot(xs, [f['m_naive'][key] for f in fr], '^:', color='#9a9890', label='Naive')
            ax.set_xticks(xs); ax.set_xticklabels(lbl, rotation=20); ax.set_title(title); ax.grid(True, alpha=0.3); ax.legend()
        last = fr[-1]
        mu, sd = last['all_preds'].mean(0), last['all_preds'].std(0)
        axes[2].plot(last['y_te'], color='#2a78d6', lw=1.0, label='Actual')
        axes[2].plot(mu, color='#eb6834', lw=1.5, ls='--', label=f'Mean ({N_RUNS} seeds)')
        axes[2].fill_between(range(len(mu)), mu - sd, mu + sd, alpha=0.2, color='#eb6834', label='±1σ')
        axes[2].set_title(f"Forecasts — test {last['label']}"); axes[2].grid(True, alpha=0.3); axes[2].legend()
        plt.tight_layout()
        fig_path = os.path.join(wd, 'figures', f"{tag}_lag{LAG}.png")
        fig.savefig(fig_path, dpi=120, bbox_inches='tight'); store.add(fig_path)
        if SHOW_LAG_FIGURES:
            show(fig)
        else:
            plt.close(fig)
        store.push(f"{tag}: lag {LAG} done", force=True)
    return {'spec': spec, 'dataset': D, 'fold_rows': fold_rows, 'pooled_rows': pooled_rows, 'pred_frames': pred_frames,
            'energy_rows': energy_rows, 'results': results, 'stopped': stopped}


def report(out):
    # cross-lag summary of one (dataset, model) + walkforward_folds / walkforward_pooled / preds / energy CSVs
    spec, D, pooled_rows = out['spec'], out['dataset'], out['pooled_rows']
    tag, wd = run_tag_for(spec, D), work_dir_for(spec, D)
    if not pooled_rows:
        print(f"{spec['name']} on {D['name']}: no completed lag to report")
        return {}
    W = 190
    print("\n" + "=" * W)
    print(f"CROSS-LAG SUMMARY — {spec['name']} on {D['name']} | level | walk-forward over "
          f"{', '.join(f['label'] for f in D['folds'])} (pooled)" + (' | PARTIAL (time budget)' if out['stopped'] else ''))
    print("=" * W)
    print(f"{'LAG':<4} | {'n':>5} | {'Ens RMSE':>9} | {'Ridge':>9} | {'Persist':>9} | {'Naive':>9} | "
          f"{'DM vs ridge (p)':>17} | {'DM vs persist (p)':>17} | {'DM vs naive (p)':>17} || "
          f"{'Ens AUC':>7} | {'Ridge':>6} | {'Naive':>6} | {'DeLong vs ridge (p)':>19} | {'fire':>5} || {'GPU kJ':>8} | {'mean W':>6}")
    print("-" * W)
    for r in pooled_rows:
        print(f"{r['lag']:<4} | {r['n_test_total']:5d} | {r['ens_rmse']:9.4g} | {r['ridge_rmse']:9.4g} | {r['persist_rmse']:9.4g} | "
              f"{r['naive_rmse']:9.4g} | {r['dm_vs_ridge']:+7.3f} ({r['dm_p_vs_ridge']:.4f}) | "
              f"{r['dm_vs_persist']:+7.3f} ({r['dm_p_vs_persist']:.4f}) | {r['dm_vs_naive']:+7.3f} ({r['dm_p_vs_naive']:.4f}) || "
              f"{r['ens_auc']:7.4f} | {r['ridge_auc']:6.4f} | {r['naive_auc']:6.4f} | "
              f"{r['delong_z_vs_ridge']:+8.3f} ({r['delong_p_vs_ridge']:.4f}) | {r['fire_rate']:5.3f} || "
              f"{r['gpu_energy_j'] / 1e3:8.2f} | {r['gpu_mean_w']:6.0f}")
    print("DM < 0: model has lower squared error.  DeLong z > 0: model has higher AUC.")
    print(f"Training (≤{spec['max_epochs']} epochs, patience {spec['patience']}): " + " | ".join(
        f"lag {r['lag']}: median best epoch {r['best_epoch_median']:.0f}, {r['pct_seeds_hit_max_epochs']:.0f}% seeds never early-stopped"
        for r in pooled_rows))
    saved = {}
    for key, df in [('walkforward_folds', pd.DataFrame(out['fold_rows'])), ('walkforward_pooled', pd.DataFrame(pooled_rows)),
                    ('energy', pd.DataFrame(out['energy_rows']))] + (
            [('preds', pd.concat(out['pred_frames'], ignore_index=True))] if SAVE_PREDS and out['pred_frames'] else []):
        p = os.path.join(wd, f"{key}_{tag}.csv")
        df.to_csv(p, index=False); store.add(p); saved[key] = p
    print(f"Saved {', '.join(os.path.basename(p) for p in saved.values())}")
    store.push(f"{tag}: results", force=True)
    return saved

## Part B — Training and fitting (cells 7–10)
The nine networks (cell 7), the closed-form TRACE-SNP v1 and its comparison models (cell 8), the networks at horizons 6 and 24 from their checkpoints (cell 9), and TRACE-SNP v2 (cell 10). Every cell is resumable.

In [ ]:
# ============================================================
# 7. NETWORK TRAINING — ALSTM-SNP and the 8 baselines on every dataset, with the walk-forward harness of cell 6
#    ALSTM-SNP: batch 8, lr 3e-3, ≤ 300 epochs, patience 40; baselines: batch 1, lr 1e-3, ≤ 100 epochs, patience 10;
#    hidden size 8 for all; 60 seeds × 5 folds × every lag in COMMON_LAGS (the 300 models of a lag train as one
#    vmapped ensemble on the GPU). Results go to LOCAL_ROOT/<dataset>/<model folder>/ (checkpoints, per-seed metrics,
#    the 60-seed ensemble predictions preds_*.csv, energy_*.json). Finished lags are skipped, so a rerun after a kernel
#    restart or the time budget continues where it stopped.
# ============================================================
TRAIN_LOG = []
if RUN_TRAINING:
    _stopped = False
    print(f"Training {[s['name'] for s in MODEL_SPECS]} on {len(DATASETS)} datasets (finished lags are skipped)")
    for _D in DATASETS:
        for _spec in MODEL_SPECS:
            try:
                check_time_budget()
            except TimeBudgetReached:
                _stopped = True
            if _stopped:
                break
            _out = walk_forward(_spec, _D)
            report(_out)
            TRAIN_LOG.append({'dataset': _D['name'], 'model': _spec['name'], 'lags_done': sorted(_out['results']),
                              'stopped': _out['stopped']})
            _stopped = _out['stopped']
            del _out; trim_ram()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
        if _stopped:
            print("*** Time budget reached — run the notebook again to continue the training. ***")
            break
    store.push("networks", force=True)
TRAIN_LOG = pd.DataFrame(TRAIN_LOG)
TRAIN_LOG

In [ ]:
# ============================================================
# 8. TRACE-SNP and the closed-form comparison models — the notebook's datasets × lag × horizon × fold
#     (all 11 datasets of cell 3; the networks of cell 7 are not refitted)
#     Same folds, τ and targets as the notebook; nothing already trained is retrained (the networks' results are read
#     from the results folder in cells 9 and 11). Resumable: each finished (dataset, lag, horizon) is saved to
#     <dataset>/closed_form/ in the results folder and reused.
#
#  TRACE-SNP (Tapped-delay, Rule-Additive, Convex, Event-driven SNP), per forecast of x_{o+h-1} from u_k = s(x_{o-k}):
#     x̂ = x_{o-1} + s⁻¹·[ b + Σ_k a_k u_k + Σ_k Σ_m w_{k,m} μ_m(u_k) + Σ_c ν_c(z)(α_c + γ_cᵀ z) ],   z = (u_1 … u_d)
#     μ_m: M triangular receptive fields per delay (Wang–Mendel fuzzy partition; 2 of M fire); ν_c: C state neurons on the
#     recent embedding, the k nearest fire (first-order Takagi–Sugeno rules). Two input encodings are candidates: levels
#     u_k = s(x_{o-k}), and level-invariant differences u_k = s'(x_{o-k} − x_{o-1}) for trending / non-stationary series. Linear in its weights → one ridge solve with a
#     curvature penalty per delay; selection by exact leave-one-out residuals on the training rows + the validation block,
#     with persistence and Ridge AR(L) as candidates (so the chosen model scores ≤ both on that criterion).
#  Ablations (same candidates, different selection rules / pathways):
#     delay pathway only · state pathway only · no curvature penalty · level inputs only · validation-only selection ·
#     no fallback candidates
#  TRACE-SNP (binary spikes): the chosen model refitted with every pulse value quantised to q spikes
#     (μ ∈ {0, 1/q, …, 1}, counts summing to q per delay / per firing state group) — a strictly binary-spike SNP.
#  NG-RC (Gauthier et al., Nat. Commun. 2021): constant + k delay taps (spacing s) + their unique quadratic monomials,
#     ridge readout on the increment; k, s, λ selected by the same criterion.
#  DLinear (Zeng et al., AAAI 2023): moving-average trend / seasonal split of the window, linear readout of both,
#     closed-form ridge; λ selected by the same criterion.
#  Horizons: direct strategy (one readout per horizon h); origins o with o+h-1 inside the fold's test block.
#  Noise robustness (h = 1): the notebook's exact input-noise draws (same scaled space, same seeds) applied to TRACE-SNP,
#     NG-RC and Ridge.
#  Energy: GPU energy (the notebook's METER) for each family's fit per (dataset, lag, horizon), plus an inference benchmark.
# ============================================================
CF_FOLDER      = 'closed_form'
CF_LAGS        = COMMON_LAGS
CF_HORIZONS    = [1, 6, 24]          # direct multi-step; a horizon is skipped where a test block has < CF_MIN_ORIGINS origins
CF_MIN_ORIGINS = 5
CF_FRESH       = False               # True refits everything (otherwise finished (dataset, lag, horizon) are reused)
CF_SPIKE_Q     = 8                   # spikes per delay for the binary-spike version
CF_GRID = dict(M=[16, 32], max_delay_units=2048, l0=[1e-6, 1e-4, 1e-2], ls=[1e-5, 1e-3, 1e-1],
               d=[5, 10], C=[32, 64], k_active=4, lst=[1e-6, 1e-4, 1e-2],
               ngrc_k=[2, 3, 4, 6], ngrc_s=[1, 2, 5, 10, 20], ngrc_lam=[1e-8, 1e-6, 1e-4, 1e-2],
               dlin_kernel=25, dlin_lam=[1e-6, 1e-4, 1e-2, 1.0])
CF_BENCH_S     = 0.3 if SMOKE else 1.0
TRACE_ENCODINGS = ['lvl', 'dif']      # input encodings TRACE-SNP chooses between: levels, and differences to the newest value
CF_MODELS = ['TRACE-SNP', 'TRACE-SNP (binary spikes)', 'TRACE: delay pathway only', 'TRACE: state pathway only',
             'TRACE: no curvature penalty', 'TRACE: level inputs only', 'TRACE: validation-only selection',
             'TRACE: no fallback candidates',
             'NG-RC', 'DLinear', 'Ridge AR(L)', 'Persistence']


def run_closed_form(datasets):
    from sklearn.preprocessing import MinMaxScaler as _MMS
    f64, dev = torch.float64, device
    G = CF_GRID
    T = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=f64, device=dev)
    sm = '_smoke' if SMOKE else ''

    # ── features ──
    def tri(U, M, q=None, lo=-1.1, hi=1.1):
        h = (hi - lo) / (M - 1)
        pos = (U.clamp(lo, hi) - lo) / h
        i = pos.floor().long().clamp(0, M - 2)
        fr = pos - i
        if q:
            fr = torch.round(fr * q) / q                      # q spikes split between the 2 firing neurons
        B = torch.zeros(U.shape + (M,), dtype=U.dtype, device=U.device)
        B.scatter_(-1, i[..., None], (1 - fr)[..., None])
        B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
        return B.reshape(U.shape[0], -1)

    def kmeans(Z, C, seed=0):
        g = torch.Generator(device='cpu').manual_seed(seed)
        idx = [int(torch.randint(len(Z), (1,), generator=g))]
        dmin = ((Z - Z[idx[0]]) ** 2).sum(1)
        for _ in range(C - 1):
            j = int(torch.multinomial((dmin / dmin.sum()).cpu(), 1, generator=g)); idx.append(j)
            dmin = torch.minimum(dmin, ((Z - Z[j]) ** 2).sum(1))
        cen = Z[idx].clone()
        for _ in range(20):
            lab = torch.cdist(Z, cen).argmin(1)
            cnt = torch.bincount(lab, minlength=C).to(Z.dtype)
            s = torch.zeros_like(cen).index_add_(0, lab, Z)
            keep = cnt > 0
            cen[keep] = s[keep] / cnt[keep, None]
        nn_d = torch.cdist(cen, cen) + torch.eye(C, dtype=Z.dtype, device=Z.device) * 1e9
        return cen, float(nn_d.min(1).values.median() ** 2)

    def state_feats(U, st, q=None):
        cen, s2, d, off = st
        Z = U[:, off:off + d]
        D2 = torch.cdist(Z, cen) ** 2
        val, idx = D2.topk(G['k_active'], dim=1, largest=False)
        g = torch.exp(-val / (2 * s2)); g = g / g.sum(1, keepdim=True)
        if q:                                                 # q spikes over the k firing state neurons (largest remainder)
            c = torch.floor(g * q); rem = (q - c.sum(1)).long()
            order = torch.argsort(g * q - c, dim=1, descending=True)
            add = (torch.arange(g.shape[1], device=g.device)[None] < rem[:, None]).to(g.dtype)
            c.scatter_add_(1, order, add); g = c / q
        Mu = torch.zeros_like(D2).scatter_(1, idx, g)
        Z1 = torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), Z], 1)
        return (Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1)

    def design(U, M=None, st=None, q=None):
        parts = [torch.ones(len(U), 1, dtype=U.dtype, device=U.device), U]
        if M:
            parts.append(tri(U, M, q))
        if st is not None:
            parts.append(state_feats(U, st, q))
        return torch.cat(parts, 1)

    def penalty(L, M, l0, ls, n_state=0, lst=0.0):
        nd = L * M if M else 0
        p = 1 + L + nd + n_state
        P = torch.zeros(p, p, dtype=f64, device=dev)
        P[1:, 1:] += l0 * torch.eye(p - 1, dtype=f64, device=dev)
        if M and ls > 0:
            D = torch.zeros(M - 2, M, dtype=f64, device=dev)
            for j in range(M - 2):
                D[j, j], D[j, j + 1], D[j, j + 2] = 1, -2, 1
            DD = ls * (D.T @ D)
            for k in range(L):
                a = 1 + L + k * M
                P[a:a + M, a:a + M] += DD
        if n_state and lst > l0:
            a = 1 + L + nd
            P[a:, a:] += (lst - l0) * torch.eye(n_state, dtype=f64, device=dev)
        P[0, 0] = 1e-10
        return P

    def fit_loo(Ptr, r, Pen, chunk=8192):
        # θ = (ΦᵀΦ/N + Λ)⁻¹ Φᵀr/N and the exact leave-one-out residuals e_i / (1 − h_ii)
        N = len(Ptr)
        A = Ptr.T @ Ptr / N + Pen
        Lc = torch.linalg.cholesky(A)
        th = torch.cholesky_solve((Ptr.T @ r / N)[:, None], Lc)[:, 0]
        loo = []
        for i in range(0, N, chunk):
            P = Ptr[i:i + chunk]
            V = torch.linalg.solve_triangular(Lc, P.T, upper=False)
            loo.append((r[i:i + chunk] - P @ th) / (1 - (V ** 2).sum(0) / N).clamp(min=1e-6))
        return th, torch.cat(loo)

    def ridge_loo(X, y, alpha):
        n = len(X); xm, ym = X.mean(0), y.mean(); Xc = X - xm
        Ai = np.linalg.inv(Xc.T @ Xc + alpha * np.eye(X.shape[1])); w = Ai @ Xc.T @ (y - ym)
        h = 1 / n + np.einsum('ij,jk,ik->i', Xc, Ai, Xc)
        return w, ym - xm @ w, (y - (Xc @ w + ym)) / (1 - h)

    def ngrc_feats(U, taps):
        V = U[:, taps]
        iu = torch.triu_indices(len(taps), len(taps), device=U.device)
        return torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), V, V[:, iu[0]] * V[:, iu[1]]], 1)

    def dlin_feats(U, kernel):
        # DLinear decomposition on the window in time order (oldest → newest), replicate padding, as in the original
        Xt = U.flip(1)
        k = kernel if kernel % 2 else kernel + 1
        pad = (k - 1) // 2
        Xp = torch.cat([Xt[:, :1].expand(-1, pad), Xt, Xt[:, -1:].expand(-1, pad)], 1)
        trend = Xp.unfold(1, k, 1).mean(-1)
        return torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), trend, Xt - trend], 1)

    def windows(x, L, o):
        return np.stack([x[o - 1 - k] for k in range(L)], 1)

    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))

    def run_fold(x, L, h, f, fi, n_folds, noise_draws):
        # origins o: window x[o-L .. o-1], target x[o+h-1]
        otr = np.arange(L, f['va_start'] - h + 1)
        ova = np.arange(f['va_start'], f['te_start'] - h + 1)
        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
        a, b = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / (b - a)
        s = lambda v: sc * (v - a) - 1
        Wtr, Wva, Wte = windows(x, L, otr), windows(x, L, ova), windows(x, L, ote)
        ytr, yva, yte = x[otr + h - 1], x[ova + h - 1], x[ote + h - 1]
        Utr, Uva, Ute = T(s(Wtr)), T(s(Wva)), T(s(Wte))
        rtr = T(s(ytr)) - Utr[:, 0]
        # level-invariant encoding: the window relative to its newest value, x_{o-k} - x_{o-1} (k ≥ 2), scaled on the
        # training rows; delay 1 carries no information here (always 0) and is kept only so the layout is unchanged
        dd = (Wtr[:, 1:] - Wtr[:, :1]) if L > 1 else np.zeros((len(Wtr), 1))
        scd = 1.0 / max(float(np.abs(dd).max()), 1e-12)
        enc = {'lvl': lambda W: T(s(W)),
               'dif': lambda W: T(np.column_stack([np.zeros(len(W)), (W[:, 1:] - W[:, :1]) * scd]) if L > 1 else np.zeros((len(W), 1)))}
        UU = {e: (f(Wtr), f(Wva), f(Wte)) for e, f in enc.items()}
        back = lambda W, rhat: W[:, 0] + rhat.cpu().numpy() / sc           # raw prediction from an increment in scaled units
        crit = lambda loo_raw, val_err: (rmse(np.concatenate([np.asarray(loo_raw), np.asarray(val_err)])), rmse(val_err))
        cands = []                     # dicts: fam, cfg, loo (LOO+val criterion), val, pred (test), fn (raw windows → raw preds)

        def add(fam, cfg, loo_raw, Wv_pred, te_pred, fn):
            c1, c2 = crit(loo_raw, yva - Wv_pred)
            cands.append({'fam': fam, 'cfg': cfg, 'loo': c1, 'val': c2, 'pred': te_pred, 'fn': fn})

        # persistence and the notebook's Ridge AR(L) (MinMax per column on the training rows, level target, closed form)
        add('persistence', {}, ytr - Wtr[:, 0], Wva[:, 0], Wte[:, 0], lambda W: W[:, 0])
        mm = _MMS((-1, 1)).fit(np.column_stack([Wtr, ytr]))
        tf = lambda W: (W * mm.scale_[:-1] + mm.min_[:-1])
        w_r, b_r, loo_r = ridge_loo(tf(Wtr), ytr * mm.scale_[-1] + mm.min_[-1], RIDGE_ALPHA)
        ridge_fn = lambda W: ((tf(W) @ w_r + b_r) - mm.min_[-1]) / mm.scale_[-1]
        add('ridge', {}, loo_r / mm.scale_[-1], ridge_fn(Wva), ridge_fn(Wte), ridge_fn)
        e_seg = {}

        # ── TRACE-SNP family ──
        seg = EnergySegment(METER); seg.resume()

        def trace_fn(th, M, st, q=None, e='lvl'):
            return lambda W: back(W, design(enc[e](W), M, st, q) @ th)

        def state_of(e, d, C):
            # state neurons on the newest d inputs (level) or the newest d differences x_{o-k} - x_{o-1}, k = 2..d+1
            Z = UU[e][0][:, :d] if e == 'lvl' else UU[e][0][:, 1:d + 1]
            cen, s2 = kmeans(Z, C)
            return cen, s2, d, (0 if e == 'lvl' else 1)

        pre = {'lvl': '', 'dif': 'Δ'}                        # family prefix: 'delay' (level inputs), 'Δdelay' (differences)
        for e in TRACE_ENCODINGS:
            if e == 'dif' and L < 3:
                continue
            Utr_, Uva_, Ute_ = UU[e]
            best_delay = {}
            for M in G['M']:
                if L * M > G['max_delay_units'] and M != min(G['M']):
                    continue
                Ptr, Pva, Pte = design(Utr_, M), design(Uva_, M), design(Ute_, M)
                for l0 in G['l0']:
                    for ls in G['ls'] + [0.0]:
                        th, loo = fit_loo(Ptr, rtr, penalty(L, M, l0, ls))
                        add(pre[e] + ('delay' if ls > 0 else 'delay_nocurv'), {'enc': e, 'M': M, 'l0': l0, 'ls': ls},
                            loo.cpu().numpy() / sc, back(Wva, Pva @ th), back(Wte, Pte @ th), trace_fn(th, M, None, None, e))
            for fam in ('delay', 'delay_nocurv'):
                best_delay[fam] = min([c for c in cands if c['fam'] == pre[e] + fam], key=lambda c: c['loo'])
            states = {}
            dmax = L if e == 'lvl' else L - 1
            for d in sorted({min(dmax, dd) for dd in G['d']}):
                for C in G['C']:
                    if C * (d + 1) > len(otr):
                        continue
                    states[(d, C)] = state_of(e, d, C)
            for base_fam, fam in (('delay', 'delay+state'), ('delay_nocurv', 'delay_nocurv+state'), (None, 'state')):
                if base_fam:
                    M, l0, ls = (best_delay[base_fam]['cfg'][k] for k in ('M', 'l0', 'ls'))
                else:
                    M, l0, ls = None, G['l0'][1], 0.0
                for (d, C), st in states.items():
                    Ptr, Pva, Pte = design(Utr_, M, st), design(Uva_, M, st), design(Ute_, M, st)
                    nst = C * (d + 1)
                    for lst in G['lst']:
                        th, loo = fit_loo(Ptr, rtr, penalty(L, M, l0, ls, nst, max(lst, l0)))
                        add(pre[e] + fam, {'enc': e, 'M': M, 'l0': l0, 'ls': ls, 'd': d, 'C': C, 'lst': lst},
                            loo.cpu().numpy() / sc, back(Wva, Pva @ th), back(Wte, Pte @ th), trace_fn(th, M, st, None, e))
        seg.pause(); e_seg['TRACE-SNP'] = seg.state()

        # ── NG-RC ──
        seg = EnergySegment(METER); seg.resume()
        tapsets = set()
        for k in G['ngrc_k']:
            for sp in G['ngrc_s']:
                if (k - 1) * sp <= L - 1:
                    tapsets.add(tuple(range(0, (k - 1) * sp + 1, sp)))
        if L == 1 or not tapsets:
            tapsets.add((0,))
        for taps in sorted(tapsets):
            Ptr, Pva, Pte = ngrc_feats(Utr, list(taps)), ngrc_feats(Uva, list(taps)), ngrc_feats(Ute, list(taps))
            for lam in G['ngrc_lam']:
                Pen = lam * torch.eye(Ptr.shape[1], dtype=f64, device=dev); Pen[0, 0] = 1e-10
                th, loo = fit_loo(Ptr, rtr, Pen)
                fn = (lambda th, taps: lambda W: back(W, ngrc_feats(T(s(W)), list(taps)) @ th))(th, taps)
                add('ngrc', {'taps': len(taps), 'span': taps[-1] + 1, 'lam': lam}, loo.cpu().numpy() / sc,
                    back(Wva, Pva @ th), back(Wte, Pte @ th), fn)
        seg.pause(); e_seg['NG-RC'] = seg.state()

        # ── DLinear (level target in scaled units) ──
        seg = EnergySegment(METER); seg.resume()
        ytr_s = T(s(ytr))
        Ptr, Pva, Pte = (dlin_feats(U, G['dlin_kernel']) for U in (Utr, Uva, Ute))
        for lam in G['dlin_lam']:
            Pen = lam * torch.eye(Ptr.shape[1], dtype=f64, device=dev); Pen[0, 0] = 1e-10
            th, loo = fit_loo(Ptr, ytr_s, Pen)
            lvl = lambda P, th=th: (P @ th).cpu().numpy() / sc + a + 1 / sc           # inverse of s
            fn = (lambda th: lambda W: (dlin_feats(T(s(W)), G['dlin_kernel']) @ th).cpu().numpy() / sc + a + 1 / sc)(th)
            add('dlinear', {'lam': lam}, loo.cpu().numpy() / sc, lvl(Pva), lvl(Pte), fn)
        seg.pause(); e_seg['DLinear'] = seg.state()

        # ── selection rules → one predictor per model ──
        def pick(fams, key='loo'):
            pool = [c for c in cands if c['fam'] in fams]
            best = pool[0]
            for c in pool[1:]:                                 # strict improvement: ties go to the earlier (simpler) one
                if c[key] < best[key]:
                    best = c
            return best
        base = ['persistence', 'ridge']
        both = lambda *f: [p + x for p in ('', 'Δ') for x in f]
        chosen = {
            'TRACE-SNP': pick(base + both('delay', 'delay+state')),
            'TRACE: delay pathway only': pick(base + both('delay')),
            'TRACE: state pathway only': pick(base + both('state')),
            'TRACE: no curvature penalty': pick(base + both('delay_nocurv', 'delay_nocurv+state')),
            'TRACE: level inputs only': pick(base + ['delay', 'delay+state']),
            'TRACE: validation-only selection': pick(base + both('delay', 'delay+state'), 'val'),
            'TRACE: no fallback candidates': pick(both('delay', 'delay+state')),
            'NG-RC': pick(['ngrc']), 'DLinear': pick(['dlinear']),
            'Ridge AR(L)': pick(['ridge']), 'Persistence': pick(['persistence']),
        }
        # binary-spike TRACE-SNP: the chosen configuration refitted on q-spike features
        full = chosen['TRACE-SNP']
        is_trace = full['fam'] in both('delay', 'delay+state')
        if is_trace:
            cfg = full['cfg']; e = cfg['enc']
            st = state_of(e, cfg['d'], cfg['C']) if 'd' in cfg else None
            q = CF_SPIKE_Q
            Ptr = design(UU[e][0], cfg['M'], st, q)
            nst = cfg['C'] * (cfg['d'] + 1) if st is not None else 0
            th, _ = fit_loo(Ptr, rtr, penalty(L, cfg['M'], cfg['l0'], cfg['ls'], nst, max(cfg.get('lst', 0), cfg['l0'])))
            fn = trace_fn(th, cfg['M'], st, q, e)
            chosen['TRACE-SNP (binary spikes)'] = {'fam': full['fam'] + ' (q=%d)' % q, 'cfg': cfg, 'loo': np.nan, 'val': np.nan,
                                                   'pred': fn(Wte), 'fn': fn}
        else:
            chosen['TRACE-SNP (binary spikes)'] = full

        # noise robustness (h = 1): the notebook's draws — scaled per column on its training rows, std of the scaled inputs
        noise = []
        if h == 1 and noise_draws:
            mmn = _MMS((-1, 1)).fit(np.column_stack([Wtr, ytr]))
            Xtr_s = Wtr * mmn.scale_[:-1] + mmn.min_[:-1]; Xte_s = Wte * mmn.scale_[:-1] + mmn.min_[:-1]
            ref_std = Xtr_s.std()
            for li, nl in enumerate(NOISE_LEVELS):
                for dd in range(noise_draws):
                    Xn = Xte_s + np.random.default_rng([L, fi, li, dd]).normal(0.0, nl * ref_std, size=Xte_s.shape)
                    Wn = (Xn - mmn.min_[:-1]) / mmn.scale_[:-1]
                    for m in ('TRACE-SNP', 'NG-RC', 'DLinear', 'Ridge AR(L)', 'Persistence'):
                        noise.append({'model': m, 'noise': nl, 'draw': dd, 'rmse': rmse(yte - chosen[m]['fn'](Wn))})

        # learned delay functions of TRACE-SNP (raw units, contribution to the increment) for the interpretability figure
        curves = None
        if h == 1 and is_trace and fi == n_folds - 1:
            M = full['cfg']['M']; e = full['cfg']['enc']; grid = torch.linspace(-1, 1, 60, dtype=f64, device=dev)
            Ptr = design(UU[e][0], M, state_of(e, full['cfg']['d'], full['cfg']['C']) if 'd' in full['cfg'] else None)
            nst = Ptr.shape[1] - (1 + L + L * M)
            th, _ = fit_loo(Ptr, rtr, penalty(L, M, full['cfg']['l0'], full['cfg']['ls'], nst, max(full['cfg'].get('lst', 0), full['cfg']['l0'])))
            Bg = tri(grid[:, None], M)                                         # (60, M)
            rows = []
            for k in range(L):
                fk = (grid * th[1 + k] + Bg @ th[1 + L + k * M:1 + L + (k + 1) * M]).cpu().numpy() / sc
                rows.append(fk)
            gx = grid.cpu().numpy()
            curves = {'u_raw': (gx + 1) / sc + a if e == 'lvl' else gx / scd, 'f': np.array(rows),
                      'input': 'x(t-k)' if e == 'lvl' else 'x(t-k) - x(t-1)', 'skip_first': e == 'dif'}
        return chosen, ote, e_seg, noise, curves, Ute, Wte

    def bench(fn, W):
        seg = EnergySegment(METER); reps, t_end = 0, time.perf_counter() + CF_BENCH_S
        seg.resume()
        while reps < 1 or time.perf_counter() < t_end:
            fn(W); reps += 1
        seg.pause(); st = seg.state(); n = reps * len(W)
        return st['j'] / n, st['s'] / n

    all_D = list(datasets)
    preds, energy, choices, noise_rows, curve_rows = [], [], [], [], []

    def one_dataset(D):
        x = D['values']
        for L in CF_LAGS:
            if not lag_ok(None, D, L):
                print(f"{D['name']} lag {L}: skipped (too short) — as for every model"); continue
            for h in CF_HORIZONS:
                if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 < CF_MIN_ORIGINS:
                    print(f"{D['name']} lag {L} h {h}: skipped (test blocks shorter than h + {CF_MIN_ORIGINS - 1})"); continue
                stem = f"{D['slug']}/{CF_FOLDER}/cf{sm}_v2_lag{L}_h{h}"
                rels = {k: f"{stem}_{k}.csv" for k in ('preds', 'energy', 'choices', 'noise', 'curves')}
                if not CF_FRESH and all(store.fetch(rels[k]) for k in ('preds', 'energy', 'choices')):
                    for k, bucket in (('preds', preds), ('energy', energy), ('choices', choices), ('noise', noise_rows), ('curves', curve_rows)):
                        p = store.fetch(rels[k])
                        if p and os.path.getsize(p) > 2:
                            bucket.append(pd.read_csv(p))
                    continue
                check_time_budget()
                t0 = time.time()
                P, E, Ch, Nz, Cv = [], {}, [], [], []
                for fi, f in enumerate(D['folds']):
                    chosen, ote, e_seg, noise, curves, Ute, Wte = run_fold(x, L, h, f, fi, len(D['folds']), N_NOISE_DRAWS)
                    pre = x[:f['te_start']]
                    tau_h = round(float(np.median(pre[h:] - pre[:-h]) if h > 1 else np.median(np.diff(pre))), 6)
                    y, prev = x[ote + h - 1], x[ote - 1]
                    for m in CF_MODELS:
                        c = chosen[m]
                        P.append(pd.DataFrame({'dataset': D['name'], 'model': m, 'lag': L, 'h': h, 'test_fold': f['label'],
                                               'origin': ote, 'target_idx': ote + h - 1, 'y': y, 'prev': prev, 'tau': tau_h,
                                               'label': ((y - prev) > tau_h).astype(int), 'ens_pred': c['pred']}))
                        Ch.append({'dataset': D['name'], 'model': m, 'lag': L, 'h': h, 'fold': f['label'], 'family': c['fam'],
                                   'criterion_loo_val': c['loo'], 'criterion_val': c['val'], **{k: v for k, v in c['cfg'].items()}})
                    for fam, st in e_seg.items():
                        E.setdefault(fam, {'j': 0.0, 's': 0.0, 'peak_w': 0.0})
                        E[fam]['j'] += st['j']; E[fam]['s'] += st['s']; E[fam]['peak_w'] = max(E[fam]['peak_w'], st['peak_w'])
                    for r in noise:
                        Nz.append({'dataset': D['name'], 'lag': L, 'fold': f['label'], **r})
                    if curves is not None:
                        for k in range(int(curves['skip_first']), curves['f'].shape[0]):
                            Cv.append(pd.DataFrame({'dataset': D['name'], 'lag': L, 'delay': k + 1, 'input': curves['input'],
                                                    'u': curves['u_raw'], 'f': curves['f'][k]}))
                    if fi == len(D['folds']) - 1:
                        for fam, m in (('TRACE-SNP', 'TRACE-SNP'), ('NG-RC', 'NG-RC'), ('DLinear', 'DLinear')):
                            ej, es = bench(chosen[m]['fn'], Wte)
                            E[fam]['infer_j_per_forecast'], E[fam]['infer_s_per_forecast'] = ej, es
                Pdf = pd.concat(P, ignore_index=True)
                Edf = pd.DataFrame([{'dataset': D['name'], 'model': fam, 'lag': L, 'h': h, 'energy_j': v['j'], 'seconds': v['s'],
                                     'mean_w': v['j'] / v['s'] if v['s'] else np.nan, 'peak_w': v['peak_w'],
                                     'infer_j_per_forecast': v.get('infer_j_per_forecast', np.nan),
                                     'infer_s_per_forecast': v.get('infer_s_per_forecast', np.nan), 'method': METER.method}
                                    for fam, v in E.items()])
                Chdf, Nzdf = pd.DataFrame(Ch), pd.DataFrame(Nz)
                Cvdf = pd.concat(Cv, ignore_index=True) if Cv else pd.DataFrame()
                for k, df in (('preds', Pdf), ('energy', Edf), ('choices', Chdf), ('noise', Nzdf), ('curves', Cvdf)):
                    p = os.path.join(LOCAL_ROOT, rels[k]); os.makedirs(os.path.dirname(p), exist_ok=True)
                    df.to_csv(p, index=False); store.add(p)
                preds.append(Pdf); energy.append(Edf); choices.append(Chdf); noise_rows.append(Nzdf); curve_rows.append(Cvdf)
                g = Pdf.groupby('model').apply(lambda d: rmse(d['y'] - d['ens_pred']), include_groups=False)
                print(f"{D['name']:<16} lag {L:2d} h {h:2d}: TRACE-SNP {g['TRACE-SNP']:.4g} | NG-RC {g['NG-RC']:.4g} | "
                      f"DLinear {g['DLinear']:.4g} | Ridge {g['Ridge AR(L)']:.4g} | persistence {g['Persistence']:.4g} | "
                      f"chosen {Chdf[Chdf['model'] == 'TRACE-SNP']['family'].value_counts().to_dict()} | {time.time() - t0:.0f} s")
                store.push(f"closed-form {D['slug']} lag {L} h {h}")
            store.push(f"closed-form {D['slug']}", force=True)

    try:
        for D in all_D:
            one_dataset(D)
            trim_ram()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    except TimeBudgetReached:
        store.push('closed-form: time budget', force=True)
        print('*** Time budget reached — rerun to continue; finished (dataset, lag, horizon) are kept. ***')
    cat = lambda lst: pd.concat([d for d in lst if len(d)], ignore_index=True) if any(len(d) for d in lst) else pd.DataFrame()
    return {'preds': cat(preds), 'energy': cat(energy), 'choices': cat(choices), 'noise': cat(noise_rows), 'curves': cat(curve_rows)}


CF_RESULTS_MAIN = run_closed_form(DATASETS)
CF_RESULTS_EXTRA = {k: pd.DataFrame() for k in CF_RESULTS_MAIN}   # every dataset is in DATASETS

In [ ]:
# ============================================================
# 9. NETWORKS AT LONGER HORIZONS — from the saved checkpoints, no retraining (all 11 datasets)
#     For every network, dataset, lag and fold, the 60 saved best-validation models (runs.pt) forecast
#     recursively: each one-step prediction is fed back into the window, h times. The ensemble is the mean over seeds,
#     as for the one-step results. Scaling is rebuilt exactly as in prepare_fold (MinMax per column on the fold's
#     training rows). Check: the first recursive step must reproduce the saved one-step predictions (printed per model).
#     Same forecast origins, targets and τ_h as cell 8, so the models are compared on identical points.
#     Saved to <dataset>/<model>/multih_*.csv in the results folder; finished (model, dataset, lag, horizon) are reused.
# ============================================================
MH_HORIZONS    = [6, 24]          # same horizons as TRACE-SNP's CF_HORIZONS (h = 1 is the saved one-step result)
MH_MIN_ORIGINS = 5                # a horizon is skipped where a test block has fewer forecast origins
MH_FRESH       = False


def run_multi_horizon(datasets):
    sm = '_smoke' if SMOKE else ''
    all_D = list(datasets)
    rows, checks = [], []

    def forecast(spec, D, L, fi, horizons):
        f = D['folds'][fi]
        p = store.fetch(store.rel(os.path.join(ckpt_dir_for(spec, D, L, f['tag']), 'runs.pt')))
        if p is None:
            return None, None
        done = torch.load(p, map_location='cpu', weights_only=False)
        runs = [done[r] for r in sorted(done)]
        if not runs:
            return None, None
        x = D['values']
        sup, t_idx = make_supervised(x, L)
        n_train = int(np.sum(t_idx < f['va_start'])); n = int(np.sum(t_idx < f['te_end']))
        sup = sup[:n].copy()
        resid = TARGET_MODE == 'resid'
        if resid:
            sup[:, -1] = sup[:, -1] - sup[:, 0]
        sc = MinMaxScaler(feature_range=(-1, 1)).fit(sup[:n_train])
        smin, sscale = torch.tensor(sc.min_, device=device), torch.tensor(sc.scale_, device=device)
        keys = list(runs[0]['state_dict'])
        stacked = {k: torch.stack([r['state_dict'][k] for r in runs]).to(device) for k in keys}
        base = spec['build']().to('meta')
        if spec['spikes']:
            one = lambda pb, xx: functional_call(base, pb, (xx,), {'return_spikes': False})
        else:
            one = lambda pb, xx: functional_call(base, pb, (xx,))
        ens = vmap(one, in_dims=(0, 0))
        out, check = {}, None
        hmax = max(horizons + [1])
        o = np.arange(f['te_start'], f['te_end'] - 1 + 1)               # all one-step origins (for the check)
        W = torch.tensor(np.stack([x[o - 1 - k] for k in range(L)], 1), device=device)        # (N, L) raw, col 0 = newest
        Wk = W[None].expand(len(runs), -1, -1).clone()                                         # (K, N, L)
        with torch.no_grad():
            for step in range(1, hmax + 1):
                Xs = Wk * sscale[:L] + smin[:L]
                seq = Xs.flip(-1)[..., None].float()
                y_s = ens(stacked, seq)[..., 0].double()
                y = (y_s - smin[-1]) / sscale[-1] + (Wk[..., 0] if resid else 0.0)            # (K, N) raw
                if step == 1:
                    saved = np.mean([r['test_pred'] for r in runs], 0)
                    check = float(np.max(np.abs(y.mean(0).cpu().numpy() - saved)))
                if step in horizons:
                    # origin o predicts x[o + step - 1]; keep origins whose target lies inside the test block
                    keep = (o + step - 1) < f['te_end']
                    out[step] = (o[keep], y.mean(0).cpu().numpy()[keep])
                Wk = torch.cat([y[..., None], Wk[..., :-1]], -1)
        return out, check

    for D in all_D:
        x = D['values']
        for spec in MODEL_SPECS:
            for L in spec['lags']:
                if not lag_ok(spec, D, L):
                    continue
                hs = [h for h in MH_HORIZONS
                      if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 >= MH_MIN_ORIGINS]
                rels = {h: store.rel(os.path.join(work_dir_for(spec, D), f"multih{sm}_{run_tag_for(spec, D)}_lag{L}_h{h}.csv")) for h in hs}
                todo = [h for h in hs if MH_FRESH or store.fetch(rels[h]) is None]
                for h in hs:
                    if h not in todo:
                        rows.append(pd.read_csv(os.path.join(LOCAL_ROOT, rels[h])))
                if not todo:
                    continue
                parts = {h: [] for h in todo}; ok = True
                for fi, f in enumerate(D['folds']):
                    out, check = forecast(spec, D, L, fi, todo)
                    if out is None:
                        ok = False; break
                    checks.append({'dataset': D['name'], 'model': spec['name'], 'lag': L, 'fold': f['label'], 'max_abs_diff_1step': check})
                    pre = x[:f['te_start']]
                    for h in todo:
                        o, pred = out[h]
                        tau_h = round(float(np.median(pre[h:] - pre[:-h])), 6)
                        y, prev = x[o + h - 1], x[o - 1]
                        parts[h].append(pd.DataFrame({'dataset': D['name'], 'model': spec['name'], 'lag': L, 'h': h,
                                                      'test_fold': f['label'], 'origin': o, 'target_idx': o + h - 1, 'y': y,
                                                      'prev': prev, 'tau': tau_h, 'label': ((y - prev) > tau_h).astype(int),
                                                      'ens_pred': pred}))
                if not ok:
                    continue                                    # this (model, lag) is not trained (yet) on this dataset
                for h in todo:
                    df = pd.concat(parts[h], ignore_index=True)
                    p = os.path.join(LOCAL_ROOT, rels[h]); os.makedirs(os.path.dirname(p), exist_ok=True)
                    df.to_csv(p, index=False); store.add(p); rows.append(df)
                c = [r['max_abs_diff_1step'] for r in checks if r['dataset'] == D['name'] and r['model'] == spec['name'] and r['lag'] == L]
                print(f"{D['name']:<16} {spec['name']:<16} lag {L:2d}: horizons {todo} done | 1-step check: max |recursive − saved| = {max(c):.2e}")
            store.push(f"multi-horizon {D['slug']} {spec['folder']}")
        store.push(f"multi-horizon {D['slug']}", force=True)
    CH = pd.DataFrame(checks)
    if len(CH):
        bad = CH[CH['max_abs_diff_1step'] > 1e-3 * max(1.0, float(np.nanmax(np.abs(np.concatenate([d['values'] for d in all_D])))))]
        print(f"\n1-step reproduction check: worst |recursive − saved| = {CH['max_abs_diff_1step'].max():.2e} "
              + ("— OK" if bad.empty else f"— {len(bad)} (model, lag, fold) differ: check those before using their horizons"))
    return (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()), CH


MH_PREDS_MAIN, MH_CHECKS_MAIN = run_multi_horizon(DATASETS)
MH_PREDS_EXTRA, MH_CHECKS_EXTRA = pd.DataFrame(), pd.DataFrame()   # every dataset is in DATASETS

In [ ]:
# ============================================================
# 10. TRACE-SNP v2 — the dataset × every lag × horizons 1, 6, 24 × the 5 walk-forward folds
#     (the binary-spike model's spike count q per delay is chosen per fold on the VALIDATION block:
#      the fewest spikes in T2_SPIKE_Q_GRID whose validation RMSE is within T2_SPIKE_TOL of the analog model's,
#      otherwise the q with the lowest validation RMSE. Test blocks are never consulted.)
#     Designed ONLY on development folds (each series cut before its first real test block); the real test blocks are
#     used once, here. Changes from TRACE-SNP (cell 8), each chosen on the development folds:
#       1. Reference point: the forecast is a correction to either the newest value x_{o-1} (persistent / chaotic series)
#          or the window mean (noisy, mean-reverting series), u-encoding levels or differences; all are candidates.
#       2. Direct vs iterated multi-step: for h > 1 the h-step direct readout competes with the one-step model applied h
#          times; the iterated forecast is used only when its validation RMSE is ≥ T2_ITER_MARGIN lower (a margin rule,
#          so a single short validation block cannot flip the choice on noise).
#       3. One model per forecast (no averaging), so the per-forecast cost stays one TRACE-SNP evaluation (× h if iterated).
#     Selection as before: exact leave-one-out residuals on the training rows + the validation block, with persistence,
#     window mean and Ridge AR(L) as candidates. Binary-spike version: the chosen model refitted on q-spike features.
#     Energy: GPU energy of the whole fit (all candidates, all folds) per (dataset, lag, horizon), and an inference
#     benchmark on ≥ T2_BENCH_N forecasts per call (large batches, like the networks' benchmark). Operations per forecast
#     are counted exactly from the chosen model. Resumable: <dataset>/trace_snp_v2/ in the results folder.
# ============================================================
T2_FOLDER       = 'trace_snp_v2b'
T2_LAGS         = COMMON_LAGS
T2_HORIZONS     = [1, 6, 24]
T2_MIN_ORIGINS  = 5
T2_ITER_MARGIN  = 0.25
T2_SPIKE_Q_GRID = [8, 16, 32]   # spikes per delay tried for the binary model
T2_SPIKE_TOL    = 0.02          # fewest spikes within 2 % of the analog model's validation RMSE
T2_BENCH_N      = 18000
T2_BENCH_S      = 0.3 if SMOKE else 1.0
T2_FRESH        = False
T2_GRID = dict(M=[16, 32], max_delay_units=2048, l0=[1e-6, 1e-4, 1e-2, 1.0], ls=[1e-5, 1e-3, 1e-1, 10.0],
               d=[5, 10], C=[32, 64], k_active=4, lst=[1e-4, 1e-2, 1.0])
T2_NAME, T2_NAME_BIN = 'TRACE-SNP v2', 'TRACE-SNP v2 (binary spikes)'


def run_trace_v2(datasets):
    from sklearn.preprocessing import MinMaxScaler as _MMS
    f64, dev, G = torch.float64, device, T2_GRID
    T = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=f64, device=dev)
    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
    sm = '_smoke' if SMOKE else ''

    def tri(U, M, q=None, lo=-1.1, hi=1.1):
        h = (hi - lo) / (M - 1)
        pos = (U.clamp(lo, hi) - lo) / h
        i = pos.floor().long().clamp(0, M - 2)
        fr = pos - i
        if q:
            fr = torch.round(fr * q) / q
        B = torch.zeros(U.shape + (M,), dtype=U.dtype, device=U.device)
        B.scatter_(-1, i[..., None], (1 - fr)[..., None])
        B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
        return B.reshape(U.shape[0], -1)

    def kmeans(Z, C, seed=0):
        g = torch.Generator(device='cpu').manual_seed(seed)
        idx = [int(torch.randint(len(Z), (1,), generator=g))]
        dmin = ((Z - Z[idx[0]]) ** 2).sum(1)
        for _ in range(C - 1):
            j = int(torch.multinomial((dmin / dmin.sum()).cpu(), 1, generator=g)); idx.append(j)
            dmin = torch.minimum(dmin, ((Z - Z[j]) ** 2).sum(1))
        cen = Z[idx].clone()
        for _ in range(20):
            lab = torch.cdist(Z, cen).argmin(1)
            cnt = torch.bincount(lab, minlength=C).to(Z.dtype)
            s_ = torch.zeros_like(cen).index_add_(0, lab, Z)
            keep = cnt > 0
            cen[keep] = s_[keep] / cnt[keep, None]
        nn_d = torch.cdist(cen, cen) + torch.eye(C, dtype=Z.dtype, device=Z.device) * 1e9
        return cen, float(nn_d.min(1).values.median() ** 2)

    def state_feats(U, st, q=None):
        cen, s2, d, off = st
        Z = U[:, off:off + d]
        D2 = torch.cdist(Z, cen) ** 2
        val, idx = D2.topk(G['k_active'], dim=1, largest=False)
        g = torch.exp(-val / (2 * s2)); g = g / g.sum(1, keepdim=True)
        if q:
            c = torch.floor(g * q); rem = (q - c.sum(1)).long()
            order = torch.argsort(g * q - c, dim=1, descending=True)
            add_ = (torch.arange(g.shape[1], device=g.device)[None] < rem[:, None]).to(g.dtype)
            c.scatter_add_(1, order, add_); g = c / q
        Mu = torch.zeros_like(D2).scatter_(1, idx, g)
        Z1 = torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), Z], 1)
        return (Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1)

    def design(U, M=None, st=None, q=None):
        parts = [torch.ones(len(U), 1, dtype=U.dtype, device=U.device), U]
        if M:
            parts.append(tri(U, M, q))
        if st is not None:
            parts.append(state_feats(U, st, q))
        return torch.cat(parts, 1)

    def penalty(L, M, l0, ls, n_state=0, lst=0.0):
        nd = L * M if M else 0
        p = 1 + L + nd + n_state
        P = torch.zeros(p, p, dtype=f64, device=dev)
        P[1:, 1:] += l0 * torch.eye(p - 1, dtype=f64, device=dev)
        if M and ls > 0:
            D = torch.zeros(M - 2, M, dtype=f64, device=dev)
            for j in range(M - 2):
                D[j, j], D[j, j + 1], D[j, j + 2] = 1, -2, 1
            DD = ls * (D.T @ D)
            for k in range(L):
                a0 = 1 + L + k * M
                P[a0:a0 + M, a0:a0 + M] += DD
        if n_state and lst > l0:
            a0 = 1 + L + nd
            P[a0:, a0:] += (lst - l0) * torch.eye(n_state, dtype=f64, device=dev)
        P[0, 0] = 1e-10
        return P

    def fit_loo(Ptr, R, Pen, chunk=8192):
        N = len(Ptr)
        A = Ptr.T @ Ptr / N + Pen
        Lc = torch.linalg.cholesky(A)
        th = torch.cholesky_solve(Ptr.T @ R / N, Lc)
        loo = []
        for i in range(0, N, chunk):
            P = Ptr[i:i + chunk]
            V = torch.linalg.solve_triangular(Lc, P.T, upper=False)
            hd = ((V ** 2).sum(0) / N).clamp(max=1 - 1e-6)
            loo.append((R[i:i + chunk] - P @ th) / (1 - hd)[:, None])
        return th, torch.cat(loo)

    def windows(x, L, o):
        return np.stack([x[o - 1 - k] for k in range(L)], 1)

    def run_fold(x, L, h, f):
        otr = np.arange(L, f['va_start'] - h + 1)
        ova = np.arange(f['va_start'], f['te_start'] - h + 1)
        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
        a, b = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / (b - a)
        s = lambda v: sc * (v - a) - 1
        W = {'tr': windows(x, L, otr), 'va': windows(x, L, ova), 'te': windows(x, L, ote)}
        y = {'tr': x[otr + h - 1], 'va': x[ova + h - 1], 'te': x[ote + h - 1]}
        anc_raw = {'last': lambda Wr: Wr[:, 0], 'mean': lambda Wr: Wr.mean(1)}
        dd = (W['tr'][:, 1:] - W['tr'][:, :1]) if L > 1 else np.zeros((len(otr), 1))
        scd = 1.0 / max(float(np.abs(dd).max()), 1e-12)
        enc = {'lvl': lambda Wr: T(s(Wr)),
               'dif': lambda Wr: T(np.column_stack([np.zeros(len(Wr)), (Wr[:, 1:] - Wr[:, :1]) * scd]) if L > 1
                                   else np.zeros((len(Wr), 1)))}
        cands = []

        def add(fam, cfg, loo_raw, va_pred, te_pred, fn, refit=None):
            e_va = y['va'] - va_pred
            cands.append({'fam': fam, 'cfg': cfg, 'pool': rmse(np.concatenate([loo_raw, e_va])), 'va_pred': va_pred,
                          'te_pred': te_pred, 'fn': fn, 'refit': refit})

        for an in ('last', 'mean'):
            add('persistence' if an == 'last' else 'window-mean', {'anchor': an}, y['tr'] - anc_raw[an](W['tr']),
                anc_raw[an](W['va']), anc_raw[an](W['te']), anc_raw[an])
        mm = _MMS((-1, 1)).fit(np.column_stack([W['tr'], y['tr']]))
        tf = lambda Wr: Wr * mm.scale_[:-1] + mm.min_[:-1]
        Xc = tf(W['tr']); yy = y['tr'] * mm.scale_[-1] + mm.min_[-1]
        xm, ym = Xc.mean(0), yy.mean(); Xc0 = Xc - xm
        Ai = np.linalg.inv(Xc0.T @ Xc0 + RIDGE_ALPHA * np.eye(L)); wr = Ai @ Xc0.T @ (yy - ym)
        hr = 1 / len(Xc) + np.einsum('ij,jk,ik->i', Xc0, Ai, Xc0)
        rf = lambda Wr: ((tf(Wr) - xm) @ wr + ym - mm.min_[-1]) / mm.scale_[-1]
        add('ridge', {}, ((yy - (Xc0 @ wr + ym)) / (1 - hr)) / mm.scale_[-1], rf(W['va']), rf(W['te']), rf)

        ys = T(s(y['tr']))

        def make_fn(e, an, M, st, th, q=None):
            def fn(Wr):
                return ((T(s(anc_raw[an](Wr))) + design(enc[e](Wr), M, st, q) @ th).cpu().numpy() + 1) / sc + a
            return fn

        def make_refit(e, an, M, st, pen_args, j):
            # pen_args, not the penalty matrix: a matrix per candidate would hold GBs at long lags
            def refit(q):
                Ptr = design(enc[e](W['tr']), M, st, q)
                th, _ = fit_loo(Ptr, (ys - T(s(anc_raw[an](W['tr']))))[:, None], penalty(*pen_args))
                return make_fn(e, an, M, st, th[:, 0], q)
            return refit

        anchors = ('last', 'mean')
        for e in ('lvl', 'dif'):
            if e == 'dif' and L < 3:
                continue
            U = {k: enc[e](W[k]) for k in W}
            A_ = {k: torch.stack([T(s(anc_raw[an](W[k]))) for an in anchors], 1) for k in W}
            R = ys[:, None] - A_['tr']
            best = {}
            for M in G['M']:
                if L * M > G['max_delay_units'] and M != min(G['M']):
                    continue
                P = {k: design(U[k], M) for k in U}
                for l0 in G['l0']:
                    for ls in G['ls']:
                        pen_args = (L, M, l0, ls)
                        th, loo = fit_loo(P['tr'], R, penalty(*pen_args))
                        pv, pt = (A_['va'] + P['va'] @ th).cpu().numpy(), (A_['te'] + P['te'] @ th).cpu().numpy()
                        for j, an in enumerate(anchors):
                            add(f"{an}|{e}|delay", {'anchor': an, 'enc': e, 'M': M, 'l0': l0, 'ls': ls},
                                loo[:, j].cpu().numpy() / sc, (pv[:, j] + 1) / sc + a, (pt[:, j] + 1) / sc + a,
                                make_fn(e, an, M, None, th[:, j].clone()), make_refit(e, an, M, None, pen_args, j))
            for an in anchors:
                best[an] = min([c for c in cands if c['fam'] == f"{an}|{e}|delay"], key=lambda c: c['pool'])['cfg']
            states = {}
            dmax = L if e == 'lvl' else L - 1
            for d in sorted({min(dmax, v) for v in G['d']}):
                for C in G['C']:
                    if C * (d + 1) > len(otr):
                        continue
                    Z = U['tr'][:, :d] if e == 'lvl' else U['tr'][:, 1:d + 1]
                    cen, s2 = kmeans(Z, C)
                    states[(d, C)] = (cen, s2, d, 0 if e == 'lvl' else 1)
            for j, an in enumerate(anchors):
                M, l0, ls = best[an]['M'], best[an]['l0'], best[an]['ls']
                for (d, C), st in states.items():
                    P = {k: design(U[k], M, st) for k in U}
                    for lst in G['lst']:
                        pen_args = (L, M, l0, ls, C * (d + 1), max(lst, l0))
                        th, loo = fit_loo(P['tr'], R[:, j:j + 1], penalty(*pen_args))
                        pv = (A_['va'][:, j:j + 1] + P['va'] @ th).cpu().numpy()[:, 0]
                        pt = (A_['te'][:, j:j + 1] + P['te'] @ th).cpu().numpy()[:, 0]
                        add(f"{an}|{e}|delay+state", {'anchor': an, 'enc': e, 'M': M, 'l0': l0, 'ls': ls, 'd': d, 'C': C,
                                                        'lst': lst}, loo[:, 0].cpu().numpy() / sc, (pv + 1) / sc + a,
                            (pt + 1) / sc + a, make_fn(e, an, M, st, th[:, 0].clone()), make_refit(e, an, M, st, pen_args, j))
        return cands, y, ova, ote, s

    def iterate(fn, x, L, origins, h):
        Wr = windows(x, L, origins)
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    def ops_of(c, L, binary, q=8):
        # exact operation count of one forecast by candidate c: MACs (multiply-accumulate) and ACs (accumulate only)
        cfg, fam = c['cfg'], c['fam']
        if fam in ('persistence',):
            return 0.0, 0.0
        if fam == 'window-mean':
            return 0.0, float(L)
        if fam == 'ridge':
            return float(L), 0.0
        mac = 2.0 * L + L                                   # encode (scale + locate the 2 firing fields) + linear term
        ac = float(L) if cfg['anchor'] == 'mean' else 0.0   # window mean
        if cfg['enc'] == 'dif':
            ac += L
        if binary:
            ac += q * L
        else:
            mac += 2.0 * L                                  # 2 graded spikes per delay
        if 'C' in cfg:
            d, C, k = float(cfg['d']), float(cfg['C']), float(G['k_active'])
            mac += C * d + k                                # distances + k exponentials
            if binary:
                ac += q * (d + 1)
            else:
                mac += k * (d + 1)
        return mac, ac

    def bench(fn, Wr):
        reps_needed = max(1, int(np.ceil(T2_BENCH_N / len(Wr))))
        Wb = np.tile(Wr, (reps_needed, 1))
        seg = EnergySegment(METER); reps, t_end = 0, time.perf_counter() + T2_BENCH_S
        seg.resume()
        while reps < 1 or time.perf_counter() < t_end:
            fn(Wb); reps += 1
        seg.pause(); st = seg.state(); n = reps * len(Wb)
        return st['j'] / n, st['s'] / n

    preds, energy, choices, noise_rows, ops_rows = [], [], [], [], []
    # notebook noise draws (h = 1), as in cell 8
    def noisy_windows(Wtr, ytr, Wte, L, fi, li, dd, nl):
        mmn = _MMS((-1, 1)).fit(np.column_stack([Wtr, ytr]))
        Xtr_s = Wtr * mmn.scale_[:-1] + mmn.min_[:-1]; Xte_s = Wte * mmn.scale_[:-1] + mmn.min_[:-1]
        Xn = Xte_s + np.random.default_rng([L, fi, li, dd]).normal(0.0, nl * Xtr_s.std(), size=Xte_s.shape)
        return (Xn - mmn.min_[:-1]) / mmn.scale_[:-1]

    def one_dataset(D):
        x = D['values']
        for L in T2_LAGS:
            if not lag_ok(None, D, L):
                continue
            hs = [h for h in T2_HORIZONS if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 >= T2_MIN_ORIGINS]
            stem = f"{D['slug']}/{T2_FOLDER}/t2{sm}_lag{L}"
            rels = {k: f"{stem}_{k}.csv" for k in ('preds', 'energy', 'choices', 'noise', 'ops')}
            if not T2_FRESH and all(store.fetch(rels[k]) for k in ('preds', 'energy', 'choices', 'ops')):
                for k, bucket in (('preds', preds), ('energy', energy), ('choices', choices), ('noise', noise_rows), ('ops', ops_rows)):
                    p = store.fetch(rels[k])
                    if p and os.path.getsize(p) > 2:
                        bucket.append(pd.read_csv(p))
                continue
            check_time_budget()
            t0 = time.time()
            P, E, Ch, Nz, Op = [], {}, [], [], []
            for fi, f in enumerate(D['folds']):
                pre = x[:f['te_start']]
                seg = EnergySegment(METER); seg.resume()
                c1, y1, ova1, ote1, _ = run_fold(x, L, 1, f)
                b1 = min(c1, key=lambda c: c['pool'])
                F1 = b1['fn']
                F1b = {q: (b1['refit'](q) if b1['refit'] else F1) for q in T2_SPIKE_Q_GRID}
                fold_fits = {1: (c1, y1, ova1, ote1)}
                for h in hs:
                    if h > 1:
                        fold_fits[h] = run_fold(x, L, h, f)[:4]
                seg.pause()
                for h in hs:
                    ch, y, ova, ote = fold_fits[h]
                    bd = min(ch, key=lambda c: c['pool'])
                    d_va, d_te = bd['va_pred'], bd['te_pred']
                    it = False
                    if h > 1:
                        i_va = iterate(F1, x, L, ova, h)
                        it = rmse(y['va'] - i_va) < (1 - T2_ITER_MARGIN) * rmse(y['va'] - d_va)
                    # binary model: one candidate per spike count, judged on the validation block only
                    if it:
                        ra = rmse(y['va'] - i_va)
                        bins = {q: (lambda Wo, q=q: iterate(F1b[q], x, L, Wo, h)) for q in T2_SPIKE_Q_GRID}
                    else:
                        ra = rmse(y['va'] - d_va)
                        fbq = {q: (bd['refit'](q) if bd['refit'] else bd['fn']) for q in T2_SPIKE_Q_GRID}
                        bins = {q: (lambda Wo, q=q: fbq[q](windows(x, L, Wo))) for q in T2_SPIKE_Q_GRID}
                    rb = {q: rmse(y['va'] - bins[q](ova)) for q in T2_SPIKE_Q_GRID}
                    ok_q = [q for q in T2_SPIKE_Q_GRID if rb[q] <= (1 + T2_SPIKE_TOL) * ra]
                    qs = min(ok_q) if ok_q else min(rb, key=rb.get)
                    pred_b = bins[qs](ote)
                    if it:
                        pred = iterate(F1, x, L, ote, h)
                        mac, ac = ops_of(b1, L, False); macb, acb = ops_of(b1, L, True, qs)
                        mac, ac, macb, acb = h * mac, h * ac, h * macb, h * acb
                        fn_bench = lambda Wr, F1=F1, h=h: iterate_windows(F1, Wr, h)
                        used = b1
                    else:
                        pred = d_te
                        mac, ac = ops_of(bd, L, False); macb, acb = ops_of(bd, L, True, qs)
                        fn_bench = bd['fn']
                        used = bd
                    tau_h = round(float(np.median(pre[h:] - pre[:-h]) if h > 1 else np.median(np.diff(pre))), 6)
                    yt, prev = x[ote + h - 1], x[ote - 1]
                    for name, pr in ((T2_NAME, pred), (T2_NAME_BIN, pred_b)):
                        P.append(pd.DataFrame({'dataset': D['name'], 'model': name, 'lag': L, 'h': h, 'test_fold': f['label'],
                                               'origin': ote, 'target_idx': ote + h - 1, 'y': yt, 'prev': prev, 'tau': tau_h,
                                               'label': ((yt - prev) > tau_h).astype(int), 'ens_pred': pr}))
                    Ch.append({'dataset': D['name'], 'model': T2_NAME, 'lag': L, 'h': h, 'fold': f['label'],
                               'family': used['fam'], 'iterated': bool(it), 'criterion': used['pool'], 'spike_q': qs,
                               'val_rmse_analog': ra, **{f'val_rmse_q{q}': v for q, v in rb.items()},
                               **{k: v for k, v in used['cfg'].items()}})
                    for name, (m_, a_) in ((T2_NAME, (mac, ac)), (T2_NAME_BIN, (macb, acb))):
                        Op.append({'dataset': D['name'], 'model': name, 'lag': L, 'h': h, 'fold': f['label'], 'MAC': m_, 'AC': a_,
                                   'pJ': FINAL_PJ_MAC_V2 * m_ + FINAL_PJ_AC_V2 * a_})
                    E.setdefault(h, {'j': 0.0, 's': 0.0, 'peak_w': 0.0})
                    if fi == len(D['folds']) - 1:
                        ej, es = bench(fn_bench, windows(x, L, ote))
                        E[h]['infer_j_per_forecast'], E[h]['infer_s_per_forecast'] = ej, es
                    if h == 1 and N_NOISE_DRAWS:
                        otr = np.arange(L, f['va_start'])
                        Wtr, ytr, Wte = windows(x, L, otr), x[otr], windows(x, L, ote)
                        for li, nl in enumerate(NOISE_LEVELS):
                            for dd in range(N_NOISE_DRAWS):
                                Wn = noisy_windows(Wtr, ytr, Wte, L, fi, li, dd, nl)
                                Nz.append({'dataset': D['name'], 'lag': L, 'fold': f['label'], 'model': T2_NAME, 'noise': nl,
                                           'draw': dd, 'rmse': rmse(yt - F1(Wn))})
                st = seg.state()
                del fold_fits, c1
                for h in hs:                                  # fit energy of the fold, shared equally by its horizons
                    E[h]['j'] += st['j'] / len(hs); E[h]['s'] += st['s'] / len(hs); E[h]['peak_w'] = max(E[h]['peak_w'], st['peak_w'])
            Pdf = pd.concat(P, ignore_index=True)
            Edf = pd.DataFrame([{'dataset': D['name'], 'model': T2_NAME, 'lag': L, 'h': h, 'energy_j': v['j'], 'seconds': v['s'],
                                 'mean_w': v['j'] / v['s'] if v['s'] else np.nan, 'peak_w': v['peak_w'],
                                 'infer_j_per_forecast': v.get('infer_j_per_forecast', np.nan),
                                 'infer_s_per_forecast': v.get('infer_s_per_forecast', np.nan), 'method': METER.method}
                                for h, v in E.items()])
            Chdf, Nzdf, Opdf = pd.DataFrame(Ch), pd.DataFrame(Nz), pd.DataFrame(Op)
            for k, df in (('preds', Pdf), ('energy', Edf), ('choices', Chdf), ('noise', Nzdf), ('ops', Opdf)):
                p = os.path.join(LOCAL_ROOT, rels[k]); os.makedirs(os.path.dirname(p), exist_ok=True)
                df.to_csv(p, index=False); store.add(p)
            preds.append(Pdf); energy.append(Edf); choices.append(Chdf); noise_rows.append(Nzdf); ops_rows.append(Opdf)
            g = Pdf[Pdf['model'] == T2_NAME].groupby('h').apply(lambda d: rmse(d['y'] - d['ens_pred']), include_groups=False)
            it_share = Chdf.groupby('h')['iterated'].mean()
            gb = Pdf[Pdf['model'] == T2_NAME_BIN].groupby('h').apply(lambda d: rmse(d['y'] - d['ens_pred']), include_groups=False)
            print(f"{'':<16}          binary RMSE " + ' | '.join(f"h{h} {gb[h]:.4g}" for h in gb.index)
                  + f" | spikes per delay chosen: {Chdf['spike_q'].value_counts().sort_index().to_dict()}")
            print(f"{D['name']:<16} lag {L:2d}: TRACE-SNP v2 RMSE " + ' | '.join(f"h{h} {g[h]:.4g}" + (f" (iterated {it_share[h]:.0%})" if h > 1 else '')
                                                                            for h in g.index) + f" | {time.time() - t0:.0f} s")
            store.push(f"trace v2 {D['slug']} lag {L}")
        store.push(f"trace v2 {D['slug']}", force=True)

    def iterate_windows(fn, Wr, h):
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    try:
        for D in datasets:
            one_dataset(D)
            trim_ram()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    except TimeBudgetReached:
        store.push('trace v2: time budget', force=True)
        print('*** Time budget reached — rerun to continue; finished (dataset, lag) are kept. ***')
    cat = lambda lst: pd.concat([d for d in lst if len(d)], ignore_index=True) if any(len(d) for d in lst) else pd.DataFrame()
    return {'preds': cat(preds), 'energy': cat(energy), 'choices': cat(choices), 'noise': cat(noise_rows), 'ops': cat(ops_rows)}


FINAL_PJ_MAC_V2, FINAL_PJ_AC_V2 = 4.6, 0.9      # Horowitz 2014, 45 nm: 32-bit MAC, 32-bit add


T2_RESULTS = run_trace_v2(DATASETS)

## Part C — Analysis, ablations and TRACE-SNP-AF (cells 11–19)
Cells 11, 12, 14, 15, 16 and 19 only read results; cells 13, 17 and 18 fit closed-form models (minutes to about an hour on a GPU).

In [ ]:
# ============================================================
# 11. ANALYSIS OVER ALL DATASETS — every model on identical test targets (TRACE-SNP v2 as the model under test)
#     As cell 11, with TRACE-SNP v2 (cell 10) as the model under test and TRACE-SNP (cell 8) kept as a rival.
#     Network inference is also reported per ENSEMBLE forecast (× 60 seeds): every network result is a 60-seed mean,
#     while the notebook's benchmark measures one member model per forecast.
#     Sources: the networks' one-step preds_*.csv (cell 7), their longer horizons (cells 9, 3), the closed-form models
#     (cells 8, 3), the networks' energy_*.json and walkforward_folds_*.csv. A "problem" = (dataset, lag, horizon).
#       1. Leaderboard: pooled test RMSE and AUC over the 5 walk-forward blocks, per problem; relative RMSE vs persistence.
#       2. TRACE-SNP vs every other model per problem: Diebold–Mariano (HLN, h-step), DeLong on the direction score;
#          Holm within the dataset and over everything; win / tie / loss at α = 0.05 after Holm.
#       3. Friedman test over problems + Nemenyi critical-difference diagram (Demšar 2006).
#       4. Ablations (relative RMSE to the full TRACE-SNP), horizon profile, noise robustness.
#       5. Energy: measured GPU energy (or seconds when no meter) to train and per forecast; operation counts per
#          forecast and a 45 nm neuromorphic estimate (Horowitz 2014: 4.6 pJ / 32-bit MAC, 0.9 pJ / AC).
#       6. Interpretability: TRACE-SNP's learned delay functions f_k(x_{t-k}).
#       7. LaTeX tables. Everything goes to <LOCAL_ROOT>/comparison/final_*.
#     Horizons > 1: closed-form models are direct (one readout per h), networks iterate their one-step model (cells 9, 3).
# ============================================================
FINAL_ALPHA        = 0.05
FINAL_MAIN_MODEL   = 'TRACE-SNP v2'
FINAL_EXCLUDE_CD   = ([m for m in CF_MODELS if m.startswith('TRACE') and m != 'TRACE-SNP']
                      + ['TRACE-SNP v2 (binary spikes)'])        # variants: not ranked as rivals (TRACE-SNP v1 stays a rival)
FINAL_PJ_MAC, FINAL_PJ_AC = 4.6, 0.9


def _final_analysis():
    from scipy import stats as _st
    from torch.utils.flop_counter import FlopCounterMode
    sm = '_smoke' if SMOKE else ''
    out_dir = os.path.join(LOCAL_ROOT, 'comparison'); os.makedirs(out_dir, exist_ok=True)
    all_D = list(DATASETS)                           # every dataset
    tag_ = 'all11' if len(all_D) == 11 else '_'.join(d['slug'] for d in all_D)
    cat_ = lambda *d: pd.concat([x for x in d if len(x)], ignore_index=True) if any(len(x) for x in d) else pd.DataFrame()
    CF_RESULTS = {k: cat_(CF_RESULTS_MAIN[k], CF_RESULTS_EXTRA[k]) for k in CF_RESULTS_MAIN}
    for k in ('preds', 'energy', 'noise'):                      # TRACE-SNP v2 joins the closed-form results
        CF_RESULTS[k] = cat_(CF_RESULTS[k], T2_RESULTS[k])
    MH_PREDS = cat_(MH_PREDS_MAIN, MH_PREDS_EXTRA)
    names = [D['name'] for D in all_D]
    saved = []
    ABL, PJT, WTL = None, None, None

    def save_df(df, stem, index=False):
        p = os.path.join(out_dir, f"final_{tag_}_{stem}{sm}.csv"); df.to_csv(p, index=index); store.add(p); saved.append(p)

    def save_fig(fig, stem):
        p = os.path.join(out_dir, f"final_{tag_}_{stem}{sm}.png"); fig.savefig(p, dpi=150, bbox_inches='tight'); store.add(p)
        saved.append(p); show(fig)

    # ── 0. one long table of predictions ──
    cols = ['dataset', 'model', 'lag', 'h', 'test_fold', 'target_idx', 'y', 'prev', 'tau', 'label', 'ens_pred']
    parts = []
    for D in all_D:
        for spec in MODEL_SPECS:
            p = store.fetch(store.rel(os.path.join(work_dir_for(spec, D), f"preds_{run_tag_for(spec, D)}.csv")))
            if p:
                df = pd.read_csv(p); df['h'] = 1; df['dataset'] = D['name']; df['model'] = spec['name']
                parts.append(df[cols])
    if len(MH_PREDS):
        parts.append(MH_PREDS[cols])
    if len(CF_RESULTS['preds']):
        parts.append(CF_RESULTS['preds'][cols])
    A = pd.concat(parts, ignore_index=True)
    A['family'] = np.where(A['model'].isin([s['name'] for s in MODEL_SPECS]), 'network', 'closed-form')
    net_names = [s['name'] for s in MODEL_SPECS]
    # diverged forecasts (NaN / inf, e.g. a network whose recursive forecast blew up): reported, then replaced by the
    # persistence forecast so every model keeps a finite score (this favours the failing model, never TRACE-SNP)
    bad = ~np.isfinite(A['ens_pred'].to_numpy(float))
    A['diverged'] = bad
    if bad.any():
        DV = (A[bad].groupby(['model', 'dataset', 'h', 'lag']).size().rename('diverged_forecasts')
                .to_frame().join(A.groupby(['model', 'dataset', 'h', 'lag']).size().rename('of')).reset_index())
        print(f"WARNING: {int(bad.sum()):,} non-finite forecasts — replaced by persistence (x_(t-1)) for scoring:")
        print(DV.to_string(index=False))
        save_df(DV, 'diverged_forecasts')
        A.loc[bad, 'ens_pred'] = A.loc[bad, 'prev']
    print(f"{len(A):,} test predictions | {A['dataset'].nunique()} datasets | {A['model'].nunique()} models | "
          f"{A.groupby(['dataset', 'lag', 'h']).ngroups} (dataset, lag, horizon) problems")

    # ── 1. leaderboard ──
    def auc_of(d):
        s = d['ens_pred'] - d['prev'] - d['tau']
        return safe_auc(d['label'], s)
    LB = (A.groupby(['dataset', 'h', 'lag', 'model'])
            .apply(lambda d: pd.Series({'rmse': float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2))), 'auc': auc_of(d),
                                        'n': len(d)}), include_groups=False).reset_index())
    pers = LB[LB['model'] == 'Persistence'].set_index(['dataset', 'h', 'lag'])['rmse']
    LB['rel_rmse'] = LB['rmse'] / LB.set_index(['dataset', 'h', 'lag']).index.map(pers).to_numpy()
    LB['rank'] = LB[~LB['model'].isin(FINAL_EXCLUDE_CD)].groupby(['dataset', 'h', 'lag'])['rmse'].rank(method='average')
    save_df(LB, 'leaderboard')
    for (dname, h), g in LB[~LB['model'].isin(FINAL_EXCLUDE_CD)].groupby(['dataset', 'h'], sort=False):
        piv = g.pivot(index='model', columns='lag', values='rmse')
        piv = piv.loc[piv.min(axis=1).sort_values().index]
        print(f"\n{dname} | horizon {h} | pooled test RMSE by lag (best per lag marked *)")
        txt = piv.map(lambda v: f"{v:.4g}" if np.isfinite(v) else '—')
        for L in piv.columns:
            j = piv[L].idxmin()
            if isinstance(j, str):
                txt.loc[j, L] = txt.loc[j, L] + '*'
        print(txt.to_string())

    # ── 2. pairwise tests: TRACE-SNP vs each other model, matched targets ──
    rows = []
    for (dname, h, L), g in A.groupby(['dataset', 'h', 'lag']):
        main = g[g['model'] == FINAL_MAIN_MODEL]
        if main.empty:
            continue
        main = main.set_index('target_idx').sort_index()
        for m, gm in g[g['model'] != FINAL_MAIN_MODEL].groupby('model'):
            gm = gm.set_index('target_idx').sort_index()
            idx = main.index.intersection(gm.index)
            if len(idx) < 8:
                continue
            a, b = main.loc[idx], gm.loc[idx]
            e1, e2 = (a['y'] - a['ens_pred']).to_numpy(), (b['y'] - b['ens_pred']).to_numpy()
            dm, p_dm = dm_test(e1, e2, h=int(h))
            s1, s2 = (a['ens_pred'] - a['prev'] - a['tau']).to_numpy(), (b['ens_pred'] - b['prev'] - b['tau']).to_numpy()
            auc1, auc2, z, p_auc = delong_test(a['label'].to_numpy(), s1, s2)
            rows.append({'dataset': dname, 'h': h, 'lag': L, 'other': m, 'family': b['family'].iloc[0], 'n': len(idx),
                         'rmse_trace': float(np.sqrt(np.mean(e1 ** 2))), 'rmse_other': float(np.sqrt(np.mean(e2 ** 2))),
                         'dm': dm, 'dm_p': p_dm, 'auc_trace': auc1, 'auc_other': auc2, 'delong_z': z, 'delong_p': p_auc})
    PW = pd.DataFrame(rows)
    if len(PW):
        PW['rmse_ratio'] = PW['rmse_trace'] / PW['rmse_other']
        PW['dm_p_holm_dataset'] = PW.groupby('dataset')['dm_p'].transform(lambda p: holm(p.to_numpy()))
        PW['dm_p_holm_global'] = holm(PW['dm_p'].to_numpy())
        PW['delong_p_holm_global'] = holm(PW['delong_p'].to_numpy())
        PW['outcome'] = np.select([(PW['dm_p_holm_dataset'] < FINAL_ALPHA) & (PW['dm'] < 0),
                                   (PW['dm_p_holm_dataset'] < FINAL_ALPHA) & (PW['dm'] > 0)], ['win', 'loss'], 'tie')
        save_df(PW, 'pairwise_tests')
        WTL = (PW[~PW['other'].isin(FINAL_EXCLUDE_CD)].groupby(['other', 'outcome']).size().unstack(fill_value=0)
               .reindex(columns=['win', 'tie', 'loss'], fill_value=0))
        WTL['median RMSE ratio'] = PW.groupby('other')['rmse_ratio'].median()
        WTL['geo-mean RMSE ratio'] = PW.groupby('other')['rmse_ratio'].apply(lambda r: float(np.exp(np.mean(np.log(r)))))
        WTL = WTL.sort_values('geo-mean RMSE ratio')
        save_df(WTL.reset_index(), 'win_tie_loss')
        print(f"\n{FINAL_MAIN_MODEL} vs each model over all (dataset, lag, horizon): DM test, Holm within dataset, α = {FINAL_ALPHA}"
              f"\n(ratio < 1 → {FINAL_MAIN_MODEL} has lower RMSE)")
        print(WTL.to_string(float_format=lambda v: f"{v:.3f}"))
        by_h = (PW[~PW['other'].isin(FINAL_EXCLUDE_CD)].groupby(['h', 'outcome']).size().unstack(fill_value=0)
                .reindex(columns=['win', 'tie', 'loss'], fill_value=0))
        print("\nby horizon:\n" + by_h.to_string())

    # ── 3. Friedman + Nemenyi CD diagram ──
    def friedman_cd(sub, title, stem, every_model=False):
        piv = sub.pivot_table(index=['dataset', 'h', 'lag'], columns='model', values='rmse')
        if every_model:           # every model, on the problems where all of them ran
            piv = piv.dropna()
        else:                     # the models present on at least half the problems, then the problems where all of those ran
            piv = piv.loc[:, piv.notna().mean() >= 0.5].dropna()
        if piv.shape[0] < 3 or piv.shape[1] < 3:
            print(f"{title}: not enough complete problems for a Friedman test"); return None
        R = piv.rank(axis=1, method='average')
        k, N = piv.shape[1], piv.shape[0]
        chi2, p = _st.friedmanchisquare(*[piv[c].to_numpy() for c in piv.columns])
        q = _st.studentized_range.ppf(1 - FINAL_ALPHA, k, np.inf) / np.sqrt(2)
        cd = q * np.sqrt(k * (k + 1) / (6.0 * N))
        mr = R.mean().sort_values()
        print(f"\n{title}: Friedman χ² = {chi2:.1f}, p = {p:.2e} over {N} problems × {k} models | Nemenyi CD = {cd:.2f}")
        print(mr.to_string(float_format=lambda v: f"{v:.2f}"))
        # diagram
        fig, ax = plt.subplots(figsize=(11, 1.6 + 0.22 * k))
        lo, hi = 1, k
        half = (k + 1) // 2
        ax.set_xlim(lo - 0.3, hi + 0.3); ax.set_ylim(-(half * 0.45 + 0.4), 2.3); ax.axis('off')
        ax.hlines(1, lo, hi, color='k', lw=1)
        for r in range(lo, hi + 1):
            ax.vlines(r, 1, 1.18, color='k', lw=1); ax.text(r, 1.35, str(r), ha='center', fontsize=8)
        ax.hlines(1.9, lo, lo + cd, color='#c0392b', lw=2); ax.text(lo + cd / 2, 2.05, f"CD = {cd:.2f}", ha='center', fontsize=8, color='#c0392b')
        for i, (m, r) in enumerate(mr.items()):
            left = i < half
            yy = -(i if left else (k - 1 - i)) * 0.45 - 0.35
            xe = lo - 0.25 if left else hi + 0.25
            col = '#eb6834' if m == FINAL_MAIN_MODEL else ('#2a78d6' if m in net_names else '#555')
            ax.plot([r, r, xe], [1, yy, yy], color=col, lw=1.2)
            ax.text(xe + (-0.03 if left else 0.03), yy, f"{m} ({r:.2f})", ha='right' if left else 'left', va='center', fontsize=8, color=col,
                    fontweight='bold' if m == FINAL_MAIN_MODEL else 'normal')
        # cliques: maximal runs of models whose mean ranks differ by < CD
        vals = mr.to_numpy(); cl = []
        for i in range(k):
            j = i
            while j + 1 < k and vals[j + 1] - vals[i] < cd:
                j += 1
            if j > i and not any(a <= i and j <= b for a, b in cl):
                cl.append((i, j))
        for n_, (i, j) in enumerate(cl):
            yy = 0.82 - 0.16 * (n_ % 3)
            ax.hlines(yy, vals[i] - 0.03, vals[j] + 0.03, color='k', lw=2)
        ax.set_title(f"{title} — mean rank by RMSE (lower is better); bars join models not significantly different (Nemenyi, α={FINAL_ALPHA})",
                     fontsize=9, loc='left')
        save_fig(fig, stem)
        return {'chi2': chi2, 'p': p, 'cd': cd, 'N': N, 'k': k, 'mean_rank': mr}

    core = LB[~LB['model'].isin(FINAL_EXCLUDE_CD)]
    main_names = [D['name'] for D in all_D]
    FR = {}
    FR['main, all horizons'] = friedman_cd(core[core['dataset'].isin(main_names)], f"{tag_}: every lag and horizon", 'cd_main')
    FR['main, every model'] = friedman_cd(core[core['dataset'].isin(main_names)],
                                          f"{tag_}: all 9 networks + closed-form models (problems where every model ran)",
                                          'cd_main_every_model', every_model=True)
    FR['main, h=1'] = friedman_cd(core[core['dataset'].isin(main_names) & (core['h'] == 1)], f"{tag_}: one-step", 'cd_main_h1')
    if False:
        FR['all datasets'] = friedman_cd(core, 'All datasets (models present on every problem)', 'cd_all')
    save_df(pd.DataFrame([{'setting': k, **{kk: v[kk] for kk in ('chi2', 'p', 'cd', 'N', 'k')}} for k, v in FR.items() if v]), 'friedman')

    # ── 4a. ablations ──
    abl_models = [m for m in CF_MODELS if m.startswith('TRACE')] + ['TRACE-SNP v2', 'TRACE-SNP v2 (binary spikes)']
    piv = LB[LB['model'].isin(abl_models)].pivot_table(index=['dataset', 'h', 'lag'], columns='model', values='rmse')
    if 'TRACE-SNP' in piv:
        rel = piv.div(piv['TRACE-SNP'], axis=0)             # every TRACE variant relative to TRACE-SNP (v1)
        ABL = pd.DataFrame({'geo-mean RMSE / TRACE-SNP': np.exp(np.log(rel).mean()),
                            'worst ratio': rel.max(), 'share of problems worse by > 1 %': (rel > 1.01).mean() * 100})
        ABL = ABL.drop(index='TRACE-SNP').sort_values('geo-mean RMSE / TRACE-SNP')
        ABLd = np.exp(np.log(rel).groupby(level='dataset').mean()).drop(columns='TRACE-SNP')
        print("\nTRACE variants relative to TRACE-SNP (v1) — ablations (> 1 → removing that part hurts) and v2 (< 1 → v2 is better)")
        print(ABL.to_string(float_format=lambda v: f"{v:.3f}"))
        print(ABLd.T.to_string(float_format=lambda v: f"{v:.3f}"))
        save_df(ABL.reset_index().rename(columns={'model': 'variant'}), 'ablations')
        save_df(ABLd.reset_index(), 'ablations_by_dataset')
        ch = T2_RESULTS['choices']
        if len(ch):
            fam = ch.groupby(['dataset', 'family']).size().unstack(fill_value=0)
            print("\nWhich TRACE-SNP v2 candidate the LOO+validation criterion picked (count over lags × horizons × folds):")
            print(fam.to_string())
            save_df(fam.reset_index(), 'selected_family')
            itr = ch[ch['h'] > 1].groupby(['dataset', 'h'])['iterated'].mean().unstack('h')
            print("\nShare of folds where TRACE-SNP v2 used the iterated one-step model (validation margin rule):")
            print(itr.to_string(float_format=lambda v: f"{v:.0%}"))
            save_df(itr.reset_index(), 'iterated_share')

    # ── 4b. horizon profile: relative RMSE (geo-mean over lags) per dataset × horizon ──
    HP = (core.groupby(['dataset', 'h', 'model'])['rel_rmse'].apply(lambda r: float(np.exp(np.mean(np.log(r)))))
             .unstack('model'))
    save_df(HP.reset_index(), 'horizon_profile')
    hs = sorted(core['h'].unique())
    if len(hs) > 1:
        ds = list(dict.fromkeys(core['dataset']))
        nc = min(4, len(ds)); nr = int(np.ceil(len(ds) / nc))
        fig, axes = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.0 * nr), squeeze=False)
        for ax, dname in zip(axes.ravel(), ds):
            t = HP.loc[dname] if dname in HP.index.get_level_values(0) else None
            if t is None:
                continue
            for m in t.columns:
                if t[m].notna().sum() < 2:
                    continue
                st_ = dict(color='#eb6834', lw=2.4, zorder=5) if m == FINAL_MAIN_MODEL else \
                      dict(color='#2a78d6', lw=1, alpha=0.7) if m in net_names else dict(color='#777', lw=1, ls='--')
                ax.plot(t.index, t[m], marker='o', ms=3, label=m, **st_)
            ax.axhline(1, color='k', lw=0.6); ax.set_yscale('log'); ax.set_xticks(hs)
            ax.set_title(dname, fontsize=9, loc='left'); ax.grid(True, alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
        for ax in axes.ravel()[len(ds):]:
            ax.axis('off')
        axes[0, 0].set_ylabel('RMSE / persistence (geo-mean over lags)')
        hh, ll = axes[0, 0].get_legend_handles_labels()
        fig.legend(hh, ll, loc='lower center', ncol=6, fontsize=7, frameon=False, bbox_to_anchor=(0.5, -0.06))
        fig.suptitle('Error growth with the forecast horizon (orange: TRACE-SNP; blue: networks; grey dashed: closed-form baselines)', fontsize=10)
        fig.tight_layout(); save_fig(fig, 'horizon_profile')

    # ── 4c. noise robustness (h = 1): mean over folds of the per-fold RMSE (networks: mean over seeds, as in cell 7) ──
    nz = []
    if len(CF_RESULTS['noise']):
        cfn = CF_RESULTS['noise'].groupby(['dataset', 'lag', 'model', 'noise', 'fold'])['rmse'].mean().reset_index()
        nz.append(cfn.groupby(['dataset', 'lag', 'model', 'noise'])['rmse'].mean().reset_index())
    for D in all_D:
        for spec in MODEL_SPECS:
            p = store.fetch(store.rel(os.path.join(work_dir_for(spec, D), f"walkforward_folds_{run_tag_for(spec, D)}.csv")))
            if not p:
                continue
            wf = pd.read_csv(p)
            for nl in NOISE_LEVELS:
                c = f"noise{nl}_rmse"
                if c in wf:
                    g = wf.groupby('lag')[[c, 'seedmean_rmse']].mean().reset_index().rename(columns={c: 'rmse', 'seedmean_rmse': 'clean'})
                    g['dataset'], g['model'], g['noise'] = D['name'], spec['name'], nl
                    nz.append(g)
    NZ = pd.DataFrame()
    if nz:
        NZ = pd.concat(nz, ignore_index=True)
        # clean reference on the same footing: mean over folds of the per-fold RMSE (networks: of the seed-mean RMSE)
        cf1 = A[(A['h'] == 1) & (A['family'] == 'closed-form')]
        clean = (cf1.groupby(['dataset', 'lag', 'model', 'test_fold']).apply(lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2))),
                                                                             include_groups=False)
                    .groupby(['dataset', 'lag', 'model']).mean().rename('clean_cf').reset_index())
        NZ = NZ.merge(clean, on=['dataset', 'lag', 'model'], how='left')
        if 'clean' not in NZ:
            NZ['clean'] = np.nan
        NZ['clean'] = NZ['clean'].fillna(NZ['clean_cf']); NZ = NZ.drop(columns='clean_cf')
        NZ['degradation_%'] = (NZ['rmse'] / NZ['clean'] - 1) * 100
        p1 = LB[(LB['h'] == 1) & (LB['model'] == 'Persistence')].set_index(['dataset', 'lag'])['rmse']
        NZ['rmse_rel_persistence'] = NZ['rmse'] / NZ.set_index(['dataset', 'lag']).index.map(p1).to_numpy()
        save_df(NZ, 'noise_robustness')
        T_ = NZ.groupby(['model', 'noise'])['rmse_rel_persistence'].median().unstack('noise')
        R_ = NZ.groupby(['model', 'noise'])['degradation_%'].median().unstack('noise')
        print("\nInput-noise robustness (h = 1), median over (dataset, lag): RMSE under noise / clean persistence RMSE, and the % rise "
              "over the model's own clean RMSE\n(networks: mean over seeds of the per-seed RMSE, as stored by cell 7; closed-form: the "
              "single fitted model under the same noise draws)")
        print(pd.concat({'RMSE / persistence': T_, 'rise %': R_}, axis=1)
                .sort_values(('RMSE / persistence', NOISE_LEVELS[-1])).to_string(float_format=lambda v: f"{v:.4g}"))

    # ── 5. energy ──
    er = []
    for D in all_D:
        for spec in MODEL_SPECS:
            for L in spec['lags']:
                p = store.fetch(store.rel(energy_path_for(spec, D, L)))
                if p:
                    e = energy_total(load_json(p))
                    er.append({'dataset': D['name'], 'model': spec['name'], 'lag': L, 'train_j': e['energy_j'], 'train_s': e['seconds'],
                               'mean_w': e['mean_w'], 'infer_j': e['infer_j_per_forecast'], 'infer_s': e['infer_s_per_forecast'],
                               'infer_j_ensemble': e['infer_j_per_forecast'] * spec['n_runs'],
                               'infer_s_ensemble': e['infer_s_per_forecast'] * spec['n_runs'],
                               'gpu': e['gpu'], 'method': e['method']})
    ce = CF_RESULTS['energy']
    if len(ce):
        c1 = ce[ce['h'] == 1]
        for _, r in c1.iterrows():
            er.append({'dataset': r['dataset'], 'model': r['model'], 'lag': r['lag'], 'train_j': r['energy_j'], 'train_s': r['seconds'],
                       'mean_w': r['mean_w'], 'infer_j': r['infer_j_per_forecast'], 'infer_s': r['infer_s_per_forecast'],
                       'infer_j_ensemble': r['infer_j_per_forecast'], 'infer_s_ensemble': r['infer_s_per_forecast'],
                       'gpu': METER.name, 'method': r['method']})
    EN = pd.DataFrame(er)
    if len(EN):
        save_df(EN, 'energy_measured')
        for col, lab in (('train_j', 'J to train (all folds; networks: 60 seeds)'), ('train_s', 's to train'),
                         ('infer_j', 'J per forecast (networks: one member model)'), ('infer_s', 's per forecast (one member)'),
                         ('infer_j_ensemble', 'J per forecast as reported (networks: the 60-seed ensemble)'),
                         ('infer_s_ensemble', 's per forecast as reported (60-seed ensemble)')):
            if EN[col].notna().any():
                t = EN.groupby(['model', 'lag'])[col].median().unstack('lag')
                print(f"\nMeasured {lab}: median over datasets")
                print(t.loc[t.median(axis=1).sort_values().index].to_string(float_format=lambda v: f"{v:.3g}"))
        tr = EN[EN['model'] == FINAL_MAIN_MODEL].set_index(['dataset', 'lag'])
        ratios = []
        for m in net_names:
            o = EN[EN['model'] == m].set_index(['dataset', 'lag'])
            idx = tr.index.intersection(o.index)
            for col in ('train_j', 'train_s', 'infer_j', 'infer_s', 'infer_j_ensemble', 'infer_s_ensemble'):
                rr = (o.loc[idx, col] / tr.loc[idx, col]).replace([np.inf, -np.inf], np.nan).dropna()
                if len(rr):
                    ratios.append({'network': m, 'quantity': col, 'median ×': float(rr.median()), 'min ×': float(rr.min()), 'n': len(rr)})
        if ratios:
            RT = pd.DataFrame(ratios).pivot(index='network', columns='quantity', values='median ×')
            print(f"\nHow many times more than {FINAL_MAIN_MODEL} each network uses (median over (dataset, lag)):")
            print(RT.to_string(float_format=lambda v: f"{v:.3g}"))
            save_df(pd.DataFrame(ratios), 'energy_ratios')
        # bar chart: training energy (or time) per lag, log scale
        col = 'train_j' if EN['train_j'].notna().any() else 'train_s'
        t = EN.groupby(['model', 'lag'])[col].median().unstack('lag')
        t = t.loc[t.median(axis=1).sort_values().index]
        fig, ax = plt.subplots(figsize=(13, 4.2))
        lags = list(t.columns); w = 0.8 / len(lags); xs = np.arange(len(t))
        for i, L in enumerate(lags):
            ax.bar(xs + i * w, t[L], w, label=f"lag {L}", color=plt.cm.viridis(i / max(1, len(lags) - 1)))
        ax.set_xticks(xs + 0.4 - w / 2); ax.set_xticklabels(t.index, rotation=30, ha='right', fontsize=8)
        ax.set_yscale('log'); ax.set_ylabel('J' if col == 'train_j' else 's')
        ax.set_title(f"Training {'energy (GPU, measured)' if col == 'train_j' else 'time'} per (model, lag), median over datasets — log scale",
                     fontsize=10, loc='left')
        ax.legend(fontsize=7, ncol=len(lags), frameon=False); ax.grid(True, axis='y', alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
        save_fig(fig, 'energy_train')

    # ── 5b. operations per forecast and a neuromorphic estimate ──
    ops = []
    fr = {}
    for D in all_D:
        for spec in MODEL_SPECS:
            p = store.fetch(store.rel(os.path.join(work_dir_for(spec, D), f"walkforward_folds_{run_tag_for(spec, D)}.csv")))
            if p:
                wf = pd.read_csv(p)
                for L, g in wf.groupby('lag'):
                    fr[(D['name'], spec['name'], L)] = float(g['fire_rate'].mean())
    macs_cache = {}
    for spec in MODEL_SPECS:
        for L in sorted({L for s in MODEL_SPECS for L in s['lags']}):
            if L not in spec['lags']:
                continue
            m = spec['build']().to('cpu').eval()
            x = torch.zeros(1, L, 1)
            with FlopCounterMode(display=False) as fc, torch.no_grad():
                m(x)
            macs_cache[(spec['name'], L)] = fc.get_total_flops() / 2
    ch = CF_RESULTS['choices']
    for D in all_D:
        for L in CF_LAGS:
            for spec in MODEL_SPECS:
                if (spec['name'], L) not in macs_cache:
                    continue
                mac = macs_cache[(spec['name'], L)]
                rate = fr.get((D['name'], spec['name'], L), np.nan)
                if spec['spikes'] and np.isfinite(rate):
                    pj = FINAL_PJ_AC * rate * mac                    # lower bound: every operation an AC, scaled by the firing rate
                    acs, macs_ = rate * mac, 0.0
                else:
                    pj = FINAL_PJ_MAC * mac; acs, macs_ = 0.0, mac
                ops.append({'dataset': D['name'], 'model': spec['name'], 'lag': L, 'MAC': macs_, 'AC': acs, 'fire_rate': rate, 'pJ': pj,
                            'pJ_ensemble': pj * spec['n_runs']})
            if len(ch):
                c = ch[(ch['dataset'] == D['name']) & (ch['lag'] == L) & (ch['h'] == 1)]
                for mname, binary in (('TRACE-SNP', False), ('TRACE-SNP (binary spikes)', True)):
                    cc = c[c['model'] == mname]
                    if cc.empty:
                        continue
                    vals = []
                    for _, r in cc.iterrows():
                        fam = str(r['family'])
                        if fam.startswith('persistence'):
                            mac, ac = 0.0, 0.0
                        elif fam.startswith('ridge'):
                            mac, ac = float(L), 0.0
                        else:
                            enc = 2.0 * L                               # scale + locate the 2 firing fields per delay
                            lin = float(L)
                            if binary:
                                ac = float(CF_SPIKE_Q * L); dly = 0.0
                            else:
                                ac = 0.0; dly = 2.0 * L
                            stt = 0.0
                            if 'state' in fam and np.isfinite(r.get('C', np.nan)):
                                d_, C_ = float(r['d']), float(r['C'])
                                stt = C_ * d_ + CF_GRID['k_active'] * (d_ + 1) + CF_GRID['k_active']
                                if binary:
                                    ac += CF_SPIKE_Q * (d_ + 1); stt -= CF_GRID['k_active'] * (d_ + 1)
                            mac = enc + lin + dly + stt
                        vals.append((mac, ac))
                    mac, ac = np.mean(vals, 0)
                    ops.append({'dataset': D['name'], 'model': mname, 'lag': L, 'MAC': mac, 'AC': ac, 'fire_rate': np.nan,
                                'pJ': FINAL_PJ_MAC * mac + FINAL_PJ_AC * ac, 'pJ_ensemble': FINAL_PJ_MAC * mac + FINAL_PJ_AC * ac})
    if len(T2_RESULTS['ops']):
        o2 = T2_RESULTS['ops']
        o2 = o2[o2['h'] == 1].groupby(['dataset', 'model', 'lag'])[['MAC', 'AC', 'pJ']].mean().reset_index()
        for _, r in o2.iterrows():
            ops.append({'dataset': r['dataset'], 'model': r['model'], 'lag': r['lag'], 'MAC': r['MAC'], 'AC': r['AC'],
                        'fire_rate': np.nan, 'pJ': r['pJ'], 'pJ_ensemble': r['pJ']})
    OPS = pd.DataFrame(ops)
    if len(OPS):
        save_df(OPS, 'ops_neuromorphic')
        PJE = OPS.groupby(['model', 'lag'])['pJ_ensemble'].median().unstack('lag')
        PJE = PJE.loc[PJE.median(axis=1).sort_values().index]
        print("\nThe same per forecast AS REPORTED (networks: the 60-seed ensemble; TRACE-SNP: one model), pJ:")
        print(PJE.to_string(float_format=lambda v: f"{v:,.0f}"))
        save_df(PJE.reset_index(), 'neuromorphic_pj_ensemble_table')
        PJT = OPS.groupby(['model', 'lag'])['pJ'].median().unstack('lag')
        PJT = PJT.loc[PJT.median(axis=1).sort_values().index]
        print(f"\nEstimated energy per forecast on 45 nm neuromorphic hardware, pJ (MAC {FINAL_PJ_MAC} pJ, AC {FINAL_PJ_AC} pJ; "
              "spiking networks counted as ACs × firing rate — a lower bound for them):")
        print(PJT.to_string(float_format=lambda v: f"{v:,.0f}"))
        save_df(PJT.reset_index(), 'neuromorphic_pj_table')

    # ── 6. interpretability: TRACE-SNP's delay functions ──
    CV = CF_RESULTS['curves']
    if len(CV):
        ds = list(dict.fromkeys(CV['dataset']))
        nc = min(4, len(ds)); nr = int(np.ceil(len(ds) / nc))
        fig, axes = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.0 * nr), squeeze=False)
        for ax, dname in zip(axes.ravel(), ds):
            c = CV[CV['dataset'] == dname]
            L = int(c['lag'].max()); c = c[c['lag'] == L]
            amp = c.groupby('delay')['f'].apply(lambda f: f.max() - f.min()).sort_values(ascending=False)
            top = list(amp.index[:5])
            for k_, kk in enumerate(top):
                cc = c[c['delay'] == kk]
                ax.plot(cc['u'], cc['f'] - cc['f'].mean(), lw=1.6, label=f"delay {kk}", color=plt.cm.plasma(k_ / 5))
            ax.set_title(f"{dname}, lag {L}: the 5 strongest delay functions", fontsize=8, loc='left')
            ax.set_xlabel(c['input'].iloc[0] if 'input' in c else 'x(t-k)', fontsize=8); ax.legend(fontsize=6, frameon=False)
            ax.grid(True, alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
        for ax in axes.ravel()[len(ds):]:
            ax.axis('off')
        axes[0, 0].set_ylabel('contribution to x̂(t) − x(t−1)')
        fig.suptitle('TRACE-SNP is readable: each delay\'s learned function f_k (last walk-forward fold, one-step)', fontsize=10)
        fig.tight_layout(); save_fig(fig, 'delay_functions')
        dom = (CV.groupby(['dataset', 'lag', 'delay'])['f'].apply(lambda f: f.max() - f.min()).reset_index(name='amplitude'))
        save_df(dom, 'delay_function_amplitude')

    # ── 7. LaTeX ──
    def tex(df, caption, label, fmt=lambda v: f"{v:.4g}", bold_min_cols=None):
        cols_ = list(df.columns)
        lines = [r"\begin{table}[t]", r"\centering\small", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
                 r"\begin{tabular}{l" + "r" * len(cols_) + "}", r"\toprule",
                 " & ".join([str(df.index.name or '')] + [str(c) for c in cols_]) + r" \\", r"\midrule"]
        mins = {c: df[c].min() for c in (bold_min_cols or [])}
        for i, r in df.iterrows():
            cells = []
            for c in cols_:
                v = r[c]
                s = fmt(v) if isinstance(v, (float, np.floating)) and np.isfinite(v) else ('--' if isinstance(v, (float, np.floating)) else str(v))
                if c in mins and isinstance(v, (float, np.floating)) and v == mins[c]:
                    s = r"\textbf{" + s + "}"
                cells.append(s)
            lines.append(" & ".join([str(i).replace('_', r'\_').replace('&', r'\&')] + cells) + r" \\")
        lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]
        return "\n".join(lines)
    blocks = []
    for dname in names:
        g = core[(core['dataset'] == dname) & (core['h'] == 1)]
        if g.empty:
            continue
        piv = g.pivot(index='model', columns='lag', values='rmse'); piv.index.name = 'Model'
        piv.columns = [f"$L={c}$" for c in piv.columns]
        piv = piv.loc[piv.mean(axis=1).sort_values().index]
        blocks.append(tex(piv, f"{dname}: one-step test RMSE over the five walk-forward blocks (best per lag in bold).",
                          f"tab:rmse_{slugify(dname)}", bold_min_cols=list(piv.columns)))
    if WTL is not None:
        w = WTL.copy(); w.index.name = 'vs.'
        blocks.append(tex(w, f"{FINAL_MAIN_MODEL} against each model over all datasets, lags and horizons: Diebold--Mariano test, "
                             f"Holm-corrected within each dataset, $\\alpha={FINAL_ALPHA}$; ratio $<1$ means {FINAL_MAIN_MODEL} is more accurate.",
                          'tab:wtl', fmt=lambda v: f"{v:.3f}"))
    if ABL is not None:
        a_ = ABL.copy(); a_.index.name = 'Variant'
        blocks.append(tex(a_, "Ablations: RMSE relative to the full TRACE-SNP (geometric mean over all problems).", 'tab:ablation',
                          fmt=lambda v: f"{v:.3f}"))
    if PJT is not None:
        t2 = PJT.copy(); t2.index.name = 'Model'; t2.columns = [f"$L={c}$" for c in t2.columns]
        blocks.append(tex(t2, "Estimated energy per forecast (pJ) on 45\\,nm hardware: 4.6\\,pJ per MAC, 0.9\\,pJ per AC; spiking "
                              "networks counted as ACs scaled by their measured firing rate.", 'tab:pj', fmt=lambda v: f"{v:,.0f}"))
    tp = os.path.join(out_dir, f"final_{tag_}_tables{sm}.tex")
    with open(tp, 'w') as fh:
        fh.write("% generated by cell 11 — needs \\usepackage{booktabs}\n\n" + "\n".join(blocks))
    store.add(tp); saved.append(tp)
    store.push('final analysis', force=True)
    print(f"\nSaved {len(saved)} files to {store.rel(out_dir)}/ (final_*) — LaTeX tables: {store.rel(tp)}")
    return {'preds': A, 'leaderboard': LB, 'pairwise': PW, 'friedman': FR, 'noise': NZ, 'energy': EN, 'ops': OPS}


FINAL_V2 = _final_analysis()

In [ ]:
# ============================================================
# 12. ALL 11 DATASETS + EVERYTHING ALREADY IN THE RESULTS FOLDER (results and parameters) — no training, no refitting
#     The 11 series and their folds are those of cell 3.
#     Writes <LOCAL_ROOT>/revisions/:
#       inventory.csv                      every file in the results folder: top folder, sub-folder, kind, size class
#       params_trace_v1_choices.csv        TRACE-SNP (v1) configuration chosen per dataset × lag × horizon × fold
#       params_trace_v2_choices.csv        TRACE-SNP v2 configuration (anchor, encoding, M, λ's, d, C, iterated, spikes)
#       params_networks.csv                every network: training setup and number of trainable parameters
#       params_summary.md                  the above in words, for the methods section
# ============================================================
REV            = 'revisions'                       # every output of this notebook goes under <LOCAL_ROOT>/revisions/
REV_DIR        = os.path.join(LOCAL_ROOT, REV)
EXTRA_NAMES    = ['etth1', 'etth2', 'melbourne_temp', 'co2_weekly', 'elnino_monthly', 'lorenz', 'lorenz_noisy', 'rossler']
CHAOTIC_SLUGS  = {'mackeyglass', 'mackey_glass', 'lorenz', 'lorenz_noisy', 'rossler'}
SM             = '_smoke' if SMOKE else ''
os.makedirs(REV_DIR, exist_ok=True)


def rev_path(*parts):
    p = os.path.join(REV_DIR, *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def rev_csv(df, name, index=False):
    p = rev_path(f"{name}{SM}.csv"); df.to_csv(p, index=index); store.add(p); return p


def _all11_datasets():
    return [dict(D, kind=D.get('kind') or ('chaotic' if D['slug'] in CHAOTIC_SLUGS else 'real')) for D in DATASETS]


ALL11 = _all11_datasets()
ALL11_NAMES = [D['name'] for D in ALL11]
KIND_OF = {D['name']: D['kind'] for D in ALL11}
print(f"{len(ALL11)} datasets: " + ", ".join(f"{D['name']} ({D['kind']}, {D['n']} pts)" for D in ALL11))
if len(ALL11) != 11:
    print(f"WARNING: expected 11 datasets, have {len(ALL11)} — the N in every dataset-level test below is {len(ALL11)}")

# ── inventory of the results folder ──
store.refresh()
_files = sorted(store.remote) if store.enabled else sorted(
    os.path.relpath(p, LOCAL_ROOT).replace(os.sep, '/') for p in glob.glob(os.path.join(LOCAL_ROOT, '**', '*'), recursive=True)
    if os.path.isfile(p))


def _kind(f):
    b = os.path.basename(f)
    for k in ('runs.pt', 'checkpoints_', 'preds_', 'multih', 'walkforward_folds_', 'walkforward_', 'energy_', '_choices', '_curves',
              '_noise', '_ops', '_preds', '_energy', 'final_', 'fig_', 'tab_'):
        if k in f:
            return k.strip('_')
    return os.path.splitext(b)[1].lstrip('.') or 'other'


INVENTORY = pd.DataFrame([{'path': f, 'top': f.split('/')[0], 'sub': f.split('/')[1] if f.count('/') >= 1 else '',
                           'kind': _kind(f)} for f in _files])
rev_csv(INVENTORY, 'inventory')
if len(INVENTORY):
    print(f"\n{LOCAL_ROOT}: {len(INVENTORY):,} files")
    print(INVENTORY[~INVENTORY['kind'].isin(['checkpoints', 'runs.pt'])].groupby(['top', 'sub']).size()
          .unstack(fill_value=0).to_string(max_cols=30))

def read_many(pattern):
    # every local CSV under LOCAL_ROOT matching the glob pattern, concatenated (smoke / full runs kept apart)
    fs = [f for f in glob.glob(os.path.join(LOCAL_ROOT, pattern)) if os.path.getsize(f) > 2 and (('_smoke' in f) == SMOKE)]
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def comp(stem, group='all11'):
    # comparison/final_<group>_<stem>.csv (cell 11); if missing, any per-dataset files stacked
    p = store.fetch(f"comparison/final_{group}_{stem}{SM}.csv")
    if p and os.path.getsize(p) > 2:
        return pd.read_csv(p)
    fs = [f for f in glob.glob(os.path.join(LOCAL_ROOT, 'comparison', f"final_*_{stem}{SM}.csv"))
          if f"final_{group}_" not in f and 'final_all11_' not in f and 'final_noise_' not in f
          and (('_smoke' in f) == SMOKE) and os.path.getsize(f) > 2]
    if not fs:
        return pd.DataFrame()
    print(f"(final_{group}_{stem} not found — stacking {len(fs)} per-dataset files)")
    df = pd.concat([pd.read_csv(f) for f in sorted(fs, key=os.path.getmtime)], ignore_index=True)
    keys = [c for c in ('dataset', 'model', 'lag', 'h', 'noise', 'other') if c in df]
    return df.drop_duplicates(subset=keys, keep='last') if keys else df


LB = comp('leaderboard')
assert len(LB), "No leaderboard in the results folder (comparison/final_all11_leaderboard.csv or final_<dataset>_leaderboard.csv)"
LB = LB[LB['dataset'].isin(ALL11_NAMES)].copy()
print(f"\nLeaderboard: {LB['dataset'].nunique()} datasets × {LB['model'].nunique()} models, "
      f"{LB.groupby(['dataset', 'lag', 'h']).ngroups} (dataset, lag, horizon) problems")

# ── parameters ──
P1 = read_many(f"*/closed_form/cf{SM}_v2_lag*_h*_choices.csv")
P2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_choices.csv")
if len(P1):
    P1 = P1[P1['model'] == 'TRACE-SNP']; rev_csv(P1, 'params_trace_v1_choices')
if len(P2):
    rev_csv(P2, 'params_trace_v2_choices')
_net_rows = []
for _spec in MODEL_SPECS:
    _m = _spec['build']()                               # input size 1: the parameter count does not depend on the lag
    _net_rows.append({'model': _spec['name'], 'folder': _spec['folder'],
                      'trainable_params': int(sum(p.numel() for p in _m.parameters() if p.requires_grad)),
                      'hidden': ALSTM_CFG['hidden'] if _spec['name'] == ALSTM_CFG['name'] else BASELINE_TRAIN['hidden'],
                      'seeds (ensemble members)': _spec['n_runs'], 'batch': _spec['batch'], 'lr': _spec['lr'],
                      'max_epochs': _spec['max_epochs'], 'patience': _spec['patience'], 'lags': str(_spec['lags'])})
NET_PARAMS = pd.DataFrame(_net_rows)
rev_csv(NET_PARAMS, 'params_networks')

_md = ["# Parameters actually used (read from the results folder)\n",
       "Networks (hidden size, training setup, trainable parameters):\n",
       NET_PARAMS.set_index('model').to_string(), "",
       f"ALSTM-SNP: {ALSTM_CFG} | cell {ALSTM_CELL}", f"Baselines: {BASELINE_TRAIN}", ""]
if len(P2):
    _cols = [c for c in ('anchor', 'enc', 'M', 'l0', 'ls', 'd', 'C', 'lst', 'spike_q') if c in P2]
    _md += ["TRACE-SNP v2 — chosen family (count over dataset × lag × horizon × fold):",
            P2['family'].astype(str).str.split('|').str[-1].value_counts().to_string(), "",
            "TRACE-SNP v2 — most frequent value of each hyper-parameter per dataset:",
            P2.groupby('dataset')[_cols].agg(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan).to_string(), "",
            "Share of folds where the iterated one-step model was used (h > 1):",
            P2[P2['h'] > 1].groupby(['dataset', 'h'])['iterated'].mean().unstack('h').to_string(float_format=lambda v: f"{v:.0%}"), ""]
if len(P1):
    _md += ["TRACE-SNP (v1) — chosen family (count):", P1['family'].astype(str).value_counts().to_string(), ""]
with open(rev_path(f"params_summary{SM}.md"), 'w') as _fh:
    _fh.write("\n".join(_md))
store.add(rev_path(f"params_summary{SM}.md"))
print("\n".join(_md))
store.push('revisions: inventory + parameters', force=True)

In [ ]:
# ============================================================
# 13. FIRING-CONDITION ABLATION (closed form, minutes on a GPU) — gated receptive fields vs a NON-GATED dense RBF basis
#     Question: is the firing condition what makes TRACE-SNP work, or is the convex ridge solver doing the work?
#     TRACE-SNP v2 is re-run with ONE change, everything else identical (same 11 datasets, lags, horizons 1/6/24, the 5
#     walk-forward folds, scaling, encodings, anchors, penalties, λ grids, exact-LOO + validation selection, the
#     iterated-vs-direct margin rule, the Cholesky ridge solve):
#       gated  (TRACE-SNP v2):  delay pathway  μ_m(u) = Wang–Mendel triangular fields, exactly 2 of M neurons fire per delay;
#                               state pathway  only the k_active = 4 nearest state neurons fire (top-k gate).
#       non-gated (RBF):        delay pathway  φ_m(u) = exp(−(u − c_m)² / 2(w·Δ)²) on the SAME M centres c_m (Δ = spacing),
#                               every neuron is active for every input — no firing condition; width w ∈ RBF_WIDTHS is chosen
#                               by the same LOO + validation criterion (a fair, tuned dense basis);
#                               state pathway  all C state neurons active (normalised Gaussian over every centre, no top-k).
#     Same number of features and weights in both (iso-parametric), so only the firing condition differs.
#     Each variant is reported twice: as the full model (persistence / window mean / Ridge AR(L) remain candidates, as in
#     TRACE-SNP v2) and with "no fallback" (only receptive-field candidates), which isolates the basis completely.
#     A reproduction check compares the recomputed gated model with the stored TRACE-SNP v2 forecasts.
#     Resumable: <LOCAL_ROOT>/revisions/rbf_ablation/<dataset>_lag<L>_*.csv; finished (dataset, lag) are reused.
# ============================================================
ABL_FOLDER    = 'rbf_ablation'
ABL_LAGS      = [1, 5] if SMOKE else COMMON_LAGS
ABL_HORIZONS  = [1, 6, 24]
RBF_WIDTHS    = [1.0, 2.0]          # Gaussian width in units of the centre spacing (1: neighbours overlap at e^-1/2)
ABL_FRESH     = False
ABL_GRID      = T2_GRID if 'T2_GRID' in globals() else dict(
    M=[16, 32], max_delay_units=2048, l0=[1e-6, 1e-4, 1e-2, 1.0], ls=[1e-5, 1e-3, 1e-1, 10.0],
    d=[5, 10], C=[32, 64], k_active=4, lst=[1e-4, 1e-2, 1.0])       # identical to TRACE-SNP v2's grid (cell 10)
ABL_ITER_MARGIN = 0.25              # TRACE-SNP v2's T2_ITER_MARGIN
ABL_MIN_ORIGINS = 5
ABL_NAMES = {('tri', True): 'TRACE-SNP v2 (gated)', ('rbf', True): 'Non-gated RBF',
             ('tri', False): 'Gated, no fallback', ('rbf', False): 'Non-gated RBF, no fallback'}


def run_rbf_ablation(datasets):
    from sklearn.preprocessing import MinMaxScaler as _MMS
    f64, dev, G = torch.float64, device, ABL_GRID
    T = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=f64, device=dev)
    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))

    def tri(U, M, lo=-1.1, hi=1.1):
        h = (hi - lo) / (M - 1)
        pos = (U.clamp(lo, hi) - lo) / h
        i = pos.floor().long().clamp(0, M - 2)
        fr = pos - i
        B = torch.zeros(U.shape + (M,), dtype=U.dtype, device=U.device)
        B.scatter_(-1, i[..., None], (1 - fr)[..., None])
        B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
        return B.reshape(U.shape[0], -1)

    def rbf(U, M, w, lo=-1.1, hi=1.1):
        h = (hi - lo) / (M - 1)
        c = torch.linspace(lo, hi, M, dtype=U.dtype, device=U.device)
        B = torch.exp(-(U.clamp(lo, hi)[..., None] - c) ** 2 / (2 * (w * h) ** 2))
        return B.reshape(U.shape[0], -1)

    def kmeans(Z, C, seed=0):                               # identical to cell 10 (same seeds → same state centres)
        g = torch.Generator(device='cpu').manual_seed(seed)
        idx = [int(torch.randint(len(Z), (1,), generator=g))]
        dmin = ((Z - Z[idx[0]]) ** 2).sum(1)
        for _ in range(C - 1):
            j = int(torch.multinomial((dmin / dmin.sum()).cpu(), 1, generator=g)); idx.append(j)
            dmin = torch.minimum(dmin, ((Z - Z[j]) ** 2).sum(1))
        cen = Z[idx].clone()
        for _ in range(20):
            lab = torch.cdist(Z, cen).argmin(1)
            cnt = torch.bincount(lab, minlength=C).to(Z.dtype)
            s_ = torch.zeros_like(cen).index_add_(0, lab, Z)
            keep = cnt > 0
            cen[keep] = s_[keep] / cnt[keep, None]
        nn_d = torch.cdist(cen, cen) + torch.eye(C, dtype=Z.dtype, device=Z.device) * 1e9
        return cen, float(nn_d.min(1).values.median() ** 2)

    def state_feats(U, st, gated):
        cen, s2, d, off = st
        Z = U[:, off:off + d]
        D2 = torch.cdist(Z, cen) ** 2
        if gated:                                           # firing condition: only the k nearest state neurons fire
            val, idx = D2.topk(G['k_active'], dim=1, largest=False)
            g = torch.exp(-val / (2 * s2)); g = g / g.sum(1, keepdim=True)
            Mu = torch.zeros_like(D2).scatter_(1, idx, g)
        else:                                               # no firing condition: every state neuron active
            Mu = torch.softmax(-D2 / (2 * s2), dim=1)
        Z1 = torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), Z], 1)
        return (Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1)

    def design(U, basis, M=None, w=None, st=None):
        parts = [torch.ones(len(U), 1, dtype=U.dtype, device=U.device), U]
        if M:
            parts.append(tri(U, M) if basis == 'tri' else rbf(U, M, w))
        if st is not None:
            parts.append(state_feats(U, st, basis == 'tri'))
        return torch.cat(parts, 1)

    def penalty(L, M, l0, ls, n_state=0, lst=0.0):         # identical to cell 10
        nd = L * M if M else 0
        p = 1 + L + nd + n_state
        P = torch.zeros(p, p, dtype=f64, device=dev)
        P[1:, 1:] += l0 * torch.eye(p - 1, dtype=f64, device=dev)
        if M and ls > 0:
            D = torch.zeros(M - 2, M, dtype=f64, device=dev)
            for j in range(M - 2):
                D[j, j], D[j, j + 1], D[j, j + 2] = 1, -2, 1
            DD = ls * (D.T @ D)
            for k in range(L):
                a0 = 1 + L + k * M
                P[a0:a0 + M, a0:a0 + M] += DD
        if n_state and lst > l0:
            a0 = 1 + L + nd
            P[a0:, a0:] += (lst - l0) * torch.eye(n_state, dtype=f64, device=dev)
        P[0, 0] = 1e-10
        return P

    def fit_loo(Ptr, R, Pen, chunk=8192):                   # identical to cell 10: one Cholesky solve + exact LOO residuals
        N = len(Ptr)
        A = Ptr.T @ Ptr / N + Pen
        Lc = torch.linalg.cholesky(A)
        th = torch.cholesky_solve(Ptr.T @ R / N, Lc)
        loo = []
        for i in range(0, N, chunk):
            P = Ptr[i:i + chunk]
            V = torch.linalg.solve_triangular(Lc, P.T, upper=False)
            hd = ((V ** 2).sum(0) / N).clamp(max=1 - 1e-6)
            loo.append((R[i:i + chunk] - P @ th) / (1 - hd)[:, None])
        return th, torch.cat(loo)

    def windows(x, L, o):
        return np.stack([x[o - 1 - k] for k in range(L)], 1)

    def run_fold(x, L, h, f):
        # TRACE-SNP v2's candidate set (cell 10), built for both bases on identical data; returns the candidates
        otr = np.arange(L, f['va_start'] - h + 1)
        ova = np.arange(f['va_start'], f['te_start'] - h + 1)
        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
        a, b = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / (b - a)
        s = lambda v: sc * (v - a) - 1
        W = {'tr': windows(x, L, otr), 'va': windows(x, L, ova), 'te': windows(x, L, ote)}
        y = {'tr': x[otr + h - 1], 'va': x[ova + h - 1], 'te': x[ote + h - 1]}
        anc_raw = {'last': lambda Wr: Wr[:, 0], 'mean': lambda Wr: Wr.mean(1)}
        dd = (W['tr'][:, 1:] - W['tr'][:, :1]) if L > 1 else np.zeros((len(otr), 1))
        scd = 1.0 / max(float(np.abs(dd).max()), 1e-12)
        enc = {'lvl': lambda Wr: T(s(Wr)),
               'dif': lambda Wr: T(np.column_stack([np.zeros(len(Wr)), (Wr[:, 1:] - Wr[:, :1]) * scd]) if L > 1
                                   else np.zeros((len(Wr), 1)))}
        cands = []

        def add(fam, basis, cfg, loo_raw, va_pred, te_pred, fn):
            e_va = y['va'] - va_pred
            cands.append({'fam': fam, 'basis': basis, 'cfg': cfg, 'pool': rmse(np.concatenate([loo_raw, e_va])),
                          'va_pred': va_pred, 'te_pred': te_pred, 'fn': fn})

        for an in ('last', 'mean'):
            add('persistence' if an == 'last' else 'window-mean', 'fallback', {'anchor': an}, y['tr'] - anc_raw[an](W['tr']),
                anc_raw[an](W['va']), anc_raw[an](W['te']), anc_raw[an])
        mm = _MMS((-1, 1)).fit(np.column_stack([W['tr'], y['tr']]))
        tf = lambda Wr: Wr * mm.scale_[:-1] + mm.min_[:-1]
        Xc = tf(W['tr']); yy = y['tr'] * mm.scale_[-1] + mm.min_[-1]
        xm, ym = Xc.mean(0), yy.mean(); Xc0 = Xc - xm
        Ai = np.linalg.inv(Xc0.T @ Xc0 + RIDGE_ALPHA * np.eye(L)); wr = Ai @ Xc0.T @ (yy - ym)
        hr = 1 / len(Xc) + np.einsum('ij,jk,ik->i', Xc0, Ai, Xc0)
        rf = lambda Wr: ((tf(Wr) - xm) @ wr + ym - mm.min_[-1]) / mm.scale_[-1]
        add('ridge', 'fallback', {}, ((yy - (Xc0 @ wr + ym)) / (1 - hr)) / mm.scale_[-1], rf(W['va']), rf(W['te']), rf)

        ys = T(s(y['tr']))

        def make_fn(e, an, basis, M, w, st, th):
            def fn(Wr):
                return ((T(s(anc_raw[an](Wr))) + design(enc[e](Wr), basis, M, w, st) @ th).cpu().numpy() + 1) / sc + a
            return fn

        anchors = ('last', 'mean')
        for e in ('lvl', 'dif'):
            if e == 'dif' and L < 3:
                continue
            U = {k: enc[e](W[k]) for k in W}
            A_ = {k: torch.stack([T(s(anc_raw[an](W[k]))) for an in anchors], 1) for k in W}
            R = ys[:, None] - A_['tr']
            states = {}
            dmax = L if e == 'lvl' else L - 1
            for d in sorted({min(dmax, v) for v in G['d']}):
                for C in G['C']:
                    if C * (d + 1) > len(otr):
                        continue
                    Z = U['tr'][:, :d] if e == 'lvl' else U['tr'][:, 1:d + 1]
                    cen, s2 = kmeans(Z, C)
                    states[(d, C)] = (cen, s2, d, 0 if e == 'lvl' else 1)
            for basis, widths in (('tri', [None]), ('rbf', RBF_WIDTHS)):
                fam0 = f"{basis}|{e}"
                for M in G['M']:
                    if L * M > G['max_delay_units'] and M != min(G['M']):
                        continue
                    for w in widths:
                        P = {k: design(U[k], basis, M, w) for k in U}
                        for l0 in G['l0']:
                            for ls in G['ls']:
                                th, loo = fit_loo(P['tr'], R, penalty(L, M, l0, ls))
                                pv, pt = (A_['va'] + P['va'] @ th).cpu().numpy(), (A_['te'] + P['te'] @ th).cpu().numpy()
                                for j, an in enumerate(anchors):
                                    add(f"{an}|{fam0}|delay", basis, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls},
                                        loo[:, j].cpu().numpy() / sc, (pv[:, j] + 1) / sc + a, (pt[:, j] + 1) / sc + a,
                                        make_fn(e, an, basis, M, w, None, th[:, j].clone()))
                        del P
                for j, an in enumerate(anchors):
                    best = min([c for c in cands if c['fam'] == f"{an}|{fam0}|delay"], key=lambda c: c['pool'])['cfg']
                    M, w, l0, ls = best['M'], best['w'], best['l0'], best['ls']
                    for (d, C), st in states.items():
                        P = {k: design(U[k], basis, M, w, st) for k in U}
                        for lst in G['lst']:
                            th, loo = fit_loo(P['tr'], R[:, j:j + 1], penalty(L, M, l0, ls, C * (d + 1), max(lst, l0)))
                            pv = (A_['va'][:, j:j + 1] + P['va'] @ th).cpu().numpy()[:, 0]
                            pt = (A_['te'][:, j:j + 1] + P['te'] @ th).cpu().numpy()[:, 0]
                            add(f"{an}|{fam0}|delay+state", basis, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls,
                                                                    'd': d, 'C': C, 'lst': lst},
                                loo[:, 0].cpu().numpy() / sc, (pv + 1) / sc + a, (pt + 1) / sc + a,
                                make_fn(e, an, basis, M, w, st, th[:, 0].clone()))
                        del P
        return cands, y, ova, ote

    def pick(cands, basis, fallback):
        pool = [c for c in cands if c['basis'] == basis or (fallback and c['basis'] == 'fallback')]
        return min(pool, key=lambda c: c['pool'])

    def iterate(fn, x, L, origins, h):
        Wr = windows(x, L, origins)
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    def active_fraction(c, L):
        # share of receptive-field / state neurons that are active (non-zero) for one forecast
        if c['basis'] == 'fallback':
            return np.nan
        M = c['cfg']['M']; n_on, n_all = (2.0 * L if c['basis'] == 'tri' else float(L * M)), float(L * M)
        if 'C' in c['cfg']:
            C = c['cfg']['C']; n_on += G['k_active'] if c['basis'] == 'tri' else C; n_all += C
        return n_on / n_all

    preds, choices = [], []
    for D in datasets:
        x = D['values']
        for L in ABL_LAGS:
            if not lag_ok(None, D, L):
                continue
            hs = [h for h in ABL_HORIZONS if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 >= ABL_MIN_ORIGINS]
            stem = f"{REV}/{ABL_FOLDER}/{D['slug']}{SM}_lag{L}"
            rels = {k: f"{stem}_{k}.csv" for k in ('preds', 'choices')}
            if not ABL_FRESH and all(store.fetch(r) for r in rels.values()):
                preds.append(pd.read_csv(store.fetch(rels['preds']))); choices.append(pd.read_csv(store.fetch(rels['choices'])))
                continue
            check_time_budget()
            t0 = time.time(); P_, C_ = [], []
            for f in D['folds']:
                fits = {h: run_fold(x, L, h, f) for h in sorted(set([1] + hs))}
                c1 = fits[1][0]
                for h in hs:
                    cands, y, ova, ote = fits[h]
                    yt, prev = x[ote + h - 1], x[ote - 1]
                    for (basis, fb), name in ABL_NAMES.items():
                        bd = pick(cands, basis, fb)
                        pred, it = bd['te_pred'], False
                        if h > 1:                                    # TRACE-SNP v2's direct-vs-iterated margin rule
                            b1 = pick(c1, basis, fb)
                            i_va = iterate(b1['fn'], x, L, ova, h)
                            it = rmse(y['va'] - i_va) < (1 - ABL_ITER_MARGIN) * rmse(y['va'] - bd['va_pred'])
                            if it:
                                pred = iterate(b1['fn'], x, L, ote, h)
                        used = pick(c1, basis, fb) if it else bd
                        P_.append(pd.DataFrame({'dataset': D['name'], 'model': name, 'lag': L, 'h': h, 'test_fold': f['label'],
                                                'origin': ote, 'target_idx': ote + h - 1, 'y': yt, 'prev': prev, 'ens_pred': pred}))
                        C_.append({'dataset': D['name'], 'model': name, 'lag': L, 'h': h, 'fold': f['label'], 'family': used['fam'],
                                   'basis_used': used['basis'], 'iterated': bool(it), 'criterion': used['pool'],
                                   'active_fraction': active_fraction(used, L), **used['cfg']})
                del fits, c1
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
            Pdf, Cdf = pd.DataFrame(pd.concat(P_, ignore_index=True)), pd.DataFrame(C_)
            for k, df in (('preds', Pdf), ('choices', Cdf)):
                p = os.path.join(LOCAL_ROOT, rels[k]); os.makedirs(os.path.dirname(p), exist_ok=True)
                df.to_csv(p, index=False); store.add(p)
            preds.append(Pdf); choices.append(Cdf)
            g = Pdf.groupby(['model', 'h']).apply(lambda d: rmse(d['y'] - d['ens_pred']), include_groups=False).unstack('h')
            print(f"{D['name']:<16} lag {L:2d} | " + " | ".join(
                f"h{h}: gated {g.loc[ABL_NAMES[('tri', True)], h]:.4g} vs RBF {g.loc[ABL_NAMES[('rbf', True)], h]:.4g}" for h in g.columns)
                + f" | {time.time() - t0:.0f} s")
            store.push(f"rbf ablation {D['slug']} lag {L}")
        store.push(f"rbf ablation {D['slug']}", force=True)
        trim_ram()
    cat = lambda lst: pd.concat(lst, ignore_index=True) if lst else pd.DataFrame()
    return {'preds': cat(preds), 'choices': cat(choices)}


try:
    ABL_RESULTS = run_rbf_ablation(ALL11)
except TimeBudgetReached:
    store.push('rbf ablation: time budget', force=True)
    raise RuntimeError('Time budget reached — rerun this cell; finished (dataset, lag) are reused.')


def _rbf_summary():
    from scipy import stats as _st
    A = ABL_RESULTS['preds']
    if A.empty:
        print('No ablation results.'); return None
    rm = lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2)))
    R = A.groupby(['dataset', 'lag', 'h', 'model']).apply(rm, include_groups=False).unstack('model')
    G_, NG = ABL_NAMES[('tri', True)], ABL_NAMES[('rbf', True)]
    G0, NG0 = ABL_NAMES[('tri', False)], ABL_NAMES[('rbf', False)]
    R['ratio'] = R[NG] / R[G_]                              # > 1: removing the firing condition hurts
    R['ratio_nofallback'] = R[NG0] / R[G0]
    # DM test per problem (matched targets), Holm within dataset
    rows = []
    for (dname, L, h), g in A.groupby(['dataset', 'lag', 'h']):
        a = g[g['model'] == G_].set_index(['test_fold', 'target_idx']).sort_index()
        b = g[g['model'] == NG].set_index(['test_fold', 'target_idx']).sort_index()
        idx = a.index.intersection(b.index)
        e1, e2 = (a.loc[idx, 'y'] - a.loc[idx, 'ens_pred']).to_numpy(), (b.loc[idx, 'y'] - b.loc[idx, 'ens_pred']).to_numpy()
        same = np.allclose(e1, e2)
        dm, p = (0.0, 1.0) if same else dm_test(e1, e2, h=int(h))
        rows.append({'dataset': dname, 'lag': L, 'h': h, 'dm': dm, 'dm_p': p, 'identical': same})
    DMt = pd.DataFrame(rows).set_index(['dataset', 'lag', 'h'])
    R = R.join(DMt)
    R['dm_p_holm'] = R.groupby(level='dataset')['dm_p'].transform(lambda p: holm(p.to_numpy()))
    R['outcome'] = np.select([(R['dm_p_holm'] < 0.05) & (R['dm'] < 0), (R['dm_p_holm'] < 0.05) & (R['dm'] > 0)],
                             ['gated better', 'RBF better'], 'no sig. difference')
    rev_csv(R.reset_index(), 'rbf_ablation_by_problem')
    gm = lambda r: float(np.exp(np.mean(np.log(r))))
    byd = R.groupby(level='dataset').agg(**{
        'RMSE ratio RBF/gated (geo-mean)': ('ratio', gm), 'ratio, no fallback': ('ratio_nofallback', gm),
        'gated better': ('outcome', lambda o: int((o == 'gated better').sum())),
        'no sig. diff.': ('outcome', lambda o: int((o == 'no sig. difference').sum())),
        'RBF better': ('outcome', lambda o: int((o == 'RBF better').sum()))})
    byd['type'] = [KIND_OF.get(d, '') for d in byd.index]
    byd = byd.loc[[d for d in ALL11_NAMES if d in byd.index]]
    byh = R.groupby(level='h').agg(**{'RMSE ratio RBF/gated (geo-mean)': ('ratio', gm), 'ratio, no fallback': ('ratio_nofallback', gm),
                                      'gated better': ('outcome', lambda o: int((o == 'gated better').sum())),
                                      'RBF better': ('outcome', lambda o: int((o == 'RBF better').sum())), 'problems': ('ratio', 'size')})
    # dataset-level test (N = number of datasets): one-sided Wilcoxon signed-rank on the per-dataset log ratios
    lr = np.log(byd['RMSE ratio RBF/gated (geo-mean)'].to_numpy())
    lr0 = np.log(byd['ratio, no fallback'].to_numpy())
    wil = _st.wilcoxon(lr, alternative='greater', zero_method='zsplit') if np.any(lr != 0) else None
    wil0 = _st.wilcoxon(lr0, alternative='greater', zero_method='zsplit') if np.any(lr0 != 0) else None
    overall = gm(R['ratio']); overall0 = gm(R['ratio_nofallback'])
    ch = ABL_RESULTS['choices']
    act = ch.groupby('model')['active_fraction'].median()
    fb = ch.groupby('model')['basis_used'].apply(lambda b: float((b == 'fallback').mean()))
    rev_csv(byd.reset_index(), 'rbf_ablation_by_dataset'); rev_csv(byh.reset_index(), 'rbf_ablation_by_horizon')
    lines = [f"## Firing-condition ablation: gated receptive fields (TRACE-SNP v2) vs non-gated dense RBF, same solver",
             f"RMSE(non-gated RBF) / RMSE(gated), geometric mean over {len(R)} (dataset, lag, horizon) problems: {overall:.3f} "
             f"(no-fallback variants: {overall0:.3f}); > 1 means the gated model is more accurate.",
             f"Datasets where the gated model is more accurate (geo-mean ratio > 1): {int((byd.iloc[:, 0] > 1).sum())} of {len(byd)}.",
             (f"One-sided Wilcoxon signed-rank over the {len(byd)} datasets (H1: gated < RBF): W = {wil.statistic:.1f}, p = {wil.pvalue:.3g}"
              if wil is not None else "Wilcoxon: all ratios equal to 1."),
             (f"  … no-fallback variants: W = {wil0.statistic:.1f}, p = {wil0.pvalue:.3g}" if wil0 is not None else ""),
             f"Per-problem DM tests (Holm within dataset, α = 0.05): {int((R['outcome'] == 'gated better').sum())} gated better, "
             f"{int((R['outcome'] == 'no sig. difference').sum())} no significant difference, {int((R['outcome'] == 'RBF better').sum())} RBF better.",
             "Active (firing) fraction of receptive-field + state neurons per forecast (median): "
             + ", ".join(f"{m}: {v:.1%}" for m, v in act.items() if np.isfinite(v)),
             "Share of folds that fell back to persistence / window mean / Ridge: " + ", ".join(f"{m}: {v:.0%}" for m, v in fb.items()),
             "", byd.to_string(float_format=lambda v: f"{v:.3f}"), "", byh.to_string(float_format=lambda v: f"{v:.3f}")]
    # reproduction check against the stored TRACE-SNP v2 forecasts
    st = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_preds.csv")
    if len(st):
        st = st[st['model'] == 'TRACE-SNP v2']
        s_r = st.groupby(['dataset', 'lag', 'h']).apply(rm, include_groups=False).rename('stored')
        chk = R[[G_]].join(s_r, how='inner')
        if len(chk):
            chk['rel_diff'] = (chk[G_] / chk['stored'] - 1).abs()
            rev_csv(chk.reset_index(), 'rbf_ablation_reproduction_check')
            lines.append(f"\nReproduction check — recomputed gated model vs stored TRACE-SNP v2 RMSE over {len(chk)} problems: "
                         f"median |rel. diff| {chk['rel_diff'].median():.1e}, max {chk['rel_diff'].max():.1e}, "
                         f"{int((chk['rel_diff'] < 1e-3).sum())} of {len(chk)} within 0.1 %")
    txt = "\n".join(l for l in lines if l is not None)
    print(txt)
    with open(rev_path(f"rbf_ablation_summary{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"rbf_ablation_summary{SM}.md"))
    return {'by_problem': R, 'by_dataset': byd, 'by_horizon': byh, 'overall': overall, 'overall_nofallback': overall0,
            'wilcoxon': wil, 'wilcoxon_nofallback': wil0, 'active': act, 'fallback_share': fb, 'text': txt}


RBF = _rbf_summary()
store.push('revisions: rbf ablation', force=True)

In [ ]:
# ============================================================
# 14. RANKING OVER THE N = 11 INDEPENDENT DATASETS (replaces the Friedman / Nemenyi test over 198 dependent problems)
#     The 198 (dataset, lag, horizon) problems share data, so they are not independent blocks. Here the block is the
#     DATASET (Demšar 2006; Benavoli et al. 2016):
#       1. per dataset and model, one score = geometric mean over that dataset's (lag, horizon) problems of
#          RMSE / persistence RMSE, taken over the problems on which every compared model ran (scale-free, so lags and
#          horizons count equally);
#       2. models ranked within each dataset (rank 1 = lowest score) → Friedman χ² and the Iman–Davenport F over N datasets;
#       3. Nemenyi critical difference CD = q_α √(k(k+1)/6N), and the CD diagram (non-overlapping labels);
#       4. post-hoc against the proposed model (TRACE-SNP v2): Wilcoxon signed-rank over the N datasets, Holm-corrected
#          (the comparison-with-a-control procedure recommended over Nemenyi by Benavoli et al. 2016).
#     Robustness: the same with the ranks averaged within each dataset first, per horizon (N = 11 each), and the old
#     198-problem numbers side by side so the change can be reported.
#     Also the facts behind two sentences of the paper: where TRACE-SNP v2's margin is largest (chaotic vs real), and who
#     wins on real series at h = 24.
# ============================================================
STAT_ALPHA   = 0.05
STAT_CONTROL = 'TRACE-SNP v2'
STAT_EXCLUDE = ['TRACE-SNP (binary spikes)', 'TRACE-SNP v2 (binary spikes)', 'TRACE: delay pathway only',
                'TRACE: state pathway only', 'TRACE: no curvature penalty', 'TRACE: level inputs only',
                'TRACE: validation-only selection', 'TRACE: no fallback candidates']   # variants / ablations: not rivals
STAT_MIN_COVER = 0.5               # a model must have run on ≥ this share of a dataset's problems to be ranked there

# colour-blind-safe identity colours (Okabe & Ito 2008) + markers / line styles as a second channel
OI = {'vermillion': '#D55E00', 'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'purple': '#CC79A7',
      'sky': '#56B4E9', 'yellow': '#F0E442', 'black': '#000000'}
STYLE = {
    'TRACE-SNP v2':     dict(color=OI['vermillion'], ls='-',  marker='o', lw=2.0),
    'TRACE-SNP':        dict(color=OI['orange'],     ls='-',  marker='s', lw=1.6),
    'ALSTM-SNP':        dict(color=OI['purple'],     ls='-',  marker='D', lw=1.6),
    'LSTM [34]':        dict(color=OI['blue'],       ls='-',  marker='v', lw=0.9),
    'GRU [39]':         dict(color=OI['blue'],       ls='--', marker='^', lw=0.9),
    'LSTM-SNP [25]':    dict(color=OI['sky'],        ls='-',  marker='<', lw=0.9),
    'BiLSTM-SNP [26]':  dict(color=OI['sky'],        ls='--', marker='>', lw=0.9),
    'Attention':        dict(color=OI['green'],      ls='-',  marker='P', lw=0.9),
    'Informer [42]':    dict(color=OI['green'],      ls='--', marker='X', lw=0.9),
    'Autoformer [40]':  dict(color=OI['green'],      ls=':',  marker='*', lw=0.9),
    'FEDformer [41]':   dict(color=OI['green'],      ls='-.', marker='h', lw=0.9),
    'NG-RC':            dict(color='#4d4d4d',        ls='--', marker='p', lw=0.9),
    'DLinear':          dict(color='#4d4d4d',        ls=':',  marker='d', lw=0.9),
    'Ridge AR(L)':      dict(color='#8c8c8c',        ls='--', marker='8', lw=0.9),
    'Persistence':      dict(color='#8c8c8c',        ls=':',  marker='H', lw=0.9),
}
OURS = ['TRACE-SNP v2', 'TRACE-SNP', 'ALSTM-SNP']
sty = lambda m: STYLE.get(m, dict(color='#999999', ls='-', marker='.', lw=0.8))
short = lambda m: m.replace(' (binary spikes)', ' (bin.)').replace('TRACE: ', '')

plt.rcParams.update({'font.family': 'serif', 'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'legend.fontsize': 7,
                     'xtick.labelsize': 7, 'ytick.labelsize': 7, 'axes.spines.top': False, 'axes.spines.right': False,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02})
W1, W2 = 3.5, 7.2
FIG_DIR = os.path.join(REV_DIR, 'figures')


def save_fig(fig, name):
    os.makedirs(FIG_DIR, exist_ok=True)
    for ext in ('pdf', 'png'):
        p = os.path.join(FIG_DIR, f"{name}{SM}.{ext}"); fig.savefig(p, dpi=300); store.add(p)
    show(fig)


def tex_table(df, name, caption, label, fmt=lambda v: f"{v:.3g}", bold=None, idx_name=''):
    # booktabs table; bold: {'min'|'max': [columns]} — the best entry per column in bold
    esc = lambda s: str(s).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')
    best = {}
    for how, cols_ in (bold or {}).items():
        for c in cols_:
            v = pd.to_numeric(df[c], errors='coerce')
            if v.notna().any():
                best[c] = v.min() if how == 'min' else v.max()
    body = []
    for i, r in df.iterrows():
        cells = []
        for c in df.columns:
            v = r[c]
            if isinstance(v, (float, np.floating, int, np.integer)) and not isinstance(v, bool):
                s = fmt(float(v)) if np.isfinite(float(v)) else '--'
                if c in best and np.isfinite(float(v)) and float(v) == best[c]:
                    s = r"\textbf{" + s + "}"
            else:
                s = esc(v)
            cells.append(s)
        body.append(" & ".join([esc(i)] + cells) + r" \\")
    L_ = [r"\begin{table}[t]", r"\centering\footnotesize", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
          r"\resizebox{\linewidth}{!}{%", r"\begin{tabular}{l" + "r" * df.shape[1] + "}", r"\toprule",
          " & ".join([esc(idx_name)] + [esc(c) for c in df.columns]) + r" \\", r"\midrule"] + body + \
         [r"\bottomrule", r"\end{tabular}}", r"\end{table}", ""]
    p = rev_path('tables', f"{name}{SM}.tex")
    with open(p, 'w') as fh:
        fh.write("% generated by the revisions notebook — \\usepackage{booktabs,graphicx}\n" + "\n".join(L_))
    store.add(p)


def dataset_scores(lb, agg='geomean'):
    # one score per (dataset, model); models / datasets kept so the matrix is complete
    rows = {}
    for d, g in lb.groupby('dataset', sort=False):
        piv = g.pivot_table(index=['lag', 'h'], columns='model', values='rel_rmse' if agg == 'geomean' else 'rmse')
        piv = piv.loc[:, piv.notna().mean() >= STAT_MIN_COVER].dropna()
        if piv.empty:
            continue
        if agg == 'geomean':
            rows[d] = np.exp(np.log(piv).mean())
        else:                                                # mean within-problem rank
            rows[d] = piv.rank(axis=1, method='average').mean()
    S = pd.DataFrame(rows).T
    gone = [m for m in S.columns if S[m].isna().any()]
    if gone:
        print(f"(left out of this ranking — missing on some dataset: " + ", ".join(
            f"{m} [{', '.join(S.index[S[m].isna()])}]" for m in gone) + ")")
    S = S.loc[:, S.notna().all()]                            # models present on every dataset
    return S


def friedman_n(S):
    from scipy import stats as _st
    N, k = S.shape
    Rk = S.rank(axis=1, method='average')
    chi2, p = _st.friedmanchisquare(*[S[c].to_numpy() for c in S.columns])
    Ff = (N - 1) * chi2 / (N * (k - 1) - chi2) if N * (k - 1) - chi2 > 0 else np.inf
    p_id = float(_st.f.sf(Ff, k - 1, (k - 1) * (N - 1))) if np.isfinite(Ff) else 0.0
    q = _st.studentized_range.ppf(1 - STAT_ALPHA, k, np.inf) / np.sqrt(2)
    cd = q * np.sqrt(k * (k + 1) / (6.0 * N))
    return {'N': N, 'k': k, 'chi2': float(chi2), 'p': float(p), 'F_ID': float(Ff), 'p_ID': p_id, 'cd': float(cd),
            'mean_rank': Rk.mean().sort_values(), 'ranks': Rk}


def wilcoxon_holm(S, control):
    from scipy import stats as _st
    rows = []
    for m in S.columns:
        if m == control:
            continue
        d = np.log(S[m].to_numpy()) - np.log(S[control].to_numpy())    # > 0: control better
        if np.allclose(d, 0):
            rows.append({'model': m, 'W': np.nan, 'p': 1.0, 'wins': 0, 'losses': 0}); continue
        w = _st.wilcoxon(d, zero_method='zsplit')
        rows.append({'model': m, 'W': float(w.statistic), 'p': float(w.pvalue), 'wins': int((d > 0).sum()), 'losses': int((d < 0).sum()),
                     'geo-mean RMSE ratio (control / model)': float(np.exp(-d.mean()))})
    W_ = pd.DataFrame(rows).set_index('model')
    W_['p_holm'] = holm(W_['p'].to_numpy())
    W_['significant'] = W_['p_holm'] < STAT_ALPHA
    return W_.sort_values('p')


def cd_diagram(ax, fr, title):
    # Demšar-style CD diagram; labels on two sides with fixed spacing (no overlap), clique bars on separate rows
    mr, cd, k = fr['mean_rank'], fr['cd'], fr['k']
    vals = mr.to_numpy(); half = (k + 1) // 2
    cl = []
    for i in range(k):
        j = i
        while j + 1 < k and vals[j + 1] - vals[i] < cd:
            j += 1
        if j > i and not any(a <= i and j <= b for a, b in cl):
            cl.append((i, j))
    row_h, bar_h = 0.34, 0.14
    y_top = 0.0                                                # axis line
    y_bars0 = -0.25
    y_lab0 = y_bars0 - bar_h * max(1, len(cl)) - 0.25
    ax.set_xlim(0.2 - 0.1 * k, k + 0.8 + 0.1 * k)
    ax.set_ylim(y_lab0 - row_h * (half - 1) - 0.3, 0.95)
    ax.axis('off')
    ax.hlines(y_top, 1, k, color='k', lw=0.8)
    for r in range(1, k + 1):
        ax.vlines(r, y_top, y_top + 0.1, color='k', lw=0.8); ax.text(r, y_top + 0.17, str(r), ha='center', va='bottom', fontsize=6.5)
    ax.hlines(y_top + 0.62, 1, 1 + cd, color='k', lw=1.4)
    ax.vlines([1, 1 + cd], y_top + 0.57, y_top + 0.67, color='k', lw=1.0)
    ax.text(1 + cd / 2, y_top + 0.72, f"CD = {cd:.2f}", ha='center', va='bottom', fontsize=6.5)
    for n_, (i, j) in enumerate(cl):
        yy = y_bars0 - bar_h * n_
        ax.hlines(yy, vals[i] - 0.05, vals[j] + 0.05, color='k', lw=2.0)
    for i, (m, r) in enumerate(mr.items()):
        left = i < half
        row = i if left else (k - 1 - i)
        yy = y_lab0 - row * row_h
        xe = 0.65 if left else k + 0.35
        st_ = sty(m)
        ax.plot([r, r], [y_top, yy], color=st_['color'], lw=1.4 if m in OURS else 0.8, zorder=1)
        ax.plot([r, xe], [yy, yy], color=st_['color'], lw=1.4 if m in OURS else 0.8, zorder=1)
        ax.text(xe + (-0.06 if left else 0.06), yy, f"{short(m)} ({r:.2f})", ha='right' if left else 'left', va='center',
                fontsize=6.5, fontweight='bold' if m in OURS else 'normal')
    ax.set_title(title, loc='left', fontsize=7.5)


def _ranking_n11():
    core = LB[~LB['model'].isin(STAT_EXCLUDE)].copy()
    out, lines = {}, []
    S = dataset_scores(core)
    rev_csv(S.reset_index().rename(columns={'index': 'dataset'}), 'n11_dataset_scores_geomean_rel_rmse')
    FR = {'all': friedman_n(S)}
    for h in sorted(core['h'].unique()):
        Sh = dataset_scores(core[core['h'] == h])
        if Sh.shape[0] >= 3 and Sh.shape[1] >= 3:
            FR[f'h={h}'] = friedman_n(Sh)
    Sr = dataset_scores(core, agg='rank')
    FR['all (mean within-dataset rank)'] = friedman_n(Sr)
    for kind in ('chaotic', 'real'):
        Sk = S.loc[[d for d in S.index if KIND_OF.get(d) == kind]]
        if Sk.shape[0] >= 3:
            FR[kind] = friedman_n(Sk)
    # the old N = 198 analysis, for the response letter
    piv = core.pivot_table(index=['dataset', 'h', 'lag'], columns='model', values='rmse')
    piv = piv.loc[:, piv.notna().mean() >= 0.5].dropna()
    from scipy import stats as _st
    chi_old, p_old = _st.friedmanchisquare(*[piv[c].to_numpy() for c in piv.columns])
    meta = pd.DataFrame({k: {'N (blocks)': v['N'], 'k (models)': v['k'], 'Friedman chi2': v['chi2'], 'p': v['p'],
                             'Iman-Davenport F': v['F_ID'], 'p (F)': v['p_ID'], 'Nemenyi CD': v['cd']} for k, v in FR.items()}).T
    meta.loc['OLD: every (dataset, lag, h) as a block'] = {'N (blocks)': piv.shape[0], 'k (models)': piv.shape[1],
                                                         'Friedman chi2': chi_old, 'p': p_old, 'Iman-Davenport F': np.nan,
                                                         'p (F)': np.nan, 'Nemenyi CD': np.nan}
    rev_csv(meta.reset_index().rename(columns={'index': 'setting'}), 'n11_friedman')
    MR = pd.DataFrame({k: v['mean_rank'] for k, v in FR.items()})
    MR = MR.loc[FR['all']['mean_rank'].index]
    rev_csv(MR.reset_index().rename(columns={'index': 'model'}), 'n11_mean_ranks')
    WH = wilcoxon_holm(S, STAT_CONTROL) if STAT_CONTROL in S else pd.DataFrame()
    if len(WH):
        rev_csv(WH.reset_index(), 'n11_wilcoxon_holm_vs_trace_v2')
    a = FR['all']
    lines += [f"## Ranking over N = {a['N']} datasets (k = {a['k']} models; score = geo-mean RMSE/persistence over each dataset's problems)",
              f"Friedman χ²({a['k'] - 1}) = {a['chi2']:.2f}, p = {a['p']:.2e}; Iman–Davenport F({a['k'] - 1}, {(a['k'] - 1) * (a['N'] - 1)}) = "
              f"{a['F_ID']:.2f}, p = {a['p_ID']:.2e}; Nemenyi CD (α = {STAT_ALPHA}) = {a['cd']:.2f}",
              f"(previously, with each of the {piv.shape[0]} (dataset, lag, horizon) problems as a block: χ² = {chi_old:.1f}, p = {p_old:.1e})",
              "", "Mean ranks:", MR.to_string(float_format=lambda v: f"{v:.2f}"), "",
              "Friedman by setting:", meta.to_string(float_format=lambda v: f"{v:.3g}")]
    if len(WH):
        lines += ["", f"{STAT_CONTROL} vs each model over the {a['N']} datasets — two-sided Wilcoxon signed-rank on log score, Holm:",
                  WH.to_string(float_format=lambda v: f"{v:.3g}")]
        top = a['mean_rank'].index[0]
        lines.append(f"Best mean rank: {top} ({a['mean_rank'].iloc[0]:.2f}). {STAT_CONTROL} is significantly better (Holm, α = {STAT_ALPHA}) "
                     f"than {int(WH['significant'].sum())} of {len(WH)} models: "
                     + ", ".join(m for m in WH.index[WH['significant'] & (WH['wins'] > WH['losses'])]))
    # tables + figures
    t = MR.copy(); t.index = [short(m) for m in t.index]
    t.columns = [c.replace('all (mean within-dataset rank)', 'All (rank-avg.)').replace('all', 'All') for c in t.columns]
    tex_table(t, 'tab_n11_mean_ranks', f"Mean rank over the N = {a['N']} datasets (lower is better). Each dataset contributes one "
              f"score per model (geometric mean over its lags and horizons of RMSE relative to persistence). Friedman "
              f"$\\chi^2 = {a['chi2']:.1f}$, $p = {a['p']:.1e}$; Iman--Davenport $p = {a['p_ID']:.1e}$; Nemenyi CD = {a['cd']:.2f} at "
              f"$\\alpha = {STAT_ALPHA}$.", 'tab:n11_ranks', fmt=lambda v: f"{v:.2f}", bold={'min': list(t.columns)}, idx_name='Model')
    if len(WH):
        w_ = WH[['geo-mean RMSE ratio (control / model)', 'wins', 'losses', 'p', 'p_holm']].copy()
        w_.columns = ['RMSE ratio', 'datasets better', 'datasets worse', '$p$', '$p_{\\mathrm{Holm}}$']
        w_.index = [short(m) for m in w_.index]
        tex_table(w_, 'tab_n11_wilcoxon', f"{STAT_CONTROL} against each model over the N = {a['N']} datasets: Wilcoxon signed-rank test "
                  "on the per-dataset log scores, Holm-corrected. RMSE ratio $<1$: " + STAT_CONTROL + " is more accurate.",
                  'tab:n11_wilcoxon', fmt=lambda v: f"{v:.3g}", idx_name='vs.')
    k = a['k']
    fig, ax = plt.subplots(figsize=(W2, 1.2 + 0.19 * k))
    cd_diagram(ax, a, f"Mean rank over N = {a['N']} datasets ({k} models; Friedman p = {a['p']:.1e}; bars: no significant "
                      f"difference, Nemenyi α = {STAT_ALPHA})")
    save_fig(fig, 'fig_cd_n11')
    hk = [kk for kk in FR if kk.startswith('h=')]
    if hk:
        fig, axes = plt.subplots(len(hk), 1, figsize=(W2, (1.2 + 0.19 * k) * len(hk)), squeeze=False)
        for ax, kk in zip(axes[:, 0], hk):
            cd_diagram(ax, FR[kk], f"Horizon {kk[2:]} — N = {FR[kk]['N']} datasets (Friedman p = {FR[kk]['p']:.1e})")
        fig.tight_layout(); save_fig(fig, 'fig_cd_n11_by_horizon')
    # chaotic vs real margin, and who wins on real series at h = 24
    ctrl = core[core['model'] == STAT_CONTROL].set_index(['dataset', 'lag', 'h'])['rmse']
    best_rival = (core[core['model'] != STAT_CONTROL].groupby(['dataset', 'lag', 'h'])['rmse'].min())
    mg = pd.concat({'trace': ctrl, 'best_rival': best_rival}, axis=1).dropna()
    mg['ratio'] = mg['trace'] / mg['best_rival']
    MGd = mg.groupby(level=['dataset', 'h'])['ratio'].apply(lambda r: float(np.exp(np.mean(np.log(r))))).unstack('h')
    MGd['type'] = [KIND_OF.get(d, '') for d in MGd.index]
    rev_csv(MGd.reset_index(), 'margin_vs_best_rival_by_dataset')
    by_type = mg.assign(type=[KIND_OF.get(d, '') for d in mg.index.get_level_values(0)]).groupby(['type', mg.index.get_level_values('h')])['ratio'] \
                .apply(lambda r: float(np.exp(np.mean(np.log(r))))).unstack()
    S24 = dataset_scores(core[core['h'] == 24]) if (core['h'] == 24).any() else pd.DataFrame()
    win24 = S24.idxmin(axis=1) if len(S24) else pd.Series(dtype=str)
    lines += ["", f"## Where {STAT_CONTROL} wins by most: RMSE / best rival's RMSE (geo-mean over lags; < 1: {STAT_CONTROL} best)",
              MGd.to_string(float_format=lambda v: f"{v:.3f}"), "", "By series type:", by_type.to_string(float_format=lambda v: f"{v:.3f}")]
    if len(win24):
        lines += ["", "Best model per dataset at h = 24 (geo-mean over lags of RMSE/persistence):",
                  "\n".join(f"  {d} ({KIND_OF.get(d, '')}): {m} — {S24.loc[d, m]:.3f}" + (f" | {STAT_CONTROL}: {S24.loc[d, STAT_CONTROL]:.3f}"
                                                                                          if STAT_CONTROL in S24 else '')
                            for d, m in win24.items())]
    out = {'FR': FR, 'S': S, 'WH': WH, 'meta': meta, 'margin': MGd, 'margin_by_type': by_type, 'win24': win24, 'S24': S24}
    txt = "\n".join(lines)
    print(txt)
    with open(rev_path(f"n11_statistics{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"n11_statistics{SM}.md"))
    out['text'] = txt
    return out


STATS = _ranking_n11()
store.push('revisions: N = 11 statistics', force=True)

In [ ]:
# ============================================================
# 15. ENERGY — per-member vs ensemble inference (one TRACE-SNP model vs a 60-seed network ensemble)
#     Every network accuracy in the paper is the mean of 60 seeds, so one reported forecast costs 60 member forward
#     passes; TRACE-SNP is one closed-form model, so its member and its "ensemble" are the same thing. The table therefore
#     shows both columns for every model, measured (GPU) and estimated (45 nm, Horowitz 2014: 4.6 pJ / MAC, 0.9 pJ / AC):
#       per-member inference energy   one model, one forecast            → the like-for-like architecture comparison
#       ensemble inference energy     what one REPORTED forecast cost    → members × per-member
#     Medians over datasets × lags at h = 1 (the measured benchmark), training energy for context.
#     Then TRACE-SNP v1 vs v2 inference cost per horizon (measured and operation counts), with the reasons it differs.
#     Caveat written into every caption: the 45 nm figure counts arithmetic (MAC / AC) only and abstracts away memory
#     access (weight / state fetches), which on real hardware can dominate; it is an estimate, not a measurement.
# ============================================================
PJ_MAC, PJ_AC = 4.6, 0.9
K_ACTIVE, SPIKE_Q_V1 = 4, 8          # TRACE-SNP v1: k_active state neurons, q spikes per delay (cell 8)
NET_MEMBERS = {s['name']: s['n_runs'] for s in MODEL_SPECS}


def _energy():
    lines = []
    EN = comp('energy_measured')
    OPS = comp('ops_neuromorphic')
    if EN.empty:
        print("No energy_measured results in the results folder."); return None
    EN = EN[EN['dataset'].isin(ALL11_NAMES) & ~EN['model'].isin(STAT_EXCLUDE)].copy()
    EN['members'] = EN['model'].map(NET_MEMBERS).fillna(1).astype(int)
    meas = 'infer_j' if EN['infer_j'].fillna(0).gt(0).any() else 'infer_s'      # no GPU meter → seconds
    unit, scale = ('µJ', 1e6) if meas == 'infer_j' else ('µs', 1e6)
    EN['member'] = EN[meas] * scale
    EN['ensemble'] = EN['member'] * EN['members']          # recomputed from the member figure, so both columns are consistent
    T_ = EN.groupby('model').agg(members=('members', 'first'), member=('member', 'median'), ensemble=('ensemble', 'median'),
                                 train=('train_j' if meas == 'infer_j' else 'train_s', 'median'))
    if len(OPS):
        O = OPS[OPS['dataset'].isin(ALL11_NAMES)].copy()
        O['members'] = O['model'].map(NET_MEMBERS).fillna(1).astype(int)
        O['pJ_ens'] = O['pJ'] * O['members']
        T_ = T_.join(O.groupby('model').agg(pj_member=('pJ', 'median'), pj_ens=('pJ_ens', 'median'),
                                            mac=('MAC', 'median'), ac=('AC', 'median')), how='left')
    ref = 'TRACE-SNP v2' if 'TRACE-SNP v2' in T_.index else T_['member'].idxmin()
    T_['member × ' + ref] = T_['member'] / T_.loc[ref, 'member']
    T_['ensemble × ' + ref] = T_['ensemble'] / T_.loc[ref, 'ensemble']
    T_ = T_.sort_values('member')
    nice = {'members': 'Ensemble members', 'member': f'Per-member inference ({unit}/forecast, GPU)',
            'ensemble': f'Ensemble inference ({unit}/reported forecast, GPU)',
            'pj_member': 'Per-member, 45 nm (pJ)', 'pj_ens': 'Ensemble, 45 nm (pJ)', 'mac': 'MAC / forecast (member)',
            'ac': 'AC / forecast (member)', 'train': 'Training energy (J, all folds and seeds)' if meas == 'infer_j' else 'Training time (s)'}
    out = T_.rename(columns=nice)
    rev_csv(out.reset_index(), 'energy_per_member_vs_ensemble')
    lines += [f"## Inference energy: per member vs ensemble (median over datasets × lags, h = 1; {meas})",
              out.to_string(float_format=lambda v: f"{v:.3g}")]
    main_cols = [nice[c] for c in ('members', 'member', 'ensemble', 'pj_member', 'pj_ens', 'train') if nice[c] in out]
    tt = out[main_cols].copy(); tt.index = [short(m) for m in tt.index]
    tex_table(tt, 'tab_energy_member_vs_ensemble',
              "Inference energy per forecast, per member model and per reported (ensemble) forecast; medians over datasets and "
              "lags at $h=1$. Networks report the mean of " + f"{max(NET_MEMBERS.values()) if NET_MEMBERS else 60}" + " seeds, so an "
              "ensemble forecast costs members $\\times$ per-member; TRACE-SNP is a single closed-form model (one member). GPU: "
              "measured board energy above idle in a large-batch benchmark. 45\\,nm: operation counts $\\times$ 4.6\\,pJ/MAC and "
              "0.9\\,pJ/AC (Horowitz, 2014); spiking networks' ACs scaled by their measured firing rate. The 45\\,nm estimate counts "
              "arithmetic only and abstracts away memory-access (weight and state fetch) energy.",
              'tab:energy_dual', fmt=lambda v: f"{v:,.3g}", bold={'min': main_cols[1:]}, idx_name='Model')

    # ── TRACE-SNP v1 vs v2 inference cost, per horizon ──
    e1 = read_many(f"*/closed_form/cf{SM}_v2_lag*_h*_energy.csv")
    e2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_energy.csv")
    c1 = read_many(f"*/closed_form/cf{SM}_v2_lag*_h*_choices.csv")
    o2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_ops.csv")
    c2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_choices.csv")
    V = {}
    if len(e1):
        e1 = e1[(e1['model'] == 'TRACE-SNP') & e1['dataset'].isin(ALL11_NAMES)]
        V['v1 measured (µJ/forecast)'] = e1.groupby('h')['infer_j_per_forecast'].median() * 1e6
        V['v1 measured (µs/forecast)'] = e1.groupby('h')['infer_s_per_forecast'].median() * 1e6
    if len(e2):
        e2 = e2[(e2['model'] == 'TRACE-SNP v2') & e2['dataset'].isin(ALL11_NAMES)]
        V['v2 measured (µJ/forecast)'] = e2.groupby('h')['infer_j_per_forecast'].median() * 1e6
        V['v2 measured (µs/forecast)'] = e2.groupby('h')['infer_s_per_forecast'].median() * 1e6
    if len(c1):
        c1 = c1[(c1['model'] == 'TRACE-SNP') & c1['dataset'].isin(ALL11_NAMES)].copy()

        def ops_v1(r):        # the analysis cell's exact count for TRACE-SNP v1 (direct forecast, one model per horizon)
            fam, L = str(r['family']), float(r['lag'])
            if fam.startswith('persistence'):
                return 0.0, 0.0
            if fam.startswith('ridge'):
                return L, 0.0
            mac = 2.0 * L + L + 2.0 * L
            if 'state' in fam and np.isfinite(r.get('C', np.nan)):
                d_, C_ = float(r['d']), float(r['C'])
                mac += C_ * d_ + K_ACTIVE * (d_ + 1) + K_ACTIVE
            return mac, 0.0
        c1[['MAC', 'AC']] = c1.apply(lambda r: pd.Series(ops_v1(r)), axis=1)
        c1['pJ'] = PJ_MAC * c1['MAC'] + PJ_AC * c1['AC']
        V['v1 MAC'] = c1.groupby('h')['MAC'].median(); V['v1 45 nm (pJ)'] = c1.groupby('h')['pJ'].median()
    if len(o2):
        o2 = o2[(o2['model'] == 'TRACE-SNP v2') & o2['dataset'].isin(ALL11_NAMES)]
        V['v2 MAC'] = o2.groupby('h')['MAC'].median(); V['v2 AC'] = o2.groupby('h')['AC'].median()
        V['v2 45 nm (pJ)'] = o2.groupby('h')['pJ'].median()
    if len(c2):
        c2 = c2[c2['dataset'].isin(ALL11_NAMES)]
        V['v2 share iterated'] = c2.groupby('h')['iterated'].mean()
        V['v2 share window-mean anchor'] = c2.groupby('h')['anchor'].apply(lambda a: float((a == 'mean').mean())) if 'anchor' in c2 else np.nan
    VV = pd.DataFrame(V)
    if len(VV):
        rev_csv(VV.reset_index(), 'trace_v1_vs_v2_inference')
        lines += ["", "## TRACE-SNP v1 vs v2 inference cost per horizon (medians over datasets × lags)", VV.to_string(float_format=lambda v: f"{v:.3g}"),
                  "Why they differ:",
                  " • v2 may forecast h > 1 by iterating its one-step model h times (chosen on validation with a 25 % margin): "
                  "those forecasts cost h one-step evaluations; v1 always uses one direct readout per horizon.",
                  " • v2 can anchor on the window mean (L extra additions) and use difference inputs (L extra subtractions).",
                  " • Measured GPU numbers use different benchmark batches: v1 timed calls on one test block (≈ block-length "
                  "forecasts per call, launch-overhead bound), v2 on ≥ 18,000 forecasts per call like the networks — compare the "
                  "operation counts (hardware-independent) for the architectural difference."]
        tv = VV.copy(); tv.index = [f"h = {h}" for h in tv.index]
        tex_table(tv, 'tab_trace_v1_vs_v2_inference', "TRACE-SNP v1 and v2: inference cost per forecast by horizon (medians over "
                  "datasets and lags). v2 forecasts some $h>1$ problems by iterating its one-step model (share shown), which costs "
                  "$h$ evaluations. 45\\,nm estimates count arithmetic only (no memory access).", 'tab:trace_v1_v2',
                  fmt=lambda v: f"{v:,.3g}", idx_name='Horizon')
    txt = "\n".join(lines)
    print(txt)
    with open(rev_path(f"energy{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"energy{SM}.md"))
    return {'table': out, 'v1_v2': VV, 'text': txt}


ENERGY = _energy()
store.push('revisions: energy tables', force=True)

In [ ]:
# ============================================================
# 16. FIGURES (PDF + PNG in <LOCAL_ROOT>/revisions/figures/)
#     All figures: colour-blind-safe Okabe–Ito colours, each model also distinguished by marker / line style, our models
#     drawn thicker; text never relies on colour alone.
#       fig_cd_n11, fig_cd_n11_by_horizon    the N = 11 CD diagrams (cell 14) 
#       fig_delay_functions                  the learned delay functions; the Mackey-Glass panel always shows k = 18 and
#                                            its legend names exactly the delays drawn 
#       fig_horizon_profile                  error growth with horizon, colour-blind safe
#       fig_accuracy_vs_energy               N = 11 mean rank vs per-member AND ensemble inference energy, labels placed so
#                                            they never overlap (leader lines where moved)
#       fig_accuracy_vs_noise, fig_noise_degradation_heatmap   colour-blind safe (sequential cividis instead of red–green)
#       fig_rbf_ablation                     per-dataset RMSE ratio, non-gated RBF / gated
#     Also checks the noise claim (is ALSTM-SNP the most accurate model at 10 % input noise?) from the stored
#     noise study, and says plainly if the data do not support it.
# ============================================================
FIG4_DATASETS   = ['MackeyGlass', 'mackey_glass', 'mackeyglass', 'lorenz', 'Sunspots', 'sunspot_monthly', 'co2_weekly', 'etth1']
FIG4_FORCE      = {'mackey': 18}      # dataset-name keyword → delay that must be drawn (and named) in that panel
FIG4_LAG        = 20                  # window length shown (must be ≥ every forced delay)
FIG4_N_DELAYS   = 4                   # delays drawn per panel (forced ones first, then the largest-amplitude ones)


def place_labels(ax, xs, ys, labels, fontsize=6, bold=(), dx=6):
    # labels beside their points (right of points in the left half of the axes, left of points in the right half);
    # placed top to bottom, each moved down just enough to clear the labels already placed on its side (greedy sweep in
    # display space), with a leader line when a label had to move. Offsets in points, so they survive savefig(dpi=…).
    fig = ax.figure
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    px2pt = 72.0 / fig.dpi
    pts = ax.transData.transform(np.column_stack([xs, ys]))
    xmid = ax.transAxes.transform((0.5, 0.5))[0]
    order = np.argsort(-pts[:, 1])
    placed, texts = [], []
    for i in order:
        side = 1 if pts[i, 0] < xmid else -1
        kw = dict(textcoords='offset points', fontsize=fontsize, va='center', ha='left' if side > 0 else 'right',
                  fontweight='bold' if labels[i] in bold else 'normal')
        t = ax.annotate(labels[i], (xs[i], ys[i]), xytext=(side * dx, 0), **kw)
        bb = t.get_window_extent(r)
        w, h = bb.width, bb.height * 1.2
        x0 = pts[i, 0] + side * dx / px2pt - (w if side < 0 else 0)
        y = pts[i, 1]
        moved = True
        while moved:
            moved = False
            for (px0, px1, py0, py1) in placed:
                if x0 < px1 and x0 + w > px0 and y - h / 2 < py1 and y + h / 2 > py0:
                    y = py0 - h / 2 - 0.5; moved = True
        placed.append((x0, x0 + w, y - h / 2, y + h / 2))
        dy = y - pts[i, 1]
        if abs(dy) > 1:
            t.remove()
            t = ax.annotate(labels[i], (xs[i], ys[i]), xytext=(side * (dx + 6), dy * px2pt),
                            arrowprops=dict(arrowstyle='-', lw=0.4, color='#666666', shrinkA=0, shrinkB=3), **kw)
        texts.append(t)
    return texts


def _figures():
    lines = []
    # ── Fig. 4: delay functions, with k = 18 for Mackey-Glass ──
    CV = read_many(f"*/closed_form/cf{SM}_v2_lag*_h1_curves.csv")
    if len(CV):
        pick = [d for d in FIG4_DATASETS if d in set(CV['dataset'])][:4]
        fig, axes = plt.subplots(1, len(pick), figsize=(W2, 2.3), squeeze=False)
        dcols = [OI['vermillion'], OI['blue'], OI['green'], OI['purple'], OI['orange']]
        dls = ['-', '--', '-.', ':', '-']
        for ax, d in zip(axes[0], pick):
            c = CV[CV['dataset'] == d]
            L = FIG4_LAG if FIG4_LAG in set(c['lag']) else int(c['lag'].max()); c = c[c['lag'] == L]
            amp = c.groupby('delay')['f'].apply(lambda f: f.max() - f.min()).sort_values(ascending=False)
            forced = [k for kw, k in FIG4_FORCE.items() if kw in d.lower() and k in amp.index]
            for kw, k in FIG4_FORCE.items():
                if kw in d.lower():
                    rk = list(amp.index).index(k) + 1 if k in amp.index else None
                    lines.append(f"Fig. 4 check — {d}, L = {L}: delay k = {k} has amplitude rank {rk} of {len(amp)}"
                                 + (f" ({amp[k]:.4g}; largest: k = {amp.index[0]} with {amp.iloc[0]:.4g})" if rk else " (not available at this L)"))
                    for L2, c2 in CV[CV['dataset'] == d].groupby('lag'):
                        a2 = c2.groupby('delay')['f'].apply(lambda f: f.max() - f.min()).sort_values(ascending=False)
                        if k in a2.index:
                            lines.append(f"    L = {L2}: k = {k} rank {list(a2.index).index(k) + 1}; top delays {list(a2.index[:4])}")
            show_k = forced + [k for k in amp.index if k not in forced][:FIG4_N_DELAYS - len(forced)]
            for i_, k_ in enumerate(show_k):
                cc = c[c['delay'] == k_].sort_values('u')
                ax.plot(cc['u'], cc['f'] - cc['f'].mean(), lw=1.6 if k_ in forced else 1.0, color=dcols[i_ % 5], ls=dls[i_ % 5],
                        label=f"k = {k_}" + (" (τ+1)" if k_ in forced and 'mackey' in d.lower() else ''))
            ax.set_title(f"{d} (L = {L})", loc='left', fontsize=7)
            ax.legend(frameon=False, fontsize=5.5, handlelength=1.8, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.32),
                      columnspacing=0.8)
            ax.set_xlabel(str(c['input'].iloc[0]).replace('t-k', 't−k').replace('t-1', 't−1') if 'input' in c else 'x(t−k)', fontsize=6.5)
            ax.tick_params(labelsize=5.5)
        axes[0, 0].set_ylabel('contribution $f_k$ (centred)')
        fig.tight_layout(); save_fig(fig, 'fig_delay_functions')
    else:
        lines.append("Fig. 4: no closed_form/*_h1_curves.csv in the results folder — not regenerated.")

    # ── horizon profile (colour-blind safe) ──
    core = LB[~LB['model'].isin(STAT_EXCLUDE)]
    HP = core.groupby(['dataset', 'h', 'model'])['rel_rmse'].apply(lambda r: float(np.exp(np.mean(np.log(r))))).reset_index()
    hs = sorted(HP['h'].unique())
    if len(hs) > 1:
        ds = [d for d in ALL11_NAMES if d in set(HP['dataset'])]
        nc = 4; nr = int(np.ceil(len(ds) / nc))
        fig, axes = plt.subplots(nr, nc, figsize=(W2, 1.55 * nr + 0.5), squeeze=False)
        order = list(STATS['FR']['all']['mean_rank'].index) if STATS else sorted(HP['model'].unique())
        for ax, d in zip(axes.ravel(), ds):
            t = HP[HP['dataset'] == d].pivot(index='h', columns='model', values='rel_rmse')
            for m in [m for m in order if m in t.columns][::-1]:
                st_ = sty(m)
                ax.plot(t.index, t[m], color=st_['color'], ls=st_['ls'], marker=st_['marker'], ms=2.6 if m in OURS else 2.0,
                        lw=st_['lw'], zorder=5 if m in OURS else 2, label=short(m))
            ax.set_yscale('log'); ax.set_xticks(hs); ax.axhline(1, color='k', lw=0.4)
            ax.set_title(f"{d} ({KIND_OF.get(d, '')})", loc='left', fontsize=7); ax.tick_params(labelsize=5.5)
        for ax in axes.ravel()[len(ds):]:
            ax.axis('off')
        for r_ in range(nr):
            axes[r_, 0].set_ylabel('RMSE / persistence', fontsize=6.5)
        hh, ll = axes[0, 0].get_legend_handles_labels()
        o_ = sorted(range(len(ll)), key=lambda i: order.index(next(m for m in order if short(m) == ll[i])) if any(short(m) == ll[i] for m in order) else 99)
        fig.legend([hh[i] for i in o_], [ll[i] for i in o_], loc='lower center', ncol=5, frameon=False, fontsize=6,
                   bbox_to_anchor=(0.5, -0.09))
        fig.tight_layout(); save_fig(fig, 'fig_horizon_profile')

    # ── accuracy vs energy: N = 11 mean rank against per-member (filled) and ensemble (open) inference energy ──
    if ENERGY is not None and STATS is not None:
        T_ = ENERGY['table']; mr = STATS['FR']['all']['mean_rank']
        cm = [c for c in T_.columns if c.startswith('Per-member inference')][0]
        ce = [c for c in T_.columns if c.startswith('Ensemble inference')][0]
        pts = [(m, T_.loc[m, cm], T_.loc[m, ce], mr[m]) for m in mr.index if m in T_.index and np.isfinite(T_.loc[m, cm])]
        if pts:
            fig, ax = plt.subplots(figsize=(W2 * 0.62, 2.9))
            for m, em, ee, r in pts:
                st_ = sty(m)
                if ee > em * 1.0001:
                    ax.plot([em, ee], [r, r], color=st_['color'], lw=0.6, ls=':', zorder=1)
                    ax.scatter(ee, r, s=22, facecolor='white', edgecolor=st_['color'], lw=1.0, marker=st_['marker'], zorder=3)
                ax.scatter(em, r, s=34 if m in OURS else 20, color=st_['color'], edgecolor='k', lw=0.3, marker=st_['marker'], zorder=4)
            ax.set_xscale('log'); ax.invert_yaxis(); ax.grid(True, alpha=0.25, lw=0.4)
            ax.set_xlabel(cm.replace('Per-member inference', 'inference energy').replace(', GPU)', ', GPU; filled: one member, open: ensemble)'))
            ax.set_ylabel(f"mean rank over N = {STATS['FR']['all']['N']} datasets")
            ax.set_title('Accuracy vs inference energy (upper left is better)', loc='left')
            place_labels(ax, [p[1] for p in pts], [p[3] for p in pts], [short(p[0]) for p in pts], fontsize=5.8, bold=[short(m) for m in OURS])
            save_fig(fig, 'fig_accuracy_vs_energy')
            rev_csv(pd.DataFrame(pts, columns=['model', 'per_member', 'ensemble', 'mean_rank_n11']), 'accuracy_vs_energy')

    # ── noise robustness (colour-blind safe) + the noise claim check ──
    NP_ = comp('pooled_rmse', group='noise')
    if len(NP_):
        NP_ = NP_[NP_['dataset'].isin(ALL11_NAMES) & ~NP_['model'].isin(STAT_EXCLUDE)] if 'dataset' in NP_ else NP_
        levels = sorted(NP_['noise'].unique()); top = levels[-1]
        acc = NP_.groupby(['model', 'noise'])['ratio_to_clean_persistence'].median().unstack('noise')
        acc = acc.sort_values(top)
        rev_csv(acc.reset_index(), 'noise_accuracy_median')
        lines += ["", f"## Input-noise claim check: RMSE / clean persistence, median over datasets × lags × horizons",
                  acc.to_string(float_format=lambda v: f"{v:.3f}")]
        if 'dataset' in NP_:
            Sn = NP_[NP_['noise'] == top].groupby(['dataset', 'model'])['ratio_to_clean_persistence'] \
                    .apply(lambda r: float(np.exp(np.mean(np.log(r))))).unstack('model')
            Sn = Sn.loc[:, Sn.notna().all()]
            if Sn.shape[0] >= 3 and Sn.shape[1] >= 3:
                frn = friedman_n(Sn)
                lines += [f"At {top:.0%} noise, mean rank over N = {frn['N']} datasets (Friedman p = {frn['p']:.2e}, CD = {frn['cd']:.2f}):",
                          frn['mean_rank'].to_string(float_format=lambda v: f"{v:.2f}")]
                best_rank = frn['mean_rank'].index[0]
                rev_csv(frn['mean_rank'].rename('mean_rank').reset_index(), f"noise_{int(round(top * 100))}pct_mean_rank_n11")
            else:
                best_rank = None
        else:
            best_rank = None
        best_med = acc[top].idxmin()
        ok = best_med == 'ALSTM-SNP' and (best_rank in (None, 'ALSTM-SNP'))
        lines.append(f"CLAIM 'ALSTM-SNP is the most accurate model at {top:.0%} input noise': "
                     + ("SUPPORTED" if ok else "NOT SUPPORTED as stated")
                     + f" — lowest median error: {best_med}" + (f"; best mean rank over datasets: {best_rank}" if best_rank else '')
                     + ("" if ok else ". Reword it (e.g. 'among the networks' / 'degrades least') to what the numbers show."))
        deg = NP_[NP_['noise'] > 0].groupby(['model', 'noise'])['ratio_to_clean'].median().unstack('noise') if 'ratio_to_clean' in NP_ else None
        if deg is not None:
            nets_only = deg.loc[[m for m in deg.index if m in NET_MEMBERS]]
            lines.append(f"Least degradation at {top:.0%} (own noisy / own clean RMSE, median): overall {deg[top].idxmin()} "
                         f"({deg[top].min():.3f}); among networks {nets_only[top].idxmin() if len(nets_only) else '—'}")
        hs_n = sorted(NP_['h'].unique()) if 'h' in NP_ else [1]
        fig, axes = plt.subplots(1, len(hs_n), figsize=(W2, 2.4), squeeze=False)
        for ax, h in zip(axes[0], hs_n):
            t = (NP_[NP_['h'] == h] if 'h' in NP_ else NP_).groupby(['model', 'noise'])['ratio_to_clean_persistence'].median().unstack('noise')
            for m in t.index:
                st_ = sty(m)
                ax.plot(range(len(t.columns)), t.loc[m].values, color=st_['color'], ls=st_['ls'], marker=st_['marker'], ms=2.4,
                        lw=st_['lw'], label=short(m), zorder=5 if m in OURS else 2)
            ax.set_xticks(range(len(t.columns))); ax.set_xticklabels(['0'] + [f"{v:.1%}" for v in t.columns[1:]], fontsize=5.5)
            ax.set_yscale('log'); ax.set_title(f"h = {h}", loc='left'); ax.axhline(1, color='k', lw=0.4)
            ax.set_xlabel('input noise (σ / std of inputs)', fontsize=6)
        axes[0, 0].set_ylabel('RMSE / clean persistence')
        hh, ll = axes[0, 0].get_legend_handles_labels()
        fig.legend(hh, ll, loc='lower center', ncol=5, frameon=False, fontsize=5.8, bbox_to_anchor=(0.5, -0.2))
        fig.tight_layout(); save_fig(fig, 'fig_accuracy_vs_noise')
        if 'dataset' in NP_ and 'ratio_to_clean' in NP_:
            hm = NP_[NP_['noise'] == top].groupby(['model', 'dataset'])['ratio_to_clean'].median().unstack('dataset')
            hm = hm.loc[hm.median(axis=1).sort_values().index, [d for d in ALL11_NAMES if d in hm.columns]]
            fig, ax = plt.subplots(figsize=(W2, 0.27 * len(hm) + 1.2))
            Z = np.log10(hm.to_numpy(float))
            im = ax.imshow(Z, cmap='cividis', aspect='auto', vmin=0, vmax=max(0.05, float(np.nanmax(Z))))
            ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns, rotation=35, ha='right')
            ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels([short(m) for m in hm.index])
            vmax = max(0.05, float(np.nanmax(Z)))
            for i in range(hm.shape[0]):
                for j in range(hm.shape[1]):
                    if np.isfinite(hm.iat[i, j]):
                        ax.text(j, i, f"{hm.iat[i, j]:.2f}", ha='center', va='center', fontsize=5,
                                color='white' if Z[i, j] < 0.5 * vmax else 'black')
            fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01, label='log$_{10}$(noisy / clean RMSE)')
            ax.set_title(f"Degradation at {top:.0%} input noise", loc='left'); save_fig(fig, 'fig_noise_degradation_heatmap')
    else:
        lines.append("Noise study: comparison/final_noise_pooled_rmse.csv not found — the noise claim could not be checked here.")

    # ── RBF ablation figure ──
    if RBF is not None:
        b = RBF['by_dataset']
        fig, ax = plt.subplots(figsize=(W1, 0.2 * len(b) + 0.9))
        y_ = np.arange(len(b))
        ax.barh(y_ - 0.18, b.iloc[:, 0], 0.36, color=OI['vermillion'], label='full model (with fallbacks)')
        ax.barh(y_ + 0.18, b['ratio, no fallback'], 0.36, color=OI['blue'], label='receptive fields only (no fallback)')
        ax.axvline(1, color='k', lw=0.6); ax.set_yticks(y_); ax.set_yticklabels([f"{d} ({KIND_OF.get(d, '')[0] if KIND_OF.get(d) else ''})" for d in b.index])
        ax.invert_yaxis(); ax.set_xscale('log')
        ax.set_xlabel('RMSE non-gated RBF / RMSE gated (> 1: firing condition helps)')
        ax.legend(frameon=False, fontsize=5.8, loc='lower right')
        save_fig(fig, 'fig_rbf_ablation')
        t = b.copy(); t.index = [str(i) for i in t.index]
        tex_table(t, 'tab_rbf_ablation', "Firing-condition ablation. TRACE-SNP v2 with its gated receptive fields (2 of $M$ "
                  "triangular fields per delay; $k=4$ of $C$ state neurons fire) against the same model with a non-gated dense "
                  "Gaussian RBF basis on the same centres (all neurons active; width selected by the same criterion), fitted by "
                  "the identical convex solver and selection rule. Ratio $>1$: the gated model has the lower RMSE (geometric mean "
                  "over lags and horizons). Counts: Diebold--Mariano tests per (lag, horizon), Holm within dataset, $\\alpha=0.05$. "
                  f"Overall ratio {RBF['overall']:.3f}" + (f"; one-sided Wilcoxon over datasets $p = {RBF['wilcoxon'].pvalue:.3g}$" if RBF['wilcoxon'] is not None else '') + ".",
                  'tab:rbf_ablation', fmt=lambda v: f"{v:.3f}", idx_name='Dataset')
    txt = "\n".join(lines)
    print(txt)
    with open(rev_path(f"figures_and_claims{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"figures_and_claims{SM}.md"))
    return {'text': txt}


FIGS = _figures()
store.push('revisions: figures', force=True)

In [ ]:
# ============================================================
# 17. WHAT DOES THE FIRING CONDITION BUY? — receptive-field SHAPE × GATING, and accuracy vs number of active neurons
#     Cell 13 compared two things at once: triangular fields that are gated (2 of M fire) against Gaussian fields that are
#     dense (all M fire). This cell separates the two factors, every variant through the identical solver, grids,
#     encodings, anchors, LOO + validation selection and iterated/direct rule of TRACE-SNP v2:
#        variant       field shape                 firing condition per delay        state pathway
#        tri  (v2)     piecewise-linear (hat)      gated: the 2 nearest fire         gated: 4 nearest
#        hat_dense     piecewise-linear, wide      none: all M active                dense: all C
#        gauss_k2      Gaussian                    gated: the 2 nearest fire         gated: 4 nearest
#        gauss_k4      Gaussian                    gated: the 4 nearest fire         gated: 4 nearest
#        gauss_k8      Gaussian                    gated: the 8 nearest fire         gated: 4 nearest
#        gauss_dense   Gaussian                    none: all M active (= cell 13)    dense: all C
#     Gaussian widths w ∈ RBF_WIDTHS (× centre spacing) are chosen by the same criterion for every Gaussian variant.
#     Gated Gaussian = the dense Gaussian with every field outside the k nearest set to zero (a pure firing condition,
#     no renormalisation). Reading: tri vs gauss_k2 isolates the field SHAPE at equal firing; gauss_k2/k4/k8 vs
#     gauss_dense isolates the FIRING CONDITION at equal shape; hat_dense vs tri isolates gating for the hat shape.
#     Reported: RMSE of every variant relative to the dense Gaussian and to TRACE-SNP v2, per dataset, chaotic vs real;
#     active neurons and readout operations per forecast; one-sided Wilcoxon and Bayesian signed-rank tests (ROPE) over
#     the 11 datasets. Resumable: <LOCAL_ROOT>/revisions/basis_study/<dataset>_lag<L>_*.csv.
#     Cost: ≈ 3–4 × cell 13 on a GPU (several hours for all 11 datasets); rerun after a kernel death continues.
# ============================================================
BS_FOLDER   = 'basis_study'
BS_LAGS     = ABL_LAGS
BS_HORIZONS = ABL_HORIZONS
BS_FRESH    = False
BASES = {   # name: (shape, k active per delay or None = dense, state gated?)
    'tri':         ('tri',   2,    True),
    'hat_dense':   ('hat',   None, False),
    'gauss_k2':    ('gauss', 2,    True),
    'gauss_k4':    ('gauss', 4,    True),
    'gauss_k8':    ('gauss', 8,    True),
    'gauss_dense': ('gauss', None, False),
}
BS_LABEL = {'tri': 'Triangular, gated k=2 (TRACE-SNP v2)', 'hat_dense': 'Piecewise-linear, dense',
            'gauss_k2': 'Gaussian, gated k=2', 'gauss_k4': 'Gaussian, gated k=4', 'gauss_k8': 'Gaussian, gated k=8',
            'gauss_dense': 'Gaussian, dense (non-gated RBF)'}
BS_ROPE = 0.01        # Bayesian signed-rank: |log RMSE ratio| < 1 % counts as practically equivalent


def run_basis_study(datasets):
    from sklearn.preprocessing import MinMaxScaler as _MMS
    f64, dev, G = torch.float64, device, ABL_GRID
    T = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=f64, device=dev)
    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
    LO, HI = -1.1, 1.1

    def fields(U, base, M, w):
        shape, k, _ = BASES[base]
        h = (HI - LO) / (M - 1)
        if shape == 'tri':                                   # identical to TRACE-SNP v2 (2 neighbouring fields fire)
            pos = (U.clamp(LO, HI) - LO) / h
            i = pos.floor().long().clamp(0, M - 2)
            fr = pos - i
            B = torch.zeros(U.shape + (M,), dtype=U.dtype, device=U.device)
            B.scatter_(-1, i[..., None], (1 - fr)[..., None])
            B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
            return B.reshape(U.shape[0], -1)
        c = torch.linspace(LO, HI, M, dtype=U.dtype, device=U.device)
        d = (U.clamp(LO, HI)[..., None] - c).abs()
        if shape == 'hat':                                   # hat of half-width M·h: every field > 0 on the whole range
            B = 1 - d / (M * h)
        else:
            B = torch.exp(-d ** 2 / (2 * (w * h) ** 2))
            if k is not None:                                # firing condition: only the k nearest fields fire
                thr = d.kthvalue(k, dim=-1, keepdim=True).values
                B = B * (d <= thr)
        return B.reshape(U.shape[0], -1)

    def kmeans(Z, C, seed=0):                                 # identical to cell 10
        g = torch.Generator(device='cpu').manual_seed(seed)
        idx = [int(torch.randint(len(Z), (1,), generator=g))]
        dmin = ((Z - Z[idx[0]]) ** 2).sum(1)
        for _ in range(C - 1):
            j = int(torch.multinomial((dmin / dmin.sum()).cpu(), 1, generator=g)); idx.append(j)
            dmin = torch.minimum(dmin, ((Z - Z[j]) ** 2).sum(1))
        cen = Z[idx].clone()
        for _ in range(20):
            lab = torch.cdist(Z, cen).argmin(1)
            cnt = torch.bincount(lab, minlength=C).to(Z.dtype)
            s_ = torch.zeros_like(cen).index_add_(0, lab, Z)
            keep = cnt > 0
            cen[keep] = s_[keep] / cnt[keep, None]
        nn_d = torch.cdist(cen, cen) + torch.eye(C, dtype=Z.dtype, device=Z.device) * 1e9
        return cen, float(nn_d.min(1).values.median() ** 2)

    def state_feats(U, st, gated):
        cen, s2, d, off = st
        Z = U[:, off:off + d]
        D2 = torch.cdist(Z, cen) ** 2
        if gated:
            val, idx = D2.topk(G['k_active'], dim=1, largest=False)
            g = torch.exp(-val / (2 * s2)); g = g / g.sum(1, keepdim=True)
            Mu = torch.zeros_like(D2).scatter_(1, idx, g)
        else:
            Mu = torch.softmax(-D2 / (2 * s2), dim=1)
        Z1 = torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), Z], 1)
        return (Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1)

    def design(U, base, M=None, w=None, st=None):
        parts = [torch.ones(len(U), 1, dtype=U.dtype, device=U.device), U]
        if M:
            parts.append(fields(U, base, M, w))
        if st is not None:
            parts.append(state_feats(U, st, BASES[base][2]))
        return torch.cat(parts, 1)

    def penalty(L, M, l0, ls, n_state=0, lst=0.0):            # identical to cell 10
        nd = L * M if M else 0
        p = 1 + L + nd + n_state
        P = torch.zeros(p, p, dtype=f64, device=dev)
        P[1:, 1:] += l0 * torch.eye(p - 1, dtype=f64, device=dev)
        if M and ls > 0:
            D = torch.zeros(M - 2, M, dtype=f64, device=dev)
            for j in range(M - 2):
                D[j, j], D[j, j + 1], D[j, j + 2] = 1, -2, 1
            DD = ls * (D.T @ D)
            for k in range(L):
                a0 = 1 + L + k * M
                P[a0:a0 + M, a0:a0 + M] += DD
        if n_state and lst > l0:
            a0 = 1 + L + nd
            P[a0:, a0:] += (lst - l0) * torch.eye(n_state, dtype=f64, device=dev)
        P[0, 0] = 1e-10
        return P

    def fit_loo(Ptr, R, Pen, chunk=8192):                     # identical to cell 10
        N = len(Ptr)
        A = Ptr.T @ Ptr / N + Pen
        Lc = torch.linalg.cholesky(A)
        th = torch.cholesky_solve(Ptr.T @ R / N, Lc)
        loo = []
        for i in range(0, N, chunk):
            P = Ptr[i:i + chunk]
            V = torch.linalg.solve_triangular(Lc, P.T, upper=False)
            hd = ((V ** 2).sum(0) / N).clamp(max=1 - 1e-6)
            loo.append((R[i:i + chunk] - P @ th) / (1 - hd)[:, None])
        return th, torch.cat(loo)

    def windows(x, L, o):
        return np.stack([x[o - 1 - k] for k in range(L)], 1)

    def run_fold(x, L, h, f):
        otr = np.arange(L, f['va_start'] - h + 1)
        ova = np.arange(f['va_start'], f['te_start'] - h + 1)
        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
        a, b = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / (b - a)
        s = lambda v: sc * (v - a) - 1
        W = {'tr': windows(x, L, otr), 'va': windows(x, L, ova), 'te': windows(x, L, ote)}
        y = {'tr': x[otr + h - 1], 'va': x[ova + h - 1], 'te': x[ote + h - 1]}
        anc_raw = {'last': lambda Wr: Wr[:, 0], 'mean': lambda Wr: Wr.mean(1)}
        dd = (W['tr'][:, 1:] - W['tr'][:, :1]) if L > 1 else np.zeros((len(otr), 1))
        scd = 1.0 / max(float(np.abs(dd).max()), 1e-12)
        enc = {'lvl': lambda Wr: T(s(Wr)),
               'dif': lambda Wr: T(np.column_stack([np.zeros(len(Wr)), (Wr[:, 1:] - Wr[:, :1]) * scd]) if L > 1
                                   else np.zeros((len(Wr), 1)))}
        cands = []

        def add(fam, base, cfg, loo_raw, va_pred, te_pred, fn):
            e_va = y['va'] - va_pred
            cands.append({'fam': fam, 'basis': base, 'cfg': cfg, 'pool': rmse(np.concatenate([loo_raw, e_va])),
                          'va_pred': va_pred, 'te_pred': te_pred, 'fn': fn})

        for an in ('last', 'mean'):
            add('persistence' if an == 'last' else 'window-mean', 'fallback', {'anchor': an}, y['tr'] - anc_raw[an](W['tr']),
                anc_raw[an](W['va']), anc_raw[an](W['te']), anc_raw[an])
        mm = _MMS((-1, 1)).fit(np.column_stack([W['tr'], y['tr']]))
        tf = lambda Wr: Wr * mm.scale_[:-1] + mm.min_[:-1]
        Xc = tf(W['tr']); yy = y['tr'] * mm.scale_[-1] + mm.min_[-1]
        xm, ym = Xc.mean(0), yy.mean(); Xc0 = Xc - xm
        Ai = np.linalg.inv(Xc0.T @ Xc0 + RIDGE_ALPHA * np.eye(L)); wr = Ai @ Xc0.T @ (yy - ym)
        hr = 1 / len(Xc) + np.einsum('ij,jk,ik->i', Xc0, Ai, Xc0)
        rf = lambda Wr: ((tf(Wr) - xm) @ wr + ym - mm.min_[-1]) / mm.scale_[-1]
        add('ridge', 'fallback', {}, ((yy - (Xc0 @ wr + ym)) / (1 - hr)) / mm.scale_[-1], rf(W['va']), rf(W['te']), rf)
        ys = T(s(y['tr']))

        def make_fn(e, an, base, M, w, st, th):
            def fn(Wr):
                return ((T(s(anc_raw[an](Wr))) + design(enc[e](Wr), base, M, w, st) @ th).cpu().numpy() + 1) / sc + a
            return fn

        anchors = ('last', 'mean')
        for e in ('lvl', 'dif'):
            if e == 'dif' and L < 3:
                continue
            U = {k: enc[e](W[k]) for k in W}
            A_ = {k: torch.stack([T(s(anc_raw[an](W[k]))) for an in anchors], 1) for k in W}
            R = ys[:, None] - A_['tr']
            states = {}
            dmax = L if e == 'lvl' else L - 1
            for d in sorted({min(dmax, v) for v in G['d']}):
                for C in G['C']:
                    if C * (d + 1) > len(otr):
                        continue
                    Z = U['tr'][:, :d] if e == 'lvl' else U['tr'][:, 1:d + 1]
                    cen, s2 = kmeans(Z, C)
                    states[(d, C)] = (cen, s2, d, 0 if e == 'lvl' else 1)
            for base in BASES:
                widths = RBF_WIDTHS if BASES[base][0] == 'gauss' else [None]
                fam0 = f"{base}|{e}"
                for M in G['M']:
                    if L * M > G['max_delay_units'] and M != min(G['M']):
                        continue
                    if BASES[base][1] is not None and BASES[base][1] >= M:
                        continue
                    for w in widths:
                        P = {k: design(U[k], base, M, w) for k in U}
                        for l0 in G['l0']:
                            for ls in G['ls']:
                                th, loo = fit_loo(P['tr'], R, penalty(L, M, l0, ls))
                                pv, pt = (A_['va'] + P['va'] @ th).cpu().numpy(), (A_['te'] + P['te'] @ th).cpu().numpy()
                                for j, an in enumerate(anchors):
                                    add(f"{an}|{fam0}|delay", base, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls},
                                        loo[:, j].cpu().numpy() / sc, (pv[:, j] + 1) / sc + a, (pt[:, j] + 1) / sc + a,
                                        make_fn(e, an, base, M, w, None, th[:, j].clone()))
                        del P
                for j, an in enumerate(anchors):
                    pool = [c for c in cands if c['fam'] == f"{an}|{fam0}|delay"]
                    if not pool:
                        continue
                    best = min(pool, key=lambda c: c['pool'])['cfg']
                    M, w, l0, ls = best['M'], best['w'], best['l0'], best['ls']
                    for (d, C), st in states.items():
                        P = {k: design(U[k], base, M, w, st) for k in U}
                        for lst in G['lst']:
                            th, loo = fit_loo(P['tr'], R[:, j:j + 1], penalty(L, M, l0, ls, C * (d + 1), max(lst, l0)))
                            pv = (A_['va'][:, j:j + 1] + P['va'] @ th).cpu().numpy()[:, 0]
                            pt = (A_['te'][:, j:j + 1] + P['te'] @ th).cpu().numpy()[:, 0]
                            add(f"{an}|{fam0}|delay+state", base, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls,
                                                                   'd': d, 'C': C, 'lst': lst},
                                loo[:, 0].cpu().numpy() / sc, (pv + 1) / sc + a, (pt + 1) / sc + a,
                                make_fn(e, an, base, M, w, st, th[:, 0].clone()))
                        del P
        return cands, y, ova, ote

    def pick(cands, base, fallback=True):
        pool = [c for c in cands if c['basis'] == base or (fallback and c['basis'] == 'fallback')]
        return min(pool, key=lambda c: c['pool'])

    def iterate(fn, x, L, origins, h):
        Wr = windows(x, L, origins)
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    def active(c, L):
        # active receptive-field + state neurons per one-step evaluation, and all of them
        if c['basis'] == 'fallback':
            return 0.0, 0.0
        M = c['cfg']['M']; k = BASES[c['basis']][1]
        on, tot = float(L * (k if k is not None else M)), float(L * M)
        if 'C' in c['cfg'] and np.isfinite(c['cfg'].get('C', np.nan)):
            C = c['cfg']['C']; on += G['k_active'] if BASES[c['basis']][2] else C; tot += C
        return on, tot

    preds, choices = [], []
    for D in datasets:
        x = D['values']
        for L in BS_LAGS:
            if not lag_ok(None, D, L):
                continue
            hs = [h for h in BS_HORIZONS if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 >= ABL_MIN_ORIGINS]
            stem = f"{REV}/{BS_FOLDER}/{D['slug']}{SM}_lag{L}"
            rels = {k: f"{stem}_{k}.csv" for k in ('preds', 'choices')}
            if not BS_FRESH and all(store.fetch(r) for r in rels.values()):
                preds.append(pd.read_csv(store.fetch(rels['preds']))); choices.append(pd.read_csv(store.fetch(rels['choices'])))
                continue
            check_time_budget()
            t0 = time.time(); P_, C_ = [], []
            for f in D['folds']:
                fits = {h: run_fold(x, L, h, f) for h in sorted(set([1] + hs))}
                c1 = fits[1][0]
                for h in hs:
                    cands, y, ova, ote = fits[h]
                    yt, prev = x[ote + h - 1], x[ote - 1]
                    for base in BASES:
                        if not any(c['basis'] == base for c in cands):
                            continue
                        bd = pick(cands, base)
                        pred, it = bd['te_pred'], False
                        if h > 1:
                            b1 = pick(c1, base)
                            i_va = iterate(b1['fn'], x, L, ova, h)
                            it = rmse(y['va'] - i_va) < (1 - ABL_ITER_MARGIN) * rmse(y['va'] - bd['va_pred'])
                            if it:
                                pred = iterate(b1['fn'], x, L, ote, h)
                        used = pick(c1, base) if it else bd
                        on, tot = active(used, L)
                        P_.append(pd.DataFrame({'dataset': D['name'], 'model': base, 'lag': L, 'h': h, 'test_fold': f['label'],
                                                'origin': ote, 'target_idx': ote + h - 1, 'y': yt, 'prev': prev, 'ens_pred': pred}))
                        C_.append({'dataset': D['name'], 'model': base, 'lag': L, 'h': h, 'fold': f['label'], 'family': used['fam'],
                                   'basis_used': used['basis'], 'iterated': bool(it), 'criterion': used['pool'],
                                   'active_neurons': on * (h if it else 1), 'total_neurons': tot, **used['cfg']})
                del fits, c1
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
            Pdf, Cdf = pd.concat(P_, ignore_index=True), pd.DataFrame(C_)
            for k, df in (('preds', Pdf), ('choices', Cdf)):
                p = os.path.join(LOCAL_ROOT, rels[k]); os.makedirs(os.path.dirname(p), exist_ok=True)
                df.to_csv(p, index=False); store.add(p)
            preds.append(Pdf); choices.append(Cdf)
            g = Pdf.groupby('model').apply(lambda d: rmse(d['y'] - d['ens_pred']), include_groups=False)
            print(f"{D['name']:<16} lag {L:2d} | " + " | ".join(f"{b} {g[b]:.4g}" for b in BASES if b in g) + f" | {time.time() - t0:.0f} s")
            store.push(f"basis study {D['slug']} lag {L}")
        store.push(f"basis study {D['slug']}", force=True)
        trim_ram()
    cat = lambda lst: pd.concat(lst, ignore_index=True) if lst else pd.DataFrame()
    return {'preds': cat(preds), 'choices': cat(choices)}


try:
    BS_RESULTS = run_basis_study(ALL11)
except TimeBudgetReached:
    store.push('basis study: time budget', force=True)
    raise RuntimeError('Time budget reached — rerun this cell; finished (dataset, lag) are reused.')


def bayes_signrank(d, rope=BS_ROPE, prior=0.5, n=50000, seed=0):
    # Bayesian signed-rank test (Benavoli et al., JMLR 2017) on paired differences d (here: log RMSE ratios).
    # Returns P(left: d < -rope), P(rope), P(right: d > rope).
    d = np.concatenate([[0.0], np.asarray(d, float)])
    w = np.ones(len(d)); w[0] = prior
    S = np.random.default_rng(seed).dirichlet(w, n)
    pair = d[:, None] + d[None, :]
    lft, rgt = (pair < -2 * rope).astype(float), (pair > 2 * rope).astype(float)
    mid = 1.0 - lft - rgt
    pl, pr, pm = (np.einsum('si,ij,sj->s', S, X, S) for X in (lft, rgt, mid))
    win = np.argmax(np.stack([pl, pm, pr], 1), 1)
    return float((win == 0).mean()), float((win == 1).mean()), float((win == 2).mean())


def _basis_summary():
    from scipy import stats as _st
    A = BS_RESULTS['preds']
    if A.empty:
        print('No basis-study results.'); return None
    rm = lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2)))
    R = A.groupby(['dataset', 'lag', 'h', 'model']).apply(rm, include_groups=False).unstack('model')
    bases = [b for b in BASES if b in R]
    gm = lambda r: float(np.exp(np.nanmean(np.log(r))))
    ref_d, ref_g = 'gauss_dense', 'tri'
    kind = pd.Series([KIND_OF.get(d, '') for d in R.index.get_level_values('dataset')], index=R.index)
    rows = []
    ch = BS_RESULTS['choices']
    for b in bases:
        r_d, r_g = R[b] / R[ref_d], R[b] / R[ref_g]
        byd_d = np.log(r_d).groupby(level='dataset').mean()
        lt, rp, rt = bayes_signrank(byd_d.to_numpy())
        cb = ch[ch['model'] == b]
        rows.append({'variant': BS_LABEL[b], 'key': b,
                     'RMSE / dense Gaussian': gm(r_d), '… chaotic': gm(r_d[kind == 'chaotic']), '… real': gm(r_d[kind == 'real']),
                     'RMSE / TRACE-SNP v2': gm(r_g),
                     'active neurons (median)': float(cb['active_neurons'].median()),
                     'active share': float((cb['active_neurons'] / cb['total_neurons'].replace(0, np.nan)).median()),
                     'datasets better than dense': int((byd_d < 0).sum()),
                     'P(better than dense)': lt, 'P(equivalent, ROPE 1%)': rp, 'P(worse than dense)': rt})
    TAB = pd.DataFrame(rows).set_index('variant')
    rev_csv(TAB.reset_index(), 'basis_study_summary')
    byd = pd.DataFrame({BS_LABEL[b]: (R[b] / R[ref_d]).groupby(level='dataset').apply(gm) for b in bases})
    byd = byd.loc[[d for d in ALL11_NAMES if d in byd.index]]
    byd['type'] = [KIND_OF.get(d, '') for d in byd.index]
    rev_csv(byd.reset_index(), 'basis_study_by_dataset')
    # the two clean contrasts, one-sided Wilcoxon over datasets
    def contrast(a, b, alt):
        d = np.log(R[a] / R[b]).groupby(level='dataset').mean()
        w = _st.wilcoxon(d, alternative=alt, zero_method='zsplit') if np.any(d != 0) else None
        return float(np.exp(d.mean())), (w.pvalue if w is not None else np.nan), bayes_signrank(d.to_numpy())
    C = {}
    if 'gauss_k2' in R:
        C['shape: Gaussian k=2 vs triangular k=2 (equal firing)'] = contrast('gauss_k2', 'tri', 'less')
    for k_ in ('gauss_k2', 'gauss_k4', 'gauss_k8'):
        if k_ in R:
            C[f'firing condition: {BS_LABEL[k_]} vs dense Gaussian (equal shape)'] = contrast(k_, 'gauss_dense', 'less')
    if 'hat_dense' in R:
        C['firing condition: triangular k=2 vs piecewise-linear dense'] = contrast('tri', 'hat_dense', 'less')
    lines = ["## Field shape × firing condition (identical solver and selection; RMSE ratios are geometric means over problems)",
             TAB.drop(columns='key').to_string(float_format=lambda v: f"{v:.3f}"), "",
             "Per dataset, RMSE / dense Gaussian:", byd.to_string(float_format=lambda v: f"{v:.3f}"), "",
             "Contrasts over the datasets (ratio < 1: first is more accurate; one-sided Wilcoxon H1: first < second; "
             "Bayesian signed-rank P(first better, equivalent, second better), ROPE ±1 %):"]
    for k_, (ratio, p, (pl, pm, pr)) in C.items():
        lines.append(f"  {k_}: ratio {ratio:.3f}, Wilcoxon p = {p:.3g}, Bayes P = ({pl:.2f}, {pm:.2f}, {pr:.2f})")
    # Bayesian signed-rank for the main N = 11 comparison too (TRACE-SNP v2 vs every rival)
    if STATS is not None and STAT_CONTROL in STATS['S']:
        S = STATS['S']; br = []
        for m in S.columns:
            if m == STAT_CONTROL:
                continue
            pl, pm, pr = bayes_signrank((np.log(S[STAT_CONTROL]) - np.log(S[m])).to_numpy())
            br.append({'model': m, f'P({STAT_CONTROL} better)': pl, 'P(equivalent)': pm, f'P({m} better)': pr})
        BR = pd.DataFrame(br).set_index('model').sort_values(f'P({STAT_CONTROL} better)', ascending=False)
        rev_csv(BR.reset_index(), 'n11_bayesian_signed_rank')
        lines += ["", f"Bayesian signed-rank over N = {S.shape[0]} datasets, {STAT_CONTROL} vs each model (ROPE ±1 % RMSE):",
                  BR.iloc[:, :2].to_string(float_format=lambda v: f"{v:.3f}")]
    # figure: accuracy vs active neurons, Gaussian family, chaotic vs real
    gk = [b for b in ('gauss_k2', 'gauss_k4', 'gauss_k8', 'gauss_dense') if b in R]
    if len(gk) >= 2:
        fig, ax = plt.subplots(figsize=(W1, 2.5))
        for kd, st_ in (('chaotic', dict(color=OI['vermillion'], marker='o', ls='-')), ('real', dict(color=OI['blue'], marker='s', ls='--'))):
            xs_ = [float((ch[(ch['model'] == b) & ch['dataset'].map(lambda d: KIND_OF.get(d) == kd)]['active_neurons'] /
                          ch[(ch['model'] == b) & ch['dataset'].map(lambda d: KIND_OF.get(d) == kd)]['total_neurons'].replace(0, np.nan)).median())
                   for b in gk]
            ys_ = [gm((R[b] / R['gauss_dense'])[kind == kd]) for b in gk]
            ax.plot(xs_, ys_, lw=1.4, ms=4, label=f"{kd} series (Gaussian fields, k = 2, 4, 8, all)", **st_)
            if 'tri' in R:
                ax.scatter([float((ch[(ch['model'] == 'tri') & ch['dataset'].map(lambda d: KIND_OF.get(d) == kd)]['active_neurons'] /
                                   ch[(ch['model'] == 'tri') & ch['dataset'].map(lambda d: KIND_OF.get(d) == kd)]['total_neurons'].replace(0, np.nan)).median())],
                           [gm((R['tri'] / R['gauss_dense'])[kind == kd])], marker='*', s=60, color=st_['color'], edgecolor='k', lw=0.4, zorder=5)
        ax.axhline(1, color='k', lw=0.5); ax.set_xscale('log')
        ax.set_xlabel('share of neurons active per forecast'); ax.set_ylabel('RMSE / dense Gaussian (geo-mean)')
        ax.set_title('Accuracy vs firing sparsity (star: TRACE-SNP v2 triangular fields)', loc='left', fontsize=7)
        ax.legend(frameon=False, fontsize=5.6); ax.grid(True, alpha=0.25, lw=0.4)
        save_fig(fig, 'fig_accuracy_vs_active_share')
    t = TAB.drop(columns=['key', 'P(better than dense)', 'P(worse than dense)'])
    tex_table(t, 'tab_basis_study', "Receptive-field shape $\\times$ firing condition, every variant fitted by the identical convex "
              "solver and selection rule of TRACE-SNP v2. Gated variants keep only the $k$ nearest fields per delay (and the 4 "
              "nearest state neurons); dense variants keep all. RMSE ratios: geometric means over all (dataset, lag, horizon) "
              "problems. P(equivalent): Bayesian signed-rank test over the 11 datasets, region of practical equivalence $\\pm1\\%$.",
              'tab:basis_study', fmt=lambda v: f"{v:.3f}", idx_name='Variant')
    txt = "\n".join(lines)
    print(txt)
    with open(rev_path(f"basis_study{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"basis_study{SM}.md"))
    store.push('revisions: basis study', force=True)
    return {'table': TAB, 'by_dataset': byd, 'contrasts': C, 'text': txt}


BASIS = _basis_summary()

In [ ]:
# ============================================================
# 18. TRACE-SNP-AF — TRACE-SNP v2 with an ADAPTIVE FIRING CONDITION (closed form; no network is retrained)
#     Cell 17 showed that how many receptive fields fire matters: 2 of M (TRACE-SNP v2) is enough on real series but too
#     sparse for chaotic ones, while Gaussian fields with 8 of 16 firing match a dense basis. TRACE-SNP-AF therefore lets
#     the model choose its firing condition per series, with the SAME selection machinery as TRACE-SNP v2 and nothing
#     that looks at a test block:
#       candidates   every TRACE-SNP v2 candidate (anchors, encodings, M, λ grids, state neurons, persistence / window
#                    mean / Ridge fallbacks) for each gated field family F ∈ AF_BASES:
#                    triangular with 2 firing (= TRACE-SNP v2), Gaussian with k ∈ {2, 4, 8} firing (width chosen too);
#                    no dense (non-gated) family is ever a candidate, so the model stays condition-gated
#       selection    exact LOO residuals on the training rows + the validation block (the v2 criterion); then a
#                    parsimony rule: among receptive-field candidates within AF_TOL (1 %) of the best criterion, the one
#                    with the FEWEST firing neurons is kept (sparsest adequate firing condition)
#       horizons     direct vs iterated one-step forecast, the v2 margin rule (25 %), per fold
#     Same 11 datasets, lags, horizons 1/6/24 and 5 walk-forward folds as everything else. Also stored: GPU inference
#     energy (large-batch benchmark, as for TRACE-SNP v2), exact operation counts, the 10 % input-noise test with the
#     notebook's own noise draws, and the firing condition chosen in every fold.
#     Then the N = 11 ranking is recomputed with TRACE-SNP-AF as the model under test (Friedman, Iman–Davenport,
#     Nemenyi CD diagram, Wilcoxon–Holm, Bayesian signed-rank), and TRACE-SNP-AF vs TRACE-SNP v2 per dataset.
#     Resumable: <LOCAL_ROOT>/revisions/trace_af/<dataset>_lag<L>_*.csv. Cost ≈ 0.7 × cell 17 on a GPU.
# ============================================================
AF_NAME    = 'TRACE-SNP-AF'
AF_FOLDER  = 'trace_af'
AF_BASES   = {'tri': ('tri', 2, True), 'gauss_k2': ('gauss', 2, True), 'gauss_k4': ('gauss', 4, True),
              'gauss_k8': ('gauss', 8, True)}                  # (shape, fields firing per delay, state gated)
AF_TOL     = 0.01
AF_BENCH_N = 18000                                             # forecasts per inference-benchmark call (as TRACE-SNP v2)
AF_BENCH_S = 0.3 if SMOKE else 1.0
AF_FRESH   = False
AF_NOISE   = NOISE_LEVELS


def run_trace_af(datasets):
    from sklearn.preprocessing import MinMaxScaler as _MMS
    f64, dev, G = torch.float64, device, ABL_GRID
    T = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=f64, device=dev)
    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
    LO, HI = -1.1, 1.1
    BASES = AF_BASES                                           # the engine below is cell 17's, on the gated families only

    def fields(U, base, M, w):
        shape, k, _ = BASES[base]
        h = (HI - LO) / (M - 1)
        if shape == 'tri':                                   # identical to TRACE-SNP v2 (2 neighbouring fields fire)
            pos = (U.clamp(LO, HI) - LO) / h
            i = pos.floor().long().clamp(0, M - 2)
            fr = pos - i
            B = torch.zeros(U.shape + (M,), dtype=U.dtype, device=U.device)
            B.scatter_(-1, i[..., None], (1 - fr)[..., None])
            B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
            return B.reshape(U.shape[0], -1)
        c = torch.linspace(LO, HI, M, dtype=U.dtype, device=U.device)
        d = (U.clamp(LO, HI)[..., None] - c).abs()
        if shape == 'hat':                                   # hat of half-width M·h: every field > 0 on the whole range
            B = 1 - d / (M * h)
        else:
            B = torch.exp(-d ** 2 / (2 * (w * h) ** 2))
            if k is not None:                                # firing condition: only the k nearest fields fire
                thr = d.kthvalue(k, dim=-1, keepdim=True).values
                B = B * (d <= thr)
        return B.reshape(U.shape[0], -1)

    def kmeans(Z, C, seed=0):                                 # identical to cell 10
        g = torch.Generator(device='cpu').manual_seed(seed)
        idx = [int(torch.randint(len(Z), (1,), generator=g))]
        dmin = ((Z - Z[idx[0]]) ** 2).sum(1)
        for _ in range(C - 1):
            j = int(torch.multinomial((dmin / dmin.sum()).cpu(), 1, generator=g)); idx.append(j)
            dmin = torch.minimum(dmin, ((Z - Z[j]) ** 2).sum(1))
        cen = Z[idx].clone()
        for _ in range(20):
            lab = torch.cdist(Z, cen).argmin(1)
            cnt = torch.bincount(lab, minlength=C).to(Z.dtype)
            s_ = torch.zeros_like(cen).index_add_(0, lab, Z)
            keep = cnt > 0
            cen[keep] = s_[keep] / cnt[keep, None]
        nn_d = torch.cdist(cen, cen) + torch.eye(C, dtype=Z.dtype, device=Z.device) * 1e9
        return cen, float(nn_d.min(1).values.median() ** 2)

    def state_feats(U, st, gated):
        cen, s2, d, off = st
        Z = U[:, off:off + d]
        D2 = torch.cdist(Z, cen) ** 2
        if gated:
            val, idx = D2.topk(G['k_active'], dim=1, largest=False)
            g = torch.exp(-val / (2 * s2)); g = g / g.sum(1, keepdim=True)
            Mu = torch.zeros_like(D2).scatter_(1, idx, g)
        else:
            Mu = torch.softmax(-D2 / (2 * s2), dim=1)
        Z1 = torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), Z], 1)
        return (Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1)

    def design(U, base, M=None, w=None, st=None):
        parts = [torch.ones(len(U), 1, dtype=U.dtype, device=U.device), U]
        if M:
            parts.append(fields(U, base, M, w))
        if st is not None:
            parts.append(state_feats(U, st, BASES[base][2]))
        return torch.cat(parts, 1)

    def penalty(L, M, l0, ls, n_state=0, lst=0.0):            # identical to cell 10
        nd = L * M if M else 0
        p = 1 + L + nd + n_state
        P = torch.zeros(p, p, dtype=f64, device=dev)
        P[1:, 1:] += l0 * torch.eye(p - 1, dtype=f64, device=dev)
        if M and ls > 0:
            D = torch.zeros(M - 2, M, dtype=f64, device=dev)
            for j in range(M - 2):
                D[j, j], D[j, j + 1], D[j, j + 2] = 1, -2, 1
            DD = ls * (D.T @ D)
            for k in range(L):
                a0 = 1 + L + k * M
                P[a0:a0 + M, a0:a0 + M] += DD
        if n_state and lst > l0:
            a0 = 1 + L + nd
            P[a0:, a0:] += (lst - l0) * torch.eye(n_state, dtype=f64, device=dev)
        P[0, 0] = 1e-10
        return P

    def fit_loo(Ptr, R, Pen, chunk=8192):                     # identical to cell 10
        N = len(Ptr)
        A = Ptr.T @ Ptr / N + Pen
        Lc = torch.linalg.cholesky(A)
        th = torch.cholesky_solve(Ptr.T @ R / N, Lc)
        loo = []
        for i in range(0, N, chunk):
            P = Ptr[i:i + chunk]
            V = torch.linalg.solve_triangular(Lc, P.T, upper=False)
            hd = ((V ** 2).sum(0) / N).clamp(max=1 - 1e-6)
            loo.append((R[i:i + chunk] - P @ th) / (1 - hd)[:, None])
        return th, torch.cat(loo)

    def windows(x, L, o):
        return np.stack([x[o - 1 - k] for k in range(L)], 1)

    def run_fold(x, L, h, f):
        otr = np.arange(L, f['va_start'] - h + 1)
        ova = np.arange(f['va_start'], f['te_start'] - h + 1)
        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
        a, b = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / (b - a)
        s = lambda v: sc * (v - a) - 1
        W = {'tr': windows(x, L, otr), 'va': windows(x, L, ova), 'te': windows(x, L, ote)}
        y = {'tr': x[otr + h - 1], 'va': x[ova + h - 1], 'te': x[ote + h - 1]}
        anc_raw = {'last': lambda Wr: Wr[:, 0], 'mean': lambda Wr: Wr.mean(1)}
        dd = (W['tr'][:, 1:] - W['tr'][:, :1]) if L > 1 else np.zeros((len(otr), 1))
        scd = 1.0 / max(float(np.abs(dd).max()), 1e-12)
        enc = {'lvl': lambda Wr: T(s(Wr)),
               'dif': lambda Wr: T(np.column_stack([np.zeros(len(Wr)), (Wr[:, 1:] - Wr[:, :1]) * scd]) if L > 1
                                   else np.zeros((len(Wr), 1)))}
        cands = []

        def add(fam, base, cfg, loo_raw, va_pred, te_pred, fn):
            e_va = y['va'] - va_pred
            cands.append({'fam': fam, 'basis': base, 'cfg': cfg, 'pool': rmse(np.concatenate([loo_raw, e_va])),
                          'va_pred': va_pred, 'te_pred': te_pred, 'fn': fn})

        for an in ('last', 'mean'):
            add('persistence' if an == 'last' else 'window-mean', 'fallback', {'anchor': an}, y['tr'] - anc_raw[an](W['tr']),
                anc_raw[an](W['va']), anc_raw[an](W['te']), anc_raw[an])
        mm = _MMS((-1, 1)).fit(np.column_stack([W['tr'], y['tr']]))
        tf = lambda Wr: Wr * mm.scale_[:-1] + mm.min_[:-1]
        Xc = tf(W['tr']); yy = y['tr'] * mm.scale_[-1] + mm.min_[-1]
        xm, ym = Xc.mean(0), yy.mean(); Xc0 = Xc - xm
        Ai = np.linalg.inv(Xc0.T @ Xc0 + RIDGE_ALPHA * np.eye(L)); wr = Ai @ Xc0.T @ (yy - ym)
        hr = 1 / len(Xc) + np.einsum('ij,jk,ik->i', Xc0, Ai, Xc0)
        rf = lambda Wr: ((tf(Wr) - xm) @ wr + ym - mm.min_[-1]) / mm.scale_[-1]
        add('ridge', 'fallback', {}, ((yy - (Xc0 @ wr + ym)) / (1 - hr)) / mm.scale_[-1], rf(W['va']), rf(W['te']), rf)
        ys = T(s(y['tr']))

        def make_fn(e, an, base, M, w, st, th):
            def fn(Wr):
                return ((T(s(anc_raw[an](Wr))) + design(enc[e](Wr), base, M, w, st) @ th).cpu().numpy() + 1) / sc + a
            return fn

        anchors = ('last', 'mean')
        for e in ('lvl', 'dif'):
            if e == 'dif' and L < 3:
                continue
            U = {k: enc[e](W[k]) for k in W}
            A_ = {k: torch.stack([T(s(anc_raw[an](W[k]))) for an in anchors], 1) for k in W}
            R = ys[:, None] - A_['tr']
            states = {}
            dmax = L if e == 'lvl' else L - 1
            for d in sorted({min(dmax, v) for v in G['d']}):
                for C in G['C']:
                    if C * (d + 1) > len(otr):
                        continue
                    Z = U['tr'][:, :d] if e == 'lvl' else U['tr'][:, 1:d + 1]
                    cen, s2 = kmeans(Z, C)
                    states[(d, C)] = (cen, s2, d, 0 if e == 'lvl' else 1)
            for base in BASES:
                widths = RBF_WIDTHS if BASES[base][0] == 'gauss' else [None]
                fam0 = f"{base}|{e}"
                for M in G['M']:
                    if L * M > G['max_delay_units'] and M != min(G['M']):
                        continue
                    if BASES[base][1] is not None and BASES[base][1] >= M:
                        continue
                    for w in widths:
                        P = {k: design(U[k], base, M, w) for k in U}
                        for l0 in G['l0']:
                            for ls in G['ls']:
                                th, loo = fit_loo(P['tr'], R, penalty(L, M, l0, ls))
                                pv, pt = (A_['va'] + P['va'] @ th).cpu().numpy(), (A_['te'] + P['te'] @ th).cpu().numpy()
                                for j, an in enumerate(anchors):
                                    add(f"{an}|{fam0}|delay", base, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls},
                                        loo[:, j].cpu().numpy() / sc, (pv[:, j] + 1) / sc + a, (pt[:, j] + 1) / sc + a,
                                        make_fn(e, an, base, M, w, None, th[:, j].clone()))
                        del P
                for j, an in enumerate(anchors):
                    pool = [c for c in cands if c['fam'] == f"{an}|{fam0}|delay"]
                    if not pool:
                        continue
                    best = min(pool, key=lambda c: c['pool'])['cfg']
                    M, w, l0, ls = best['M'], best['w'], best['l0'], best['ls']
                    for (d, C), st in states.items():
                        P = {k: design(U[k], base, M, w, st) for k in U}
                        for lst in G['lst']:
                            th, loo = fit_loo(P['tr'], R[:, j:j + 1], penalty(L, M, l0, ls, C * (d + 1), max(lst, l0)))
                            pv = (A_['va'][:, j:j + 1] + P['va'] @ th).cpu().numpy()[:, 0]
                            pt = (A_['te'][:, j:j + 1] + P['te'] @ th).cpu().numpy()[:, 0]
                            add(f"{an}|{fam0}|delay+state", base, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls,
                                                                   'd': d, 'C': C, 'lst': lst},
                                loo[:, 0].cpu().numpy() / sc, (pv + 1) / sc + a, (pt + 1) / sc + a,
                                make_fn(e, an, base, M, w, st, th[:, 0].clone()))
                        del P
        return cands, y, ova, ote

    def pick(cands, base, fallback=True):
        pool = [c for c in cands if c['basis'] == base or (fallback and c['basis'] == 'fallback')]
        return min(pool, key=lambda c: c['pool'])

    def iterate(fn, x, L, origins, h):
        Wr = windows(x, L, origins)
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    def active(c, L):
        # active receptive-field + state neurons per one-step evaluation, and all of them
        if c['basis'] == 'fallback':
            return 0.0, 0.0
        M = c['cfg']['M']; k = BASES[c['basis']][1]
        on, tot = float(L * (k if k is not None else M)), float(L * M)
        if 'C' in c['cfg'] and np.isfinite(c['cfg'].get('C', np.nan)):
            C = c['cfg']['C']; on += G['k_active'] if BASES[c['basis']][2] else C; tot += C
        return on, tot

    def n_fire(c, L):
        # firing receptive-field + state neurons in one one-step evaluation (0 for the fallbacks)
        if c['basis'] == 'fallback':
            return 0.0
        n = float(L * BASES[c['basis']][1])
        if 'C' in c['cfg'] and np.isfinite(c['cfg'].get('C', np.nan)):
            n += G['k_active']
        return n

    def choose(cands, L):
        # v2 criterion, then the sparsest receptive-field candidate within AF_TOL of the best
        best = min(cands, key=lambda c: c['pool'])
        if best['basis'] == 'fallback':
            return best
        ok = [c for c in cands if c['basis'] != 'fallback' and c['pool'] <= (1 + AF_TOL) * best['pool']]
        return min(ok, key=lambda c: (n_fire(c, L), c['pool']))

    def ops(c, L):
        # multiply-accumulates of one one-step forecast (exp of a Gaussian field counted as one MAC: a lower bound)
        if c['basis'] == 'fallback':
            return {'persistence': 0.0, 'window-mean': 0.0, 'ridge': float(L)}.get(c['fam'], float(L)), \
                   (float(L) if c['fam'] == 'window-mean' else 0.0)
        shape, k, _ = BASES[c['basis']]
        mac = 2.0 * L + L + k * L + (k * L if shape == 'gauss' else 0.0)    # scale+locate, linear, firing weights, exps
        ac = (float(L) if c['cfg']['anchor'] == 'mean' else 0.0) + (float(L) if c['cfg']['enc'] == 'dif' else 0.0)
        if 'C' in c['cfg'] and np.isfinite(c['cfg'].get('C', np.nan)):
            d, C, ka = float(c['cfg']['d']), float(c['cfg']['C']), float(G['k_active'])
            mac += C * d + ka + ka * (d + 1)
        return mac, ac

    def bench(fn, Wr):
        reps_needed = max(1, int(np.ceil(AF_BENCH_N / len(Wr))))
        Wb = np.tile(Wr, (reps_needed, 1))
        seg = EnergySegment(METER); reps, t_end = 0, time.perf_counter() + AF_BENCH_S
        seg.resume()
        while reps < 1 or time.perf_counter() < t_end:
            fn(Wb); reps += 1
        seg.pause(); st = seg.state(); n = reps * len(Wb)
        return st['j'] / n, st['s'] / n

    def iterate_windows(fn, Wr, h):
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    def noisy_windows(Wtr, ytr, Wte, L, fi, li, dd, nl):          # the notebook's noise draws (as cells 8 and 10)
        mmn = _MMS((-1, 1)).fit(np.column_stack([Wtr, ytr]))
        Xtr_s = Wtr * mmn.scale_[:-1] + mmn.min_[:-1]; Xte_s = Wte * mmn.scale_[:-1] + mmn.min_[:-1]
        Xn = Xte_s + np.random.default_rng([L, fi, li, dd]).normal(0.0, nl * Xtr_s.std(), size=Xte_s.shape)
        return (Xn - mmn.min_[:-1]) / mmn.scale_[:-1]

    preds, choices, energy, noise_rows = [], [], [], []
    for D in datasets:
        x = D['values']
        for L in BS_LAGS:
            if not lag_ok(None, D, L):
                continue
            hs = [h for h in BS_HORIZONS if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 >= ABL_MIN_ORIGINS]
            stem = f"{REV}/{AF_FOLDER}/{D['slug']}{SM}_lag{L}"
            rels = {k: f"{stem}_{k}.csv" for k in ('preds', 'choices', 'energy', 'noise')}
            if not AF_FRESH and all(store.fetch(rels[k]) for k in ('preds', 'choices', 'energy')):
                for k, lst in (('preds', preds), ('choices', choices), ('energy', energy), ('noise', noise_rows)):
                    p = store.fetch(rels[k])
                    if p and os.path.getsize(p) > 2:
                        lst.append(pd.read_csv(p))
                continue
            check_time_budget()
            t0 = time.time(); P_, C_, Nz = [], [], []
            E = {h: {'j': 0.0, 's': 0.0} for h in hs}
            for fi, f in enumerate(D['folds']):
                seg = EnergySegment(METER); seg.resume()
                fits = {h: run_fold(x, L, h, f) for h in sorted(set([1] + hs))}
                seg.pause(); st = seg.state()
                c1 = fits[1][0]
                b1 = choose(c1, L)
                for h in hs:
                    cands, y, ova, ote = fits[h]
                    yt, prev = x[ote + h - 1], x[ote - 1]
                    bd = choose(cands, L)
                    pred, it = bd['te_pred'], False
                    if h > 1:
                        i_va = iterate(b1['fn'], x, L, ova, h)
                        it = rmse(y['va'] - i_va) < (1 - ABL_ITER_MARGIN) * rmse(y['va'] - bd['va_pred'])
                        if it:
                            pred = iterate(b1['fn'], x, L, ote, h)
                    used = b1 if it else bd
                    mac, ac = ops(used, L)
                    mult = h if it else 1
                    tau_h = round(float(np.median(x[:f['te_start']][h:] - x[:f['te_start']][:-h]) if h > 1
                                        else np.median(np.diff(x[:f['te_start']]))), 6)
                    P_.append(pd.DataFrame({'dataset': D['name'], 'model': AF_NAME, 'lag': L, 'h': h, 'test_fold': f['label'],
                                            'origin': ote, 'target_idx': ote + h - 1, 'y': yt, 'prev': prev, 'tau': tau_h,
                                            'label': ((yt - prev) > tau_h).astype(int), 'ens_pred': pred}))
                    C_.append({'dataset': D['name'], 'model': AF_NAME, 'lag': L, 'h': h, 'fold': f['label'],
                               'field_family': used['basis'], 'family': used['fam'], 'iterated': bool(it), 'criterion': used['pool'],
                               'firing_neurons': n_fire(used, L) * mult, 'MAC': mac * mult, 'AC': ac * mult,
                               'pJ': PJ_MAC * mac * mult + PJ_AC * ac * mult, **used['cfg']})
                    E[h]['j'] += st['j'] / len(hs); E[h]['s'] += st['s'] / len(hs)
                    if fi == len(D['folds']) - 1:
                        fn_b = (lambda Wr, F=b1['fn'], h=h: iterate_windows(F, Wr, h)) if it else used['fn']
                        E[h]['infer_j_per_forecast'], E[h]['infer_s_per_forecast'] = bench(fn_b, windows(x, L, ote))
                    if h == 1 and N_NOISE_DRAWS:
                        otr = np.arange(L, f['va_start'])
                        Wtr, ytr, Wte = windows(x, L, otr), x[otr], windows(x, L, ote)
                        for li, nl in enumerate(AF_NOISE):
                            for dd in range(N_NOISE_DRAWS):
                                Wn = noisy_windows(Wtr, ytr, Wte, L, fi, li, dd, nl)
                                Nz.append({'dataset': D['name'], 'lag': L, 'fold': f['label'], 'model': AF_NAME, 'noise': nl,
                                           'draw': dd, 'rmse': rmse(yt - b1['fn'](Wn))})
                del fits, c1
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
            Pdf, Cdf, Nzdf = pd.concat(P_, ignore_index=True), pd.DataFrame(C_), pd.DataFrame(Nz)
            Edf = pd.DataFrame([{'dataset': D['name'], 'model': AF_NAME, 'lag': L, 'h': h, 'energy_j': v['j'], 'seconds': v['s'],
                                 'infer_j_per_forecast': v.get('infer_j_per_forecast', np.nan),
                                 'infer_s_per_forecast': v.get('infer_s_per_forecast', np.nan), 'method': METER.method}
                                for h, v in E.items()])
            for k, df in (('preds', Pdf), ('choices', Cdf), ('energy', Edf), ('noise', Nzdf)):
                p = os.path.join(LOCAL_ROOT, rels[k]); os.makedirs(os.path.dirname(p), exist_ok=True)
                df.to_csv(p, index=False); store.add(p)
            preds.append(Pdf); choices.append(Cdf); energy.append(Edf); noise_rows.append(Nzdf)
            g = Pdf.groupby('h').apply(lambda d: rmse(d['y'] - d['ens_pred']), include_groups=False)
            print(f"{D['name']:<16} lag {L:2d} | {AF_NAME} " + " | ".join(f"h{h} {g[h]:.4g}" for h in g.index)
                  + f" | firing families {Cdf['field_family'].value_counts().to_dict()} | {time.time() - t0:.0f} s")
            store.push(f"trace af {D['slug']} lag {L}")
        store.push(f"trace af {D['slug']}", force=True)
        trim_ram()
    cat = lambda lst: pd.concat([d for d in lst if len(d)], ignore_index=True) if any(len(d) for d in lst) else pd.DataFrame()
    return {'preds': cat(preds), 'choices': cat(choices), 'energy': cat(energy), 'noise': cat(noise_rows)}


try:
    AF_RESULTS = run_trace_af(ALL11)
except TimeBudgetReached:
    store.push('trace af: time budget', force=True)
    raise RuntimeError('Time budget reached — rerun this cell; finished (dataset, lag) are reused.')


def _af_analysis():
    from scipy import stats as _st
    A = AF_RESULTS['preds']
    if A.empty:
        print('No TRACE-SNP-AF results.'); return None
    rm = lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2)))
    lb_af = A.groupby(['dataset', 'h', 'lag']).apply(rm, include_groups=False).rename('rmse').reset_index()
    pers = LB[LB['model'] == 'Persistence'].set_index(['dataset', 'h', 'lag'])['rmse']
    lb_af['rel_rmse'] = lb_af['rmse'] / lb_af.set_index(['dataset', 'h', 'lag']).index.map(pers).to_numpy()
    lb_af['model'] = AF_NAME
    lb_af['n'] = A.groupby(['dataset', 'h', 'lag']).size().to_numpy()
    LB2 = pd.concat([LB, lb_af], ignore_index=True)
    rev_csv(LB2, 'leaderboard_with_trace_af')
    core = LB2[~LB2['model'].isin(STAT_EXCLUDE)]
    keys = pd.MultiIndex.from_frame(lb_af[['dataset', 'lag', 'h']])          # rank on the problems TRACE-SNP-AF ran on
    core = core[pd.MultiIndex.from_frame(core[['dataset', 'lag', 'h']]).isin(keys)]
    lines = []
    # 1. TRACE-SNP-AF vs TRACE-SNP v2 and vs the dense Gaussian (cell 17)
    v2 = LB[LB['model'] == 'TRACE-SNP v2'].set_index(['dataset', 'lag', 'h'])['rmse']
    af = lb_af.set_index(['dataset', 'lag', 'h'])['rmse']
    rr = (af / v2).dropna()
    kind = pd.Series([KIND_OF.get(d, '') for d in rr.index.get_level_values('dataset')], index=rr.index)
    gm = lambda r: float(np.exp(np.mean(np.log(r))))
    per_d = rr.groupby(level='dataset').apply(gm)
    w = _st.wilcoxon(np.log(per_d), alternative='less', zero_method='zsplit') if np.any(per_d != 1) else None
    bl, bm, br_ = bayes_signrank(np.log(per_d).to_numpy())
    lines += [f"## {AF_NAME}: adaptive firing condition (gated families only, chosen on training + validation data)",
              f"RMSE {AF_NAME} / TRACE-SNP v2: {gm(rr):.3f} overall | chaotic {gm(rr[kind == 'chaotic']):.3f} | real {gm(rr[kind == 'real']):.3f} "
              f"| better on {int((per_d < 1).sum())} of {len(per_d)} datasets"
              + (f" | one-sided Wilcoxon p = {w.pvalue:.3g}" if w is not None else '')
              + f" | Bayes P(AF better, equivalent, v2 better) = ({bl:.2f}, {bm:.2f}, {br_:.2f})",
              "Per dataset: " + ", ".join(f"{d} {v:.3f}" for d, v in per_d.items())]
    if 'BS_RESULTS' in globals() and len(BS_RESULTS['preds']):
        B = BS_RESULTS['preds']
        dn = B[B['model'] == 'gauss_dense'].groupby(['dataset', 'lag', 'h']).apply(rm, include_groups=False)
        r2 = (af / dn).dropna()
        k2 = pd.Series([KIND_OF.get(d, '') for d in r2.index.get_level_values('dataset')], index=r2.index)
        lines.append(f"RMSE {AF_NAME} / dense (non-gated) Gaussian: {gm(r2):.3f} overall | chaotic {gm(r2[k2 == 'chaotic']):.3f} | "
                     f"real {gm(r2[k2 == 'real']):.3f}")
    ch = AF_RESULTS['choices']
    fam = ch.groupby(['dataset', 'field_family']).size().unstack(fill_value=0)
    fam = fam.div(fam.sum(axis=1), axis=0)
    rev_csv(fam.reset_index(), 'trace_af_chosen_firing')
    act = ch.assign(share=ch['firing_neurons'] / (ch['lag'] * ch['M'].fillna(1) + ch['C'].fillna(0)))
    lines += ["", "Firing condition chosen (share of folds × lags × horizons):", fam.to_string(float_format=lambda v: f"{v:.0%}"),
              f"Median firing neurons per forecast: {ch['firing_neurons'].median():.0f}; median share of neurons firing "
              f"{act['share'].median():.1%} (chaotic {act[act['dataset'].map(KIND_OF) == 'chaotic']['share'].median():.1%}, "
              f"real {act[act['dataset'].map(KIND_OF) == 'real']['share'].median():.1%})"]
    # 2. the N = 11 ranking with TRACE-SNP-AF as the model under test
    S = dataset_scores(core)
    fr = friedman_n(S)
    WH = wilcoxon_holm(S, AF_NAME) if AF_NAME in S else pd.DataFrame()
    br = []
    for m in S.columns:
        if m != AF_NAME and AF_NAME in S:
            pl, pm, pr = bayes_signrank((np.log(S[AF_NAME]) - np.log(S[m])).to_numpy())
            br.append({'model': m, f'P({AF_NAME} better)': pl, 'P(equivalent)': pm, 'P(rival better)': pr})
    BR = pd.DataFrame(br).set_index('model') if br else pd.DataFrame()
    FRh = {}
    for h in sorted(core['h'].unique()):
        Sh = dataset_scores(core[core['h'] == h])
        if Sh.shape[0] >= 3:
            FRh[f'h={h}'] = friedman_n(Sh)
    for kd in ('chaotic', 'real'):
        Sk = S.loc[[d for d in S.index if KIND_OF.get(d) == kd]]
        if Sk.shape[0] >= 3:
            FRh[kd] = friedman_n(Sk)
    MR = pd.DataFrame({'all': fr['mean_rank'], **{k: v['mean_rank'] for k, v in FRh.items()}}).loc[fr['mean_rank'].index]
    rev_csv(MR.reset_index().rename(columns={'index': 'model'}), 'n11_mean_ranks_with_trace_af')
    if len(WH):
        rev_csv(WH.reset_index(), 'n11_wilcoxon_holm_vs_trace_af')
    if len(BR):
        rev_csv(BR.reset_index(), 'n11_bayesian_signed_rank_vs_trace_af')
    lines += ["", f"## Ranking over N = {fr['N']} datasets with {AF_NAME} (k = {fr['k']} models)",
              f"Friedman χ²({fr['k'] - 1}) = {fr['chi2']:.2f}, p = {fr['p']:.2e}; Iman–Davenport p = {fr['p_ID']:.2e}; Nemenyi CD = {fr['cd']:.2f}",
              MR.to_string(float_format=lambda v: f"{v:.2f}")]
    if len(WH):
        lines += ["", f"{AF_NAME} vs each model — Wilcoxon signed-rank over datasets, Holm:", WH.to_string(float_format=lambda v: f"{v:.3g}")]
    if len(BR):
        lines += ["", f"Bayesian signed-rank (ROPE ±1 %):", BR.to_string(float_format=lambda v: f"{v:.3f}")]
    k = fr['k']
    fig, ax = plt.subplots(figsize=(W2, 1.2 + 0.19 * k))
    cd_diagram(ax, fr, f"Mean rank over N = {fr['N']} datasets ({k} models; Friedman p = {fr['p']:.1e}; bars: no significant "
                       f"difference, Nemenyi α = {STAT_ALPHA})")
    save_fig(fig, 'fig_cd_n11_with_trace_af')
    t = MR.copy(); t.index = [short(m) for m in t.index]; t.columns = [c.replace('all', 'All') for c in t.columns]
    tex_table(t, 'tab_n11_mean_ranks_with_trace_af', f"Mean rank over the N = {fr['N']} datasets including {AF_NAME} (lower is "
              f"better; one score per dataset and model). Friedman $\\chi^2 = {fr['chi2']:.1f}$, $p = {fr['p']:.1e}$; Nemenyi CD = "
              f"{fr['cd']:.2f}.", 'tab:n11_ranks_af', fmt=lambda v: f"{v:.2f}", bold={'min': list(t.columns)}, idx_name='Model')
    # 3. energy, operations and noise of TRACE-SNP-AF next to TRACE-SNP v2
    en = AF_RESULTS['energy']
    o2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_ops.csv")
    e2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_energy.csv")
    EV = pd.DataFrame({
        f'{AF_NAME} µJ/forecast (GPU)': en.groupby('h')['infer_j_per_forecast'].median() * 1e6,
        f'{AF_NAME} MAC': ch.groupby('h')['MAC'].median(), f'{AF_NAME} 45 nm pJ': ch.groupby('h')['pJ'].median(),
        **({'TRACE-SNP v2 µJ/forecast (GPU)': e2[e2['model'] == 'TRACE-SNP v2'].groupby('h')['infer_j_per_forecast'].median() * 1e6} if len(e2) else {}),
        **({'TRACE-SNP v2 MAC': o2[o2['model'] == 'TRACE-SNP v2'].groupby('h')['MAC'].median(),
            'TRACE-SNP v2 45 nm pJ': o2[o2['model'] == 'TRACE-SNP v2'].groupby('h')['pJ'].median()} if len(o2) else {})})
    rev_csv(EV.reset_index(), 'trace_af_energy_ops')
    lines += ["", f"Inference cost per forecast (medians over datasets × lags):", EV.to_string(float_format=lambda v: f"{v:,.3g}")]
    nz = AF_RESULTS['noise']
    n2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_noise.csv")
    if len(nz):
        allnz = pd.concat([nz, n2[n2['model'] == 'TRACE-SNP v2']] if len(n2) else [nz], ignore_index=True)
        p1 = LB[(LB['h'] == 1) & (LB['model'] == 'Persistence')].set_index(['dataset', 'lag'])['rmse']
        g = allnz.groupby(['dataset', 'lag', 'model', 'noise'])['rmse'].mean().reset_index()
        g['rel_persistence'] = g['rmse'] / g.set_index(['dataset', 'lag']).index.map(p1).to_numpy()
        NT = g.groupby(['model', 'noise'])['rel_persistence'].median().unstack('noise')
        rev_csv(NT.reset_index(), 'trace_af_noise')
        lines += ["", "Input noise, h = 1 (RMSE / clean persistence, median over datasets × lags; same draws as every model):",
                  NT.to_string(float_format=lambda v: f"{v:.3f}")]
    txt = "\n".join(lines)
    print(txt)
    with open(rev_path(f"trace_af{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"trace_af{SM}.md"))
    store.push('revisions: TRACE-SNP-AF', force=True)
    return {'LB2': LB2, 'FR': fr, 'WH': WH, 'BR': BR, 'ratio_v2': rr, 'text': txt}


AF = _af_analysis()

In [ ]:
# ============================================================
# 19. PAPER NUMBERS — variance, uncertainty bands, and every table the manuscript cites (no network is retrained)
#     a) Seed spread of every network: RMSE of the 60 individual seeds per (dataset, lag, test block), from the
#        walkforward_folds_* files the training notebooks wrote (seedmean_rmse, seedstd_rmse, ens_rmse; h = 1).
#        The closed-form models have no seed: refitting gives the identical model.
#     b) Block-to-block variability of every model: RMSE / persistence RMSE on each of the 5 walk-forward test blocks
#        (h = 1), mean ± s.d. over blocks per dataset, and the share of blocks on which TRACE-SNP-AF is more accurate.
#     c) Per-problem Diebold–Mariano tests (HLN, Holm within dataset), TRACE-SNP-AF vs TRACE-SNP v2, all horizons.
#     d) Fig. 4 with uncertainty: the delay functions f_k of TRACE-SNP v2 refitted on every one of the 5 training
#        windows (the exact v2 solver and grid), with a pointwise 95 % band that combines the ridge sampling variance
#        (sandwich estimate) and the spread over the 5 refits; Mackey-Glass always shows k = 18.
#     e) paper_numbers.md: the dataset table, every setting, the N = 11 score matrix (overall and per horizon) with
#        TRACE-SNP-AF, and the tables above, in one file to send for the manuscript.
#     Cost: a few minutes (the refits in d are 4 datasets × 5 blocks at L = 20, h = 1).
# ============================================================
VAR_FIG_LAG   = FIG4_LAG
VAR_FIG_GRID  = 81
VAR_KEY       = [AF_NAME, 'TRACE-SNP v2', 'TRACE-SNP', 'ALSTM-SNP', 'LSTM-SNP [25]', 'BiLSTM-SNP [26]', 'GRU [39]',
                 'LSTM [34]', 'Attention', 'Informer [42]', 'FEDformer [41]', 'Autoformer [40]', 'NG-RC', 'DLinear',
                 'Ridge AR(L)']


def run_delay_bands(datasets):
    from sklearn.preprocessing import MinMaxScaler as _MMS
    f64, dev, G = torch.float64, device, ABL_GRID
    T = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=f64, device=dev)
    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
    LO, HI = -1.1, 1.1
    BASES = {'tri': ('tri', 2, True)}                          # TRACE-SNP v2 only; the engine below is cell 17's

    def fields(U, base, M, w):
        shape, k, _ = BASES[base]
        h = (HI - LO) / (M - 1)
        if shape == 'tri':                                   # identical to TRACE-SNP v2 (2 neighbouring fields fire)
            pos = (U.clamp(LO, HI) - LO) / h
            i = pos.floor().long().clamp(0, M - 2)
            fr = pos - i
            B = torch.zeros(U.shape + (M,), dtype=U.dtype, device=U.device)
            B.scatter_(-1, i[..., None], (1 - fr)[..., None])
            B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
            return B.reshape(U.shape[0], -1)
        c = torch.linspace(LO, HI, M, dtype=U.dtype, device=U.device)
        d = (U.clamp(LO, HI)[..., None] - c).abs()
        if shape == 'hat':                                   # hat of half-width M·h: every field > 0 on the whole range
            B = 1 - d / (M * h)
        else:
            B = torch.exp(-d ** 2 / (2 * (w * h) ** 2))
            if k is not None:                                # firing condition: only the k nearest fields fire
                thr = d.kthvalue(k, dim=-1, keepdim=True).values
                B = B * (d <= thr)
        return B.reshape(U.shape[0], -1)

    def kmeans(Z, C, seed=0):                                 # identical to cell 10
        g = torch.Generator(device='cpu').manual_seed(seed)
        idx = [int(torch.randint(len(Z), (1,), generator=g))]
        dmin = ((Z - Z[idx[0]]) ** 2).sum(1)
        for _ in range(C - 1):
            j = int(torch.multinomial((dmin / dmin.sum()).cpu(), 1, generator=g)); idx.append(j)
            dmin = torch.minimum(dmin, ((Z - Z[j]) ** 2).sum(1))
        cen = Z[idx].clone()
        for _ in range(20):
            lab = torch.cdist(Z, cen).argmin(1)
            cnt = torch.bincount(lab, minlength=C).to(Z.dtype)
            s_ = torch.zeros_like(cen).index_add_(0, lab, Z)
            keep = cnt > 0
            cen[keep] = s_[keep] / cnt[keep, None]
        nn_d = torch.cdist(cen, cen) + torch.eye(C, dtype=Z.dtype, device=Z.device) * 1e9
        return cen, float(nn_d.min(1).values.median() ** 2)

    def state_feats(U, st, gated):
        cen, s2, d, off = st
        Z = U[:, off:off + d]
        D2 = torch.cdist(Z, cen) ** 2
        if gated:
            val, idx = D2.topk(G['k_active'], dim=1, largest=False)
            g = torch.exp(-val / (2 * s2)); g = g / g.sum(1, keepdim=True)
            Mu = torch.zeros_like(D2).scatter_(1, idx, g)
        else:
            Mu = torch.softmax(-D2 / (2 * s2), dim=1)
        Z1 = torch.cat([torch.ones(len(U), 1, dtype=U.dtype, device=U.device), Z], 1)
        return (Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1)

    def design(U, base, M=None, w=None, st=None):
        parts = [torch.ones(len(U), 1, dtype=U.dtype, device=U.device), U]
        if M:
            parts.append(fields(U, base, M, w))
        if st is not None:
            parts.append(state_feats(U, st, BASES[base][2]))
        return torch.cat(parts, 1)

    def penalty(L, M, l0, ls, n_state=0, lst=0.0):            # identical to cell 10
        nd = L * M if M else 0
        p = 1 + L + nd + n_state
        P = torch.zeros(p, p, dtype=f64, device=dev)
        P[1:, 1:] += l0 * torch.eye(p - 1, dtype=f64, device=dev)
        if M and ls > 0:
            D = torch.zeros(M - 2, M, dtype=f64, device=dev)
            for j in range(M - 2):
                D[j, j], D[j, j + 1], D[j, j + 2] = 1, -2, 1
            DD = ls * (D.T @ D)
            for k in range(L):
                a0 = 1 + L + k * M
                P[a0:a0 + M, a0:a0 + M] += DD
        if n_state and lst > l0:
            a0 = 1 + L + nd
            P[a0:, a0:] += (lst - l0) * torch.eye(n_state, dtype=f64, device=dev)
        P[0, 0] = 1e-10
        return P

    def fit_loo(Ptr, R, Pen, chunk=8192):                     # identical to cell 10
        N = len(Ptr)
        A = Ptr.T @ Ptr / N + Pen
        Lc = torch.linalg.cholesky(A)
        th = torch.cholesky_solve(Ptr.T @ R / N, Lc)
        loo = []
        for i in range(0, N, chunk):
            P = Ptr[i:i + chunk]
            V = torch.linalg.solve_triangular(Lc, P.T, upper=False)
            hd = ((V ** 2).sum(0) / N).clamp(max=1 - 1e-6)
            loo.append((R[i:i + chunk] - P @ th) / (1 - hd)[:, None])
        return th, torch.cat(loo)

    def windows(x, L, o):
        return np.stack([x[o - 1 - k] for k in range(L)], 1)

    def run_fold(x, L, h, f):
        otr = np.arange(L, f['va_start'] - h + 1)
        ova = np.arange(f['va_start'], f['te_start'] - h + 1)
        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
        a, b = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / (b - a)
        s = lambda v: sc * (v - a) - 1
        W = {'tr': windows(x, L, otr), 'va': windows(x, L, ova), 'te': windows(x, L, ote)}
        y = {'tr': x[otr + h - 1], 'va': x[ova + h - 1], 'te': x[ote + h - 1]}
        anc_raw = {'last': lambda Wr: Wr[:, 0], 'mean': lambda Wr: Wr.mean(1)}
        dd = (W['tr'][:, 1:] - W['tr'][:, :1]) if L > 1 else np.zeros((len(otr), 1))
        scd = 1.0 / max(float(np.abs(dd).max()), 1e-12)
        enc = {'lvl': lambda Wr: T(s(Wr)),
               'dif': lambda Wr: T(np.column_stack([np.zeros(len(Wr)), (Wr[:, 1:] - Wr[:, :1]) * scd]) if L > 1
                                   else np.zeros((len(Wr), 1)))}
        cands = []

        def add(fam, base, cfg, loo_raw, va_pred, te_pred, fn):
            e_va = y['va'] - va_pred
            cands.append({'fam': fam, 'basis': base, 'cfg': cfg, 'pool': rmse(np.concatenate([loo_raw, e_va])),
                          'va_pred': va_pred, 'te_pred': te_pred, 'fn': fn})

        for an in ('last', 'mean'):
            add('persistence' if an == 'last' else 'window-mean', 'fallback', {'anchor': an}, y['tr'] - anc_raw[an](W['tr']),
                anc_raw[an](W['va']), anc_raw[an](W['te']), anc_raw[an])
        mm = _MMS((-1, 1)).fit(np.column_stack([W['tr'], y['tr']]))
        tf = lambda Wr: Wr * mm.scale_[:-1] + mm.min_[:-1]
        Xc = tf(W['tr']); yy = y['tr'] * mm.scale_[-1] + mm.min_[-1]
        xm, ym = Xc.mean(0), yy.mean(); Xc0 = Xc - xm
        Ai = np.linalg.inv(Xc0.T @ Xc0 + RIDGE_ALPHA * np.eye(L)); wr = Ai @ Xc0.T @ (yy - ym)
        hr = 1 / len(Xc) + np.einsum('ij,jk,ik->i', Xc0, Ai, Xc0)
        rf = lambda Wr: ((tf(Wr) - xm) @ wr + ym - mm.min_[-1]) / mm.scale_[-1]
        add('ridge', 'fallback', {}, ((yy - (Xc0 @ wr + ym)) / (1 - hr)) / mm.scale_[-1], rf(W['va']), rf(W['te']), rf)
        ys = T(s(y['tr']))

        def make_fn(e, an, base, M, w, st, th):
            def fn(Wr):
                return ((T(s(anc_raw[an](Wr))) + design(enc[e](Wr), base, M, w, st) @ th).cpu().numpy() + 1) / sc + a
            fn.meta = {'e': e, 'an': an, 'M': M, 'w': w, 'st': st, 'th': th, 'sc': sc, 'a': a, 'scd': scd,
                       'Utr': lambda: enc[e](W['tr']), 'R': ys - T(s(anc_raw[an](W['tr'])))}
            return fn

        anchors = ('last', 'mean')
        for e in ('lvl', 'dif'):
            if e == 'dif' and L < 3:
                continue
            U = {k: enc[e](W[k]) for k in W}
            A_ = {k: torch.stack([T(s(anc_raw[an](W[k]))) for an in anchors], 1) for k in W}
            R = ys[:, None] - A_['tr']
            states = {}
            dmax = L if e == 'lvl' else L - 1
            for d in sorted({min(dmax, v) for v in G['d']}):
                for C in G['C']:
                    if C * (d + 1) > len(otr):
                        continue
                    Z = U['tr'][:, :d] if e == 'lvl' else U['tr'][:, 1:d + 1]
                    cen, s2 = kmeans(Z, C)
                    states[(d, C)] = (cen, s2, d, 0 if e == 'lvl' else 1)
            for base in BASES:
                widths = RBF_WIDTHS if BASES[base][0] == 'gauss' else [None]
                fam0 = f"{base}|{e}"
                for M in G['M']:
                    if L * M > G['max_delay_units'] and M != min(G['M']):
                        continue
                    if BASES[base][1] is not None and BASES[base][1] >= M:
                        continue
                    for w in widths:
                        P = {k: design(U[k], base, M, w) for k in U}
                        for l0 in G['l0']:
                            for ls in G['ls']:
                                th, loo = fit_loo(P['tr'], R, penalty(L, M, l0, ls))
                                pv, pt = (A_['va'] + P['va'] @ th).cpu().numpy(), (A_['te'] + P['te'] @ th).cpu().numpy()
                                for j, an in enumerate(anchors):
                                    add(f"{an}|{fam0}|delay", base, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls},
                                        loo[:, j].cpu().numpy() / sc, (pv[:, j] + 1) / sc + a, (pt[:, j] + 1) / sc + a,
                                        make_fn(e, an, base, M, w, None, th[:, j].clone()))
                        del P
                for j, an in enumerate(anchors):
                    pool = [c for c in cands if c['fam'] == f"{an}|{fam0}|delay"]
                    if not pool:
                        continue
                    best = min(pool, key=lambda c: c['pool'])['cfg']
                    M, w, l0, ls = best['M'], best['w'], best['l0'], best['ls']
                    for (d, C), st in states.items():
                        P = {k: design(U[k], base, M, w, st) for k in U}
                        for lst in G['lst']:
                            th, loo = fit_loo(P['tr'], R[:, j:j + 1], penalty(L, M, l0, ls, C * (d + 1), max(lst, l0)))
                            pv = (A_['va'][:, j:j + 1] + P['va'] @ th).cpu().numpy()[:, 0]
                            pt = (A_['te'][:, j:j + 1] + P['te'] @ th).cpu().numpy()[:, 0]
                            add(f"{an}|{fam0}|delay+state", base, {'anchor': an, 'enc': e, 'M': M, 'w': w, 'l0': l0, 'ls': ls,
                                                                   'd': d, 'C': C, 'lst': lst},
                                loo[:, 0].cpu().numpy() / sc, (pv + 1) / sc + a, (pt + 1) / sc + a,
                                make_fn(e, an, base, M, w, st, th[:, 0].clone()))
                        del P
        return cands, y, ova, ote

    def pick(cands, base, fallback=True):
        pool = [c for c in cands if c['basis'] == base or (fallback and c['basis'] == 'fallback')]
        return min(pool, key=lambda c: c['pool'])

    def iterate(fn, x, L, origins, h):
        Wr = windows(x, L, origins)
        for _ in range(h):
            yhat = fn(Wr)
            Wr = np.column_stack([yhat, Wr[:, :-1]])
        return yhat

    def active(c, L):
        # active receptive-field + state neurons per one-step evaluation, and all of them
        if c['basis'] == 'fallback':
            return 0.0, 0.0
        M = c['cfg']['M']; k = BASES[c['basis']][1]
        on, tot = float(L * (k if k is not None else M)), float(L * M)
        if 'C' in c['cfg'] and np.isfinite(c['cfg'].get('C', np.nan)):
            C = c['cfg']['C']; on += G['k_active'] if BASES[c['basis']][2] else C; tot += C
        return on, tot

    rows, checks = [], []
    for D in datasets:
        x, L = D['values'], VAR_FIG_LAG
        if not lag_ok(None, D, L):
            checks.append(f"{D['name']}: L = {L} too long for this series"); continue
        for f in D['folds']:
            cands, y, ova, ote = run_fold(x, L, 1, f)
            c = pick(cands, 'tri')
            m = getattr(c['fn'], 'meta', None)
            if c['basis'] == 'fallback' or m is None:
                checks.append(f"{D['name']} {f['label']}: fallback chosen ({c['fam']}), no delay functions"); continue
            cfg, M = c['cfg'], m['M']
            Ptr = design(m['Utr'](), 'tri', M, m['w'], m['st'])
            R = m['R'][:, None]
            N, p = Ptr.shape
            n_state = p - (1 + L + L * M)
            Pen = penalty(L, M, cfg['l0'], cfg['ls'], n_state, max(cfg.get('lst', 0.0) or 0.0, cfg['l0']))
            Gm = Ptr.T @ Ptr / N
            A = Gm + Pen
            th = torch.linalg.solve(A, Ptr.T @ R / N)
            dth = float((th[:, 0] - m['th'].reshape(-1)).abs().max())
            s2 = float(((R - Ptr @ th) ** 2).mean())
            Ai = torch.linalg.inv(A)
            Cov = s2 / N * Ai @ Gm @ Ai                         # sandwich covariance of the ridge estimate
            grid = torch.linspace(-1, 1, VAR_FIG_GRID, dtype=f64, device=dev)
            Bg = fields(grid[:, None].repeat(1, L), 'tri', M, m['w'])
            u_raw = ((grid + 1) / m['sc'] + m['a']) if m['e'] == 'lvl' else grid / m['scd']
            for k in range(1 if m['e'] == 'dif' else 0, L):
                Gk = torch.zeros(len(grid), p, dtype=f64, device=dev)
                Gk[:, 1 + k] = grid
                Gk[:, 1 + L + k * M:1 + L + (k + 1) * M] = Bg[:, k * M:(k + 1) * M]
                Gk = Gk - Gk.mean(0, keepdim=True)             # centred: the level of each f_k is not identified
                fk = (Gk @ th)[:, 0] / m['sc']
                se = torch.sqrt(((Gk @ Cov) * Gk).sum(1).clamp(min=0)) / m['sc']
                rows.append(pd.DataFrame({'dataset': D['name'], 'lag': L, 'fold': f['label'], 'enc': m['e'], 'anchor': m['an'],
                                          'delay': k + 1, 'u': u_raw.cpu().numpy(), 'f': fk.cpu().numpy(), 'se': se.cpu().numpy()}))
            checks.append(f"{D['name']} {f['label']}: {c['fam']}, M = {M} | refit vs selected θ, max |Δ| = {dth:.1e}")
            del cands
    return (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()), checks


def _paper_numbers():
    gm = lambda v: float(np.exp(np.mean(np.log(np.asarray(v, float)))))
    f3 = lambda v: f"{v:.3f}"
    lines = []

    # ── a) seed spread of the networks (h = 1) ──
    WF = read_many('*/*/walkforward_folds_*.csv')
    SEED = pd.DataFrame()
    if len(WF):
        WF = WF[WF['dataset'].isin(ALL11_NAMES)].copy()
        WF['seed_cv'] = WF['seedstd_rmse'] / WF['seedmean_rmse']
        WF['ens_over_seedmean'] = WF['ens_rmse'] / WF['seedmean_rmse']
        WF['seedstd_over_persist'] = WF['seedstd_rmse'] / WF['persist_rmse']
        SEED = WF.groupby('model').agg(blocks=('seed_cv', 'size'), seed_cv_median=('seed_cv', 'median'),
                                       seed_cv_p90=('seed_cv', lambda s: s.quantile(0.9)),
                                       ens_over_seedmean=('ens_over_seedmean', 'median'),
                                       seedstd_over_persist=('seedstd_over_persist', 'median')).sort_values('seed_cv_median')
        rev_csv(SEED.reset_index(), 'variance_seed_spread')
        rev_csv(WF.groupby(['model', 'dataset'])['seed_cv'].median().unstack().reset_index(), 'variance_seed_cv_by_dataset')
        t = SEED.reset_index()[['model', 'seed_cv_median', 'seed_cv_p90', 'ens_over_seedmean', 'seedstd_over_persist']]
        t.columns = ['Model', 'Seed CV (median)', 'Seed CV (90th pct.)', 'Ensemble / mean seed RMSE', 'Seed s.d. / persistence RMSE']
        tex_table(t, 'tab_seed_spread', "Spread of the test RMSE over the 60 seeds of every network, one-step forecasts, per "
                  "(dataset, lag, test block); CV = s.d./mean over seeds. The closed-form models have no seed.",
                  'tab:seed', fmt=lambda v: f"{v:.3f}")
        lines += ["## Seed spread of the networks (h = 1; per dataset × lag × test block, 60 seeds)",
                  SEED.to_string(float_format=f3)]
    else:
        lines.append("No walkforward_folds_* files: seed spread not available.")

    # ── b) block-to-block variability (h = 1 for every model; all horizons for the closed-form models) ──
    A = AF_RESULTS['preds'] if 'AF_RESULTS' in globals() and len(AF_RESULTS['preds']) else read_many(
        f"{REV}/{AF_FOLDER}/*_lag*_preds.csv")
    V2 = read_many(f"*/trace_snp_v2b/t2{SM}_lag*_preds.csv")
    V2 = V2[V2['model'] == 'TRACE-SNP v2'] if len(V2) else V2
    V1 = read_many(f"*/closed_form/cf{SM}_v2_lag*_h*_preds.csv")
    V1 = V1[V1['model'].isin(VAR_KEY)] if len(V1) else V1
    allp = pd.concat([d for d in (A, V2, V1) if len(d)], ignore_index=True)
    allp = allp[allp['dataset'].isin(ALL11_NAMES)]
    g = allp.groupby(['dataset', 'model', 'lag', 'h', 'test_fold'])
    FS = pd.DataFrame({'rmse': g.apply(lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2))), include_groups=False),
                       'persist': g.apply(lambda d: float(np.sqrt(np.mean((d['y'] - d['prev']) ** 2))), include_groups=False)}
                      ).reset_index()
    key = ['dataset', 'lag', 'h', 'test_fold']
    PERS = FS[FS['model'] == AF_NAME].set_index(key)['persist']
    FS['rel'] = FS['rmse'] / FS.set_index(key).index.map(PERS).to_numpy()
    if len(WF):
        wf = WF[['dataset', 'model', 'lag', 'test_fold', 'ens_rmse']].rename(columns={'ens_rmse': 'rmse'}).assign(h=1)
        wf['rel'] = wf['rmse'] / wf.set_index(key).index.map(PERS).to_numpy()
        FS = pd.concat([FS, wf], ignore_index=True)
    FS = FS[FS['rel'].notna() & (FS['rel'] > 0) & FS['model'].isin(VAR_KEY)]
    rev_csv(FS, 'variance_block_scores')
    B1 = FS[FS['h'] == 1]
    BLK = B1.groupby(['model', 'dataset', 'test_fold'])['rel'].apply(gm).groupby(['model', 'dataset']).agg(['mean', 'std', 'count'])
    rev_csv(BLK.reset_index(), 'variance_block_by_dataset')
    cell = BLK.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1).unstack('model')
    cell = cell[[m for m in VAR_KEY if m in cell.columns]]
    blk_cv = (BLK['std'] / BLK['mean']).groupby('model').median().reindex([m for m in VAR_KEY if m in BLK.index.get_level_values(0)])
    piv = FS.pivot_table(index=key, columns='model', values='rel')
    win = {}
    if AF_NAME in piv:
        for m in piv.columns:
            if m == AF_NAME:
                continue
            ok = piv[[AF_NAME, m]].dropna()
            ok1 = ok[ok.index.get_level_values('h') == 1]
            win[m] = {'blocks (h = 1)': len(ok1), 'AF more accurate (h = 1)': float((ok1[AF_NAME] < ok1[m]).mean()) if len(ok1) else np.nan,
                      'blocks (all h)': len(ok), 'AF more accurate (all h)': float((ok[AF_NAME] < ok[m]).mean()) if len(ok) else np.nan}
    WIN = pd.DataFrame(win).T
    if len(WIN):
        rev_csv(WIN.reset_index().rename(columns={'index': 'model'}), 'variance_block_win_rate_trace_af')
    if len(cell):
        tcols = [m for m in [AF_NAME, 'TRACE-SNP v2', 'ALSTM-SNP', 'LSTM-SNP [25]', 'BiLSTM-SNP [26]', 'Informer [42]'] if m in cell]
        tex_table(cell[tcols].reset_index().rename(columns={'dataset': 'Dataset'}), 'tab_block_variability',
                  "RMSE relative to persistence on each of the five walk-forward test blocks (one-step forecasts, geometric "
                  "mean over lags): mean $\\pm$ s.d. over blocks.", 'tab:blocks', fmt=str)
    lines += ["", "## Block-to-block variability, h = 1 (RMSE / persistence per test block, geo-mean over lags; mean ± s.d. over the 5 blocks)",
              cell.to_string(), "", "Median over datasets of the block CV (s.d. / mean):", blk_cv.to_string(float_format=f3),
              "", f"Share of test blocks on which {AF_NAME} is more accurate:", WIN.to_string(float_format=f3) if len(WIN) else "(none)"]

    # ── c) per-problem DM tests, TRACE-SNP-AF vs TRACE-SNP v2 ──
    if len(A) and len(V2):
        Mg = A.merge(V2, on=['dataset', 'lag', 'h', 'test_fold', 'origin'], suffixes=('_af', '_v2'))
        recs = []
        for (d, L, h), gg in Mg.groupby(['dataset', 'lag', 'h']):
            e1, e2 = (gg['y_af'] - gg['ens_pred_af']).to_numpy(), (gg['y_v2'] - gg['ens_pred_v2']).to_numpy()
            if len(e1) < 10 or np.allclose(e1, e2):
                recs.append({'dataset': d, 'lag': L, 'h': h, 'dm': 0.0, 'p': 1.0}); continue
            st_, p_ = dm_test(e1, e2, h)
            recs.append({'dataset': d, 'lag': L, 'h': h, 'dm': st_, 'p': p_})
        DM = pd.DataFrame(recs)
        if len(DM):
            DM['p_holm'] = DM.groupby('dataset')['p'].transform(lambda s: pd.Series(holm(s.to_numpy()), index=s.index))
            DM['outcome'] = np.where(DM['p_holm'] >= STAT_ALPHA, 'no sig. diff.', np.where(DM['dm'] < 0, 'AF better', 'v2 better'))
            rev_csv(DM, 'dm_trace_af_vs_v2')
            T_ = DM.groupby(['dataset', 'outcome']).size().unstack(fill_value=0)
            lines += ["", f"## Diebold–Mariano per problem, {AF_NAME} vs TRACE-SNP v2 (HLN, Holm within dataset, α = {STAT_ALPHA})",
                      T_.to_string(), "Total: " + ", ".join(f"{k} {v}" for k, v in DM['outcome'].value_counts().items())]

    # ── d) Fig. 4 with uncertainty bands (TRACE-SNP v2 refitted on each of the 5 training windows) ──
    pick = [D for n in FIG4_DATASETS for D in ALL11 if D['name'] == n][:4]
    CVb, chk = run_delay_bands(pick)
    lines += ["", f"## Fig. 4 refits (L = {VAR_FIG_LAG}, h = 1)"] + chk
    if len(CVb):
        rev_csv(CVb, 'delay_functions_with_se')
        fig, axes = plt.subplots(1, len(pick), figsize=(W2, 2.4), squeeze=False)
        dcols = [OI['vermillion'], OI['blue'], OI['green'], OI['purple'], OI['orange']]
        dls = ['-', '--', '-.', ':', '-']
        for ax, D in zip(axes[0], pick):
            c = CVb[CVb['dataset'] == D['name']]
            if c.empty:
                ax.set_axis_off(); continue
            enc = c['enc'].mode().iloc[0]; c = c[c['enc'] == enc]
            folds = list(c['fold'].unique())
            lo = c.groupby('fold')['u'].min().max(); hi = c.groupby('fold')['u'].max().min()
            uu = np.linspace(lo, hi, VAR_FIG_GRID)
            curves = {}
            for k, ck in c.groupby('delay'):
                F = np.array([np.interp(uu, cf['u'], cf['f']) for _, cf in ck.groupby('fold')])
                S = np.array([np.interp(uu, cf['u'], cf['se']) for _, cf in ck.groupby('fold')])
                mu = F.mean(0)
                band = 1.96 * np.sqrt((S ** 2).mean(0) + (F.var(0, ddof=1) if len(F) > 1 else 0))
                curves[k] = (mu, band)
            amp = pd.Series({k: v[0].max() - v[0].min() for k, v in curves.items()}).sort_values(ascending=False)
            forced = [k for kw, k in FIG4_FORCE.items() if kw in D['name'].lower() and k in curves]
            for k in forced:
                mu, band = curves[k]
                lines.append(f"{D['name']}: k = {k} amplitude rank {list(amp.index).index(k) + 1} of {len(amp)} (mean over "
                             f"{len(folds)} refits); its 95 % band excludes 0 on {np.mean(np.abs(mu) > band):.0%} of the input range")
            show_k = forced + [k for k in amp.index if k not in forced][:FIG4_N_DELAYS - len(forced)]
            lines.append(f"{D['name']}: delays drawn {show_k}; top by amplitude {list(amp.index[:5])}; encoding {enc}; "
                         f"{len(folds)} refits")
            for i_, k in enumerate(show_k):
                mu, band = curves[k]
                ax.fill_between(uu, mu - band, mu + band, color=dcols[i_ % 5], alpha=0.18, lw=0)
                ax.plot(uu, mu, lw=1.6 if k in forced else 1.0, color=dcols[i_ % 5], ls=dls[i_ % 5],
                        label=f"k = {k}" + (" (τ+1)" if k in forced and 'mackey' in D['name'].lower() else ''))
            ax.axhline(0, color='0.6', lw=0.5)
            ax.set_title(f"{D['name']} (L = {VAR_FIG_LAG})", loc='left', fontsize=7)
            ax.legend(frameon=False, fontsize=5.5, handlelength=1.8, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.32),
                      columnspacing=0.8)
            ax.set_xlabel('x(t−k)' if enc == 'lvl' else 'x(t−k) − x(t−1)', fontsize=6.5)
            ax.tick_params(labelsize=5.5)
        axes[0, 0].set_ylabel('contribution $f_k$ (centred)')
        fig.tight_layout(); save_fig(fig, 'fig_delay_functions_bands')

    # ── e) everything the manuscript cites, in one file ──
    ds = pd.DataFrame([{'dataset': D['name'], 'type': D['kind'], 'points': D['n'], 'test block': D['block'],
                        'validation': D['val'], 'folds': len(D['folds']),
                        'test span': f"{D['folds'][0]['te_start']}–{D['folds'][-1]['te_end'] - 1}"} for D in ALL11])
    settings = {'lags': COMMON_LAGS, 'horizons': ABL_HORIZONS, 'walk-forward folds': WF_N_FOLDS, 'BLOCK_LEN': BLOCK_LEN,
                'VAL_LEN': VAL_LEN, 'TEST_FRAC_MAX': TEST_FRAC_MAX, 'TRACE-SNP v2 grid': ABL_GRID, 'Gaussian widths': RBF_WIDTHS,
                'iterated-forecast margin': ABL_ITER_MARGIN, 'AF tolerance': AF_TOL, 'AF families': AF_BASES,
                'Ridge AR alpha': RIDGE_ALPHA, 'noise levels': NOISE_LEVELS, 'noise draws': N_NOISE_DRAWS,
                'pJ per MAC / AC': (PJ_MAC, PJ_AC), 'ensemble members': NET_MEMBERS, 'ROPE': BS_ROPE}
    out = [f"# Paper numbers — {time.strftime('%Y-%m-%d %H:%M')} — {LOCAL_ROOT}", "", "## Datasets", ds.to_string(index=False),
           "", "## Settings"] + [f"- {k}: {v}" for k, v in settings.items()]
    if 'NET_PARAMS' in globals():
        out += ["", "## Networks", NET_PARAMS.to_string(index=False)]
    if AF is not None:
        LB2 = AF['LB2']
        core = LB2[~LB2['model'].isin(STAT_EXCLUDE)]
        ka = pd.MultiIndex.from_frame(LB2[LB2['model'] == AF_NAME][['dataset', 'lag', 'h']])
        core = core[pd.MultiIndex.from_frame(core[['dataset', 'lag', 'h']]).isin(ka)]
        S = dataset_scores(core)
        out += ["", "## N = 11 score matrix: geo-mean RMSE / persistence per dataset (all horizons)", S.T.to_string(float_format=f3)]
        rev_csv(S.reset_index().rename(columns={'index': 'dataset'}), 'n11_scores_with_trace_af')
        for h in sorted(core['h'].unique()):
            Sh = dataset_scores(core[core['h'] == h])
            out += ["", f"## Score matrix, h = {h}", Sh.T.to_string(float_format=f3)]
        r = LB2[LB2['model'].isin([AF_NAME, 'TRACE-SNP v2'])].pivot_table(index=['dataset', 'lag', 'h'], columns='model', values='rmse')
        if AF_NAME in r and 'TRACE-SNP v2' in r:
            rr = np.log(r[AF_NAME] / r['TRACE-SNP v2'])
            out += ["", f"## {AF_NAME} / TRACE-SNP v2 RMSE per dataset and horizon (geo-mean over lags)",
                    np.exp(rr.groupby(['dataset', 'h']).mean()).unstack('h').to_string(float_format=f3)]
    for nm in ('STATS', 'ENERGY', 'BASIS', 'AF'):
        obj = globals().get(nm)
        if isinstance(obj, dict) and obj.get('text'):
            out += ["", f"## Cell output: {nm}", obj['text']]
    out += [""] + lines
    txt = "\n".join(out)
    with open(rev_path(f"paper_numbers{SM}.md"), 'w') as fh:
        fh.write(txt + "\n")
    store.add(rev_path(f"paper_numbers{SM}.md"))
    store.push('revisions: paper numbers', force=True)
    print("\n".join(lines))
    print(f"\nWrote {REV}/paper_numbers{SM}.md — send this file for the manuscript.")
    return {'seed': SEED, 'blocks': BLK, 'win': WIN, 'text': txt}


PAPER = _paper_numbers()

## Part D — Control experiments (cells 20–25)
Tuned networks, networks trained directly for each horizon, ALSTM-SNP diagnostics, an echo-type SNP reservoir baseline, and the common-protocol energy benchmark. Cells 21–23 are the long ones (about 15–25 GPU hours together).

In [ ]:
# ============================================================
# 20. CONTROL EXPERIMENTS — settings and helpers of cells 21–25
#     21  networks tuned on the validation block (learning rate × width), every network on one protocol
#     22  networks trained directly for h = 6 and 24 (no iteration)
#     23  ALSTM-SNP diagnostics: matched protocol, no firing condition, no threshold adaptation, surrogate slope 1
#     24  an echo-type SNP reservoir baseline (fixed SNP reservoir + ridge readout, closed form)
#     25  GPU inference energy of TRACE-SNP v1, v2 and TRACE-SNP-AF with one benchmark code path
#     every result table → <LOCAL_ROOT>/revisions/r2_*.csv, read by the PAPER FINAL cell (26)
#     The harness takes the two new settings as spec entries, so nothing of cells 1–12 is redefined:
#     spec['lr'] may be a list (seed s trains with lr[s % len]) and spec['h'] is the forecast step of the windows.
#     Every cell is resumable: finished (dataset, model, lag) runs are reused from the results folder after a restart.
#     Time: about 15–25 GPU hours in total on an RTX 6000-class GPU (cell 21 is most of it); every cell stops cleanly at
#     TIME_BUDGET_H with everything saved, and a rerun continues where it stopped.
# ============================================================
R2_TAG        = 'r2'                                   # sub-folder of every new run: <dataset>/r2_*/...
RUN_TUNE      = True
RUN_DIRECT    = True
RUN_DIAG      = True
RUN_ECHO      = True
RUN_ENERGY    = True

# 31 — validation tuning: every network, including ALSTM-SNP, on ALSTM-SNP's protocol (batch 8, ≤300 epochs,
#      patience 40), learning rate × width grid, 5 seeds per learning rate; the published default configuration is a
#      candidate too, so the tuned network is never worse than the default one on the validation block
TUNE_LAGS     = [1, 5] if SMOKE else [5, 20]
TUNE_HIDDEN   = [8] if SMOKE else [4, 8]          # widths within the iso-parametric budget
TUNE_LRS      = [1e-3, 3e-3, 1e-2]
TUNE_SEEDS    = 1 if SMOKE else 5                 # seeds per learning rate (seed s uses TUNE_LRS[s % 3])
TUNE_TRAIN    = dict(batch=8, max_epochs=2 if SMOKE else 300, patience=2 if SMOKE else 40)

# 32 — direct multi-step training (one network per horizon), learning rate chosen on validation as in 31
DIRECT_LAGS   = [5] if SMOKE else [20]
DIRECT_H      = [6] if SMOKE else [6, 24]
DIRECT_HIDDEN = 8

# 33 — ALSTM-SNP diagnostics (5 seeds each, h = 1); the reference is retrained with the same 5 seeds
DIAG_LAGS     = [5] if SMOKE else [5, 20]
DIAG_SEEDS    = 2 if SMOKE else 5
DIAG_VARIANTS = {   # name: (cell changes, training changes)
    'reference':            ({}, {}),
    'matched protocol':     ({}, dict(batch=BASELINE_TRAIN['batch'], lr=BASELINE_TRAIN['lr'],
                                      max_epochs=BASELINE_TRAIN['max_epochs'], patience=BASELINE_TRAIN['patience'])),
    'no firing condition':  ({'spiking': False}, {}),
    'no threshold adaptation': ({'adaptive': False}, {}),
    'surrogate slope 1':    ({'gamma_sg': 1.0}, {}),
}

# 34 — echo-type SNP reservoir (closed form)
ECHO_NAME     = 'Echo-SNP reservoir'
ECHO_UNITS    = 64                                # readout: 2·64 + L + 1 weights, inside the networks' budget
ECHO_DENSITY  = 0.1
ECHO_THETA    = 0.0                               # firing threshold of a reservoir neuron
ECHO_RHO      = [0.5, 0.9]                        # spectral radius
ECHO_LEAK     = [0.3, 1.0]
ECHO_INSCALE  = [0.5, 1.0]
ECHO_LAMBDA   = [1e-4, 1e-2, 1.0]
ECHO_SEEDS    = 2 if SMOKE else 5                 # reservoirs per problem; the forecast is their mean
ECHO_LAGS     = [1, 5] if SMOKE else COMMON_LAGS
ECHO_H        = [1, 6, 24]
ECHO_MIN_ORIGINS = 5

# 35 — inference energy with one code path for the three closed-form members
EN_BENCH_N    = 18000                             # forecasts per call, as in the networks' benchmark
EN_BENCH_S    = 0.3 if SMOKE else 2.0
R2_DIR = os.path.join(LOCAL_ROOT, 'revisions')
os.makedirs(R2_DIR, exist_ok=True)


def r2_csv(df, name):
    p = os.path.join(R2_DIR, f"r2_{name}{SM}.csv"); df.to_csv(p, index=False); store.add(p); return p


def r2_read(name):
    p = store.fetch(f"revisions/r2_{name}{SM}.csv")
    return pd.read_csv(p) if p and os.path.getsize(p) > 2 else pd.DataFrame()


FOLDER_OF = {s['name']: s['folder'] for s in MODEL_SPECS}
NET_NAMES = [s['name'] for s in MODEL_SPECS]


def r2_build(base, hidden, **cell):
    if base == ALSTM_NAME:
        c = {**ALSTM_CELL, **cell}
        return lambda: ALSTMSNP(input_size=1, hidden_size=hidden, num_layers=c['layers'], frac_adaptive=c['frac_adaptive'],
                                tau_a_range=c['tau_a_range'], tau_m=c['tau_m'], gamma=c['gamma_sg'], spiking=c['spiking'],
                                adaptive=c['adaptive'], strict=c['strict'])
    return lambda cls=BASELINE_CLASSES[FOLDER_OF[base]]: cls(hidden)


def r2_spec(base, label, folder, hidden, lr, n_runs, lags, train, cell=None, h=1):
    return {'name': f"{base} {label}", 'base': base, 'folder': f"{R2_TAG}_{folder}", 'build': r2_build(base, hidden, **(cell or {})),
            'lags': list(lags), 'batch': train['batch'], 'lr': lr, 'max_epochs': train['max_epochs'],
            'patience': train['patience'], 'n_runs': n_runs, 'spikes': base == ALSTM_NAME, 'rate_reg': 0.0, 'variant': '',
            'hidden': hidden, 'h': h}


def r2_train(spec, datasets):
    # walk-forward of one spec on every dataset (resumable); returns {dataset name: report paths}
    out = {}
    for D in datasets:
        res = walk_forward(spec, D)
        out[D['name']] = report(res)
        if res['stopped']:
            raise TimeBudgetReached(f"time budget reached in {spec['name']} on {D['name']} — rerun this cell to continue")
        trim_ram()
    return out


def r2_seed_preds(spec, D, LAG):
    # per-seed test forecasts and validation RMSE of one finished (spec, dataset, lag), from the runs.pt of each fold
    rows = []
    for f in D['folds']:
        p = os.path.join(ckpt_dir_for(spec, D, LAG, f['tag']), 'runs.pt')
        if not store.fetch(store.rel(p)):
            continue
        done = torch.load(p, map_location='cpu', weights_only=False)
        for r, rec in done.items():
            rows.append({'fold': f['label'], 'te_start': f['te_start'], 'seed': int(r), 'val_rmse': rec['best_val_rmse'],
                         'fire_rate': rec.get('fire_rate', np.nan), 'best_epoch': rec.get('best_epoch', np.nan),
                         'test_pred': np.asarray(rec['test_pred'], float)})
    return rows


def r2_targets(D, LAG, h, fold):
    # the targets of one test fold, in the order the harness stores them (rows whose target lies in the test block)
    t_idx = make_supervised(D['values'], LAG, h)[1]
    return t_idx[(t_idx >= fold['te_start']) & (t_idx < fold['te_end'])]


def r2_run(fn, what):
    # one control experiment; at the time budget everything is saved and the cell stops (a rerun continues)
    try:
        return fn()
    except TimeBudgetReached as err:
        store.push(f"{what}: time budget", force=True)
        raise RuntimeError(f"{err} — finished runs are kept; rerun the notebook to continue") from None


PERSIST = LB[LB['model'] == 'Persistence'].set_index(['dataset', 'h', 'lag'])['rmse']
print(f"control experiments: tune {RUN_TUNE} (lags {TUNE_LAGS}, widths {TUNE_HIDDEN}, lr {TUNE_LRS}, {TUNE_SEEDS} seeds/lr) | "
      f"direct {RUN_DIRECT} (lags {DIRECT_LAGS}, h {DIRECT_H}) | ALSTM diagnostics {RUN_DIAG} | echo SNP {RUN_ECHO} | energy {RUN_ENERGY}")

In [ ]:
# ============================================================
# 21. NETWORKS TUNED ON THE VALIDATION BLOCK
#     Every network, ALSTM-SNP included, is retrained on one protocol (batch 8, ≤300 epochs, patience 40) for each
#     width in TUNE_HIDDEN, with the learning rates TUNE_LRS trained side by side (seed s uses TUNE_LRS[s % 3]).
#     Per (network, dataset, lag, test block) the configuration with the lowest mean validation RMSE over its seeds is
#     kept, among the grid and the published default configuration; its forecast is the mean over its seeds.
#     Writes revisions/r2_tuned_preds.csv, r2_tuned_choices.csv, r2_tuned_leaderboard.csv (h = 1).
# ============================================================
def _tune_specs(base):
    return [r2_spec(base, f"tuned H{H}", f"tune_h{H}", H, list(TUNE_LRS), TUNE_SEEDS * len(TUNE_LRS), TUNE_LAGS, TUNE_TRAIN)
            for H in TUNE_HIDDEN]


def _default_candidate(base, D, LAG, fold):
    # the published configuration of this network (60 seeds): mean validation RMSE of its seeds and its forecast
    spec = next(s for s in MODEL_SPECS if s['name'] == base)
    tag, wd = run_tag_for(spec, D), work_dir_for(spec, D)
    rp = store.fetch(store.rel(os.path.join(wd, f"runs_{tag}_lag{LAG}_{fold['tag']}.csv")))
    pp = store.fetch(store.rel(os.path.join(wd, f"preds_{tag}.csv")))
    if not rp or not pp:
        return None
    runs, pr = pd.read_csv(rp), pd.read_csv(pp)
    pr = pr[(pr['lag'] == LAG) & (pr['test_fold'] == fold['label'])].sort_values('target_idx')
    if not len(pr) or 'val_rmse' not in runs:
        return None
    return {'config': f"default (H={BASELINE_TRAIN['hidden'] if base != ALSTM_NAME else ALSTM_CFG['hidden']}, "
                      f"lr={spec['lr']}, batch={spec['batch']})",
            'val': float(runs['val_rmse'].mean()), 'pred': pr['ens_pred'].to_numpy(), 'target_idx': pr['target_idx'].to_numpy()}


def run_tuning():
    pred_rows, choice_rows = [], []
    for base in NET_NAMES:
        specs = _tune_specs(base)
        for spec in specs:
            r2_train(spec, ALL11)
        for D in ALL11:
            for LAG in TUNE_LAGS:
                if not lag_ok(None, D, LAG):
                    continue
                per = {id(s): r2_seed_preds(s, D, LAG) for s in specs}
                for f in D['folds']:
                    tt = r2_targets(D, LAG, 1, f)
                    cands = []
                    for s in specs:
                        rows = [r for r in per[id(s)] if r['fold'] == f['label']]
                        for li, lr in enumerate(TUNE_LRS):
                            sel = [r for r in rows if r['seed'] % len(TUNE_LRS) == li]
                            if sel:
                                cands.append({'config': f"H={s['hidden']}, lr={lr}, batch={TUNE_TRAIN['batch']}",
                                              'val': float(np.mean([r['val_rmse'] for r in sel])),
                                              'pred': np.mean([r['test_pred'] for r in sel], 0), 'target_idx': tt})
                    dc = _default_candidate(base, D, LAG, f)
                    if dc is not None and len(dc['pred']) == len(tt):
                        cands.append(dc)
                    if not cands:
                        continue
                    best = min(cands, key=lambda c: c['val'])
                    y = D['values'][best['target_idx']]
                    pred_rows.append(pd.DataFrame({'dataset': D['name'], 'model': base, 'lag': LAG, 'h': 1, 'test_fold': f['label'],
                                                   'target_idx': best['target_idx'], 'y': y, 'ens_pred': best['pred']}))
                    choice_rows.append({'dataset': D['name'], 'model': base, 'lag': LAG, 'fold': f['label'],
                                        'chosen': best['config'], 'val_rmse': best['val'],
                                        'default_val_rmse': dc['val'] if dc is not None else np.nan, 'n_candidates': len(cands)})
        print(f"{base}: tuned on {len(ALL11)} datasets × lags {TUNE_LAGS}")
    P = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
    C = pd.DataFrame(choice_rows)
    if len(P):
        rm = lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2)))
        lb = P.groupby(['dataset', 'model', 'lag', 'h']).apply(rm, include_groups=False).rename('rmse').reset_index()
        lb['rel_rmse'] = lb['rmse'] / lb.set_index(['dataset', 'h', 'lag']).index.map(PERSIST).to_numpy()
        dflt = LB.set_index(['dataset', 'model', 'lag', 'h'])['rel_rmse']
        lb['default_rel_rmse'] = lb.set_index(['dataset', 'model', 'lag', 'h']).index.map(dflt).to_numpy()
        r2_csv(P, 'tuned_preds'); r2_csv(C, 'tuned_choices'); r2_csv(lb, 'tuned_leaderboard')
        print(lb.groupby('model')[['rel_rmse', 'default_rel_rmse']].median().to_string())
        print(C['chosen'].value_counts().to_string())
    store.push('controls: tuned networks', force=True)
    return P


TUNED = r2_run(run_tuning, 'tuning') if RUN_TUNE else None

In [ ]:
# ============================================================
# 22. NETWORKS TRAINED DIRECTLY FOR h = 6 AND 24
#     One network per horizon maps the window x_{o-1..o-L} to x_{o+h-1}; same protocol and learning-rate selection as
#     cell 21 (width 8). Evaluated on exactly the forecast origins of the closed-form models (origin in the test block),
#     so the RMSE is comparable with the leaderboard. Writes revisions/r2_direct_preds.csv, r2_direct_leaderboard.csv.
# ============================================================
def run_direct():
    pred_rows = []
    for base in NET_NAMES:
        for h in DIRECT_H:
            spec = r2_spec(base, f"direct h{h}", f"direct_h{h}", DIRECT_HIDDEN, list(TUNE_LRS),
                           TUNE_SEEDS * len(TUNE_LRS), DIRECT_LAGS, TUNE_TRAIN, h=h)
            r2_train(spec, ALL11)
            for D in ALL11:
                for LAG in DIRECT_LAGS:
                    if not lag_ok(None, D, LAG):
                        continue
                    rows = r2_seed_preds(spec, D, LAG)
                    for f in D['folds']:
                        tt = r2_targets(D, LAG, h, f)
                        sel = [r for r in rows if r['fold'] == f['label']]
                        if not sel:
                            continue
                        by_lr = {li: [r for r in sel if r['seed'] % len(TUNE_LRS) == li] for li in range(len(TUNE_LRS))}
                        li = min((k for k in by_lr if by_lr[k]), key=lambda k: np.mean([r['val_rmse'] for r in by_lr[k]]))
                        pred = np.mean([r['test_pred'] for r in by_lr[li]], 0)
                        keep = tt >= f['te_start'] + h - 1                 # origin o = t − h + 1 inside the test block
                        pred_rows.append(pd.DataFrame({'dataset': D['name'], 'model': base, 'lag': LAG, 'h': h,
                                                       'test_fold': f['label'], 'target_idx': tt[keep],
                                                       'y': D['values'][tt[keep]], 'ens_pred': pred[keep],
                                                       'lr': TUNE_LRS[li]}))
        print(f"{base}: direct h {DIRECT_H} done")
    P = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
    if len(P):
        rm = lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2)))
        lb = P.groupby(['dataset', 'model', 'lag', 'h']).apply(rm, include_groups=False).rename('rmse').reset_index()
        lb['rel_rmse'] = lb['rmse'] / lb.set_index(['dataset', 'h', 'lag']).index.map(PERSIST).to_numpy()
        it = LB.set_index(['dataset', 'model', 'lag', 'h'])['rel_rmse']
        lb['iterated_rel_rmse'] = lb.set_index(['dataset', 'model', 'lag', 'h']).index.map(it).to_numpy()
        r2_csv(P, 'direct_preds'); r2_csv(lb, 'direct_leaderboard')
        print(lb.groupby(['model', 'h'])[['rel_rmse', 'iterated_rel_rmse']].median().unstack('h').to_string())
    store.push('controls: direct networks', force=True)
    return P


DIRECT = r2_run(run_direct, 'direct') if RUN_DIRECT else None

In [ ]:
# ============================================================
# 23. ALSTM-SNP DIAGNOSTICS
#     Five variants, same 5 seeds, h = 1, lags DIAG_LAGS: the published configuration retrained (reference); the other
#     networks' protocol (batch 1, lr 1e-3, ≤100 epochs, patience 10); no firing condition (z = 1, the dense cell);
#     no threshold adaptation (β = 0); surrogate slope γ = 1. The firing activity is measured on the test windows.
#     Writes revisions/r2_alstm_diag.csv (one row per variant × dataset × lag).
# ============================================================
def run_alstm_diag():
    rows = []
    base_train = dict(batch=ALSTM_CFG['batch'], lr=ALSTM_CFG['lr'],
                      max_epochs=TUNE_TRAIN['max_epochs'] if SMOKE else ALSTM_CFG['max_epochs'],
                      patience=TUNE_TRAIN['patience'] if SMOKE else ALSTM_CFG['patience'])
    for variant, (cell, train) in DIAG_VARIANTS.items():
        tr = {**base_train, **train}
        if SMOKE:
            tr.update(max_epochs=TUNE_TRAIN['max_epochs'], patience=TUNE_TRAIN['patience'])
        spec = r2_spec(ALSTM_NAME, f"diag {variant}", 'diag_' + slugify(variant), ALSTM_CFG['hidden'], tr['lr'],
                       DIAG_SEEDS, DIAG_LAGS, tr, cell)
        r2_train(spec, ALL11)
        for D in ALL11:
            for LAG in DIAG_LAGS:
                if not lag_ok(None, D, LAG):
                    continue
                seeds = r2_seed_preds(spec, D, LAG)
                y_all, p_all = [], []
                for f in D['folds']:
                    sel = [r for r in seeds if r['fold'] == f['label']]
                    if sel:
                        tt = r2_targets(D, LAG, 1, f)
                        y_all.append(D['values'][tt]); p_all.append(np.mean([r['test_pred'] for r in sel], 0))
                if not y_all:
                    continue
                y, p = np.concatenate(y_all), np.concatenate(p_all)
                rmse_ = float(np.sqrt(np.mean((y - p) ** 2)))
                rows.append({'dataset': D['name'], 'kind': KIND_OF.get(D['name'], ''), 'variant': variant, 'lag': LAG,
                             'rmse': rmse_, 'rel_rmse': rmse_ / PERSIST.get((D['name'], 1, LAG), np.nan),
                             'fire_rate': float(np.nanmean([r['fire_rate'] for r in seeds])),
                             'best_epoch_median': float(np.nanmedian([r['best_epoch'] for r in seeds])),
                             'batch': tr['batch'], 'lr': tr['lr'], 'max_epochs': tr['max_epochs'], 'patience': tr['patience']})
        print(f"ALSTM-SNP {variant}: done")
    out = pd.DataFrame(rows)
    if len(out):
        r2_csv(out, 'alstm_diag')
        print(out.groupby(['variant', 'kind'])[['rel_rmse', 'fire_rate']].median().unstack('kind').to_string())
    store.push('controls: ALSTM-SNP diagnostics', force=True)
    return out


DIAG = r2_run(run_alstm_diag, 'diagnostics') if RUN_DIAG else None

In [ ]:
# ============================================================
# 24. ECHO-TYPE SNP RESERVOIR BASELINE
#     Our implementation of the common design of echo-type SNP forecasters: a fixed, sparse, random reservoir of
#     ECHO_UNITS nonlinear SNP neurons and a ridge readout. Neuron j has charge u_j and fires when u_j ≥ θ:
#         u_t = (1 − a) u_{t−1} + a (s_in w_in x_t + W e_{t−1} + b),   z_t = 1[u_t ≥ θ],   e_t = z_t · tanh(u_t)
#     (W rescaled to spectral radius ρ). Readout on [1, e_L, mean_t e_t, window], fitted on the training block;
#     (ρ, a, s_in, λ) chosen on the validation block, separately per reservoir seed; the forecast is the mean over
#     ECHO_SEEDS reservoirs. Direct readout per horizon, same origins as the closed-form models.
#     Writes revisions/r2_echo_preds.csv, r2_echo_leaderboard.csv (with the measured firing activity).
# ============================================================
def run_echo():
    f64 = torch.float64
    rows, prows = [], []
    for D in ALL11:
        x = D['values']
        for L in ECHO_LAGS:
            if not lag_ok(None, D, L):
                continue
            hs = [h for h in ECHO_H if min(f['te_end'] - f['te_start'] for f in D['folds']) - h + 1 >= ECHO_MIN_ORIGINS]
            stem = f"revisions/r2_echo/{D['slug']}{SM}_lag{L}.csv"
            p = store.fetch(stem)
            if p and os.path.getsize(p) > 2:
                prows.append(pd.read_csv(p)); continue
            check_time_budget()
            out = []
            for f in D['folds']:
                o_all = np.arange(L, f['te_end'])
                a_, b_ = x[:f['va_start']].min(), x[:f['va_start']].max(); sc = 2.0 / max(b_ - a_, 1e-12)
                Wn = np.stack([x[o_all - 1 - k] for k in range(L)], 1)                       # newest first
                U = torch.tensor(sc * (Wn - a_) - 1, dtype=f64, device=device)
                preds = {h: [] for h in hs}; acts = []
                for seed in range(ECHO_SEEDS):
                    g = torch.Generator(device='cpu').manual_seed(10_007 * (seed + 1) + L)
                    w_in = (torch.rand(ECHO_UNITS, generator=g, dtype=f64) * 2 - 1).to(device)
                    Wr = (torch.rand(ECHO_UNITS, ECHO_UNITS, generator=g, dtype=f64) * 2 - 1)
                    Wr = Wr * (torch.rand(ECHO_UNITS, ECHO_UNITS, generator=g, dtype=f64) < ECHO_DENSITY)
                    Wr = (Wr / max(float(torch.linalg.eigvals(Wr).abs().max()), 1e-9)).to(device)
                    bias = ((torch.rand(ECHO_UNITS, generator=g, dtype=f64) * 2 - 1) * 0.2).to(device)
                    feats = {}
                    for rho in ECHO_RHO:
                        for leak in ECHO_LEAK:
                            for s_in in ECHO_INSCALE:
                                u = torch.zeros(len(U), ECHO_UNITS, dtype=f64, device=device); e = torch.zeros_like(u)
                                esum = torch.zeros_like(u); zsum = torch.zeros(len(U), dtype=f64, device=device)
                                for t in range(L):                                      # oldest → newest
                                    xt = U[:, L - 1 - t:L - t]
                                    u = (1 - leak) * u + leak * (s_in * xt * w_in + e @ (rho * Wr).T + bias)
                                    z = (u >= ECHO_THETA).to(f64); e = z * torch.tanh(u)
                                    esum += e; zsum += z.mean(1)
                                feats[(rho, leak, s_in)] = (torch.cat([torch.ones(len(U), 1, dtype=f64, device=device), e,
                                                                       esum / L, U], 1), zsum / L)
                    for h in hs:
                        otr = np.arange(L, f['va_start'] - h + 1); ova = np.arange(f['va_start'], f['te_start'] - h + 1)
                        ote = np.arange(f['te_start'], f['te_end'] - h + 1)
                        itr, iva, ite = otr - L, ova - L, ote - L
                        ytr = torch.tensor(sc * (x[otr + h - 1] - a_) - 1, dtype=f64, device=device)
                        best = None
                        for key, (Fm, zr) in feats.items():
                            Ftr = Fm[itr]; A0 = Ftr.T @ Ftr / len(Ftr); rhs = Ftr.T @ ytr / len(Ftr)
                            for lam in ECHO_LAMBDA:
                                Pn = lam * torch.eye(A0.shape[0], dtype=f64, device=device); Pn[0, 0] = 1e-10
                                th = torch.linalg.solve(A0 + Pn, rhs)
                                va = ((Fm[iva] @ th).cpu().numpy() + 1) / sc + a_
                                v = float(np.sqrt(np.mean((x[ova + h - 1] - va) ** 2)))
                                if best is None or v < best[0]:
                                    best = (v, key, th)
                        Fm, zr = feats[best[1]]
                        preds[h].append(((Fm[ite] @ best[2]).cpu().numpy() + 1) / sc + a_)
                        if h == hs[0]:
                            acts.append(float(zr[ite].mean()))
                for h in hs:
                    ote = np.arange(f['te_start'], f['te_end'] - h + 1)
                    out.append(pd.DataFrame({'dataset': D['name'], 'model': ECHO_NAME, 'lag': L, 'h': h, 'test_fold': f['label'],
                                             'origin': ote, 'target_idx': ote + h - 1, 'y': x[ote + h - 1],
                                             'ens_pred': np.mean(preds[h], 0), 'activity': float(np.mean(acts))}))
            df = pd.concat(out, ignore_index=True)
            lp = os.path.join(LOCAL_ROOT, stem); os.makedirs(os.path.dirname(lp), exist_ok=True)
            df.to_csv(lp, index=False); store.add(lp); store.push(f"echo SNP {D['slug']} lag {L}")
            prows.append(df)
        print(f"{ECHO_NAME}: {D['name']} done")
    P = pd.concat(prows, ignore_index=True) if prows else pd.DataFrame()
    if len(P):
        rm = lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2)))
        lb = P.groupby(['dataset', 'model', 'lag', 'h']).apply(rm, include_groups=False).rename('rmse').reset_index()
        lb['rel_rmse'] = lb['rmse'] / lb.set_index(['dataset', 'h', 'lag']).index.map(PERSIST).to_numpy()
        lb['activity'] = lb.set_index(['dataset', 'model', 'lag', 'h']).index.map(
            P.groupby(['dataset', 'model', 'lag', 'h'])['activity'].mean()).to_numpy()
        lb['trainable_params'] = 2 * ECHO_UNITS + lb['lag'] + 1
        r2_csv(lb, 'echo_leaderboard')
        print(lb.groupby('dataset')[['rel_rmse', 'activity']].median().to_string())
    store.push('controls: echo SNP', force=True)
    return P


ECHO_PREDS = r2_run(run_echo, 'echo') if RUN_ECHO else None

In [ ]:
# ============================================================
# 25. GPU INFERENCE ENERGY OF THE CLOSED-FORM MEMBERS WITH ONE CODE PATH
#     For every (dataset, lag) and each of TRACE-SNP v1, v2 and TRACE-SNP-AF, the configuration selected on the last
#     test block at h = 1 is rebuilt with the same forward code (the TRACE-SNP-AF engine: receptive fields, state
#     neurons, anchor, encoding, readout) and benchmarked on EN_BENCH_N windows per call for ≥ EN_BENCH_S seconds.
#     The energy of a forecast does not depend on the weight values, so the readout weights are random.
#     Writes revisions/r2_energy_common.csv.
# ============================================================
def _trace_forward(cfg, L, shape, k, w, gen):
    f64, LO, HI = torch.float64, -1.1, 1.1
    M = int(cfg['M']) if pd.notna(cfg.get('M', np.nan)) else 0
    d = int(cfg['d']) if pd.notna(cfg.get('d', np.nan)) else 0
    C = int(cfg['C']) if pd.notna(cfg.get('C', np.nan)) else 0
    dif = str(cfg.get('enc', 'lvl')) == 'dif' and L > 1
    mean_anchor = str(cfg.get('anchor', 'last')) == 'mean'
    hh = (HI - LO) / (M - 1) if M > 1 else 1.0
    cen = torch.linspace(LO, HI, max(M, 2), dtype=f64, device=device)
    d_eff = min(d, L - 1 if dif else L) if C else 0
    protos = torch.rand(max(C, 1), max(d_eff, 1), generator=gen, dtype=f64).to(device) * 2 - 1
    n_feat = 1 + L + L * M + C * (d_eff + 1)
    theta = torch.randn(n_feat, generator=gen, dtype=f64).to(device)

    def fn(U):
        Ue = torch.cat([torch.zeros(len(U), 1, dtype=f64, device=device), (U[:, 1:] - U[:, :1])], 1) if dif else U
        parts = [torch.ones(len(U), 1, dtype=f64, device=device), Ue]
        if M:
            if shape == 'tri':
                pos = (Ue.clamp(LO, HI) - LO) / hh
                i = pos.floor().long().clamp(0, M - 2); fr = pos - i
                B = torch.zeros(Ue.shape + (M,), dtype=f64, device=device)
                B.scatter_(-1, i[..., None], (1 - fr)[..., None]); B.scatter_add_(-1, (i + 1)[..., None], fr[..., None])
            else:
                dd = (Ue.clamp(LO, HI)[..., None] - cen[:M]).abs()
                B = torch.exp(-dd ** 2 / (2 * (w * hh) ** 2))
                thr = dd.kthvalue(k, dim=-1, keepdim=True).values
                B = B * (dd <= thr)
            parts.append(B.reshape(len(U), -1))
        if C:
            Z = Ue[:, 1:d_eff + 1] if dif else Ue[:, :d_eff]
            D2 = torch.cdist(Z, protos) ** 2
            val, idx = D2.topk(min(4, C), dim=1, largest=False)
            gg = torch.exp(-val / 2.0); gg = gg / gg.sum(1, keepdim=True)
            Mu = torch.zeros_like(D2).scatter_(1, idx, gg)
            Z1 = torch.cat([torch.ones(len(U), 1, dtype=f64, device=device), Z], 1)
            parts.append((Mu[:, :, None] * Z1[:, None, :]).reshape(len(U), -1))
        out = torch.cat(parts, 1) @ theta
        return out + (U.mean(1) if mean_anchor else U[:, 0])
    return fn


def _fallback_forward(fam, L, gen):
    wv = torch.randn(L, generator=gen, dtype=torch.float64).to(device)
    return {'persistence': lambda U: U[:, 0] + 0.0, 'window-mean': lambda U: U.mean(1)}.get(fam, lambda U: U @ wv)


def _bench(fn, L):
    U = torch.rand(EN_BENCH_N, L, dtype=torch.float64, device=device) * 2 - 1
    fn(U)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    seg = EnergySegment(METER); reps, t_end = 0, time.perf_counter() + EN_BENCH_S
    seg.resume()
    while reps < 1 or time.perf_counter() < t_end:
        fn(U); reps += 1
    if device.type == 'cuda':
        torch.cuda.synchronize()
    seg.pause(); st = seg.state(); n = reps * EN_BENCH_N
    return st['j'] / n if METER.ok else np.nan, st['s'] / n


def run_energy_common():
    af = read_many(f"revisions/trace_af/*{SM}_lag*_choices.csv")
    if not len(af):
        store.pull('revisions/trace_af/*_choices.csv'); af = read_many(f"revisions/trace_af/*{SM}_lag*_choices.csv")
    sources = {'TRACE-SNP': P1, 'TRACE-SNP v2': P2, 'TRACE-SNP-AF': af}
    rows = []
    for model, C_ in sources.items():
        if not len(C_):
            print(f"no choices for {model} — skipped"); continue
        C_ = C_[C_['h'] == 1]
        for (ds, L), g in C_.groupby(['dataset', 'lag']):
            c = g.iloc[-1].to_dict()                                       # the last test block, as in the benchmarks
            gen = torch.Generator().manual_seed(int(L))
            fam = str(c.get('family', ''))
            if model == 'TRACE-SNP-AF':
                ff = str(c.get('field_family', 'tri'))
                if ff == 'fallback':
                    fn = _fallback_forward(fam, int(L), gen)
                else:
                    shape, k, _ = AF_BASES_R2[ff]
                    fn = _trace_forward(c, int(L), shape, k, float(c.get('w', 1.0)) if pd.notna(c.get('w', np.nan)) else 1.0, gen)
            elif fam.split('|')[-1] in ('persistence', 'window-mean', 'ridge', 'fallback'):
                fn = _fallback_forward(fam.split('|')[-1], int(L), gen)
            else:
                cfg = dict(c)
                if 'state' not in fam:
                    cfg['C'] = np.nan
                if model == 'TRACE-SNP':
                    cfg['anchor'] = 'last'
                fn = _trace_forward(cfg, int(L), 'tri', 2, 1.0, gen)
            j, s = _bench(fn, int(L))
            rows.append({'dataset': ds, 'model': model, 'lag': int(L), 'infer_j_per_forecast': j, 'infer_s_per_forecast': s,
                         'forecasts_per_call': EN_BENCH_N, 'gpu': METER.name if METER.ok else 'n/a'})
    out = pd.DataFrame(rows)
    if len(out):
        r2_csv(out, 'energy_common')
        print((out.groupby('model')[['infer_j_per_forecast', 'infer_s_per_forecast']].median() * 1e6)
              .rename(columns=lambda c: c.replace('_per_forecast', ' (µJ / µs)')).to_string())
    store.push('controls: energy', force=True)
    return out


AF_BASES_R2 = {'tri': ('tri', 2, True), 'gauss_k2': ('gauss', 2, True), 'gauss_k4': ('gauss', 4, True),
               'gauss_k8': ('gauss', 8, True)}
EN_COMMON = run_energy_common() if RUN_ENERGY else None

## Part E — Extended comparison and data checks (cells 26–31)
Classical statistical forecasters on the same walk-forward folds (ARIMA, simple and damped Holt exponential smoothing, seasonal naive, training median), constant-forecast references, the horizons in Lyapunov times, a check of the operation counts behind the energy model, and checks that every stored forecast matches its series. Only cell 28 fits models (the classical forecasters, a few minutes on a CPU); no network is trained and no TRACE-SNP model is refitted. Everything is written to `revisions/extended_checks/`, which cell 32 reads.

In [ ]:
# ============================================================
# 26. EXTENDED COMPARISON — settings, data manifest and consistency checks (no training, no refitting)
#     Cells 26–31 extend the evaluation of cells 7–25 with classical statistical forecasters and a set of data and
#     protocol checks. They use the series and folds of cell 3 and the stored results in LOCAL_ROOT; only cell 28 fits
#     models (the classical forecasters, a few minutes on the CPU). Every output goes to
#     <LOCAL_ROOT>/revisions/extended_checks/, which cell 32 reads for the extended tables and figures of the paper.
#     This cell writes
#       data_manifest.csv     every series: file, SHA-256, length and source column (EPL: the column of the raw file)
#       provenance_gate.csv   stored forecasts against the series of cell 3: y = x[target], prev = x[origin − 1], the test-block
#                             labels, and the persistence RMSE of the leaderboard recomputed from the series
#       epl_columns.csv       EPL: which column of the raw file reproduces the stored targets (the series is described by it)
# ============================================================
EXT_DIR      = rev_path(f"extended_checks{SM}", '')               # <LOCAL_ROOT>/revisions/extended_checks/
EXT_HORIZONS = CF_HORIZONS                                        # h = 1, 6, 24, as every model
EXT_STRICT   = True                                               # stop if a stored forecast does not match its series
EXT_PRED_FOLDERS = ['revisions/trace_af', 'revisions/basis_study', 'revisions/rbf_ablation']


def ext_save(obj, name, index=True):
    # one output of cells 26–31: DataFrame → CSV, text → file; registered in the results folder
    p = os.path.join(EXT_DIR, name)
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        obj.to_csv(p, index=index)
    else:
        with open(p, 'w') as fh:
            fh.write(str(obj))
    store.add(p)
    return p


def wf_origins(f, h):
    # forecast origins of one test block: the target x[o + h − 1] lies inside the block
    return np.arange(f['te_start'], f['te_end'] - h + 1)


def wf_pooled(x, folds, h, predict):
    # pooled RMSE over the 5 test blocks; predict(x, f, origins, h) returns the forecasts of x[o + h − 1]
    e = np.concatenate([x[wf_origins(f, h) + h - 1] - predict(x, f, wf_origins(f, h), h) for f in folds])
    return float(np.sqrt(np.mean(e ** 2)))


fc_persistence = lambda x, f, o, h: x[o - 1]
fc_train_mean  = lambda x, f, o, h: np.full(len(o), x[:f['te_start']].mean())      # every point before the test block
fc_train_med   = lambda x, f, o, h: np.full(len(o), np.median(x[:f['te_start']]))
EXT_DS = {D['name']: D for D in ALL11}


def ext_sha256(path):
    h_ = hashlib.sha256()
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''):
            h_.update(b)
    return h_.hexdigest()


def epl_raw_columns():
    # EPL: every numeric column of the raw file in DATA_DIR, read exactly as cell 3 reads a series
    raw = os.path.join(DATA_DIR, 'EPL.csv')
    if not os.path.exists(raw):
        return raw, None, {}
    df = pd.read_csv(raw, sep=_delimiter(raw))
    out = {}
    for c in df.columns:
        try:
            v = read_series(raw, c)[0]
        except Exception:
            continue
        if len(v) >= 50 and np.isfinite(v).all():
            out[c] = v
    return raw, df, out


def ext_manifest():
    raw, df_raw, cols = epl_raw_columns()
    rows = []
    for D in ALL11:
        p = os.path.join(LOCAL_ROOT, D['file'])
        col = (SERIES_SOURCES.get(D['name'], (None, None))[1] or 'value') if D['name'] != 'EPL' else 'value'
        if D['name'] == 'EPL' and cols:
            hit = [c for c, v in cols.items() if len(v) == D['n_raw'] and np.allclose(v[-D['n']:], D['values'])]
            col = hit[0] if hit else read_series(raw, None)[1]
        rows.append({'dataset': D['name'], 'kind': D['kind'], 'file': D['file'], 'bytes': os.path.getsize(p), 'sha256': ext_sha256(p),
                     'column': col, 'n': D['n'], 'test block': D['block'], 'folds': ' '.join(f['label'] for f in D['folds'])})
    return pd.DataFrame(rows).set_index('dataset')


def stored_forecasts(D):
    # every stored forecast row of one dataset from the first folder that has them
    for folder in EXT_PRED_FOLDERS:
        fs = sorted(glob.glob(os.path.join(LOCAL_ROOT, folder, f"{D['slug']}{SM}_lag*_preds.csv")))
        fs = [f for f in fs if os.path.getsize(f) > 2 and (('_smoke' in f) == SMOKE)]
        if fs:
            P = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
            return folder, P[P['dataset'].astype(str) == D['name']] if 'dataset' in P else P
    return None, pd.DataFrame()


def match_stored(x, P):
    # largest |stored − recomputed| relative to the spread of the series, for the target y and the last input prev
    t = P['target_idx'].to_numpy(int)
    if t.min() < 0 or t.max() >= len(x):
        return np.inf, np.inf
    sc = float(np.std(x)) or 1.0
    ey = float(np.max(np.abs(P['y'].to_numpy(float) - x[t]))) / sc
    ep = (float(np.max(np.abs(P['prev'].to_numpy(float) - x[P['origin'].to_numpy(int) - 1]))) / sc
          if {'prev', 'origin'} <= set(P) else np.nan)
    return ey, ep


def ext_gate():
    rows, fail = [], []
    for D in ALL11:
        x, folds = D['values'], D['folds']
        row = {'dataset': D['name'], 'n': len(x)}
        src, P = stored_forecasts(D)
        row['checked against'] = src or '—'
        checks = []
        if len(P):
            ey, ep = match_stored(x, P)
            lab_ok = set(P['test_fold'].astype(str)) <= {f['label'] for f in folds} if 'test_fold' in P else True
            row.update({'rows': len(P), 'max |y err| / sd': ey, 'max |prev err| / sd': ep, 'folds match': lab_ok})
            checks.append(ey < 1e-8 and (np.isnan(ep) or ep < 1e-8) and lab_ok)
        pers = LB[(LB['model'] == 'Persistence') & (LB['dataset'] == D['name'])]
        if len(pers):
            rec = {h: wf_pooled(x, folds, int(h), fc_persistence) for h in pers['h'].unique()}
            rel = np.abs(pers['rmse'].to_numpy() / pers['h'].map(rec).to_numpy() - 1)
            row['persistence max rel err'] = float(np.nanmax(rel))
            checks.append(bool(np.nanmax(rel) < 1e-4))
        row['PASS'] = all(checks) if checks else None
        rows.append(row)
        if row['PASS'] is False:
            fail.append(D['name'])
    return pd.DataFrame(rows).set_index('dataset'), fail


def ext_epl_columns():
    raw, df_raw, cols = epl_raw_columns()
    D = EXT_DS.get('EPL')
    if df_raw is None or D is None:
        print(f"(no raw EPL file at {raw}: column check skipped)"); return pd.DataFrame()
    _, P = stored_forecasts(D)
    rows = []
    for c, v in cols.items():
        v = v[-D['n']:] if len(v) >= D['n'] else v
        r = {'column': c, 'n': len(v), 'mean': v.mean(), 'sd': v.std(), 'used': len(v) == D['n'] and np.allclose(v, D['values'])}
        if len(P) and len(v) == D['n']:
            r['matches stored y'] = bool(match_stored(v, P)[0] < 1e-8)
        rows.append(r)
    return pd.DataFrame(rows).set_index('column')


EXT_MANIFEST = ext_manifest()
ext_save(EXT_MANIFEST, 'data_manifest.csv')
print(EXT_MANIFEST[['kind', 'file', 'column', 'n', 'test block']].to_string())

EXT_GATE, _ext_fail = ext_gate()
ext_save(EXT_GATE, 'provenance_gate.csv')
with pd.option_context('display.float_format', lambda v: f"{v:.2e}", 'display.width', 200):
    print("\n" + EXT_GATE.to_string())
EXT_EPL_COLUMNS = ext_epl_columns()
if len(EXT_EPL_COLUMNS):
    ext_save(EXT_EPL_COLUMNS, 'epl_columns.csv')
    _used = list(EXT_EPL_COLUMNS.index[EXT_EPL_COLUMNS['used']])
    print(f"\nEPL: the series of cell 3 is column {_used} of {os.path.join(DATA_DIR, 'EPL.csv')} "
          f"({len(EXT_EPL_COLUMNS)} numeric columns checked)")
if _ext_fail:
    _msg = f"stored forecasts do not match the series of cell 3 for {_ext_fail} (see provenance_gate.csv)"
    if EXT_STRICT:
        raise RuntimeError(_msg)
    print("WARNING: " + _msg)
else:
    print("\nEvery checkable dataset reproduces its stored forecasts exactly.")


In [ ]:
# ============================================================
# 27. CONSTANT FORECASTS — the training mean and median of each walk-forward fold against persistence (no fitting)
#     Score as in the paper: geometric mean over h = 1, 6, 24 of the pooled RMSE relative to persistence. A score above
#     one means persistence is the stricter of the two references. Writes constant_forecasts.csv (with the score of
#     TRACE-SNP-AF from the leaderboard of cell 18 for comparison).
# ============================================================
def dataset_score(lb, model):
    # the paper's dataset score of one model: geometric mean of rel_rmse over every (lag, h) of that dataset
    g = lb[lb['model'] == model]
    return g.groupby('dataset')['rel_rmse'].apply(lambda s: float(np.exp(np.log(s).mean())))


def ext_constant_forecasts():
    lb_af = read_many(f"revisions/leaderboard_with_trace_af{SM}.csv")
    rows = []
    for D in ALL11:
        x, folds = D['values'], D['folds']
        P = {h: wf_pooled(x, folds, h, fc_persistence) for h in EXT_HORIZONS}
        r = {'dataset': D['name']}
        for nm, fn in (('training mean', fc_train_mean), ('training median', fc_train_med)):
            r[nm] = float(np.exp(np.mean([np.log(wf_pooled(x, folds, h, fn) / P[h]) for h in EXT_HORIZONS])))
        rows.append(r)
    M = pd.DataFrame(rows).set_index('dataset')
    if len(lb_af):
        M['TRACE-SNP-AF'] = dataset_score(lb_af, 'TRACE-SNP-AF').reindex(M.index)
    return M


EXT_CONSTANT = ext_constant_forecasts()
ext_save(EXT_CONSTANT, 'constant_forecasts.csv')
print(EXT_CONSTANT.round(3).to_string())
print(f"\nconstant forecast at the training mean below persistence on {int((EXT_CONSTANT['training mean'] < 1).sum())} of "
      f"{len(EXT_CONSTANT)} series")


In [ ]:
# ============================================================
# 28. CLASSICAL STATISTICAL FORECASTERS — ARIMA, simple and damped Holt exponential smoothing, seasonal naive, training
#     median, on the walk-forward folds of cell 3 (same targets and origins as every other model; about 5 minutes on a CPU)
#     ARIMA(p, d, q), p ≤ 3, d ∈ {0, 1}, q ≤ 2: the order by AIC on the data before the first validation block, the
#     coefficients by maximum likelihood on all data before each test block, h-step forecasts from the state-space form.
#     If the default optimiser fails or its forecasts diverge (more than 20 training s.d. from the training mean), Powell,
#     then Nelder–Mead, then ARIMA(1, 1, 0) is used, and the fold is listed.
#     SES and damped Holt (φ ∈ [0.8, 0.995]): parameters by least squares on the data before each test block.
#     Seasonal naive: periods 24 (ETTh1, ETTh2), 365 (Melbourne), 52 (CO2) and 12 (El Niño); not defined elsewhere.
#     These forecasters do not depend on the window length L: each has one relative RMSE per (dataset, horizon).
#     Writes statistical_baselines.csv (dataset score and ARIMA order) and statistical_baselines_by_h.csv (per horizon).
#     Resumable: an existing result is reused unless EXT_REFIT_CLASSICAL = True.
# ============================================================
EXT_REFIT_CLASSICAL = False
EXT_SEASON = {'etth1': 24, 'etth2': 24, 'melbourne_temp': 365, 'co2_weekly': 52, 'elnino_monthly': 12}


def run_classical_forecasters():
    import warnings
    from scipy.optimize import minimize
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'], check=True)
        from statsmodels.tsa.statespace.sarimax import SARIMAX
    warnings.filterwarnings('ignore')
    notes = []

    def arima_order(y):
        best = None
        for d in (0, 1):
            for p in range(4):
                for q in range(3):
                    if p == q == 0 and d == 0:
                        continue
                    try:
                        r = SARIMAX(y, order=(p, d, q), trend='c' if d == 0 else 'n').fit(disp=False, maxiter=200)
                        if best is None or r.aic < best[0]:
                            best = (r.aic, (p, d, q))
                    except Exception:
                        pass
        return best[1] if best else (1, 1, 0)

    def arima_fits(y, order):
        for o_, kw in ((order, {}), (order, {'method': 'powell'}), (order, {'method': 'nm', 'maxiter': 3000}), ((1, 1, 0), {})):
            try:
                yield o_, kw, SARIMAX(y, order=o_, trend='c' if o_[1] == 0 else 'n').fit(disp=False, **{'maxiter': 300, **kw})
            except (np.linalg.LinAlgError, ValueError, OverflowError, FloatingPointError):
                continue

    def arima_paths(res, x, f, h_list):
        fr = res.apply(x[:f['te_end']]).filter_results
        Tm, Z = fr.transition[:, :, 0], fr.design[:, :, 0]
        c, dd = fr.state_intercept[:, 0], fr.obs_intercept[:, 0]
        out = {}
        for h in h_list:
            a = fr.predicted_state[:, wf_origins(f, h)].copy()      # state at o given the data up to o − 1
            for _ in range(h - 1):
                a = Tm @ a + c[:, None]
            out[h] = (Z @ a + dd[:, None])[0]
        return out

    def arima_forecasts(x, f, order, h_list):
        y = x[:f['te_start']]; lim = 20 * y.std()
        for o_, kw, res in arima_fits(y, order):
            try:
                out = arima_paths(res, x, f, h_list)
            except (np.linalg.LinAlgError, ValueError, OverflowError, FloatingPointError):
                continue
            if all(np.isfinite(v).all() and np.abs(v - y.mean()).max() < lim for v in out.values()):
                if (o_, kw) != (order, {}):
                    notes.append(f"block t{f['te_start']} (n = {len(y)}) order {order}: used {o_} {kw.get('method', 'lbfgs')}")
                return out
        raise RuntimeError(f"no usable ARIMA fit (order {order}, block t{f['te_start']})")

    def ses_holt(x, f, h_list, damped):
        y = x[:f['te_start']]

        def run(params, series):
            al, be, ph = (params[0], params[1], params[2]) if damped else (params[0], 0.0, 0.0)
            l, b = series[0], (series[1] - series[0]) if damped else 0.0
            L_ = np.empty(len(series)); B_ = np.empty(len(series)); e = 0.0
            for t in range(len(series)):
                L_[t], B_[t] = l, b                                 # state before observing series[t]
                fc = l + ph * b
                err = series[t] - fc; e += err * err
                l_new = fc + al * err
                b = ph * b + be * err if damped else 0.0
                l = l_new
            return e, L_, B_

        x0, bnd = ([0.5, 0.05, 0.9], [(1e-3, 1), (1e-4, 1), (0.8, 0.995)]) if damped else ([0.5], [(1e-3, 1)])
        prm = minimize(lambda p_: run(p_, y)[0], x0, bounds=bnd, method='L-BFGS-B').x
        _, L_, B_ = run(prm, x[:f['te_end']])
        ph = prm[2] if damped else 0.0
        out = {}
        for h in h_list:
            o = wf_origins(f, h)
            out[h] = L_[o] + (sum(ph ** k for k in range(1, h + 1)) if damped else 0.0) * B_[o]
        return out

    rows, long_rows = [], []
    for D in ALL11:
        t0 = time.time()
        name, x, folds = D['name'], D['values'], D['folds']
        pers = {h: wf_pooled(x, folds, h, fc_persistence) for h in EXT_HORIZONS}
        order = arima_order(x[:folds[0]['va_start']])
        preds = {m: {h: [] for h in EXT_HORIZONS} for m in ('ARIMA', 'SES', 'Holt damped', 'Seasonal naive', 'Training median')}
        ys = {h: [] for h in EXT_HORIZONS}
        for f in folds:
            A = arima_forecasts(x, f, order, EXT_HORIZONS)
            S = ses_holt(x, f, EXT_HORIZONS, False)
            Hd = ses_holt(x, f, EXT_HORIZONS, True)
            for h in EXT_HORIZONS:
                o = wf_origins(f, h); ys[h].append(x[o + h - 1])
                preds['ARIMA'][h].append(A[h]); preds['SES'][h].append(S[h]); preds['Holt damped'][h].append(Hd[h])
                preds['Training median'][h].append(np.full(len(o), np.median(x[:f['te_start']])))
                m = EXT_SEASON.get(name)
                if m:                    # the value one season before the target that is already observed at the origin
                    preds['Seasonal naive'][h].append(x[o + h - 1 - int(np.ceil(h / m)) * m])
        r = {'dataset': name, 'ARIMA order': str(order)}
        for mname, P in preds.items():
            if not P[EXT_HORIZONS[0]]:
                continue
            rel = [float(np.sqrt(np.mean((np.concatenate(ys[h]) - np.concatenate(P[h])) ** 2))) / pers[h] for h in EXT_HORIZONS]
            r[mname] = float(np.exp(np.mean(np.log(rel))))
            long_rows += [{'dataset': name, 'model': mname, 'h': h, 'rmse': rl * pers[h], 'persistence_rmse': pers[h], 'rel_rmse': rl,
                           'order': str(order) if mname == 'ARIMA' else ''} for h, rl in zip(EXT_HORIZONS, rel)]
        rows.append(r)
        print(f"{name:15s} ARIMA{order} {time.time() - t0:5.0f} s", {k: round(v, 3) for k, v in r.items() if isinstance(v, float)},
              flush=True)
    if notes:
        print('\nARIMA blocks fitted with a fallback optimiser or order:', *notes, sep='\n  ')
    df = pd.DataFrame(rows).set_index('dataset')
    lb_af = read_many(f"revisions/leaderboard_with_trace_af{SM}.csv")
    for c in ('TRACE-SNP-AF', 'TRACE-SNP v2', 'DLinear'):
        if len(lb_af) and c in set(lb_af['model']):
            df[c] = dataset_score(lb_af, c).reindex(df.index)
    return df, pd.DataFrame(long_rows)


_p1, _p2 = os.path.join(EXT_DIR, 'statistical_baselines.csv'), os.path.join(EXT_DIR, 'statistical_baselines_by_h.csv')
if not EXT_REFIT_CLASSICAL and os.path.exists(_p1) and os.path.exists(_p2):
    print(f"reusing {_p1} (set EXT_REFIT_CLASSICAL = True to refit)")
    EXT_CLASSICAL, EXT_CLASSICAL_BY_H = pd.read_csv(_p1, index_col=0), pd.read_csv(_p2)
else:
    EXT_CLASSICAL, EXT_CLASSICAL_BY_H = run_classical_forecasters()
    ext_save(EXT_CLASSICAL, 'statistical_baselines.csv')
    ext_save(EXT_CLASSICAL_BY_H, 'statistical_baselines_by_h.csv', index=False)
print(EXT_CLASSICAL.round(3).to_string())


In [ ]:
# ============================================================
# 29. LYAPUNOV TIMES OF THE CHAOTIC GENERATORS — the horizons h = 1, 6, 24 in Lyapunov times (no fitting)
#     Largest Lyapunov exponent by the two-trajectory method of Benettin et al.: two trajectories 1e-8 apart (Mackey–Glass:
#     1e-9 over the 18-dimensional delay state), renormalised at every step, 5,000 burn-in steps; integrator and step as in
#     cell 3 (Lorenz-63: RK4, dt = 0.01, every 3rd step kept; Rössler: RK4, dt = 0.05, every 2nd step; Mackey–Glass: the
#     unit-step Euler recursion with τ = 17). Also checks that the stored Mackey–Glass series is that recursion.
#     Writes lyapunov.csv (about 2 minutes on a CPU).
# ============================================================
def ext_lyapunov():
    def rk4(f, v, dt):
        k1 = f(v); k2 = f(v + dt / 2 * k1); k3 = f(v + dt / 2 * k2); k4 = f(v + dt * k3)
        return v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

    lor = lambda v: np.array([10 * (v[1] - v[0]), v[0] * (28 - v[2]) - v[1], v[0] * v[1] - 8 / 3 * v[2]])
    ros = lambda v: np.array([-v[1] - v[2], v[0] + 0.2 * v[1], 0.2 + v[2] * (v[0] - 5.7)])

    def lyap_flow(f, dt, n_steps, burn, d0=1e-8):
        v = np.ones(3)
        for _ in range(burn):
            v = rk4(f, v, dt)
        w = v + np.array([d0, 0, 0]); s = 0.0
        for _ in range(n_steps):
            v = rk4(f, v, dt); w = rk4(f, w, dt); d = np.linalg.norm(w - v); s += np.log(d / d0); w = v + (w - v) * d0 / d
        return s / (n_steps * dt)                                   # per unit of model time

    def lyap_mg(n=200000, tau=17, burn=2000, d0=1e-9):
        step = lambda a: a[-1] + 0.2 * a[-1 - tau] / (1 + a[-1 - tau] ** 10) - 0.1 * a[-1]
        x = list(mackey_glass(burn + tau + 2))                      # the generator of cell 3
        a = np.array(x[-(tau + 1):]); b = a + d0 * np.random.default_rng(0).normal(size=tau + 1) / np.sqrt(tau + 1); s = 0.0
        for _ in range(n):
            a = np.append(a[1:], step(a)); b = np.append(b[1:], step(b))
            d = np.linalg.norm(b - a); s += np.log(d / d0); b = a + (b - a) * d0 / d
        return s / n                                                # per sample

    lam_l, lam_r, lam_m = lyap_flow(lor, 0.01, 300000, 5000), lyap_flow(ros, 0.05, 400000, 5000), lyap_mg()
    rows = [('Lorenz-63 x (dt 0.01, every 3rd step)', lam_l * 0.03),
            ('Rössler x (dt 0.05, every 2nd step)', lam_r * 0.10),
            ('Mackey–Glass (unit-step Euler map)', lam_m)]
    print(f"largest exponent: Lorenz-63 {lam_l:.3f} per unit time (literature 0.906), Rössler {lam_r:.4f} (literature about 0.071), "
          f"Mackey–Glass map {lam_m:.4f} per sample")
    # residual of x[t+1] = x[t] + 0.2 x[t−τ] / (1 + x[t−τ]^10) − 0.1 x[t] on the stored series, τ = 10 … 30
    xm = EXT_DS['MackeyGlass']['values'] if 'MackeyGlass' in EXT_DS else None
    res = {}
    if xm is not None:
        for tau in range(10, 31):
            xd = xm[:-(tau + 1)]
            r = xm[tau + 1:] - (xm[tau:-1] + 0.2 * xd / (1 + xd ** 10) - 0.1 * xm[tau:-1])
            res[tau] = float(np.max(np.abs(r)) / np.std(xm))
        print(f"stored Mackey–Glass series vs the Euler map: max |residual| / sd = {res[17]:.2e} at τ = 17")
    out = pd.DataFrame([{'system': nm_, 'Lyapunov time (samples)': 1 / ps, **{f'h={h} in Lyapunov times': h * ps for h in EXT_HORIZONS}}
                        for nm_, ps in rows]).set_index('system')
    out['note'] = ['', '', f"stored series vs Euler map: max |residual|/sd = {res[17]:.1e} (tau 17)" if res else '']
    return out


EXT_LYAPUNOV = ext_lyapunov()
ext_save(EXT_LYAPUNOV, 'lyapunov.csv')
print(EXT_LYAPUNOV.round(3).to_string())


In [ ]:
# ============================================================
# 30. OPERATION COUNTS — the counting rule of the energy model (Eq. S23 of the supplement) against the counts that
#     cell 18 stored with every TRACE-SNP-AF forecast (no fitting)
#     One-step forecast of a receptive-field model with window L, k firing fields per delay and an optional state pathway
#     (C state neurons, embedding d, k_a = 4 firing state neurons):
#         MAC = 3L + kL (+ kL for a Gaussian family, the field values) + [C d + k_a + k_a (d + 1)] with the state pathway.
#     Writes ops_formula_check.csv (share of stored rows the rule reproduces) and ops_stored_trace_af.csv (medians of the
#     configuration actually chosen per field family).
# ============================================================
def ext_operation_counts():
    ka = 4
    ch = read_many(f"revisions/trace_af/*{SM}_lag*_choices.csv")
    if not len(ch) or not {'MAC', 'field_family', 'lag'} <= set(ch):
        print("no TRACE-SNP-AF choices with stored operation counts (cell 18)"); return pd.DataFrame(), pd.DataFrame()
    c1 = ch[(ch['h'] == 1) & (ch['field_family'] != 'fallback')].copy()
    kk = c1['field_family'].map({'tri': 2, 'gauss_k2': 2, 'gauss_k4': 4, 'gauss_k8': 8}).astype(float)
    Lc = c1['lag'].astype(float)
    has_state = c1['C'].notna() if 'C' in c1 else pd.Series(False, index=c1.index)
    Cc = c1.get('C', pd.Series(0.0, index=c1.index)).fillna(0.0)
    dc = c1.get('d', pd.Series(0.0, index=c1.index)).fillna(0.0)
    c1['MAC (rule)'] = 3 * Lc + kk * Lc + np.where(c1['field_family'].str.startswith('gauss'), kk * Lc, 0) + \
        np.where(has_state, Cc * dc + ka + ka * (dc + 1), 0)
    share = float(np.isclose(c1['MAC'], c1['MAC (rule)']).mean())
    chk = pd.DataFrame([{'share': share, 'rows': len(c1)}])
    agg = {'lag': 'median', 'MAC': 'median', **{c: 'median' for c in ('M', 'd', 'C', 'pJ') if c in c1}}
    med = c1.groupby('field_family').agg(agg).assign(rows=c1.groupby('field_family').size())
    print(f"TRACE-SNP-AF, h = 1: the rule reproduces the stored MAC count on {share:.1%} of {len(c1)} receptive-field forecasts")
    return chk, med


EXT_OPS, EXT_OPS_STORED = ext_operation_counts()
if len(EXT_OPS):
    ext_save(EXT_OPS, 'ops_formula_check.csv', index=False)
    ext_save(EXT_OPS_STORED, 'ops_stored_trace_af.csv')
    print(EXT_OPS_STORED.round(1).to_string())


In [ ]:
# ============================================================
# 31. FALLBACK SELECTIONS AND SUMMARY — how often TRACE-SNP-AF selects a fallback candidate (persistence, window mean,
#     ridge AR) instead of a receptive-field model, per dataset, and a summary of cells 26–31 (no fitting)
#     Writes fallback_share_trace_af.csv and SUMMARY.md.
# ============================================================
def ext_fallback_shares():
    ch = read_many(f"revisions/trace_af/*{SM}_lag*_choices.csv")
    if not len(ch):
        return pd.DataFrame()
    ch['dataset'] = ch['dataset'].astype(str)
    n_all = ch.groupby('dataset').size()
    fb = ch[ch['field_family'] == 'fallback'].groupby(['dataset', 'family']).size().unstack(fill_value=0)
    FB = fb.reindex(n_all.index, fill_value=0).div(n_all, axis=0)
    FB['any fallback'] = FB.sum(axis=1)
    return FB


EXT_FALLBACK = ext_fallback_shares()
if len(EXT_FALLBACK):
    ext_save(EXT_FALLBACK, 'fallback_share_trace_af.csv')
    print("TRACE-SNP-AF: share of (fold, lag, horizon) cases with a fallback candidate selected")
    print(EXT_FALLBACK.round(3).to_string())

_lines = ["# Extended comparison and data checks (cells 26–31)", "",
          f"Datasets: {len(EXT_MANIFEST)} (SHA-256 in data_manifest.csv); EPL column: {EXT_MANIFEST.loc['EPL', 'column'] if 'EPL' in EXT_MANIFEST.index else '--'}",
          "Stored forecasts against the series: " + ("every checkable dataset matches exactly" if not _ext_fail else f"MISMATCH for {_ext_fail}"),
          f"Classical forecasters fitted on the walk-forward folds: {', '.join(c for c in ('ARIMA', 'SES', 'Holt damped', 'Seasonal naive', 'Training median') if c in EXT_CLASSICAL)}",
          f"Constant forecast at the training mean below persistence on {int((EXT_CONSTANT['training mean'] < 1).sum())} series",
          "Lyapunov time (samples): " + ", ".join(f"{i.split(' (')[0]} {v:.0f}" for i, v in EXT_LYAPUNOV['Lyapunov time (samples)'].items()),
          (f"Operation counts: the rule of Eq. S23 reproduces {EXT_OPS['share'].iloc[0]:.1%} of {int(EXT_OPS['rows'].iloc[0])} stored forecasts"
           if len(EXT_OPS) else "Operation counts: not checked"),
          (f"TRACE-SNP-AF fallbacks: median {EXT_FALLBACK['any fallback'].median():.1%}, at most {EXT_FALLBACK['any fallback'].max():.1%}"
           if len(EXT_FALLBACK) else ""),
          "", "Cell 32 reads these files for the extended ranking and the tables and figures of the classical forecasters."]
ext_save("\n".join(_lines) + "\n", 'SUMMARY.md')
print("\n" + "\n".join(_lines))
print(f"\n{len(os.listdir(EXT_DIR))} files in {EXT_DIR}")
store.push('extended comparison and checks', force=True)


## Part F — Every figure and table of the paper (cell 32)
Reads the results folder only (cells 7–31) and writes `paper_final/figures/`, `paper_final/tables/`, `paper_final/numbers.md` and `paper_final.zip`.

In [ ]:
# ============================================================
# 32. PAPER FINAL — every figure and table of the paper and of its supplementary material, from the results folder
#     Reads what cells 7–31 wrote to LOCAL_ROOT (no training, no refitting) and writes
#         paper_final/figures/*.pdf (+ .png)      every figure, file names exactly as the LaTeX sources use them
#         paper_final/tables/*.tex                every generated table (booktabs), \input by the LaTeX sources
#         paper_final/tables/r2_macros.tex        the numbers of cells 21–25 as LaTeX macros
#         paper_final/tables/ext_macros.tex       the numbers of the extended comparison (cells 26–31) as LaTeX macros
#         paper_final/numbers.md                  every number the text quotes, for checking
#         paper_final.zip                         the folder in one download
#     The 16-model comparison of the paper is the main ranking; the classical forecasters of cell 28 enter a second,
#     extended ranking with its own table and figures (EXTENDED_IN_MAIN_RANKING inside the function).
# ============================================================
def paper_final_cell(local_root, *after_cells_21_31):
    # ============================================================
    # PAPER FINAL — every figure and table of the manuscript and of the supplementary material
    #    Reads what cells 7–31 wrote to LOCAL_ROOT (no training, no refitting) and writes
    #        paper_final/figures/*.pdf (+ .png)      every figure, file names exactly as main.tex / supplementary.tex use them
    #        paper_final/tables/*.tex                every generated table (booktabs), \input by the two .tex files
    #        paper_final/numbers.md                  every number the text quotes, for checking
    #        paper_final.zip                         the folder in one download
    #    Everything lives inside one function so that its names stay private to this cell. Rerunning overwrites the
    #    outputs (about 1–2 minutes).
    #    LaTeX: put figures/ and tables/ next to main.tex and supplementary.tex and compile (pdfLaTeX + BibTeX).
    # ============================================================
    LOCAL_ROOT = local_root                                    # the results folder of cells 0–31
    DATA_ROOT  = local_root.rstrip('/') + '/data_main'         # exact copies of EPL, Sunspots and Mackey–Glass (cell 3)
    PAPER_DIR  = 'paper_final'

    import os, re, io, glob, time, json, math, zipfile, fnmatch, warnings
    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch, Rectangle
    from scipy import stats as st

    warnings.filterwarnings('ignore', category=RuntimeWarning)
    SMOKE      = os.environ.get('SMOKE', '0') == '1'           # smoke-test outputs are kept apart (suffix _smoke)
    SM         = '_smoke' if SMOKE else ''
    ALPHA      = 0.05
    ROPE       = 0.01
    NOISE_LEVELS = [0.005, 0.05, 0.10]
    # ── extended comparison (outputs of cells 26–31): classical statistical forecasters, the dense basis
    #    in the ranking, Lyapunov times, constant forecasts, operation-count check, EPL column, fallback shares ──
    EXTENDED_IN_MAIN_RANKING = False   # False: the 16-model comparison of every existing table and figure is unchanged and the
                                       # extended comparison has its own tables / figures; True: added to every ranking as well
    EXTENDED_IN_FIGURES = False        # the added models also appear in Figure 6 (extra rows, not ranked) and in Figures S2-S5
                                       # (extra lines); no statistic of the 16-model comparison changes
    EXT_CLASSICAL = True               # ARIMA, SES, damped Holt, seasonal naive, training median
    EXT_DENSE     = False              # the dense basis is an ablation of TRACE-SNP, reported in Section 5.2, not a ranked rival
    EXT_DIRS      = [f'revisions/extended_checks{SM}']   # written by cells 26–31

    # ── datasets: stored name → name in the paper, type, order ──
    DATASETS = [('EPL', 'EPL', 'real'), ('Sunspots', 'Sunspots', 'real'), ('etth1', 'ETTh1', 'real'), ('etth2', 'ETTh2', 'real'),
                ('melbourne_temp', 'Melbourne', 'real'), ('co2_weekly', 'CO$_2$', 'real'), ('elnino_monthly', 'El Niño', 'real'),
                ('MackeyGlass', 'Mackey–Glass', 'chaotic'), ('lorenz', 'Lorenz', 'chaotic'), ('lorenz_noisy', 'Lorenz (noisy)', 'chaotic'),
                ('rossler', 'Rössler', 'chaotic')]
    DS_ORDER = [d for d, _, _ in DATASETS]
    DS_NAME  = {d: n for d, n, _ in DATASETS}
    DS_KIND  = {d: k for d, _, k in DATASETS}
    DS_SLUG  = {'EPL': 'epl', 'Sunspots': 'sunspots', 'MackeyGlass': 'mackeyglass'}
    slug_of  = lambda d: DS_SLUG.get(d, d)
    TEX_DS   = {'CO$_2$': 'CO$_2$', 'El Niño': 'El Ni\\~no', 'Mackey–Glass': 'Mackey--Glass', 'Rössler': 'R\\"ossler'}

    # ── models: stored name → name in the paper (citation brackets dropped), style ──
    AF, V2, V1 = 'TRACE-SNP-AF', 'TRACE-SNP v2', 'TRACE-SNP'
    OI = {'vermillion': '#D55E00', 'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'purple': '#CC79A7',
          'sky': '#56B4E9', 'yellow': '#F0E442', 'black': '#000000', 'grey': '#8c8c8c', 'dark': '#4d4d4d'}
    STYLE = {
        AF:                 dict(color=OI['black'],      ls='-',  marker='*', lw=2.0, ms=7),
        V2:                 dict(color=OI['vermillion'], ls='-',  marker='o', lw=1.8, ms=5),
        V1:                 dict(color=OI['orange'],     ls='-',  marker='s', lw=1.4, ms=4.5),
        'ALSTM-SNP':        dict(color=OI['purple'],     ls='-',  marker='D', lw=1.4, ms=4.5),
        'LSTM [34]':        dict(color=OI['blue'],       ls='-',  marker='v', lw=0.9, ms=4),
        'GRU [39]':         dict(color=OI['blue'],       ls='--', marker='^', lw=0.9, ms=4),
        'LSTM-SNP [25]':    dict(color=OI['sky'],        ls='-',  marker='<', lw=0.9, ms=4),
        'BiLSTM-SNP [26]':  dict(color=OI['sky'],        ls='--', marker='>', lw=0.9, ms=4),
        'Attention':        dict(color=OI['green'],      ls='-',  marker='P', lw=0.9, ms=4),
        'Informer [42]':    dict(color=OI['green'],      ls='--', marker='X', lw=0.9, ms=4),
        'Autoformer [40]':  dict(color=OI['green'],      ls=':',  marker='h', lw=0.9, ms=4),
        'FEDformer [41]':   dict(color=OI['green'],      ls='-.', marker='H', lw=0.9, ms=4),
        'NG-RC':            dict(color=OI['dark'],       ls='--', marker='p', lw=0.9, ms=4),
        'DLinear':          dict(color=OI['dark'],       ls=':',  marker='d', lw=0.9, ms=4),
        'Ridge AR(L)':      dict(color=OI['grey'],       ls='--', marker='8', lw=0.9, ms=4),
        'Persistence':      dict(color=OI['grey'],       ls=':',  marker='.', lw=0.9, ms=4),
        # [extended] classical statistical forecasters and the dense (non-gated) Gaussian basis
        'Dense RBF (no firing condition)': dict(color='#882255', ls=':', marker='o', lw=1.0, ms=4),
        'ARIMA':            dict(color='#a6761d',        ls='-',  marker='1', lw=0.9, ms=5),
        'SES':              dict(color='#a6761d',        ls='--', marker='2', lw=0.9, ms=5),
        'Holt damped':      dict(color='#a6761d',        ls=':',  marker='3', lw=0.9, ms=5),
        'Seasonal naive':   dict(color='#a6761d',        ls='-.', marker='4', lw=0.9, ms=5),
        'Training median':  dict(color='#b3b3b3',        ls='-.', marker='_', lw=0.9, ms=5),
    }
    DENSE   = 'Dense RBF (no firing condition)'
    STATS   = ['ARIMA', 'SES', 'Holt damped', 'Seasonal naive', 'Training median']
    TRANS   = ['Informer [42]', 'Autoformer [40]', 'FEDformer [41]']
    MODELS  = list(STYLE)
    OURS    = [AF, V2, V1, 'ALSTM-SNP']
    NETS    = ['ALSTM-SNP', 'LSTM [34]', 'GRU [39]', 'LSTM-SNP [25]', 'BiLSTM-SNP [26]', 'Attention', 'Informer [42]',
               'Autoformer [40]', 'FEDformer [41]']
    KEY     = [AF, V2, 'ALSTM-SNP', 'LSTM-SNP [25]', 'BiLSTM-SNP [26]', 'GRU [39]', 'Informer [42]', 'DLinear']
    EXCLUDE = ['TRACE-SNP (binary spikes)', 'TRACE-SNP v2 (binary spikes)', 'TRACE: delay pathway only', 'TRACE: state pathway only',
               'TRACE: no curvature penalty', 'TRACE: level inputs only', 'TRACE: validation-only selection',
               'TRACE: no fallback candidates']
    name = lambda m: {V1: 'TRACE-SNP v1', 'Ridge AR(L)': 'Ridge AR($L$)', DENSE: 'Dense Gaussian basis', 'Holt damped': 'Damped Holt'}.get(
        m, re.sub(r'\s*\[\d+\]', '', m))
    plain = lambda m: name(m).replace('$', '')
    sty = lambda m: STYLE.get(m, dict(color='#999999', ls='-', marker='.', lw=0.8, ms=3))

    plt.rcParams.update({'font.family': 'serif', 'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'legend.fontsize': 6.5,
                         'xtick.labelsize': 7, 'ytick.labelsize': 7, 'axes.spines.top': False, 'axes.spines.right': False,
                         'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
                         'mathtext.fontset': 'dejavuserif'})
    W1, W2 = 3.5, 7.2                                          # Elsevier single / double column width (inches)
    FIG_DIR, TAB_DIR = os.path.join(PAPER_DIR, 'figures'), os.path.join(PAPER_DIR, 'tables')
    os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
    NUMBERS = []                                               # (key, value) lines for numbers.md
    MADE = {'figures': [], 'tables': []}


    def R(name_, required=True):
        # one revisions/<name>.csv (smoke and full runs kept apart)
        p = os.path.join(LOCAL_ROOT, 'revisions', f"{name_}{SM}.csv")
        if os.path.exists(p) and os.path.getsize(p) > 2:
            return pd.read_csv(p)
        if required:
            print(f"MISSING revisions/{name_}{SM}.csv — the outputs that need it are skipped")
        return pd.DataFrame()


    def many(pattern):
        fs = [f for f in glob.glob(os.path.join(LOCAL_ROOT, pattern)) if os.path.getsize(f) > 2 and (('_smoke' in f) == SMOKE)]
        return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


    def save(fig, fname):
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(FIG_DIR, f"{fname}.{ext}"), dpi=300)
        plt.close(fig); MADE['figures'].append(fname)


    def num(key, value):
        NUMBERS.append(f"- {key}: {value}")


    def tex_esc(s):
        s = str(s)
        parts = s.split('$')                                      # escape text only, never inside $...$
        for i in range(0, len(parts), 2):
            for a in ('&', '%', '_', '#'):                            # never escape a character twice
                parts[i] = re.sub(r'(?<!\\)' + re.escape(a), lambda m_: '\\' + a, parts[i])
        s = '$'.join(parts)
        for a, b in TEX_DS.items():
            s = s.replace(a, b)
        return s.replace('Niño', 'Ni\\~no').replace('Rössler', 'R\\"ossler').replace('–', '--').replace('µ', '$\\mu$')


    INT_COLS = {'N', 'k', 'Members', 'Blocks ($h=1$)', 'Blocks (all $h$)', 'Parameters', 'Hidden', 'Batch', 'Max.\\ epochs', 'Patience'}


    def write_table(df, fname, colfmt=None, bold_min=(), bold_max=(), fmt=lambda v: f"{v:.3f}", small='\\footnotesize', note=None):
        # booktabs tabular wrapped in adjustbox (max width = the column or page width it is placed in); no float, no caption:
        # main.tex / supplementary.tex own the captions, so a table can be placed in either
        cols = list(df.columns)
        best = {}
        for c in cols:
            v = pd.to_numeric(df[c], errors='coerce')
            if c in bold_min and v.notna().any():
                best[c] = v.min()
            if c in bold_max and v.notna().any():
                best[c] = v.max()
        lines = [small, '\\begin{adjustbox}{max width=\\linewidth}',
                 f"\\begin{{tabular}}{{{colfmt or 'l' + 'r' * (len(cols) - 1)}}}", '\\toprule',
                 ' & '.join(tex_esc(c) for c in cols) + ' \\\\', '\\midrule']
        for _, r in df.iterrows():
            cells = []
            for c in cols:
                v = r[c]
                if isinstance(v, (int, float, np.floating, np.integer)) and not isinstance(v, bool):
                    s_ = '--' if not np.isfinite(v) else fmt(v)
                    if c in INT_COLS and np.isfinite(v) and float(v).is_integer():   # counts print as integers on every pandas
                        s_ = str(int(v))
                    if c in best and np.isfinite(v) and np.isclose(v, best[c]):
                        s_ = f"\\textbf{{{s_}}}"
                else:
                    s_ = tex_esc(v)
                cells.append(s_)
            lines.append(' & '.join(cells) + ' \\\\')
        lines += ['\\bottomrule', '\\end{tabular}', '\\end{adjustbox}']
        if note:
            lines.append(f"\\par\\smallskip{{\\scriptsize {note}}}")
        with open(os.path.join(TAB_DIR, f"{fname}.tex"), 'w') as fh:
            fh.write('\n'.join(lines) + '\n')
        MADE['tables'].append(fname)


    # ── statistics over the N datasets (Demšar 2006; Benavoli et al. 2017) ──
    def holm(p):
        p = np.asarray(p, float); adj = np.full(len(p), np.nan)
        ok = np.flatnonzero(np.isfinite(p)); m = len(ok); run = 0.0
        for rank, i in enumerate(ok[np.argsort(p[ok])]):
            run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
        return adj


    def friedman(S):
        N, k = S.shape
        chi2, p = st.friedmanchisquare(*[S[c].to_numpy() for c in S.columns])
        Ff = (N - 1) * chi2 / (N * (k - 1) - chi2) if N * (k - 1) - chi2 > 0 else np.inf
        p_id = float(st.f.sf(Ff, k - 1, (k - 1) * (N - 1))) if np.isfinite(Ff) else 0.0
        cd = st.studentized_range.ppf(1 - ALPHA, k, np.inf) / np.sqrt(2) * np.sqrt(k * (k + 1) / (6.0 * N))
        return {'N': N, 'k': k, 'chi2': float(chi2), 'p': float(p), 'F': float(Ff), 'p_ID': p_id, 'cd': float(cd),
                'mean_rank': S.rank(axis=1, method='average').mean().sort_values()}


    def bayes_signrank(d, rope=ROPE, prior=0.5, n=50000, seed=0):
        # Bayesian signed-rank test (Benavoli et al. 2017): Dirichlet process on the differences, with a pseudo-observation at 0
        g = np.random.default_rng(seed)
        z = np.concatenate([[0.0], np.asarray(d, float)])
        w = g.dirichlet(np.r_[prior, np.ones(len(d))], size=n)
        s = z[:, None] + z[None, :]
        lo, mid, hi = (s < -rope), (np.abs(s) <= rope), (s > rope)
        P = np.stack([np.einsum('ni,ij,nj->n', w, m.astype(float), w) for m in (lo, mid, hi)], 1)
        win = P.argmax(1)
        return tuple(float(np.mean(win == i)) for i in range(3))


    def scores(lb):
        # one score per (dataset, model): geometric mean of RMSE / persistence over the dataset's (lag, horizon) problems
        rows = {}
        for d, g in lb.groupby('dataset', sort=False):
            piv = g.pivot_table(index=['lag', 'h'], columns='model', values='rel_rmse')
            piv = piv.loc[:, piv.notna().mean() >= 0.5].dropna()
            if len(piv):
                rows[d] = np.exp(np.log(piv).mean())
        S = pd.DataFrame(rows).T
        S = S.loc[[d for d in DS_ORDER if d in S.index], [m for m in MODELS if m in S.columns and S[m].notna().all()]]
        return S


    def scores_any(lb):
        # like scores(), but keeps a model that is missing on some datasets (NaN there), for display only
        rows = {}
        for d, g in lb.groupby('dataset', sort=False):
            piv = g.pivot_table(index=['lag', 'h'], columns='model', values='rel_rmse')
            core_ = [m for m in piv.columns if piv[m].notna().mean() >= 0.5 and m in MODELS and m not in [DENSE] + STATS]
            keep = piv[core_].dropna().index if core_ else piv.index
            p2 = piv.loc[keep]
            rows[d] = np.exp(np.log(p2).mean())
        return pd.DataFrame(rows).T


    def cd_diagram(ax, fr, title, fs=6.2, row_h=0.34):
        mr, cd, k = fr['mean_rank'], fr['cd'], fr['k']
        vals = mr.to_numpy(); half = (k + 1) // 2
        cl = []
        for i in range(k):
            j = i
            while j + 1 < k and vals[j + 1] - vals[i] < cd:
                j += 1
            if j > i and not any(a <= i and j <= b for a, b in cl):
                cl.append((i, j))
        bar_h = 0.13                                             # row_h: wider label rows for small panels
        y_bars0 = -0.25
        y_lab0 = y_bars0 - bar_h * max(1, len(cl)) - 0.25
        ax.set_xlim(0.2 - 0.12 * k, k + 0.8 + 0.12 * k)
        ax.set_ylim(y_lab0 - row_h * (half - 1) - 0.3, 0.95)
        ax.axis('off')
        ax.hlines(0, 1, k, color='k', lw=0.8)
        for r in range(1, k + 1):
            ax.vlines(r, 0, 0.1, color='k', lw=0.8); ax.text(r, 0.17, str(r), ha='center', va='bottom', fontsize=fs)
        ax.hlines(0.62, 1, 1 + cd, color='k', lw=1.4); ax.vlines([1, 1 + cd], 0.57, 0.67, color='k', lw=1.0)
        ax.text(1 + cd / 2, 0.72, f"CD = {cd:.2f}", ha='center', va='bottom', fontsize=fs)
        for n_, (i, j) in enumerate(cl):
            ax.hlines(y_bars0 - bar_h * n_, vals[i] - 0.05, vals[j] + 0.05, color='k', lw=2.0)
        for i, (m, r) in enumerate(mr.items()):
            left = i < half
            yy = y_lab0 - (i if left else (k - 1 - i)) * row_h
            xe = 0.65 if left else k + 0.35
            c_ = sty(m)['color']; lw_ = 1.5 if m in OURS else 0.8
            ax.plot([r, r], [0, yy], color=c_, lw=lw_); ax.plot([r, xe], [yy, yy], color=c_, lw=lw_)
            ax.text(xe + (-0.06 if left else 0.06), yy, f"{plain(m)} ({r:.2f})", ha='right' if left else 'left', va='center',
                    fontsize=fs, fontweight='bold' if m in OURS else 'normal')
        ax.set_title(title, loc='left', fontsize=7.5)


    def labelled(ax, txt):
        t = ax.get_title(loc='left')                             # panel letter in bold, prefixed to the panel title
        ax.set_title(r"$\bf{" + txt.replace('(', '(').replace(')', ')') + "}$" + (f"  {t}" if t else ''), loc='left',
                     fontsize=7)


    # =========================================================================================================
    # load everything once
    # =========================================================================================================
    LB      = R('leaderboard_with_trace_af')
    LB      = LB[LB['dataset'].isin(DS_ORDER) & ~LB['model'].isin(EXCLUDE)].copy() if len(LB) else LB
    if len(LB):                                                  # rank only on the problems TRACE-SNP-AF ran on
        ka = pd.MultiIndex.from_frame(LB[LB['model'] == AF][['dataset', 'lag', 'h']])
        LB = LB[pd.MultiIndex.from_frame(LB[['dataset', 'lag', 'h']]).isin(ka)]
    LB_MAIN = LB.copy()                                          # the 16-model comparison of the paper


    def ext_file(*names):
        # [extended] local path of an output of cells 26–31, or None
        for d_ in EXT_DIRS:
            for n_ in names:
                p_ = os.path.join(LOCAL_ROOT, d_, n_)
                if os.path.exists(p_) and os.path.getsize(p_) > 2:
                    return p_
        return None


    def ext_csv(*names, **kw):
        p_ = ext_file(*names)
        return pd.read_csv(p_, **kw) if p_ else pd.DataFrame()


    def extended_rows(lb):
        # [extended] rows in the leaderboard format (dataset, model, lag, h, rmse, rel_rmse) for the dense non-gated
        # Gaussian basis (basis study, exact per problem) and the classical forecasters (lag-free: one value per dataset and
        # horizon, repeated on every lag of the leaderboard, so the geometric mean over (lag, h) weights them like every model)
        add = []
        if not len(lb):
            return add
        pers = lb[lb['model'] == 'Persistence'].set_index(['dataset', 'h', 'lag'])['rmse']
        if EXT_DENSE:
            bs = many("revisions/basis_study/*_lag*_preds.csv")
            if len(bs) and 'gauss_dense' in set(bs['model']):
                b = bs[(bs['model'] == 'gauss_dense') & bs['dataset'].astype(str).isin(DS_ORDER)]
                g = b.groupby(['dataset', 'h', 'lag']).apply(lambda d: float(np.sqrt(np.mean((d['y'] - d['ens_pred']) ** 2))),
                                                            include_groups=False).rename('rmse').reset_index()
                g['rel_rmse'] = g['rmse'] / g.set_index(['dataset', 'h', 'lag']).index.map(pers).to_numpy()
                add.append(g.assign(model=DENSE))
            else:
                print("NOTE: no basis_study 'gauss_dense' forecasts in the results folder — dense basis left out of the extended ranking")
        if EXT_CLASSICAL:
            stt = ext_csv('statistical_baselines_by_h.csv')
            if len(stt):
                stt = stt[stt['dataset'].astype(str).isin(DS_ORDER) & stt['model'].isin(STATS)]
                probs = lb[['dataset', 'lag', 'h']].drop_duplicates()
                g = probs.merge(stt[['dataset', 'model', 'h', 'rel_rmse']], on=['dataset', 'h'])
                g['rmse'] = g['rel_rmse'] * g.set_index(['dataset', 'h', 'lag']).index.map(pers).to_numpy()
                add.append(g)
            else:
                print("NOTE: statistical_baselines_by_h.csv not in the results folder — run cell 28; "
                      "classical forecasters left out")
        return add


    EXT_ROWS  = extended_rows(LB)
    LB_EXT    = pd.concat([LB] + EXT_ROWS, ignore_index=True) if EXT_ROWS else LB.copy()
    EXT_ADDED = [m for m in [DENSE] + STATS if len(LB_EXT) and m in set(LB_EXT['model'])]
    if EXT_ROWS:
        print("extended comparison: " + ", ".join(f"{m} ({n} problems)" for m, n in
              pd.concat(EXT_ROWS)['model'].value_counts().items())
              + (" — also in every main ranking" if EXTENDED_IN_MAIN_RANKING else " — in its own tables and figures"))
    if EXTENDED_IN_MAIN_RANKING:
        LB = LB_EXT
    MAIN_ADDED = EXT_ADDED if EXTENDED_IN_MAIN_RANKING else []
    S_ALL   = scores(LB) if len(LB) else pd.DataFrame()
    FR_ALL  = friedman(S_ALL) if len(S_ALL) else None
    S_EXT   = scores(LB_EXT) if len(LB_EXT) else pd.DataFrame()
    FR_EXT  = friedman(S_EXT) if len(S_EXT) else None
    LYAP    = ext_csv('lyapunov.csv')                          # [extended] Lyapunov times of the three generators
    RBF_D   = R('rbf_ablation_by_dataset'); RBF_H = R('rbf_ablation_by_horizon'); RBF_P = R('rbf_ablation_by_problem')
    BASIS   = R('basis_study_summary');     BASIS_D = R('basis_study_by_dataset')
    AFCH    = R('trace_af_chosen_firing');  AFEO = R('trace_af_energy_ops'); AFNZ = R('trace_af_noise')
    DM      = R('dm_trace_af_vs_v2')
    SEED    = R('variance_seed_spread');    SEED_D = R('variance_seed_cv_by_dataset')
    BLK     = R('variance_block_by_dataset'); BLKS = R('variance_block_scores'); WIN = R('variance_block_win_rate_trace_af')
    DELAY   = R('delay_functions_with_se')
    EN      = R('energy_per_member_vs_ensemble'); AE = R('accuracy_vs_energy'); V1V2 = R('trace_v1_vs_v2_inference')
    NOISE   = R('noise_accuracy_median');    NOISE_RK = R('noise_10pct_mean_rank_n11')
    NETP    = R('params_networks');          P2 = R('params_trace_v2_choices'); P1 = R('params_trace_v1_choices', False)
    AF_CHO  = many(f"revisions/trace_af/*{SM}_lag*_choices.csv")
    AF_PRED = many(f"revisions/trace_af/*{SM}_lag*_preds.csv")
    V2_PRED = many(f"*/trace_snp_v2b/t2{SM}_lag*_preds.csv")
    print(f"leaderboard: {LB['dataset'].nunique() if len(LB) else 0} datasets × {LB['model'].nunique() if len(LB) else 0} models")


    # =========================================================================================================
    # MAIN-PAPER FIGURES
    # =========================================================================================================
    def _box(ax, x, y, w, h, txt, fc='white', ec='k', fs=6, lw=0.8, ls='-', bold=False, ha='center'):
        from matplotlib.patches import FancyBboxPatch
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.25,rounding_size=0.8', fc=fc, ec=ec, lw=lw, ls=ls,
                                    zorder=2))
        tx = x + w / 2 if ha == 'center' else x + 0.8
        ax.text(tx, y + h / 2, txt, ha=ha, va='center', fontsize=fs, zorder=3, fontweight='bold' if bold else 'normal',
                linespacing=1.25)
        return (x, y, w, h)


    def _arr(ax, p0, p1, color='k', lw=0.8, rad=0.0, ls='-'):
        ax.annotate('', xy=p1, xytext=p0, zorder=1,
                    arrowprops=dict(arrowstyle='-|>', color=color, lw=lw, ls=ls, mutation_scale=7,
                                    shrinkA=0, shrinkB=0, connectionstyle=f"arc3,rad={rad}"))


    def _canvas(w, h, xmax, ymax):
        fig, ax = plt.subplots(figsize=(w, h))
        ax.set_xlim(0, xmax); ax.set_ylim(0, ymax); ax.set_axis_off()
        return fig, ax


    def fig_arch_alstm():
        # Architecture of ALSTM-SNP (Eqs. 2-5 of the paper): one neuron's update and the forecasting model around it
        fig, ax = _canvas(W2, 3.1, 100, 46)
        ax.add_patch(plt.Rectangle((0.5, 0.5), 70, 43.5, fc='#f7f7f7', ec=OI['grey'], lw=0.6, ls='--', zorder=0))
        ax.text(1.5, 42.3, 'ALSTM-SNP neuron, step $t$', fontsize=7, fontweight='bold', va='center')
        # inputs
        ins = {'x': (35, '$x_t$'), 'h': (27, '$h_{t-1}$'), 'u': (19, '$u_{t-1}$'), 'e': (5, r'$\eta_{t-1}$')}
        for k, (y, t) in ins.items():
            ax.text(3.0, y, t, fontsize=7, va='center', ha='center')
        # gates
        G = {'r': (35, r'$r_t=\sigma(\cdot)$' + '\nreset'), 'c': (28, r'$c_t=\sigma(\cdot)$' + '\nconsume'),
             'o': (21, r'$o_t=\sigma(\cdot)$' + '\ngenerate'), 'n': (14, r'$n_t=\tanh(\cdot)$' + '\ninput')}
        for k, (y, t) in G.items():
            _box(ax, 11, y - 2.6, 11, 5.2, t, fc='#e8f1fa', ec=OI['blue'], fs=5.6)
        ax.plot([6, 8.5], [35, 35], color='k', lw=0.6); ax.plot([6, 8.5], [27, 27], color='k', lw=0.6)
        ax.plot([6, 8.5], [19, 19], color='k', lw=0.6)
        ax.plot([8.5, 8.5], [14, 35], color='k', lw=0.6)
        for y in (35, 28, 21, 14):
            _arr(ax, (8.5, y), (10.6, y), lw=0.6)
        ax.text(0.9, 10.4, r'gate input $Wx_t+Uh_{t-1}\,(+v\odot u_{t-1})$, diag $U=0$', fontsize=4.7, va='center',
                color=OI['dark'])
        # charge, threshold, fire
        _box(ax, 26.5, 30, 14, 7, 'charge\n' + r'$\tilde u_t=\alpha\odot r_t\odot u_{t-1}+n_t$', fc='white', ec='k', fs=5.6)
        _box(ax, 26.5, 2.5, 14, 7, 'adaptive threshold\n' + r'$B_t=b_0+\beta\odot\eta_{t-1}\geq b_0>0$', fc='#fdf0e6',
             ec=OI['vermillion'], fs=5.4)
        _box(ax, 43, 13.5, 11.5, 11.5, 'firing condition\n' + r'$v_t=(\tilde u_t-B_t)/B_t$' + '\n' + r'$z_t=\Theta(v_t)$'
             + '\nsurrogate gradient\n' + r'$\gamma\max(0,1-|v_t|)$',
             fc='#fdf0e6', ec=OI['vermillion'], fs=5.6, lw=1.2)
        _arr(ax, (22.3, 35), (26.7, 34.5)); _arr(ax, (22.3, 14), (26.7, 31.2), rad=0.25)
        ax.plot([5, 24], [19, 19], color='k', lw=0.6, ls=':'); _arr(ax, (24, 19), (26.7, 32.5), lw=0.6, ls=':', rad=-0.2)
        _arr(ax, (6, 5), (26.7, 5.5))
        _arr(ax, (40.8, 33.5), (46, 25.3), rad=-0.2); _arr(ax, (40.8, 6), (46, 13.2), rad=0.2)
        # updates
        _box(ax, 57.5, 31, 12.3, 7, 'reset by consumption\n' + r'$u_t=\tilde u_t-z_t\odot c_t\odot B_t$', fs=5.1)
        _box(ax, 57.5, 15.5, 12.3, 8, 'spike output\n' + r'$h_t=o_t\odot z_t$' + '\n' + r'$\quad\odot\,\mathrm{softplus}(\tilde u_t)$', fs=5.1,
             fc='#e9f6f1', ec=OI['green'])
        _box(ax, 57.5, 2.5, 12.3, 7, 'firing trace\n' + r'$\eta_t=\rho\,\eta_{t-1}+(1-\rho)z_t$', fs=5.3)
        _arr(ax, (54.8, 22.5), (57.2, 33)); _arr(ax, (54.8, 19.5), (57.2, 19.5)); _arr(ax, (54.8, 16.5), (57.2, 6.5))
        _arr(ax, (22.3, 28), (57.2, 35.5), rad=-0.12, color=OI['blue'], lw=0.6)
        _arr(ax, (22.3, 21), (57.2, 21.5), rad=-0.05, color=OI['blue'], lw=0.6)
        ax.text(40.8, 1.25, r'$\alpha,\rho\in(0,1)$, $\beta\geq0$, $b_0=\mathrm{softplus}(\cdot)+10^{-3}$', fontsize=4.8,
                color=OI['dark'])
        # model around the neuron
        ax.text(73.5, 42.3, 'Forecasting model', fontsize=7, fontweight='bold', va='center')
        _box(ax, 73.5, 34, 25, 5, r'window $x_{t-L+1},\dots,x_t$ (min–max scaled)', fs=5.8)
        _box(ax, 73.5, 24.5, 25, 6.5, 'ALSTM-SNP layer, $H=8$ neurons\nunrolled over the $L$ steps', fs=5.8,
             fc='#fdf0e6', ec=OI['vermillion'])
        _box(ax, 73.5, 15, 25, 6.5, 'linear readout on\n' + r'$[h_T,\ u_T,\ \eta_T,\ \bar h]$, $\bar h$ = mean of $h_t$', fs=5.8)
        _box(ax, 73.5, 6.5, 25, 5.5, 'one-step forecast $\\hat x_{t+1}$\n(iterated $h$ times for $h>1$)', fs=5.6)
        _box(ax, 73.5, 0.6, 25, 3.8, 'reported: mean of 60 seeds', fs=5.6, fc='#eeeeee', ec=OI['grey'])
        for y0, y1 in ((33.7, 31.3), (24.2, 21.8), (14.7, 12.3), (6.2, 4.7)):
            _arr(ax, (86, y0), (86, y1))
        _arr(ax, (70.2, 19.5), (73.2, 27.5), rad=0.2, color=OI['grey'], ls='--', lw=0.6)
        fig.tight_layout(pad=0.1); save(fig, 'fig_arch_alstm')


    def fig_arch_trace():
        # Architecture of TRACE-SNP v1, v2 and TRACE-SNP-AF (Eqs. 6-8 and Sections 3.3-3.5 of the paper)
        fig, ax = _canvas(W2, 4.0, 100, 60)
        c_v2, c_af = OI['orange'], OI['blue']
        ax.text(0.5, 58.5, 'Forecast path (one evaluation per forecast)', fontsize=7, fontweight='bold', va='center')
        _box(ax, 0.5, 38, 12, 14, 'window\n' + r'$x_{o-1},\dots,x_{o-L}$', fs=6)
        _box(ax, 16, 38, 15, 14, 'encode each delay\n' + r'$u_k=s(x_{o-k})$ (lvl) or' + '\n'
             + r'$u_k=s^\prime(x_{o-k}-x_{o-1})$ (dif)' + '\n\nanchor $\\bar x_o$: newest value\nor window mean$^{\\ast}$', fs=5.2)
        _arr(ax, (12.9, 45), (15.7, 45))
        # three pathways
        _box(ax, 35, 50, 27, 6, 'linear neurons (always active)\n' + r'$b+\Sigma_k\,a_k u_k$', fs=5.6)
        p = _box(ax, 35, 33, 27, 15, '', fs=5.6, fc='#fdf0e6', ec=OI['vermillion'], lw=1.1)
        ax.text(36, 46.4, 'receptive-field neurons, $M$ per delay', fontsize=5.6, va='center')
        ax.text(36, 35.0, r'fire: $|u_k-c_m|<\Delta$, 2 of $M$ (tri) ' + '\n' + r'or $k$ nearest of Gaussian$^{\dagger}$',
                fontsize=5.0, va='center')
        cs = np.linspace(39.5, 57.5, 9); dlt = cs[1] - cs[0]; uu = 48.6
        for i, cm in enumerate(cs):
            on = abs(uu - cm) < dlt
            ax.plot([cm - dlt, cm, cm + dlt], [38.2, 44.4, 38.2], color=OI['vermillion'] if on else '#bdbdbd',
                    lw=1.2 if on else 0.6, zorder=3)
        ax.plot([uu, uu], [38, 45], color='k', lw=0.6, ls=':', zorder=3); ax.text(uu + 0.4, 44.6, '$u_k$', fontsize=5.2)
        _box(ax, 35, 18, 27, 12, 'state neurons ($C$ $k$-means prototypes of $\\mathbf{z}$)\n'
             + 'fire: prototype among the 4 nearest\nTakagi–Sugeno consequent\n' + r'$\nu_c(\mathbf{z})(\alpha_c+\gamma_c^{\top}\mathbf{z})$',
             fs=5.3, fc='#e9f6f1', ec=OI['green'])
        for y in (53, 40.5, 24):
            _arr(ax, (31.3, 45), (34.7, y), rad=0.0)
        # sum and output
        ax.add_patch(plt.Circle((67, 40.5), 2.2, fc='white', ec='k', lw=0.8, zorder=2)); ax.text(67, 40.5, r'$\Sigma$', ha='center', va='center', fontsize=7)
        for y in (53, 40.5, 24):
            _arr(ax, (62.3, y), (64.9, 40.5 + (y - 40.5) * 0.06))
        _box(ax, 72, 34, 14, 9, r'$\hat x_{o+h-1}=\bar x_o+s^{-1}[\Sigma]$' + '\n(direct $h$-step)', fs=5.4)
        _arr(ax, (69.3, 40.5), (71.7, 39.5))
        _box(ax, 72, 18.5, 14, 12, 'multi-step rule$^{\\ast}$ ($h>1$):\niterate the one-step\nmodel $h$ times only if\nits validation RMSE\nis $\\geq25\\%$ lower',
             fs=5.0, fc='#fff6e0', ec=c_v2, ls='--')
        _arr(ax, (79, 33.7), (79, 30.8), color=c_v2)
        # fit path
        ax.text(0.5, 13.8, 'Fit path (closed form; training and validation data only)', fontsize=7, fontweight='bold', va='center')
        fit = [('sparse design $\\mathbf{\\Phi}$\n$N$ windows $\\times\\,p$', 'white', 'k', '-'),
               ('penalty $\\mathbf{P}$: ridge $\\lambda_0$,\ncurvature $\\lambda_s\\|\\mathbf{D}_2\\mathbf{w}_k\\|^2$,\nstate ridge', 'white', 'k', '-'),
               ('one Cholesky of\n$\\mathbf{A}=\\frac{1}{N}\\mathbf{\\Phi}^{\\top}\\mathbf{\\Phi}+\\mathbf{P}\\succ0$', 'white', 'k', '-'),
               ('exact leave-one-out\n$e_i/(1-H_{ii})$\n+ validation block', 'white', 'k', '-'),
               ('select best candidate\n(grid $\\cup$ persistence,\nmean, ridge AR)', 'white', 'k', '-'),
               ('parsimony$^{\\dagger}$: fewest\nfiring neurons within\n1% of the best', '#e8f1fa', c_af, '--')]
        xw = 15.2
        for i, (t, fc, ec, ls) in enumerate(fit):
            _box(ax, 0.5 + i * (xw + 1.6), 1.0, xw, 10, t, fs=5.1, fc=fc, ec=ec, ls=ls)
            if i:
                _arr(ax, (0.5 + i * (xw + 1.6) - 1.5, 6.0), (0.5 + i * (xw + 1.6) - 0.2, 6.0), color=c_af if i == 5 else 'k')
        _arr(ax, (8.3, 37.7), (8.3, 11.3), color=OI['grey'], ls='--', lw=0.6)
        _arr(ax, (93, 11.3), (86.4, 38.5), color=OI['grey'], ls='--', lw=0.6, rad=0.35)
        ax.text(86.8, 24, r'$\hat{\theta}=\mathbf{A}^{-1}\frac{1}{N}\mathbf{\Phi}^{\top}\mathbf{r}$', fontsize=6, color=OI['dark'])
        # variant key
        ax.text(72, 58.5, 'Members', fontsize=7, fontweight='bold', va='center')
        ax.text(72, 55.6, 'TRACE-SNP v1: everything unmarked', fontsize=5.4, va='center')
        ax.text(72, 53.1, r'$^{\ast}$ TRACE-SNP v2 adds the mean anchor and the', fontsize=5.4, va='center', color=c_v2)
        ax.text(73.4, 51.0, 'direct-vs-iterated rule', fontsize=5.4, va='center', color=c_v2)
        ax.text(72, 48.5, r'$^{\dagger}$ TRACE-SNP-AF adds the firing family', fontsize=5.4, va='center', color=c_af)
        ax.text(73.4, 46.4, r'$\{$tri $k$=2, Gauss $k\in\{2,4,8\}$, $w\in\{1,2\}\}$ + parsimony', fontsize=5.0, va='center', color=c_af)
        fig.tight_layout(pad=0.1); save(fig, 'fig_arch_trace')


    def fig_family():
        # Fig. 1 — the firing condition in the three members (illustration computed from the model equations)
        fig, ax = plt.subplots(1, 3, figsize=(W2, 2.05), gridspec_kw={'width_ratios': [1.35, 1, 1]})
        g = np.random.default_rng(3); T = 60
        x = np.sin(np.arange(T) / 4.0) * 0.8 + g.normal(0, 0.25, T)
        alpha, b0, beta, rho = 0.8, 0.45, 0.6, 0.85
        u = eta = 0.0; U, B, Z = [], [], []
        for t in range(T):                                     # one ALSTM-SNP neuron with r = c = 1 and n_t = 0.6 tanh(x_t)
            ut = alpha * u + 0.6 * np.tanh(x[t]); Bt = b0 + beta * eta
            z = float(ut >= Bt); u = ut - z * Bt; eta = rho * eta + (1 - rho) * z
            U.append(ut); B.append(Bt); Z.append(z)
        ax[0].plot(U, color=OI['blue'], lw=1.0, label='charge $\\tilde u_t$')
        ax[0].plot(B, color=OI['vermillion'], lw=1.2, ls='--', label='threshold $B_t=b_0+\\beta\\eta_{t-1}$')
        for t in np.flatnonzero(Z):
            ax[0].vlines(t, -1.15, -0.9, color=OI['black'], lw=1.0)
        ax[0].axhline(b0, color=OI['grey'], lw=0.6, ls=':', label='base threshold $b_0>0$')
        ax[0].set_ylim(-1.2, 3.1); ax[0].set_xlabel('step $t$'); ax[0].set_yticks([])
        ax[0].legend(frameon=False, loc='upper left', fontsize=6, ncol=1)
        ax[0].set_title(f"ALSTM-SNP: learned threshold ({int(sum(Z))}/{T} spikes)", loc='left', fontsize=7)
        uu = np.linspace(-1.1, 1.1, 400); M = 9; c = np.linspace(-1.1, 1.1, M); h = c[1] - c[0]; u0 = 0.23
        for m in range(M):
            mu = np.maximum(0, 1 - np.abs(uu - c[m]) / h)
            fire = abs(u0 - c[m]) < h
            ax[1].plot(uu, mu, color=OI['vermillion'] if fire else '#bbbbbb', lw=1.4 if fire else 0.8)
        ax[1].axvline(u0, color=OI['black'], lw=0.8, ls=':')
        ax[1].set_title('TRACE-SNP: 2 of $M$ fields fire', loc='left', fontsize=7); ax[1].set_xlabel('$u_k$'); ax[1].set_yticks([0, 1])
        for k_, col, ls in ((2, OI['sky'], ':'), (8, OI['black'], '-')):
            d = np.abs(u0 - c)
            keep = np.argsort(d)[:k_]
            act = np.exp(-d ** 2 / (2 * h ** 2))
            ax[2].bar(np.arange(M) + (-0.2 if k_ == 2 else 0.2), np.where(np.isin(np.arange(M), keep), act, 0), width=0.38,
                      color=col, alpha=0.85, label=f"Gaussian, $k={k_}$ fire")
        ax[2].plot(np.arange(M), np.exp(-(u0 - c) ** 2 / (2 * h ** 2)), color=OI['vermillion'], lw=1.0, marker='o', ms=2.5,
                   label='dense (all fire)')
        ax[2].set_xlabel('field $m$'); ax[2].set_title('TRACE-SNP-AF: width chosen', loc='left', fontsize=7)
        ax[2].set_ylim(0, 2.15); ax[2].legend(frameon=False, fontsize=5.8, loc='upper left', ncol=1)
        for a, l in zip(ax, 'abc'):
            labelled(a, f"({l})")
        fig.tight_layout(w_pad=1.2); save(fig, 'fig1_family')


    def read_series(path):
        # the series exactly as cell 3 reads it: same column, same trimming
        import csv
        with open(path, newline='', errors='replace') as fh:
            try:
                sep = csv.Sniffer().sniff(fh.read(20000), delimiters=',;\t|').delimiter
            except csv.Error:
                sep = ','
        def isnum(s):
            try:
                float(s); return True
            except (TypeError, ValueError):
                return False
        df = pd.read_csv(path, sep=sep)
        if len(df.columns) and all(isnum(c) for c in df.columns):
            df = pd.read_csv(path, sep=sep, header=None); df.columns = [f'col{i}' for i in range(df.shape[1])]
        num = df.apply(lambda c: pd.to_numeric(c, errors='coerce'))
        numeric = [c for c in df.columns if num[c].notna().mean() >= 0.9]
        index_names = {'t', 'time', 'index', 'idx', 'id', 'date', 'datetime', 'timestamp', 'year', 'month', 'day', 'hour',
                       'minute', 'step', 'unnamed: 0'}
        def index_like(c):
            if str(c).strip().lower() in index_names:
                return True
            v = num[c].dropna().to_numpy()
            if len(v) < 3:
                return False
            dv = np.diff(v)
            return bool(np.all(v == np.round(v)) and np.all(dv == dv[0]) and dv[0] != 0)
        lower = {str(c).strip().lower(): c for c in df.columns}
        mg_sheet = 't' in lower and ('t-taw' in lower or 't-tau' in lower)        # the Mackey-Glass layout
        if mg_sheet:
            col = lower['t']
        else:
            named = [c for c in numeric if str(c).strip().lower() in ('value', 'values', 'y', 'target')]
            rest = [c for c in numeric if not index_like(c)]
            col = named[0] if named else (rest[-1] if rest else numeric[-1])
        s = num[col]
        if mg_sheet:                                                              # zero padding is trimmed
            nz = np.flatnonzero(s.fillna(0).to_numpy()); s = s.iloc[nz[0]:nz[-1] + 1]
        s = s.loc[s.first_valid_index():s.last_valid_index()]
        return s.interpolate(method='linear').to_numpy(dtype=np.float64)


    def fig_datasets():
        # Fig. 2 — the 11 series (last 600 points) with the five walk-forward test blocks shaded
        fig, axes = plt.subplots(4, 3, figsize=(W2, 5.0)); axes = axes.ravel()
        for i, d in enumerate(DS_ORDER):
            ax = axes[i]
            if d in DS_SLUG:
                fs = [f for f in glob.glob(os.path.join(DATA_ROOT, '**', '*.csv'), recursive=True)
                      if {'EPL': 'epl', 'Sunspots': 'sunspot', 'MackeyGlass': 'mackey'}[d] in os.path.basename(f).lower()]
            else:
                fs = glob.glob(os.path.join(LOCAL_ROOT, 'extra_data', f"{d}.csv"))
            if not fs:
                ax.set_axis_off(); ax.set_title(f"{DS_NAME[d]} (data file not found)", fontsize=7); continue
            x = read_series(fs[0])
            if d in ('etth1', 'etth2'):
                x = x[-4000:]
            n = len(x); block = min(60, max(5, int(0.3 * n // 5)))
            lo = max(0, n - 600)
            ax.plot(np.arange(lo, n), x[lo:], lw=0.6, color=OI['blue'] if DS_KIND[d] == 'real' else OI['vermillion'])
            for j in range(5):
                s0 = n - (5 - j) * block
                ax.axvspan(s0, s0 + block, color='#000000', alpha=0.06 + 0.03 * (j % 2), lw=0)
            ax.set_title(f"{DS_NAME[d]} ({DS_KIND[d]}, n = {n:,})", loc='left', fontsize=7)
            ax.tick_params(labelsize=5.5)
            num(f"dataset {d}", f"n = {n}, test block = {block}")
        axes[-1].set_axis_off()
        axes[-1].legend(handles=[Line2D([], [], color=OI['blue'], lw=1, label='real series'),
                                 Line2D([], [], color=OI['vermillion'], lw=1, label='chaotic series'),
                                 Patch(color='#000000', alpha=0.1, label='5 walk-forward test blocks')], frameon=False, loc='center')
        fig.tight_layout(h_pad=0.6); save(fig, 'fig2_datasets')


    def fig_cd():
        if FR_ALL is None:
            return
        fig, ax = plt.subplots(figsize=(W2, max(2.55, 0.5 + 0.13 * FR_ALL['k'])))   # taller when k grows
        cd_diagram(ax, FR_ALL, f"All horizons — Friedman $\\chi^2_{{{FR_ALL['k'] - 1}}}$ = {FR_ALL['chi2']:.1f}, p = {FR_ALL['p']:.1e}; "
                               f"N = {FR_ALL['N']} datasets, k = {FR_ALL['k']} models", fs=6.3)
        save(fig, 'fig3_cd')
        num('Friedman all', f"chi2 = {FR_ALL['chi2']:.2f}, p = {FR_ALL['p']:.2e}, Iman-Davenport p = {FR_ALL['p_ID']:.2e}, CD = {FR_ALL['cd']:.2f}")
        for m, r in FR_ALL['mean_rank'].items():
            num(f"mean rank {m}", f"{r:.2f}")


    def fig_scores():
        if not len(S_ALL):
            return
        order = list(FR_ALL['mean_rank'].index)
        Z = np.log10(S_ALL[order].T.to_numpy())
        n_rank = len(order)                                      # ranked rows; the best-per-dataset frames stay among them
        extra = [m for m in EXT_ADDED if m not in order] if EXTENDED_IN_FIGURES and not EXTENDED_IN_MAIN_RANKING else []
        if extra:                                                # [extended] added models as unranked rows under a line
            SX = scores_any(LB_EXT).reindex(S_ALL.index)
            extra = [m for m in extra if m in SX.columns]
            Z = np.vstack([Z, np.log10(SX[extra].T.to_numpy(float))])
            order = order + extra
        fig, ax = plt.subplots(figsize=(W2, max(3.3, 0.6 + 0.17 * len(order))))      # taller when rows are added
        im = ax.imshow(Z, aspect='auto', cmap='cividis_r', vmin=np.nanpercentile(Z, 2), vmax=min(0.15, np.nanmax(Z)))
        ax.set_xticks(range(S_ALL.shape[0])); ax.set_xticklabels([DS_NAME[d] for d in S_ALL.index], rotation=35, ha='right')
        ax.set_yticks(range(len(order))); ax.set_yticklabels([plain(m) for m in order])
        for t in ax.get_yticklabels():
            if t.get_text() in [plain(m) for m in OURS]:
                t.set_fontweight('bold')
            if t.get_text() in [plain(m) for m in MAIN_ADDED + extra]:     # added forecasters in italics
                t.set_fontstyle('italic')
        for i in range(Z.shape[0]):
            for j in range(Z.shape[1]):
                v = 10 ** Z[i, j]
                if not np.isfinite(v):
                    ax.text(j, i, '–', ha='center', va='center', fontsize=4.6, color='0.5'); continue
                ax.text(j, i, f"{v:.2f}" if v >= 0.1 else f"{v:.3f}", ha='center', va='center', fontsize=4.6,
                        color='white' if Z[i, j] > np.nanpercentile(Z, 60) else 'black')
            best = np.nanargmin(Z[:n_rank], axis=0)
        for j, i in enumerate(best):
            ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=OI['vermillion'], lw=1.2))
        ax.axvline(6.5, color='white', lw=2)
        if len(order) > n_rank:
            ax.axhline(n_rank - 0.5, color='white', lw=2)
        cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.set_label('log$_{10}$ RMSE / persistence', fontsize=6.5)
        cb.ax.tick_params(labelsize=6)
        fig.tight_layout(); save(fig, 'fig4_scores')


    def ylog(ax):
        # log y axis with plain tick labels; a narrow range (< 4x) stays linear so its ticks remain readable
        lo, hi = ax.dataLim.intervaly                            # the data range (view limits carry a margin that can go below 0)
        if np.isfinite(lo) and np.isfinite(hi) and lo > 0 and hi / lo >= 4:
            ax.set_yscale('log')
            ax.yaxis.set_major_locator(matplotlib.ticker.LogLocator(subs=(1.0, 2.0, 5.0) if hi / lo < 30 else (1.0,), numticks=6))
            ax.yaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter('%g'))
            ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())


    def hbox(ax, data, **kw):
        try:
            return ax.boxplot(data, orientation='horizontal', **kw)
        except TypeError:                                         # matplotlib < 3.10
            return ax.boxplot(data, vert=False, **kw)


    def fig_ablation():
        if not (len(RBF_D) and len(BASIS)):
            return
        fig, ax = plt.subplots(1, 3, figsize=(W2, 2.9), gridspec_kw={'width_ratios': [1.15, 1, 1.15]})
        r = RBF_D.set_index('dataset').reindex([d for d in DS_ORDER if d in set(RBF_D['dataset'])])
        col = 'RMSE ratio RBF/gated (geo-mean)'
        cols_ = [OI['blue'] if DS_KIND[d] == 'real' else OI['vermillion'] for d in r.index]
        ax[0].barh(range(len(r)), r[col], color=cols_); ax[0].axvline(1, color='k', lw=0.7)
        ax[0].set_yticks(range(len(r))); ax[0].set_yticklabels([DS_NAME[d] for d in r.index]); ax[0].invert_yaxis()
        ax[0].set_xlabel('RMSE dense / gated (k = 2)'); ax[0].set_title('dense basis vs fixed 2-of-M', loc='left', fontsize=7)
        b = BASIS.set_index('key') if 'key' in BASIS else BASIS.set_index('variant')
        lab = {'tri': 'tri, k=2', 'hat_dense': 'pw-lin dense', 'gauss_k2': 'Gauss k=2', 'gauss_k4': 'Gauss k=4',
               'gauss_k8': 'Gauss k=8', 'gauss_dense': 'Gauss dense'}
        mk = {'tri': 's', 'hat_dense': 'D', 'gauss_k2': 'v', 'gauss_k4': '^', 'gauss_k8': 'o', 'gauss_dense': 'P'}
        for kind_, c_, colname in (('chaotic', OI['vermillion'], '… chaotic'), ('real', OI['blue'], '… real')):
            for key, rr in b.iterrows():
                ax[1].scatter(rr['active share'] * 100, rr[colname], marker=mk.get(key, 'o'), color=c_, s=22, zorder=3)
            g_ = b.loc[[k for k in ['gauss_k2', 'gauss_k4', 'gauss_k8', 'gauss_dense'] if k in b.index]]
            ax[1].plot(g_['active share'] * 100, g_[colname], color=c_, lw=0.8, label=f"{kind_} series (Gaussian)")
        if len(AFCH):
            pass
        ax[1].set_xscale('log'); ax[1].axhline(1, color='k', lw=0.6, ls=':')
        ax[1].set_xticks([5, 10, 25, 50, 100]); ax[1].set_xticklabels(['5', '10', '25', '50', '100'])
        ax[1].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax[1].yaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter('%.2f'))
        ax[1].set_xlabel('active neurons (%)'); ax[1].set_ylabel('RMSE / dense Gaussian')
        ax[1].legend(handles=[Line2D([], [], color='k', marker=mk[k], ls='', ms=4, label=lab[k]) for k in mk if k in b.index]
                     + [Line2D([], [], color=OI['vermillion'], lw=1, label='chaotic'), Line2D([], [], color=OI['blue'], lw=1, label='real')],
                     frameon=False, fontsize=5.3, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.27))
        ax[1].set_title('shape × firing width', loc='left', fontsize=7)
        if len(BASIS_D):
            bd = BASIS_D.set_index('dataset').reindex([d for d in DS_ORDER if d in set(BASIS_D['dataset'])])
            cc = [c for c in bd.columns if c != 'type']
            Zb = np.log2(bd[cc].astype(float).to_numpy())
            im = ax[2].imshow(Zb, aspect='auto', cmap='RdBu_r', vmin=-1.5, vmax=1.5)
            ax[2].set_yticks(range(len(bd))); ax[2].set_yticklabels([DS_NAME[d] for d in bd.index], fontsize=6)
            short_ = ['tri k2', 'pw-lin', 'G k2', 'G k4', 'G k8', 'G dense']
            ax[2].set_xticks(range(len(cc))); ax[2].set_xticklabels(short_[:len(cc)], rotation=90, ha='center', fontsize=6)
            cb = fig.colorbar(im, ax=ax[2], fraction=0.05, pad=0.02); cb.set_label('log$_2$ RMSE / dense', fontsize=6)
            cb.ax.tick_params(labelsize=5.5)
            ax[2].set_title('per dataset', loc='left', fontsize=7)
        for a, l in zip(ax, 'abc'):
            labelled(a, f"({l})")
        fig.tight_layout(w_pad=0.8); save(fig, 'fig5_ablation')


    def fig_af():
        if not len(AFCH):
            return
        fig, ax = plt.subplots(1, 3, figsize=(W2, 3.6), gridspec_kw={'width_ratios': [1.1, 1.1, 1]}, layout='constrained')
        c = AFCH.set_index('dataset').reindex([d for d in DS_ORDER if d in set(AFCH['dataset'])]).fillna(0)
        fams = [('tri', 'triangular, k=2', OI['vermillion']), ('gauss_k2', 'Gaussian, k=2', OI['orange']),
                ('gauss_k4', 'Gaussian, k=4', OI['sky']), ('gauss_k8', 'Gaussian, k=8', OI['blue']), ('fallback', 'fallback', OI['grey'])]
        left = np.zeros(len(c))
        for key, lab, col in fams:
            if key in c and c[key].sum() > 0:
                v = c[key].to_numpy() / c[[k for k, _, _ in fams if k in c]].sum(axis=1).to_numpy()
                ax[0].barh(range(len(c)), v, left=left, color=col, label=lab, height=0.75); left += v
        ax[0].set_yticks(range(len(c))); ax[0].set_yticklabels([DS_NAME[d] for d in c.index]); ax[0].invert_yaxis()
        ax[0].set_xlabel('share of selections')
        ax[0].set_title('firing condition chosen', loc='left', fontsize=7)
        if len(LB):
            p = LB[LB['model'].isin([AF, V2])].pivot_table(index=['dataset', 'lag', 'h'], columns='model', values='rmse')
            rr = np.log(p[AF] / p[V2]).groupby(['dataset', 'h']).mean().unstack('h')
            rr = np.exp(rr.reindex([d for d in DS_ORDER if d in rr.index]))
            for j, h in enumerate(rr.columns):
                ax[1].scatter(rr[h], np.arange(len(rr)) + (j - 1) * 0.22, marker='os^'[j % 3], s=14,
                              color=[OI['vermillion'], OI['blue'], OI['green']][j % 3], label=f"h = {h}", zorder=3)
            ax[1].axvline(1, color='k', lw=0.7); ax[1].set_xscale('log')
            ax[1].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
            ax[1].set_yticks(range(len(rr))); ax[1].set_yticklabels([DS_NAME[d] for d in rr.index]); ax[1].invert_yaxis()
            ax[1].set_xlabel('RMSE AF / v2 (geo-mean over lags)')
            ax[1].set_title('adaptive vs fixed condition', loc='left', fontsize=7)
        if len(DM):
            t = DM.groupby(['dataset', 'outcome']).size().unstack(fill_value=0).reindex([d for d in DS_ORDER if d in set(DM['dataset'])])
            left = np.zeros(len(t))
            for key, col in (('AF better', OI['black']), ('no sig. diff.', '#d9d9d9'), ('v2 better', OI['vermillion'])):
                if key in t:
                    ax[2].barh(range(len(t)), t[key], left=left, color=col, label=key, height=0.75); left += t[key].to_numpy()
            ax[2].set_yticks(range(len(t))); ax[2].set_yticklabels([]); ax[2].invert_yaxis()
            ax[2].set_xlabel('problems (DM, Holm)')
            ax[2].set_title('per-problem tests', loc='left', fontsize=7)
            num('DM AF vs v2', DM['outcome'].value_counts().to_dict())
        for a, l in zip(ax, 'abc'):
            labelled(a, f"({l})")
        hl = [a.get_legend_handles_labels() for a in ax]
        fig.legend(sum([h_[0] for h_ in hl], []), sum([h_[1] for h_ in hl], []), frameon=False, fontsize=6, ncol=4,
                   loc='outside lower center')
        save(fig, 'fig6_af')


    def fig_variance():
        if not (len(SEED) or len(BLK)):
            return
        fig = plt.figure(figsize=(W2, 4.9), layout='constrained')
        ax = list(fig.subplot_mosaic([['a', 'b'], ['c', 'c']], height_ratios=[1.15, 1]).values())
        if len(SEED_D):
            sd = SEED_D.set_index('model')
            order = [m for m in NETS if m in sd.index]
            data = [sd.loc[m].astype(float).dropna().to_numpy() * 100 for m in order]
            bp = hbox(ax[0], data, widths=0.6, showfliers=False, patch_artist=True)
            for patch, m in zip(bp['boxes'], order):
                patch.set_facecolor(sty(m)['color']); patch.set_alpha(0.35)
            for i, (m, v) in enumerate(zip(order, data)):
                ax[0].scatter(v, np.full(len(v), i + 1) + np.random.default_rng(i).uniform(-0.15, 0.15, len(v)), s=4,
                              color=sty(m)['color'], zorder=3)
            ax[0].set_yticks(range(1, len(order) + 1)); ax[0].set_yticklabels([plain(m) for m in order]); ax[0].invert_yaxis()
            ax[0].set_xlabel('seed CV of RMSE (%), median per dataset'); ax[0].set_title('60 seeds per network', loc='left', fontsize=7)
        if len(BLK):
            b = BLK.copy(); b['cv'] = b['std'] / b['mean'] * 100
            order = [m for m in MODELS if m in set(b['model']) and m != 'Persistence']
            data = [b[b['model'] == m]['cv'].dropna().to_numpy() for m in order]
            bp = hbox(ax[1], data, widths=0.6, showfliers=False, patch_artist=True)
            for patch, m in zip(bp['boxes'], order):
                patch.set_facecolor(sty(m)['color']); patch.set_alpha(0.45)
            ax[1].set_yticks(range(1, len(order) + 1)); ax[1].set_yticklabels([plain(m) for m in order], fontsize=5.8); ax[1].invert_yaxis()
            ax[1].set_xlabel('block-to-block CV (%), per dataset'); ax[1].set_title('5 test blocks', loc='left', fontsize=7)
        if len(WIN):
            w = WIN.set_index('model')
            col = 'AF more accurate (h = 1)'
            w = w.reindex([m for m in MODELS if m in w.index]).dropna(subset=[col])
            ax[2].barh(range(len(w)), w[col] * 100, color=[sty(m)['color'] for m in w.index], alpha=0.85)
            ax[2].axvline(50, color='k', lw=0.6, ls=':')
            ax[2].set_yticks(range(len(w))); ax[2].set_yticklabels([plain(m) for m in w.index], fontsize=5.8); ax[2].invert_yaxis()
            ax[2].set_xlabel('blocks where AF is more accurate (%)'); ax[2].set_xlim(0, 100)
            ax[2].set_title('block-wise wins of TRACE-SNP-AF', loc='left', fontsize=7)
        for a, l in zip(ax, 'abc'):
            labelled(a, f"({l})")
        save(fig, 'fig7_variance')


    def fig_energy():
        if not len(EN):
            return
        e = EN.set_index('model')
        mr = FR_ALL['mean_rank'] if FR_ALL else pd.Series(dtype=float)
        fig, ax = plt.subplots(1, 2, figsize=(W2, 4.7), gridspec_kw={'width_ratios': [1.25, 1]}, layout='constrained')
        pm, ens = 'Per-member inference (µJ/forecast, GPU)', 'Ensemble inference (µJ/reported forecast, GPU)'
        rows = []
        for m in e.index:
            if m in mr.index and np.isfinite(e.loc[m, pm]):
                rows.append((m, e.loc[m, pm], e.loc[m, ens], mr[m]))
        if len(AFEO) and AF in mr.index:
            v = AFEO[AFEO['h'] == 1]
            if len(v):
                gj = float(v.iloc[0][f'{AF} µJ/forecast (GPU)'])
                rows.append((AF, gj, gj, mr[AF]))
        placed = []
        for m, a, b_, r in sorted(rows, key=lambda t: t[3]):
            s_ = sty(m)
            ax[0].scatter(b_, r, marker=s_['marker'], color=s_['color'], s=34 if m in OURS else 20, zorder=3, edgecolor='k', lw=0.3)
            if abs(a - b_) / b_ > 0.05:
                ax[0].scatter(a, r, marker=s_['marker'], facecolor='none', edgecolor=s_['color'], s=20, zorder=3, lw=0.8)
                ax[0].plot([a, b_], [r, r], color=s_['color'], lw=0.5, ls=':')
            placed.append(Line2D([], [], marker=s_['marker'], color=s_['color'], mec='k', mew=0.3, ls='', ms=5, label=plain(m)))
        ax[0].set_xscale('log'); ax[0].invert_yaxis(); ax[0].set_xlabel('GPU inference energy per forecast (µJ)')
        if rows:
            ax[0].set_xlim(right=max(max(t[1], t[2]) for t in rows) * 2)
        ax[0].set_ylabel('mean rank over 11 datasets')
        fig.legend(handles=placed + [Line2D([], [], marker='o', color='k', ls='', ms=4, label='filled: per reported forecast'),
                                     Line2D([], [], marker='o', mfc='none', color='k', ls='', ms=4, label='open: per member')],
                   frameon=False, fontsize=5.5, loc='outside lower center', ncol=4)
        ax[0].set_title('accuracy vs energy', loc='left', fontsize=7)
        c45m, c45e = 'Per-member, 45 nm (pJ)', 'Ensemble, 45 nm (pJ)'
        rows = [(m, e.loc[m, c45m], e.loc[m, c45e]) for m in e.index if np.isfinite(e.loc[m, c45m])]
        if len(AFEO):
            v = AFEO[AFEO['h'] == 1]
            if len(v):
                pj = float(v.iloc[0][f'{AF} 45 nm pJ']); rows.append((AF, pj, pj))
        rows.sort(key=lambda t: t[2])
        y = np.arange(len(rows))
        ax[1].barh(y + 0.2, [r[2] for r in rows], height=0.4, color=[sty(r[0])['color'] for r in rows], label='ensemble')
        ax[1].barh(y - 0.2, [r[1] for r in rows], height=0.4, color='white', edgecolor=[sty(r[0])['color'] for r in rows],
                   lw=0.8, label='per member')
        ax[1].set_xscale('log'); ax[1].set_yticks(y); ax[1].set_yticklabels([plain(r[0]) for r in rows], fontsize=5.8)
        ax[1].set_xlabel('45 nm estimate per forecast (pJ)'); ax[1].legend(frameon=False, fontsize=5.5, loc='lower right')
        ax[1].set_title('operation-count estimate', loc='left', fontsize=7)
        for a in ax:                                              # labels on 1-2-5 ticks only, never on every minor tick
            lo_, hi_ = a.get_xlim()
            a.xaxis.set_major_locator(matplotlib.ticker.LogLocator(subs=(1.0, 2.0, 5.0) if hi_ / lo_ < 30 else (1.0,), numticks=6))
            a.xaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter('%g'))
            a.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        for a, l in zip(ax, 'ab'):
            labelled(a, f"({l})")
        save(fig, 'fig8_energy')


    def fig_noise():
        if not len(NOISE):
            return
        n = NOISE.set_index('model')
        lv = [c for c in n.columns]
        xs = [float(c) * 100 for c in lv]
        fig, ax = plt.subplots(2, 1, figsize=(W2, 6.0), height_ratios=[1, 1.25], layout='constrained')
        for m in [m for m in MODELS if m in n.index]:
            s_ = sty(m)
            ax[0].plot(np.maximum(xs, 0.1), n.loc[m].to_numpy(float), color=s_['color'], ls=s_['ls'], marker=s_['marker'],
                       lw=s_['lw'], ms=3, label=plain(m))
        if len(AFNZ) and AF in set(AFNZ['model']):               # TRACE-SNP-AF was run on the noisy inputs separately
            a_ = AFNZ.set_index('model').loc[AF]; s_ = sty(AF)
            ax[0].plot([float(c) * 100 for c in a_.index], a_.to_numpy(float), color=s_['color'], ls=s_['ls'],
                       marker=s_['marker'], lw=s_['lw'], ms=3, label=plain(AF))
        ax[0].set_xscale('log'); ax[0].set_xlabel('input noise (% of range; 0 plotted at 0.1)'); ax[0].set_ylabel('RMSE / clean persistence')
        ax[0].legend(frameon=False, fontsize=5, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0)); ax[0].set_title('median over datasets, lags, horizons', loc='left', fontsize=7)
        if len(NOISE_RK):
            r = NOISE_RK.set_index('model')['mean_rank'].sort_values()
            ax[1].barh(range(len(r)), r, color=[sty(m)['color'] for m in r.index]); ax[1].invert_yaxis()
            ax[1].set_yticks(range(len(r))); ax[1].set_yticklabels([plain(m) for m in r.index], fontsize=5.8)
            ax[1].set_xlabel('mean rank at 10 % noise (11 datasets)'); ax[1].set_title('ranking under noise', loc='left', fontsize=7)
        for a, l in zip(ax, 'ab'):
            labelled(a, f"({l})")
        save(fig, 'fig9_noise')


    def delay_curves(d):
        c = DELAY[DELAY['dataset'] == d]
        if c.empty:
            return None
        enc = c['enc'].mode().iloc[0]; c = c[c['enc'] == enc]
        lo = c.groupby('fold')['u'].min().max(); hi = c.groupby('fold')['u'].max().min()
        uu = np.linspace(lo, hi, 81); curves = {}
        for k, ck in c.groupby('delay'):
            F = np.array([np.interp(uu, cf['u'], cf['f']) for _, cf in ck.groupby('fold')])
            Sd = np.array([np.interp(uu, cf['u'], cf['se']) for _, cf in ck.groupby('fold')])
            band = 1.96 * np.sqrt((Sd ** 2).mean(0) + (F.var(0, ddof=1) if len(F) > 1 else 0))
            curves[k] = (F.mean(0), band)
        amp = pd.Series({k: v[0].max() - v[0].min() for k, v in curves.items()}).sort_values(ascending=False)
        return uu, curves, amp, enc, c['fold'].nunique(), int(c['lag'].iloc[0])


    def fig_delay():
        if not len(DELAY):
            return
        ds = [d for d in DS_ORDER if d in set(DELAY['dataset'])]
        ds = sorted(ds, key=lambda d: (d != 'MackeyGlass', DS_ORDER.index(d)))[:4]
        fig, axes = plt.subplots(1, len(ds), figsize=(W2, 2.3), squeeze=False)
        cols = [OI['vermillion'], OI['blue'], OI['green'], OI['purple']]
        for ax, d in zip(axes[0], ds):
            uu, curves, amp, enc, nf, L = delay_curves(d)
            forced = [18] if d == 'MackeyGlass' and 18 in curves else []
            show_k = forced + [k for k in amp.index if k not in forced][:4 - len(forced)]
            for i, k in enumerate(show_k):
                mu, band = curves[k]
                ax.fill_between(uu, mu - band, mu + band, color=cols[i], alpha=0.18, lw=0)
                ax.plot(uu, mu, color=cols[i], lw=1.6 if k in forced else 1.0, ls=['-', '--', '-.', ':'][i],
                        label=f"k = {k}" + (" (τ+1)" if k in forced else ''))
            ax.axhline(0, color='0.6', lw=0.5)
            ax.set_title(f"{DS_NAME[d]} (L = {L})", loc='left', fontsize=7)
            ax.set_xlabel('$x(t-k)$' if enc == 'lvl' else '$x(t-k)-x(t-1)$', fontsize=7)
            ax.legend(frameon=False, fontsize=5.5, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.3), columnspacing=0.8)
            ax.tick_params(labelsize=5.5)
            if forced:
                mu, band = curves[18]
                num('Fig delay MG k=18', f"amplitude rank {list(amp.index).index(18) + 1} of {len(amp)}; band excludes 0 on "
                                         f"{np.mean(np.abs(mu) > band):.0%} of range; top {list(amp.index[:4])}")
        axes[0, 0].set_ylabel('contribution $f_k$ (centred)')
        fig.tight_layout(); save(fig, 'fig10_delay')


    # =========================================================================================================
    # SUPPLEMENTARY FIGURES
    # =========================================================================================================
    def figS_bound():
        # the bound of Theorem S7 against the exact worst-case dropped activation, computed numerically
        fig, ax = plt.subplots(1, 2, figsize=(W2, 2.6), layout='constrained')
        ks = np.arange(1, 17)
        for w, col in ((0.5, OI['sky']), (1.0, OI['blue']), (2.0, OI['vermillion'])):
            bound = [sum(math.exp(-((n - 1) ** 2) / (8 * w * w)) for n in range(k + 1, 400)) for k in ks]
            c = np.arange(-60, 61); us = np.linspace(0, 0.5, 51); exact = []
            for k in ks:
                worst = 0.0
                for u in us:
                    d = np.abs(u - c); a = np.exp(-d ** 2 / (2 * w * w)); idx = np.argsort(d)
                    worst = max(worst, a[idx[k:]].sum())
                exact.append(worst)
            tot = np.sqrt(2 * np.pi) * w
            ax[0].semilogy(ks, bound, color=col, lw=1.2, label=f"bound, w = {w:g}")
            ax[0].semilogy(ks, exact, color=col, lw=0, marker='o', ms=3, label=f"exact worst case, w = {w:g}")
            ax[1].plot(ks, np.array(exact) / tot * 100, color=col, marker='o', ms=3, lw=1, label=f"w = {w:g}")
        ax[0].set_xlabel('firing fields k'); ax[0].set_ylabel('dropped activation $\\epsilon_k(w)$')
        ax[0].set_ylim(1e-12, 20)
        ax[1].set_xlabel('firing fields k'); ax[1].set_ylabel('dropped share of activation (%)'); ax[1].set_yscale('log'); ax[1].set_ylim(1e-6, 150)
        for a, l in zip(ax, 'ab'):
            labelled(a, f"({l})")
        fig.legend(handles=[Line2D([], [], color='k', lw=1.2, label='bound (Theorem)'),
                            Line2D([], [], color='k', lw=0, marker='o', ms=3, label='exact worst case')]
                   + [Line2D([], [], color=c_, lw=4, label=f"w = {w_:g}") for w_, c_ in ((0.5, OI['sky']), (1.0, OI['blue']), (2.0, OI['vermillion']))],
                   frameon=False, fontsize=6, ncol=3, loc='outside lower center')
        save(fig, 'figS1_bound')


    def figS_cd_settings():
        if not len(LB):
            return
        # the 16-model comparison; the extended ranking is figS18. Total height kept within the Elsevier text block (it
        # overflowed the page before), label rows widened so that no two labels touch
        LBp = LB_MAIN
        sets = [(f"h = {h}", LBp[LBp['h'] == h]) for h in sorted(LBp['h'].unique())]
        S = scores(LBp)
        fig, axes = plt.subplots(len(sets) + 2, 1, figsize=(W2, min(2.3 * (len(sets) + 2), 8.8)))
        for ax, (t, sub) in zip(axes, sets):
            Sh = scores(sub); fr = friedman(Sh)
            cd_diagram(ax, fr, f"{t}: Friedman p = {fr['p']:.1e}, N = {fr['N']}", fs=6, row_h=0.62)
            num(f"Friedman {t}", f"chi2 = {fr['chi2']:.2f}, p = {fr['p']:.2e}, CD = {fr['cd']:.2f}")
        for ax, kd in zip(axes[len(sets):], ('chaotic', 'real')):
            Sk = S.loc[[d for d in S.index if DS_KIND[d] == kd]]; fr = friedman(Sk)
            cd_diagram(ax, fr, f"{kd} series: Friedman p = {fr['p']:.1e}, N = {fr['N']}", fs=6, row_h=0.62)
            num(f"Friedman {kd}", f"chi2 = {fr['chi2']:.2f}, p = {fr['p']:.2e}, CD = {fr['cd']:.2f}")
        fig.tight_layout(); save(fig, 'figS2_cd_settings')


    def figS_lag_profiles():
        if not len(LB):
            return
        for h in sorted(LB['h'].unique()):
            LBP = LB_EXT if EXTENDED_IN_FIGURES else LB          # [extended] added models as extra lines
            sub = LBP[LBP['h'] == h]
            fig, axes = plt.subplots(4, 3, figsize=(W2, 6.6)); axes = axes.ravel()
            for ax, d in zip(axes, DS_ORDER):
                g = sub[sub['dataset'] == d]
                for m in [m for m in MODELS if m in set(g['model'])]:
                    gg = g[g['model'] == m].sort_values('lag'); s_ = sty(m)
                    ax.plot(gg['lag'], gg['rel_rmse'], color=s_['color'], ls=s_['ls'], marker=s_['marker'], ms=2.5,
                            lw=1.4 if m in OURS else 0.6, alpha=1 if m in OURS else 0.75)
                ax.set_xscale('log'); ylog(ax); ax.set_xticks([1, 5, 10, 20, 30, 60])
                ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
                ax.set_xticklabels(['1', '5', '10', '20', '30', '60'], fontsize=5.5); ax.tick_params(labelsize=5.5)
                ax.set_title(DS_NAME[d], loc='left', fontsize=7)
            axes[-1].set_axis_off()
            axes[-1].legend(handles=[Line2D([], [], color=sty(m)['color'], ls=sty(m)['ls'], marker=sty(m)['marker'], ms=3,
                                            lw=1.2, label=plain(m)) for m in MODELS if m in set(sub['model'])],
                            frameon=False, fontsize=5.3, ncol=2, loc='center')
            fig.supxlabel('window length L', fontsize=7); fig.supylabel('RMSE / persistence', fontsize=7, x=0.0)
            fig.tight_layout(rect=(0.025, 0, 1, 1)); save(fig, f"figS3_lags_h{h}")   # y label clear of the tick labels


    def lyap_note(d, h):
        # [extended] the longest horizon in Lyapunov times, for the chaotic series ('' when R10 has not been run)
        if not len(LYAP) or DS_KIND.get(d) != 'chaotic':
            return ''
        key = {'MackeyGlass': 'Mackey', 'lorenz': 'Lorenz', 'lorenz_noisy': 'Lorenz', 'rossler': 'ssler'}.get(d)
        row = LYAP[LYAP['system'].astype(str).str.contains(key)] if key else LYAP.iloc[:0]
        if row.empty:
            return ''
        t = float(h) / float(row['Lyapunov time (samples)'].iloc[0])
        num(f"Lyapunov {d}", f"h = {h} is {t:.2f} Lyapunov times")
        return f" (h={h}: {t:.2f} $T_\\lambda$)"


    def figS_horizon():
        if not len(S_ALL):
            return
        LBP = LB_EXT if EXTENDED_IN_FIGURES else LB              # [extended] added models as extra lines
        fig, axes = plt.subplots(4, 3, figsize=(W2, 6.4)); axes = axes.ravel()
        for ax, d in zip(axes, DS_ORDER):
            g = LBP[LBP['dataset'] == d]
            p = np.exp(np.log(g.pivot_table(index=['h', 'lag'], columns='model', values='rel_rmse')).groupby(level='h').mean())
            for m in [m for m in MODELS if m in p.columns]:
                s_ = sty(m)
                ax.plot(p.index, p[m], color=s_['color'], ls=s_['ls'], marker=s_['marker'], ms=2.5, lw=1.4 if m in OURS else 0.6)
            ylog(ax); ax.set_xticks(list(p.index)); ax.tick_params(labelsize=5.5)
            ax.set_title(DS_NAME[d] + lyap_note(d, max(p.index)), loc='left', fontsize=7)
        axes[-1].set_axis_off()
        axes[-1].legend(handles=[Line2D([], [], color=sty(m)['color'], ls=sty(m)['ls'], marker=sty(m)['marker'], ms=3, lw=1.2,
                                        label=plain(m)) for m in MODELS if m in set(LBP['model'])], frameon=False, fontsize=5.3, ncol=2,
                        loc='center')
        fig.supxlabel('horizon h', fontsize=7); fig.supylabel('RMSE / persistence (geo-mean over lags)', fontsize=7, x=0.0)
        fig.tight_layout(rect=(0.025, 0, 1, 1)); save(fig, 'figS4_horizons')                # y label clear of the tick labels


    def figS_rbf_problem():
        if not len(RBF_P):
            return
        fig, ax = plt.subplots(1, 2, figsize=(W2, 2.6))
        for kd, col in (('real', OI['blue']), ('chaotic', OI['vermillion'])):
            g = RBF_P[RBF_P['dataset'].map(DS_KIND) == kd]
            ax[0].scatter(g['TRACE-SNP v2 (gated)'], g['Non-gated RBF'], s=6, color=col, label=f"{kd} ({len(g)} problems)", alpha=0.8)
        lim = [np.nanmin(RBF_P[['TRACE-SNP v2 (gated)', 'Non-gated RBF']].to_numpy()) * 0.7,
               np.nanmax(RBF_P[['TRACE-SNP v2 (gated)', 'Non-gated RBF']].to_numpy()) * 1.4]
        ax[0].plot(lim, lim, color='k', lw=0.6); ax[0].set_xscale('log'); ax[0].set_yscale('log')
        ax[0].set_xlabel('RMSE, gated (2 of M)'); ax[0].set_ylabel('RMSE, dense basis'); ax[0].legend(frameon=False, fontsize=6)
        if 'outcome' in RBF_P:
            t = RBF_P.groupby(['h', 'outcome']).size().unstack(fill_value=0)
            left = np.zeros(len(t))
            for key in t.columns:
                col = {'gated better': OI['black'], 'RBF better': OI['vermillion']}.get(key, '#d9d9d9')
                ax[1].barh(range(len(t)), t[key], left=left, color=col, label=key); left += t[key].to_numpy()
            ax[1].set_yticks(range(len(t))); ax[1].set_yticklabels([f"h = {h}" for h in t.index]); ax[1].invert_yaxis()
            ax[1].set_xlabel('problems (DM, Holm within dataset)'); ax[1].legend(frameon=False, fontsize=6, loc='upper left', bbox_to_anchor=(1.01, 1.0))
        for a, l in zip(ax, 'ab'):
            labelled(a, f"({l})")
        fig.tight_layout(); save(fig, 'figS5_rbf_problems')


    def figS_af_lag():
        if not len(AF_CHO) or 'field_family' not in AF_CHO:
            return
        fams = ['tri', 'gauss_k2', 'gauss_k4', 'gauss_k8', 'fallback']
        colors = [OI['vermillion'], OI['orange'], OI['sky'], OI['blue'], OI['grey']]
        fig, axes = plt.subplots(4, 3, figsize=(W2, 6.0)); axes = axes.ravel()
        for ax, d in zip(axes, DS_ORDER):
            g = AF_CHO[AF_CHO['dataset'] == d]
            if g.empty:
                ax.set_axis_off(); continue
            t = g.groupby(['lag', 'field_family']).size().unstack(fill_value=0)
            t = t.div(t.sum(axis=1), axis=0)
            bottom = np.zeros(len(t))
            for f_, c_ in zip(fams, colors):
                if f_ in t:
                    ax.bar(range(len(t)), t[f_], bottom=bottom, color=c_, width=0.8); bottom += t[f_].to_numpy()
            ax.set_xticks(range(len(t))); ax.set_xticklabels(t.index, fontsize=5.5); ax.set_ylim(0, 1); ax.tick_params(labelsize=5.5)
            ax.set_title(DS_NAME[d], loc='left', fontsize=7)
        axes[-1].set_axis_off()
        axes[-1].legend(handles=[Patch(color=c_, label=l) for c_, l in zip(colors, ['triangular k=2', 'Gaussian k=2', 'Gaussian k=4',
                                                                                   'Gaussian k=8', 'fallback'])], frameon=False, loc='center')
        fig.supxlabel('window length L', fontsize=7); fig.supylabel('share of selections', fontsize=7)
        fig.tight_layout(); save(fig, 'figS6_af_by_lag')


    def figS_dm():
        if not len(DM):
            return
        hs = sorted(DM['h'].unique())
        fig, axes = plt.subplots(1, len(hs), figsize=(W2, 2.9), squeeze=False, layout='constrained')
        for ax, h in zip(axes[0], hs):
            g = DM[DM['h'] == h].copy()
            g['v'] = -np.sign(g['dm']) * np.minimum(-np.log10(g['p_holm'].clip(lower=1e-12)), 8)
            p = g.pivot_table(index='dataset', columns='lag', values='v').reindex([d for d in DS_ORDER if d in set(g['dataset'])])
            im = ax.imshow(p.to_numpy(), cmap='RdBu', vmin=-6, vmax=6, aspect='auto')
            ax.set_xticks(range(p.shape[1])); ax.set_xticklabels(p.columns, fontsize=6)
            ax.set_yticks(range(p.shape[0])); ax.set_yticklabels([DS_NAME[d] for d in p.index] if h == hs[0] else [], fontsize=6)
            for i in range(p.shape[0]):
                for j in range(p.shape[1]):
                    v = p.to_numpy()[i, j]
                    if np.isfinite(v) and abs(v) > -np.log10(ALPHA):
                        ax.text(j, i, '+' if v > 0 else '−', ha='center', va='center', fontsize=6, color='white')
            ax.set_title(f"h = {h}", loc='left', fontsize=7); ax.set_xlabel('L', fontsize=6.5)
        cb = fig.colorbar(im, ax=axes[0].tolist(), fraction=0.03, pad=0.02)
        cb.set_label('signed −log$_{10}$ p$_{Holm}$\n(+: AF better)', fontsize=6); cb.ax.tick_params(labelsize=5.5)
        save(fig, 'figS7_dm_af_v2')


    def figS_seed_heat():
        if not len(SEED_D):
            return
        s = SEED_D.set_index('model').reindex([m for m in NETS if m in set(SEED_D['model'])])
        s = s[[d for d in DS_ORDER if d in s.columns]] * 100
        fig, ax = plt.subplots(figsize=(W2, 2.4))
        im = ax.imshow(np.log10(s.to_numpy(float).clip(min=1e-3)), cmap='cividis', aspect='auto')
        for i in range(s.shape[0]):
            for j in range(s.shape[1]):
                ax.text(j, i, f"{s.iloc[i, j]:.1f}", ha='center', va='center', fontsize=5.3, color='white' if s.iloc[i, j] < np.nanmedian(s.to_numpy()) else 'k')
        ax.set_xticks(range(s.shape[1])); ax.set_xticklabels([DS_NAME[d] for d in s.columns], rotation=30, ha='right')
        ax.set_yticks(range(s.shape[0])); ax.set_yticklabels([plain(m) for m in s.index])
        cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.set_label('log$_{10}$ seed CV (%)', fontsize=6)
        fig.tight_layout(); save(fig, 'figS8_seed_cv')


    def figS_blocks():
        if not len(BLKS):
            return
        b = BLKS[(BLKS['h'] == 1)]
        keys = [m for m in KEY if m in set(b['model'])]
        fig, axes = plt.subplots(4, 3, figsize=(W2, 6.2)); axes = axes.ravel()
        for ax, d in zip(axes, DS_ORDER):
            g = b[b['dataset'] == d]
            if g.empty:
                ax.set_axis_off(); continue
            gb = g.groupby(['model', 'test_fold'])['rel'].apply(lambda v: float(np.exp(np.mean(np.log(v))))).reset_index()
            folds = sorted(gb['test_fold'].unique(), key=lambda s: int(re.findall(r'\d+', str(s))[0]) if re.findall(r'\d+', str(s)) else 0)
            for m in keys:
                gg = gb[gb['model'] == m].set_index('test_fold').reindex(folds); s_ = sty(m)
                ax.plot(range(len(folds)), gg['rel'], color=s_['color'], ls=s_['ls'], marker=s_['marker'], ms=2.5,
                        lw=1.4 if m in OURS else 0.7)
            ylog(ax); ax.set_xticks(range(len(folds))); ax.set_xticklabels([f"B{i + 1}" for i in range(len(folds))], fontsize=5.5)
            ax.tick_params(labelsize=5.5); ax.set_title(DS_NAME[d], loc='left', fontsize=7)
        axes[-1].set_axis_off()
        axes[-1].legend(handles=[Line2D([], [], color=sty(m)['color'], ls=sty(m)['ls'], marker=sty(m)['marker'], ms=3, lw=1.2,
                                        label=plain(m)) for m in keys], frameon=False, fontsize=5.6, ncol=1, loc='center')
        fig.supxlabel('walk-forward test block', fontsize=7); fig.supylabel('one-step RMSE / persistence (geo-mean over lags)', fontsize=7)
        fig.tight_layout(); save(fig, 'figS9_blocks')


    def figS_delay_maps():
        if not len(DELAY):
            return
        ds = [d for d in DS_ORDER if d in set(DELAY['dataset'])]
        nr = (len(ds) + 1) // 2
        fig, axes = plt.subplots(nr, 2, figsize=(W2, 2.3 * nr), squeeze=False, layout='constrained')
        for ax in axes.ravel()[len(ds):]:
            ax.set_axis_off()
        for ax, d in zip(axes.ravel(), ds):
            uu, curves, amp, enc, nf, L = delay_curves(d)
            ks = sorted(curves)
            Zm = np.array([curves[k][0] for k in ks])
            sig = np.array([np.abs(curves[k][0]) > curves[k][1] for k in ks])
            v = np.nanmax(np.abs(Zm))
            im = ax.imshow(Zm, aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v, extent=[uu[0], uu[-1], ks[-1] + 0.5, ks[0] - 0.5])
            ax.contour(uu, ks, sig.astype(float), levels=[0.5], colors='k', linewidths=0.4)
            ax.set_title(f"{DS_NAME[d]} (L = {L})", loc='left', fontsize=7); ax.set_xlabel('$x(t-k)$' if enc == 'lvl' else '$x(t-k)-x(t-1)$', fontsize=6.5)
            ax.tick_params(labelsize=5.5)
            cb = fig.colorbar(im, ax=ax, fraction=0.06, pad=0.02); cb.ax.tick_params(labelsize=5)
            cb.locator = matplotlib.ticker.MaxNLocator(nbins=4, symmetric=True); cb.update_ticks()
            ax.set_ylabel('delay k')
        save(fig, 'figS10_delay_maps')


    def figS_forecasts():
        if not (len(AF_PRED) and len(V2_PRED)):
            return
        hs_ = sorted(set(AF_PRED['h'])); hs_ = [hs_[0], hs_[-1]] if len(hs_) > 1 else hs_
        lags_ = sorted(set(AF_PRED['lag'])); LF = 20 if 20 in lags_ else min(lags_, key=lambda l: abs(l - 20))
        sets = [(d, h) for d in ('MackeyGlass', 'rossler', 'lorenz', 'Sunspots') for h in hs_]
        sets = [(d, h) for d, h in sets if d in set(AF_PRED['dataset'])]
        if not sets:
            return
        fig, axes = plt.subplots(len(sets) // 2 + len(sets) % 2, 2, figsize=(W2, 1.75 * (len(sets) // 2 + len(sets) % 2) + 0.5),
                                 squeeze=False, layout='constrained')
        for ax, (d, h) in zip(axes.ravel(), sets):
            a = AF_PRED[(AF_PRED['dataset'] == d) & (AF_PRED['h'] == h) & (AF_PRED['lag'] == LF)]
            v = V2_PRED[(V2_PRED['dataset'] == d) & (V2_PRED['h'] == h) & (V2_PRED['lag'] == LF) & (V2_PRED['model'] == V2)]
            if a.empty:
                ax.set_axis_off(); continue
            fold = a['test_fold'].iloc[-1]; a = a[a['test_fold'] == fold].sort_values('target_idx')
            v = v[v['test_fold'] == fold].sort_values('target_idx')
            ax.plot(a['target_idx'], a['y'], color='k', lw=1.0, label='observed')
            ax.plot(a['target_idx'], a['ens_pred'], color=OI['black'], lw=1.0, ls='--', label=AF)
            if len(v):
                ax.plot(v['target_idx'], v['ens_pred'], color=OI['vermillion'], lw=0.9, ls=':', label=V2)
            ra = float(np.sqrt(np.mean((a['y'] - a['ens_pred']) ** 2)))
            rv = float(np.sqrt(np.mean((v['y'] - v['ens_pred']) ** 2))) if len(v) else np.nan
            ax.set_title(f"{DS_NAME[d]}, h = {h}, L = {LF}\nRMSE: AF {ra:.3g}, v2 {rv:.3g}", loc='left', fontsize=6.3)
            ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins=5, integer=True))
            ax.tick_params(labelsize=5.5)
        for ax in axes.ravel():
            if ax.get_legend_handles_labels()[0]:
                fig.legend(*ax.get_legend_handles_labels(), frameon=False, fontsize=5.5, ncol=3, loc='outside upper center'); break
        for ax in axes.ravel()[len(sets):]:
            ax.set_axis_off()
        save(fig, 'figS11_forecasts')


    def figS_params():
        if not len(P2):
            return
        fig, ax = plt.subplots(1, 4, figsize=(W2, 2.1))
        p = P2[P2['dataset'].isin(DS_ORDER)]
        for a, col, title in zip(ax, ['anchor', 'enc', 'M', 'iterated'], ['anchor', 'encoding', 'fields per delay M', 'iterated (h > 1)']):
            if col not in p:
                a.set_axis_off(); continue
            sub = p if col != 'iterated' else p[p['h'] > 1]
            t = sub.groupby([sub['dataset'].map(DS_KIND), col]).size().unstack(fill_value=0)
            t = t.div(t.sum(axis=1), axis=0)
            bottom = np.zeros(len(t))
            for j, c_ in enumerate(t.columns):
                a.bar(range(len(t)), t[c_], bottom=bottom, color=[OI['blue'], OI['vermillion'], OI['green'], OI['orange']][j % 4],
                      label=str(c_), width=0.7); bottom += t[c_].to_numpy()
            a.set_xticks(range(len(t))); a.set_xticklabels(t.index, fontsize=6); a.set_ylim(0, 1)
            a.set_title(title, loc='left', fontsize=7); a.legend(frameon=False, fontsize=5.3, loc='lower center', bbox_to_anchor=(0.5, -0.55), ncol=2)
        fig.tight_layout(); save(fig, 'figS12_v2_choices')


    def fig_ext_cd():
        # [extended] figS18: the N = 11 ranking with the classical forecasters and the dense basis, all models and without
        # the three Transformers (the 16-model diagram is Figure 5 of the main paper)
        if FR_EXT is None or not EXT_ADDED:
            return
        sets = [('(a) all models', LB_EXT), ('(b) without the three Transformers', LB_EXT[~LB_EXT['model'].isin(TRANS)])]
        frs = []
        for t, sub in sets:
            S_ = scores(sub)
            if S_.shape[1] >= 3:
                frs.append((t, friedman(S_)))
        hs = [max(2.2, 0.45 + 0.12 * fr['k']) for _, fr in frs]
        fig, axes = plt.subplots(len(frs), 1, figsize=(W2, min(sum(hs), 8.8)), height_ratios=hs, squeeze=False)
        for ax, (t, fr) in zip(axes[:, 0], frs):
            cd_diagram(ax, fr, f"{t}: Friedman $\\chi^2_{{{fr['k'] - 1}}}$ = {fr['chi2']:.1f}, p = {fr['p']:.1e}, N = {fr['N']}, "
                               f"k = {fr['k']}", fs=6, row_h=0.42)
            num(f"extended ranking {t}", {m: round(v, 2) for m, v in fr['mean_rank'].items()} | {'CD': round(fr['cd'], 2)})
        fig.tight_layout(); save(fig, 'figS18_extended_cd')


    EXT_PAIR = {}                                                # rival -> dict(ratio, wl, p, p_holm, pb, ratio_real, ...)


    def fig_ext_h2h():
        # [extended] figS19: TRACE-SNP-AF against each classical forecaster, the dense basis, DLinear and Ridge AR(L)
        if not len(S_EXT) or AF not in S_EXT:
            return
        rivals = [m for m in EXT_ADDED + ['DLinear', 'Ridge AR(L)'] if m in S_EXT.columns]
        if not rivals:
            return
        fig, ax = plt.subplots(1, 2, figsize=(W2, 3.0), gridspec_kw={'width_ratios': [1.6, 1]}, layout='constrained')
        y = np.arange(len(S_EXT.index)); rows = []
        is_real = np.array([DS_KIND[x] == 'real' for x in S_EXT.index])
        for j, m in enumerate(rivals):
            r = S_EXT[AF] / S_EXT[m]; s_ = sty(m)
            ax[0].scatter(r, y + (j - (len(rivals) - 1) / 2) * 0.11, marker=s_['marker'], color=s_['color'], s=16, zorder=3,
                          label=plain(m), lw=0.8)
            d = np.log(r.to_numpy())
            w = st.wilcoxon(d, zero_method='zsplit') if not np.allclose(d, 0) else None
            pb = bayes_signrank(d)
            rows.append({'Rival': name(m), 'Wins/losses': f"{int((d < 0).sum())}/{int((d > 0).sum())}", 'Ratio': float(np.exp(d.mean())),
                         'Ratio (real)': float(np.exp(d[is_real].mean())), 'Ratio (chaotic)': float(np.exp(d[~is_real].mean())),
                         '$p$': float(w.pvalue) if w else 1.0, 'P(AF better)': pb[0], 'P(equiv.)': pb[1], 'P(rival better)': pb[2],
                         '_m': m, '_wl_real': f"{int((d[is_real] < 0).sum())}/{int((d[is_real] > 0).sum())}"})
        ax[0].axvline(1, color='k', lw=0.7); ax[0].set_xscale('log')
        ax[0].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax[0].set_yticks(y); ax[0].set_yticklabels([DS_NAME[d] for d in S_EXT.index]); ax[0].invert_yaxis()
        ax[0].axhline(int(is_real.sum()) - 0.5, color='0.7', lw=0.6, ls='--')
        ax[0].set_xlabel('score TRACE-SNP-AF / rival (< 1: TRACE-SNP-AF more accurate)')
        ax[0].set_title('per dataset', loc='left', fontsize=7)
        t = pd.DataFrame(rows)
        t.insert(6, '$p_{\\mathrm{Holm}}$', holm(t['$p$'].to_numpy()))
        ax[1].barh(range(len(t)), t['Ratio'], color=[sty(m)['color'] for m in rivals])
        for i, (rr, rc) in enumerate(zip(t['Ratio (real)'], t['Ratio (chaotic)'])):
            ax[1].plot([rr], [i], marker='o', color=OI['blue'], ms=3.5, ls='')
            ax[1].plot([rc], [i], marker='s', color=OI['vermillion'], ms=3.5, ls='')
        ax[1].axvline(1, color='k', lw=0.7); ax[1].set_xscale('log')
        ax[1].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax[1].set_yticks(range(len(t))); ax[1].set_yticklabels([plain(m) for m in rivals]); ax[1].invert_yaxis()
        ax[1].set_xlabel('geometric-mean ratio over the datasets')
        ax[1].set_title('bars: all; circles: real; squares: chaotic', loc='left', fontsize=7)
        fig.legend(*ax[0].get_legend_handles_labels(), frameon=False, fontsize=6, ncol=4, loc='outside lower center')
        for a, l in zip(ax, 'ab'):
            labelled(a, f"({l})")
        save(fig, 'figS19_extended_h2h')
        for _, r in t.iterrows():
            EXT_PAIR[r['_m']] = r.to_dict()
        write_table(t.drop(columns=['_m', '_wl_real']), 'tab_ext_pairwise', fmt=lambda v: f"{v:.3f}")
        num('extended pairwise TRACE-SNP-AF', t.drop(columns=['_m']).to_dict('records'))


    EXT_RANKS = {}


    def ext_tables():
        # [extended] the extended ranking in the layout of Table 6, the classical forecasters per dataset, Lyapunov times,
        # constant forecasts, and every number the text quotes as a macro (tables/ext_macros.tex)
        if len(LB_EXT) and EXT_ADDED:
            cols = {}
            for t_, sub in [('All', LB_EXT)] + [(f"$h={h}$", LB_EXT[LB_EXT['h'] == h]) for h in sorted(LB_EXT['h'].unique())]:
                S_ = scores(sub)
                if S_.shape[1] >= 3:
                    cols[t_] = friedman(S_)['mean_rank']
            for kd in ('chaotic', 'real'):
                Sk = S_EXT.loc[[d for d in S_EXT.index if DS_KIND[d] == kd]]
                if len(Sk) >= 2:
                    cols[kd.capitalize()] = friedman(Sk)['mean_rank']
            t = pd.DataFrame(cols).sort_values('All')
            EXT_RANKS.update({k: v for k, v in cols.items()})
            lab = lambda m: name(m) + (' (ours)' if m in OURS else '')
            t.index = [lab(m) for m in t.index]
            write_table(t.reset_index().rename(columns={'index': 'Model'}), 'tab_ext_ranks', bold_min=list(cols), fmt=lambda v: f"{v:.2f}")
        stw = ext_csv('statistical_baselines.csv', index_col=0)
        if len(stw):
            stw = stw.reindex([d for d in DS_ORDER if d in stw.index])
            cc = [c for c in ('ARIMA order', 'ARIMA', 'SES', 'Holt damped', 'Seasonal naive', 'Training median') if c in stw]
            t = stw[cc].copy()
            for m in (AF, V2, 'DLinear'):
                if len(S_EXT) and m in S_EXT:
                    t[name(m)] = S_EXT[m].reindex(t.index)
            t.index = [DS_NAME[d] for d in t.index]
            if 'ARIMA order' in t:
                t['ARIMA order'] = t['ARIMA order'].astype(str).str.replace(' ', '')
            t = t.rename(columns={'Holt damped': 'Damped Holt', 'ARIMA order': '$(p,d,q)$'})
            write_table(t.reset_index().rename(columns={'index': 'Dataset'}), 'tab_ext_classical',
                        bold_min=[c for c in t.columns if c != '$(p,d,q)$'], fmt=lambda v: f"{v:.3f}" if v >= 0.01 else f"{v:.1e}",
                        small='\\scriptsize')
        if len(LYAP):
            rows = []
            for _, r in LYAP.iterrows():
                T_ = float(r['Lyapunov time (samples)'])
                rows.append({'Generator': str(r['system']), '$\\lambda_{\\max}$ per sample': 1 / T_, 'Lyapunov time (samples)': T_,
                             **{f"$h={h}$": h / T_ for h in (1, 6, 24)}})
            write_table(pd.DataFrame(rows), 'tab_ext_lyapunov', fmt=lambda v: f"{v:.4f}" if v < 0.1 else f"{v:.2f}" if v < 10 else f"{v:.1f}")
        r9 = ext_csv('constant_forecasts.csv', index_col=0)
        if len(r9):
            r9 = r9.reindex([d for d in DS_ORDER if d in r9.index])[[c for c in ('training mean', 'training median') if c in r9]]
            if len(S_EXT) and AF in S_EXT:
                r9[name(AF)] = S_EXT[AF].reindex(r9.index)
            r9.index = [DS_NAME[d] for d in r9.index]
            r9.columns = [c[0].upper() + c[1:] for c in r9.columns]
            write_table(r9.reset_index().rename(columns={'index': 'Dataset'}), 'tab_ext_constant', fmt=lambda v: f"{v:.3f}" if v >= 0.01 else f"{v:.1e}")


    def write_ext_macros():
        # [extended] every number the new sentences of main.tex / supplementary.tex quote; '--' = not computed (missing input)
        M = {}
        f2 = lambda v: '--' if v is None or not np.isfinite(v) else f"{v:.2f}"
        f3 = lambda v: '--' if v is None or not np.isfinite(v) else f"{v:.3f}"
        pc = lambda v, d=1: '--' if v is None or not np.isfinite(v) else f"{100 * v:.{d}f}\\%"
        def sci(v):
            if v is None or not np.isfinite(v):
                return '--'
            m_, e_ = f"{v:.1e}".split('e'); return f"{m_}\\times10^{{{int(e_)}}}"
        if FR_EXT is not None:
            mr = FR_EXT['mean_rank']
            M.update({'ExtK': str(FR_EXT['k']), 'ExtN': str(FR_EXT['N']), 'ExtCD': f2(FR_EXT['cd']), 'ExtChi': f"{FR_EXT['chi2']:.1f}",
                      'ExtDf': str(FR_EXT['k'] - 1), 'ExtP': sci(FR_EXT['p']), 'ExtPid': sci(FR_EXT['p_ID']),
                      'ExtFirst': tex_esc(name(mr.index[0])), 'ExtFirstRank': f2(mr.iloc[0]),
                      'ExtSecond': tex_esc(name(mr.index[1])), 'ExtSecondRank': f2(mr.iloc[1])})
            for key, m in (('AF', AF), ('Vtwo', V2), ('Vone', V1)) + ((('Dense', DENSE),) if EXT_DENSE else ()) + (
                           ('Arima', 'ARIMA'), ('Ses', 'SES'),
                           ('Holt', 'Holt damped'), ('Median', 'Training median'), ('Dlinear', 'DLinear'), ('Ridge', 'Ridge AR(L)')):
                M[f'ExtRank{key}'] = f2(mr.get(m, np.nan))
                M[f'ExtPos{key}'] = str(list(mr.index).index(m) + 1) if m in mr.index else '--'
            cls = [m for m in STATS if m in mr.index]
            if cls:
                b = min(cls, key=lambda m: mr[m]); M['ExtBestClassical'] = tex_esc(name(b)); M['ExtBestClassicalRank'] = f2(mr[b])
            S_ = scores(LB_EXT[~LB_EXT['model'].isin(TRANS)])
            if S_.shape[1] >= 3:
                fr = friedman(S_); M.update({'ExtNoTK': str(fr['k']), 'ExtNoTCD': f2(fr['cd']),
                                             'ExtNoTRankAF': f2(fr['mean_rank'].get(AF, np.nan)),
                                             'ExtNoTFirst': tex_esc(name(fr['mean_rank'].index[0]))})
            for kd, key in (('Real', 'Real'), ('Chaotic', 'Chaotic')):
                if kd in EXT_RANKS:
                    r_ = EXT_RANKS[kd]; M[f'Ext{key}RankAF'] = f2(r_.get(AF, np.nan)); M[f'Ext{key}First'] = tex_esc(name(r_.index[0]))
                    M[f'Ext{key}FirstRank'] = f2(r_.iloc[0])
            seas = [d for d in DS_ORDER if len(LB_EXT[(LB_EXT['model'] == 'Seasonal naive') & (LB_EXT['dataset'] == d)])]
            M['ExtSeasonalN'] = str(len(seas))
        for key, m in (('Arima', 'ARIMA'), ('Ses', 'SES'), ('Holt', 'Holt damped'), ('Median', 'Training median')) + (
                       (('Dense', DENSE),) if EXT_DENSE else ()) + (('Dlinear', 'DLinear'),):
            r = EXT_PAIR.get(m)
            M[f'Ext{key}Ratio'] = f3(r['Ratio']) if r else '--'
            M[f'Ext{key}RatioReal'] = f3(r['Ratio (real)']) if r else '--'
            M[f'Ext{key}RatioChaotic'] = f3(r['Ratio (chaotic)']) if r else '--'
            M[f'Ext{key}WL'] = r['Wins/losses'] if r else '--'
            M[f'Ext{key}WLReal'] = r['_wl_real'] if r else '--'
            M[f'Ext{key}Pholm'] = f3(r['$p_{\\mathrm{Holm}}$']) if r else '--'
            M[f'Ext{key}Pb'] = f2(r['P(AF better)']) if r else '--'
            M[f'Ext{key}Peq'] = f2(r['P(equiv.)']) if r else '--'
            M[f'Ext{key}Prival'] = f2(r['P(rival better)']) if r else '--'
        stw = ext_csv('statistical_baselines.csv', index_col=0)
        if len(stw) and 'ARIMA order' in stw:
            oc = stw['ARIMA order'].astype(str).str.replace(' ', '').value_counts()
            M['ExtArimaModalOrder'] = oc.index[0]; M['ExtArimaModalCount'] = str(int(oc.iloc[0])); M['ExtArimaOrders'] = str(len(oc))
        for key, pat in (('Lorenz', 'Lorenz'), ('Rossler', 'ssler'), ('Mg', 'Mackey')):
            r = LYAP[LYAP['system'].astype(str).str.contains(pat)] if len(LYAP) else pd.DataFrame()
            T_ = float(r['Lyapunov time (samples)'].iloc[0]) if len(r) else np.nan
            M[f'ExtLyapTime{key}'] = '--' if not np.isfinite(T_) else f"{T_:.0f}"
            M[f'ExtLyapExp{key}'] = '--' if not np.isfinite(T_) else f"{1 / T_:.4f}"
            M[f'ExtHorizon{key}'] = '--' if not np.isfinite(T_) else f"{24 / T_:.2f}"
        mg = LYAP[LYAP['system'].astype(str).str.contains('Mackey')] if len(LYAP) and 'note' in LYAP else pd.DataFrame()
        if len(mg):
            mres = re.findall(r'=\s*([0-9.eE+-]+)', str(mg['note'].iloc[0]))
            ok = bool(mres) and float(mres[0]) < 1e-6
            M['ExtMgGenerator'] = ('the unit-step Euler discretisation $x_{t+1}=x_t+0.2\\,x_{t-17}/(1+x_{t-17}^{10})-0.1\\,x_t$'
                                   if ok else 'a discretisation of the delay equation')
        r9 = ext_csv('constant_forecasts.csv', index_col=0)
        if len(r9) and 'training mean' in r9:
            re_ = r9.loc[[d for d in r9.index if DS_KIND.get(d) == 'real'], 'training mean']
            ch_ = r9.loc[[d for d in r9.index if DS_KIND.get(d) == 'chaotic'], 'training mean']
            M['ExtMeanRealMin'] = f2(re_.min()) if len(re_) else '--'; M['ExtMeanRealMax'] = f2(re_.max()) if len(re_) else '--'
            M['ExtMeanChaoticMin'] = f2(ch_.min()) if len(ch_) else '--'; M['ExtMeanChaoticMax'] = f2(ch_.max()) if len(ch_) else '--'
            M['ExtMeanBelowOne'] = str(int((r9['training mean'] < 1).sum()))
        man = ext_csv('data_manifest.csv', index_col=0)
        M['ExtEplColumn'] = tex_esc(str(man.loc['EPL', 'column'])) if len(man) and 'EPL' in man.index else '--'
        EPL_WORDS = {'BbAvAHA': 'average Asian-handicap odds for the away team (Betbrain average, column BbAvAHA)',
                     'BbAvAHH': 'average Asian-handicap odds for the home team (Betbrain average, column BbAvAHH)',
                     'TotalGoals': 'total number of goals'}
        _c = str(man.loc['EPL', 'column']) if len(man) and 'EPL' in man.index else ''
        M['ExtEplWords'] = tex_esc(EPL_WORDS.get(_c, f"values of column {_c}")) if _c else '--'
        chk = ext_csv('ops_formula_check.csv')
        M['ExtOpsMatch'] = pc(float(chk['share'].iloc[0])) if len(chk) and 'share' in chk else '--'
        M['ExtOpsRows'] = f"{int(chk['rows'].iloc[0]):,}".replace(',', '{,}') if len(chk) and 'rows' in chk else '--'
        fb = ext_csv('fallback_share_trace_af.csv', index_col=0)
        if len(fb) and 'any fallback' in fb:
            M['ExtFallbackMedian'] = pc(float(fb['any fallback'].median())); M['ExtFallbackMax'] = pc(float(fb['any fallback'].max()))
            M['ExtFallbackMaxDataset'] = tex_esc(DS_NAME.get(fb['any fallback'].idxmax(), fb['any fallback'].idxmax()))
        if len(S_EXT) and DENSE in S_EXT and AF in S_EXT:
            d_ = S_EXT[AF] / S_EXT[DENSE]
            M['ExtDenseBetterN'] = str(int((d_ > 1).sum()))
        # sentences chosen from the numbers, so the text needs no decision (\ExtSent... macros)
        g = lambda k, d=np.nan: (float(M[k].replace('\\%', '')) if M.get(k, '--') not in ('--', '') else d)
        rr = g('ExtArimaRatioReal')
        M['ExtSentClosing'] = (
            'The classical forecasters therefore do not change the picture: on real series they are on par with the closed-form '
            'members, and on chaotic series the closed-form members remain one to two orders of magnitude more accurate.'
            if not np.isfinite(rr) or rr <= 1.02 else
            f"On real series {M.get('ExtRealFirst', 'ARIMA')} is the strongest model of the study (mean rank \\ExtRealFirstRank{{}} on "
            'the seven real series, against \\ExtRealRankAF{} for TRACE-SNP-AF), which matches our claim of parity rather than '
            'superiority on real data; on chaotic series the closed-form members remain one to two orders of magnitude more accurate.')
        nb = g('ExtMeanBelowOne', 0)
        M['ExtSentConstantMain'] = (
            'Persistence is the stricter of the two simple references: a constant forecast at the training mean has a score between '
            '\\ExtMeanRealMin{} and \\ExtMeanRealMax{} on the real series and between \\ExtMeanChaoticMin{} and '
            '\\ExtMeanChaoticMax{} on the chaotic ones (Table~S\\ref{tab:ext_constant}).' if nb == 0 else
            'A constant forecast at the training mean is reported for comparison in Table~S\\ref{tab:ext_constant}.')
        M['ExtSentConstantSupp'] = (
            'Persistence is the stricter reference: Table~\\ref{tab:ext_constant} gives the score of a constant forecast at the '
            'training mean and median, which is above one on every series.' if nb == 0 else
            'Table~\\ref{tab:ext_constant} gives the score of a constant forecast at the training mean and median, which is below '
            'one on \\ExtMeanBelowOne{} of the 11 series.')
        fm = g('ExtFallbackMax', np.nan)
        M['ExtSentFallback'] = ('TRACE-SNP-AF never selects a fallback candidate.' if fm == 0 else
                                'A fallback (persistence, window mean or ridge AR) is selected in a median \\ExtFallbackMedian{} of '
                                'the (fold, lag, horizon) cases of a dataset, at most \\ExtFallbackMax{} (\\ExtFallbackMaxDataset).'
                                if np.isfinite(fm) else '')
        pa = M.get('ExtPosAF', '--')
        top3 = {M.get('ExtPosAF'), M.get('ExtPosVtwo'), M.get('ExtPosVone')}
        if not EXT_DENSE:
            M['ExtSentConclusion'] = (
                'The three closed-form members also take the first three places when five classical statistical forecasters '
                'join the comparison, ahead of every network and of ARIMA.' if top3 == {'1', '2', '3'} else
                'TRACE-SNP-AF also ranks first when five classical statistical forecasters join the comparison.' if pa == '1' else
                'When five classical statistical forecasters join the comparison, TRACE-SNP-AF has mean rank \\ExtRankAF{} among '
                '\\ExtK{} models.')
            M['ExtSentTopThree'] = ('The three closed-form members take the first three places.' if top3 == {'1', '2', '3'} else
                                    'TRACE-SNP-AF has position \\ExtPosAF{} of \\ExtK.')
        else:
          M['ExtSentConclusion'] = (
            'TRACE-SNP-AF also ranks first when five classical statistical forecasters and the dense basis join the comparison.'
            if pa == '1' else
            'When five classical statistical forecasters and the dense basis join the comparison, only the dense basis, which keeps '
            'every neuron active, ranks ahead of TRACE-SNP-AF.' if pa == '2' and M.get('ExtPosDense') == '1' else
            'When five classical statistical forecasters and the dense basis join the comparison, TRACE-SNP-AF has mean rank '
            '\\ExtRankAF{} among \\ExtK{} models.')
        with open(os.path.join(TAB_DIR, 'ext_macros.tex'), 'w') as fh:
            fh.write("% numbers of the extended comparison, written by PAPER FINAL — \\input{tables/ext_macros} in the preamble\n")
            for k, v in sorted(M.items()):
                fh.write(f"\\providecommand{{\\{k}}}{{}}\\renewcommand{{\\{k}}}{{{v}}}\n")
        MADE['tables'].append('ext_macros')
        num('extended macros', M)


    # =========================================================================================================
    # TABLES (\input by main.tex and supplementary.tex)
    # =========================================================================================================
    def tables():
        if len(S_ALL):
            order = list(FR_ALL['mean_rank'].index)
            t = S_ALL[order].T.copy(); t.columns = [DS_NAME[d] for d in t.columns]
            t.insert(0, 'Rank', FR_ALL['mean_rank'][order].to_numpy())
            t.index = [name(m) for m in t.index]; t = t.reset_index().rename(columns={'index': 'Model'})
            write_table(t, 'tab_scores', bold_min=[c for c in t.columns if c not in ('Model',)],
                        fmt=lambda v: f"{v:.3f}" if v >= 0.01 else f"{v:.1e}", small='\\scriptsize')
            for h in sorted(LB['h'].unique()):
                Sh = scores(LB[LB['h'] == h]); fr = friedman(Sh); o = list(fr['mean_rank'].index)
                t = Sh[o].T.copy(); t.columns = [DS_NAME[d] for d in t.columns]
                t.insert(0, 'Rank', fr['mean_rank'][o].to_numpy()); t.index = [name(m) for m in t.index]
                t = t.reset_index().rename(columns={'index': 'Model'})
                write_table(t, f"tab_scores_h{h}", bold_min=[c for c in t.columns if c != 'Model'],
                            fmt=lambda v: f"{v:.3f}" if v >= 0.01 else f"{v:.1e}", small='\\scriptsize')
            # pairwise Wilcoxon–Holm and Bayesian signed-rank, control = each of our closed-form models
            for ctrl, fname in ((AF, 'tab_pairwise_af'), (V2, 'tab_pairwise_v2')):
                S = S_ALL if ctrl == AF else S_ALL.drop(columns=[AF])
                rows = []
                for m in S.columns:
                    if m == ctrl:
                        continue
                    d = np.log(S[ctrl].to_numpy()) - np.log(S[m].to_numpy())      # < 0: control better
                    w = st.wilcoxon(d, zero_method='zsplit') if not np.allclose(d, 0) else None
                    pb = bayes_signrank(d)
                    rows.append({'Model': name(m), 'Wins/losses': f"{int((d < 0).sum())}/{int((d > 0).sum())}",
                                 'Ratio': float(np.exp(d.mean())), 'p': float(w.pvalue) if w else 1.0,
                                 'P(ctrl better)': pb[0], 'P(equiv.)': pb[1], 'P(rival better)': pb[2]})
                t = pd.DataFrame(rows); t.insert(4, 'p Holm', holm(t['p'].to_numpy())); t = t.sort_values('p')
                t.columns = ['Model', 'Wins/losses', 'Ratio', '$p$', '$p_{\\mathrm{Holm}}$', f"P({'AF' if ctrl == AF else 'v2'} better)",
                             'P(equiv.)', 'P(rival better)']
                write_table(t, fname, fmt=lambda v: f"{v:.3f}")
                num(f"pairwise {ctrl}", t.to_dict('records'))
            # Friedman per setting
            rows = []
            for t_, sub in [('all', LB)] + [(f"h = {h}", LB[LB['h'] == h]) for h in sorted(LB['h'].unique())]:
                fr = friedman(scores(sub)); rows.append({'Setting': t_, 'N': fr['N'], 'k': fr['k'], '$\\chi^2$': fr['chi2'],
                                                        '$p$': fr['p'], 'Iman--Davenport $p$': fr['p_ID'], 'CD': fr['cd']})
            for kd in ('chaotic', 'real'):
                fr = friedman(S_ALL.loc[[d for d in S_ALL.index if DS_KIND[d] == kd]])
                rows.append({'Setting': kd, 'N': fr['N'], 'k': fr['k'], '$\\chi^2$': fr['chi2'], '$p$': fr['p'],
                             'Iman--Davenport $p$': fr['p_ID'], 'CD': fr['cd']})
            t = pd.DataFrame(rows)
            write_table(t, 'tab_friedman', fmt=lambda v: f"{v:.3g}" if abs(v) < 1e-2 or abs(v) >= 1e3 else f"{v:.2f}")
        if len(BLK):
            b = BLK[BLK['model'].isin(MODELS)].copy()
            cell = b.apply(lambda r: f"{r['mean']:.3f} $\\pm$ {r['std']:.3f}", axis=1)
            b['cell'] = cell
            t = b.pivot(index='dataset', columns='model', values='cell')
            t = t.reindex([d for d in DS_ORDER if d in t.index])
            for part, ms in (('a', [AF, V2, V1, 'ALSTM-SNP', 'LSTM-SNP [25]', 'BiLSTM-SNP [26]', 'GRU [39]', 'LSTM [34]']),
                             ('b', ['Attention', 'Informer [42]', 'FEDformer [41]', 'Autoformer [40]', 'NG-RC', 'DLinear', 'Ridge AR(L)'])):
                tt = t[[m for m in ms if m in t.columns]].copy(); tt.columns = [name(m) for m in tt.columns]
                tt.index = [DS_NAME[d] for d in tt.index]
                write_table(tt.reset_index().rename(columns={'index': 'Dataset'}), f"tab_blocks_full_{part}", small='\\scriptsize')
        if len(SEED_D):
            s = SEED_D.set_index('model').reindex([m for m in NETS if m in set(SEED_D['model'])])
            s = s[[d for d in DS_ORDER if d in s.columns]] * 100
            s.columns = [DS_NAME[d] for d in s.columns]; s.index = [name(m) for m in s.index]
            write_table(s.reset_index().rename(columns={'index': 'Model'}), 'tab_seed_by_dataset', fmt=lambda v: f"{v:.1f}",
                        small='\\scriptsize')
        if len(WIN):
            w = WIN.copy(); w['model'] = w['model'].map(name)
            w.columns = ['Model', 'Blocks ($h=1$)', 'AF better ($h=1$)', 'Blocks (all $h$)', 'AF better (all $h$)']
            write_table(w, 'tab_winrate', fmt=lambda v: f"{v:.0f}" if v > 1.5 else f"{v:.3f}")
        if len(DM):
            t = DM.groupby(['dataset', 'outcome']).size().unstack(fill_value=0).reindex([d for d in DS_ORDER if d in set(DM['dataset'])])
            t.index = [DS_NAME[d] for d in t.index]
            t.loc['All'] = t.sum()
            write_table(t.reset_index().rename(columns={'index': 'Dataset'}), 'tab_dm_af_v2', fmt=lambda v: f"{v:.0f}")
        if len(BASIS_D):
            bd = BASIS_D.set_index('dataset').reindex([d for d in DS_ORDER if d in set(BASIS_D['dataset'])])
            bd = bd.drop(columns=['type'], errors='ignore'); bd.index = [DS_NAME[d] for d in bd.index]
            bd.columns = ['Tri.\\ $k$=2', 'Pw-lin.\\ dense', 'Gauss.\\ $k$=2', 'Gauss.\\ $k$=4', 'Gauss.\\ $k$=8', 'Gauss.\\ dense'][:bd.shape[1]]
            write_table(bd.reset_index().rename(columns={'index': 'Dataset'}), 'tab_basis_by_dataset', fmt=lambda v: f"{v:.3f}")
        if len(P2):
            p = P2[P2['dataset'].isin(DS_ORDER)]
            rows = []
            for d in DS_ORDER:
                g = p[p['dataset'] == d]
                if g.empty:
                    continue
                md = lambda c: g[c].mode().iloc[0] if c in g and g[c].notna().any() else np.nan
                rows.append({'Dataset': DS_NAME[d], 'Anchor': md('anchor'), 'Encoding': md('enc'), '$M$': f"{md('M'):.0f}",
                             '$\\lambda_0$': f"{md('l0'):.0e}", '$\\lambda_s$': f"{md('ls'):.0e}",
                             'State share': f"{g['family'].astype(str).str.contains('state').mean():.0%}".replace('%', '\\%'),
                             'Iterated ($h>1$)': f"{g[g['h'] > 1]['iterated'].astype(bool).mean():.0%}".replace('%', '\\%')})
            write_table(pd.DataFrame(rows), 'tab_v2_params')
        if len(NETP):
            t = NETP[['model', 'trainable_params', 'hidden', 'batch', 'lr', 'max_epochs', 'patience']].copy()
            t['model'] = t['model'].map(name)
            t.columns = ['Model', 'Parameters', 'Hidden', 'Batch', 'Learning rate', 'Max.\\ epochs', 'Patience']
            write_table(t, 'tab_nets', fmt=lambda v: f"{v:g}")


    # =========================================================================================================
    # PROTOCOL CONTROLS (cells 21–25): tuned and direct networks, ALSTM-SNP diagnostics, echo-type SNP
    # reservoir, one-protocol energy, sensitivity of the Bayesian analysis. Every number the text quotes from them is a
    # LaTeX macro in tables/r2_macros.tex, so the text and the tables always agree; '--' marks a result not yet computed.
    # =========================================================================================================
    R2_TUNE = R('r2_tuned_leaderboard', False); R2_TCH = R('r2_tuned_choices', False)
    R2_DIR_ = R('r2_direct_leaderboard', False); R2_DIAG = R('r2_alstm_diag', False)
    R2_ECHO = R('r2_echo_leaderboard', False); R2_EN = R('r2_energy_common', False)
    ECHO = 'Echo-SNP reservoir'
    MACRO_KEYS = ['RtuneRatioMin', 'RtuneRatioMax', 'RtuneGainMin', 'RtuneGainMax', 'RtuneN', 'RtuneBetter', 'RtuneSig', 'RtuneBayes', 'RdirRatioMin', 'RdirRatioMax', 'RdirGainMin', 'RdirGainMax', 'RdirN', 'RdirBetter', 'RdirBayes', 'RechoRatio', 'RechoPb', 'RechoWL', 'RechoAct', 'RechoRank', 'RechoAFRank', 'RechoK', 'RechoCD', 'RdiagActRef', 'RdiagMatched', 'RdiagNofire', 'RdiagNoadapt', 'RdiagGamma', 'RenVone', 'RenVtwo', 'RenAF', 'RbayesMin', 'RbayesMax']
    MACROS = {}


    def mac(key, value):
        MACROS[key] = value


    def _f(v, d=3):
        return '--' if v is None or not np.isfinite(v) else f"{v:.{d}f}"


    def _pct(v, d=1):
        return '--' if v is None or not np.isfinite(v) else f"{100 * v:.{d}f}\\%"


    def _gm(x):
        x = np.asarray(x, float); x = x[np.isfinite(x) & (x > 0)]
        return float(np.exp(np.mean(np.log(x)))) if len(x) else np.nan


    def _per_dataset_logratio(a, b):
        # a, b: frames with dataset, lag, h, rel_rmse → per dataset mean of log(a / b) over their common problems
        m = a.merge(b, on=['dataset', 'lag', 'h'], suffixes=('_a', '_b')).dropna(subset=['rel_rmse_a', 'rel_rmse_b'])
        m = m[(m['rel_rmse_a'] > 0) & (m['rel_rmse_b'] > 0)]
        if m.empty:
            return pd.Series(dtype=float)
        d = (np.log(m['rel_rmse_a']) - np.log(m['rel_rmse_b'])).groupby(m['dataset']).mean()
        return d.reindex([x for x in DS_ORDER if x in d.index])


    def _compare(d):
        # d: per-dataset log ratio (AF / rival); < 0 means TRACE-SNP-AF is more accurate
        d = np.asarray(d, float)
        if len(d) == 0:
            return dict(ratio=np.nan, wl='--', p=np.nan, pb=(np.nan, np.nan, np.nan), n=0)
        w = st.wilcoxon(d, zero_method='zsplit') if len(d) >= 2 and not np.allclose(d, 0) else None
        return dict(ratio=float(np.exp(d.mean())), wl=f"{int((d < 0).sum())}/{int((d > 0).sum())}",
                    p=float(w.pvalue) if w else 1.0, pb=bayes_signrank(d), n=len(d))


    def protocol_controls():
        af = LB[LB['model'] == AF][['dataset', 'lag', 'h', 'rel_rmse']] if len(LB) else pd.DataFrame()
        rows = []
        # 1. tuned networks (h = 1)
        gains, tune_rows = {}, []
        if len(R2_TUNE) and len(af):
            for m in [n for n in NETS if n in set(R2_TUNE['model'])]:
                g = R2_TUNE[R2_TUNE['model'] == m]
                c = _compare(_per_dataset_logratio(af, g[['dataset', 'lag', 'h', 'rel_rmse']]))
                dl = _per_dataset_logratio(g[['dataset', 'lag', 'h', 'rel_rmse']],
                                           g[['dataset', 'lag', 'h', 'default_rel_rmse']].rename(columns={'default_rel_rmse': 'rel_rmse'}))
                gains[m] = float(np.exp(dl.mean())) if len(dl) else np.nan
                tune_rows.append({'group': 'Networks tuned on validation ($h=1$)', 'Rival': name(m), **c})
        # 2. direct multi-step networks
        dir_gain, dir_rows = {}, []
        if len(R2_DIR_) and len(af):
            for m in [n for n in NETS if n in set(R2_DIR_['model'])]:
                g = R2_DIR_[R2_DIR_['model'] == m]
                c = _compare(_per_dataset_logratio(af, g[['dataset', 'lag', 'h', 'rel_rmse']]))
                dl = _per_dataset_logratio(g[['dataset', 'lag', 'h', 'rel_rmse']],
                                           g[['dataset', 'lag', 'h', 'iterated_rel_rmse']].rename(columns={'iterated_rel_rmse': 'rel_rmse'}))
                dir_gain[m] = float(np.exp(dl.mean())) if len(dl) else np.nan
                dir_rows.append({'group': 'Networks trained directly ($h=6, 24$)', 'Rival': name(m), **c})
        # 3. echo-type SNP reservoir
        echo_rows = []
        if len(R2_ECHO) and len(af):
            c = _compare(_per_dataset_logratio(af, R2_ECHO[['dataset', 'lag', 'h', 'rel_rmse']]))
            echo_rows.append({'group': 'SNP reservoir (all problems)', 'Rival': 'Echo-type SNP reservoir', **c})
        for grp in (tune_rows, dir_rows, echo_rows):
            if grp:
                ph = holm([r['p'] for r in grp])
                for r, p_ in zip(grp, ph):
                    r['p_holm'] = p_
                rows += grp
        if rows:
            lines = ['\\footnotesize', '\\begin{adjustbox}{max width=\\linewidth}', '\\begin{tabular}{lrrrrrr}', '\\toprule',
                     'Rival & Ratio & W/L & $p_{\\mathrm{Holm}}$ & P(AF) & P(eq.) & P(rival) \\\\']
            for grp in dict.fromkeys(r['group'] for r in rows):
                lines += ['\\midrule', f"\\multicolumn{{7}}{{l}}{{\\emph{{{grp}}}}} \\\\"]
                for r in [r for r in rows if r['group'] == grp]:
                    lines.append(f"{tex_esc(r['Rival'])} & {_f(r['ratio'])} & {r['wl']} & {_f(r['p_holm'])} & {_f(r['pb'][0], 2)} & "
                                 f"{_f(r['pb'][1], 2)} & {_f(r['pb'][2], 2)} \\\\")
            lines += ['\\bottomrule', '\\end{tabular}', '\\end{adjustbox}']
            with open(os.path.join(TAB_DIR, 'tab_r2_controls.tex'), 'w') as fh:
                fh.write('\n'.join(lines) + '\n')
            MADE['tables'].append('tab_r2_controls')
            num('protocol controls', [{k: v for k, v in r.items()} for r in rows])

        def rng(vals):
            v = [x for x in vals if np.isfinite(x)]
            return (min(v), max(v)) if v else (np.nan, np.nan)
        lo, hi = rng([r['ratio'] for r in tune_rows]); mac('RtuneRatioMin', _f(lo)); mac('RtuneRatioMax', _f(hi))
        lo, hi = rng(list(gains.values())); mac('RtuneGainMin', _f(lo)); mac('RtuneGainMax', _f(hi))
        mac('RtuneN', str(len(tune_rows)) if tune_rows else '--')
        mac('RtuneBetter', str(sum(r['ratio'] < 1 for r in tune_rows)) if tune_rows else '--')
        mac('RtuneSig', str(sum(r['p_holm'] < ALPHA for r in tune_rows)) if tune_rows else '--')
        mac('RtuneBayes', str(sum(r['pb'][0] >= 0.95 for r in tune_rows)) if tune_rows else '--')
        lo, hi = rng([r['ratio'] for r in dir_rows]); mac('RdirRatioMin', _f(lo)); mac('RdirRatioMax', _f(hi))
        lo, hi = rng(list(dir_gain.values())); mac('RdirGainMin', _f(lo)); mac('RdirGainMax', _f(hi))
        mac('RdirN', str(len(dir_rows)) if dir_rows else '--')
        mac('RdirBetter', str(sum(r['ratio'] < 1 for r in dir_rows)) if dir_rows else '--')
        mac('RdirBayes', str(sum(r['pb'][0] >= 0.95 for r in dir_rows)) if dir_rows else '--')
        e = echo_rows[0] if echo_rows else None
        mac('RechoRatio', _f(e['ratio']) if e else '--'); mac('RechoPb', _f(e['pb'][0], 2) if e else '--')
        mac('RechoWL', e['wl'] if e else '--')
        mac('RechoAct', _pct(float(R2_ECHO['activity'].median())) if len(R2_ECHO) and 'activity' in R2_ECHO else '--')
        # extended ranking with the reservoir (17 models)
        if len(R2_ECHO) and len(LB):
            MODELS.append(ECHO)                                  # scores() keeps only listed models
            try:
                S17 = scores(pd.concat([LB, R2_ECHO.assign(model=ECHO)[['dataset', 'model', 'lag', 'h', 'rel_rmse']]],
                                       ignore_index=True))
            finally:
                MODELS.remove(ECHO)
            cols = [m for m in MODELS if m in S17.columns] + ([ECHO] if ECHO in S17.columns else [])
            S17 = S17[cols].dropna()
            if ECHO in S17.columns and len(S17) >= 2:
                fr = friedman(S17); mr = fr['mean_rank']
                mac('RechoRank', _f(mr[ECHO], 2)); mac('RechoAFRank', _f(mr[AF], 2)); mac('RechoK', str(fr['k']))
                mac('RechoCD', _f(fr['cd'], 2))
                t = pd.DataFrame({'Model': [name(m) if m != ECHO else 'Echo-type SNP reservoir' for m in mr.index],
                                  'Mean rank': mr.to_numpy()})
                write_table(t, 'tab_r2_rank17', fmt=lambda v: f"{v:.2f}")
                num('ranking with the echo SNP reservoir', {m: round(v, 3) for m, v in mr.items()})
            per = R2_ECHO.groupby('dataset').agg(rel=('rel_rmse', _gm), act=('activity', 'median'))
            afd = af.groupby('dataset')['rel_rmse'].apply(_gm) if len(af) else pd.Series(dtype=float)
            per = per.reindex([d for d in DS_ORDER if d in per.index])
            t = pd.DataFrame({'Dataset': [DS_NAME[d] for d in per.index], 'Reservoir': per['rel'].to_numpy(),
                              'TRACE-SNP-AF': [afd.get(d, np.nan) for d in per.index], 'Reservoir activity': per['act'].to_numpy()})
            write_table(t, 'tab_r2_echo', fmt=lambda v: f"{v:.3f}")
        for k in ('RechoRank', 'RechoAFRank', 'RechoK', 'RechoCD'):
            MACROS.setdefault(k, '--')
        # tuning details
        if len(R2_TCH):
            t = R2_TCH.copy(); t['default'] = t['chosen'].astype(str).str.startswith('default')
            g = t.groupby('model').agg(folds=('chosen', 'size'), default=('default', 'mean'),
                                       top=('chosen', lambda s: s.value_counts().index[0]))
            g = g.reindex([m for m in NETS if m in g.index])
            tt = pd.DataFrame({'Model': [name(m) for m in g.index], 'Tuned / default': [gains.get(m, np.nan) for m in g.index],
                               'Default kept': g['default'].to_numpy(), 'Most frequent choice': g['top'].to_numpy()})
            write_table(tt, 'tab_r2_tuning', fmt=lambda v: f"{v:.3f}")
        if dir_rows:
            tt = pd.DataFrame({'Model': [r['Rival'] for r in dir_rows], 'Direct / iterated': [dir_gain.get(m, np.nan) for m in
                                                                                               [n for n in NETS if n in set(R2_DIR_['model'])]],
                               'AF / direct': [r['ratio'] for r in dir_rows]})
            write_table(tt, 'tab_r2_direct', fmt=lambda v: f"{v:.3f}")
        # 4. ALSTM-SNP diagnostics
        diag = {}
        if len(R2_DIAG):
            sc = R2_DIAG.groupby(['variant', 'dataset'])['rel_rmse'].apply(_gm).unstack('variant')
            act = R2_DIAG.groupby(['variant', 'dataset'])['fire_rate'].mean().unstack('variant')
            ep = R2_DIAG.groupby('variant')['best_epoch_median'].median()
            if 'reference' in sc:
                rows_ = []
                for v in [c for c in ['reference', 'matched protocol', 'no firing condition', 'no threshold adaptation',
                                      'surrogate slope 1'] if c in sc]:
                    r = np.log(sc[v]) - np.log(sc['reference'])
                    ch = [d for d in r.index if DS_KIND.get(d) == 'chaotic']; re_ = [d for d in r.index if DS_KIND.get(d) == 'real']
                    diag[v] = (float(np.exp(r.mean())), float(act[v].median()))
                    rows_.append({'Variant': v[0].upper() + v[1:], 'All': float(np.exp(r.mean())),
                                  'Chaotic': float(np.exp(r[ch].mean())) if ch else np.nan,
                                  'Real': float(np.exp(r[re_].mean())) if re_ else np.nan,
                                  'Activity': float(act[v].median()), 'Best epoch': float(ep.get(v, np.nan))})
                write_table(pd.DataFrame(rows_), 'tab_r2_alstm', fmt=lambda v: f"{v:.3f}" if v < 10 else f"{v:.0f}")
                num('ALSTM-SNP diagnostics', rows_)
        mac('RdiagActRef', _pct(diag['reference'][1]) if 'reference' in diag else '--')
        for k, v in (('RdiagMatched', 'matched protocol'), ('RdiagNofire', 'no firing condition'),
                     ('RdiagNoadapt', 'no threshold adaptation'), ('RdiagGamma', 'surrogate slope 1')):
            mac(k, _f(diag[v][0]) if v in diag else '--')
        # 5. energy with one code path
        if len(R2_EN):
            e_ = R2_EN.groupby('model')[['infer_j_per_forecast', 'infer_s_per_forecast']].median() * 1e6
            e_ = e_.reindex([m for m in (V1, V2, AF) if m in e_.index])
            t = pd.DataFrame({'Model': [name(m) for m in e_.index], 'GPU energy (\\textmu J)': e_['infer_j_per_forecast'].to_numpy(),
                              'Time (\\textmu s)': e_['infer_s_per_forecast'].to_numpy()})
            write_table(t, 'tab_r2_energy', fmt=lambda v: f"{v:.3g}")
            for k, m in (('RenVone', V1), ('RenVtwo', V2), ('RenAF', AF)):
                mac(k, f"{e_.loc[m, 'infer_j_per_forecast']:.3g}" if m in e_.index and np.isfinite(e_.loc[m, 'infer_j_per_forecast']) else '--')
        for k in ('RenVone', 'RenVtwo', 'RenAF'):
            MACROS.setdefault(k, '--')
        # 6. sensitivity of the Bayesian signed-rank analysis (prior concentration × ROPE), TRACE-SNP-AF vs the networks
        if len(S_ALL) and AF in S_ALL:
            rows_ = []
            for prior in (0.5, 1.0, 2.0):
                for rope in (0.005, 0.01, 0.02):
                    ps = [bayes_signrank(np.log(S_ALL[AF].to_numpy()) - np.log(S_ALL[m].to_numpy()), rope=rope, prior=prior)[0]
                          for m in NETS if m in S_ALL]
                    pv2 = bayes_signrank(np.log(S_ALL[AF].to_numpy()) - np.log(S_ALL[V2].to_numpy()), rope=rope, prior=prior) \
                        if V2 in S_ALL else (np.nan,) * 3
                    rows_.append({'Prior': prior, 'ROPE': f"$\\pm${100 * rope:g}\\%", 'Min.\\ P(AF better)': min(ps),
                                  'Median': float(np.median(ps)), 'Networks with P $\\geq$ 0.95': float(sum(p >= 0.95 for p in ps)),
                                  'P(v2 better)': pv2[2]})
            t = pd.DataFrame(rows_)
            write_table(t, 'tab_r2_bayes', fmt=lambda v: f"{v:.0f}" if float(v).is_integer() and v >= 1 else f"{v:.2f}")
            mac('RbayesMin', _f(float(t['Min.\\ P(AF better)'].min()), 2))
            mac('RbayesMax', _f(float(t['Min.\\ P(AF better)'].max()), 2))
        for k in ('RbayesMin', 'RbayesMax'):
            MACROS.setdefault(k, '--')


    def write_macros():
        for k in MACRO_KEYS:
            MACROS.setdefault(k, '--')
        with open(os.path.join(TAB_DIR, 'r2_macros.tex'), 'w') as fh:
            for k, v in sorted(MACROS.items()):
                fh.write(f"\\providecommand{{\\{k}}}{{}}\\renewcommand{{\\{k}}}{{{v}}}\n")
        num('macros', MACROS)


    # =========================================================================================================
    # run
    # =========================================================================================================
    for f in (fig_arch_alstm, fig_arch_trace, fig_family, fig_datasets, fig_cd, fig_scores, fig_ablation, fig_af, fig_variance, fig_energy, fig_noise, fig_delay,
              figS_bound, figS_cd_settings, figS_lag_profiles, figS_horizon, figS_rbf_problem, figS_af_lag, figS_dm, figS_seed_heat,
              figS_blocks, figS_delay_maps, figS_forecasts, figS_params, fig_ext_cd, fig_ext_h2h, tables, ext_tables,
              protocol_controls, write_ext_macros):
        try:
            f()
        except Exception as e:
            print(f"WARNING: {f.__name__} failed: {type(e).__name__}: {e}")
            plt.close('all')
    write_macros()
    # =========================================================================================================
    # LaTeX sources: main.tex, supplementary.tex and refs.bib, written next to figures/ and tables/ so that
    # paper_final/ compiles as it is (pdfLaTeX + BibTeX; elsarticle.cls is on Overleaf and in TeX Live).
    # =========================================================================================================
    LATEX_SOURCES = {}                                         # the LaTeX sources are not needed for the figures
    for _n, _t in LATEX_SOURCES.items():
        with open(os.path.join(PAPER_DIR, _n), 'w', encoding='utf-8') as fh:
            fh.write(_t)
    with open(os.path.join(PAPER_DIR, 'numbers.md'), 'w') as fh:
        fh.write(f"# Numbers behind paper_final — {time.strftime('%Y-%m-%d %H:%M')}\n\n" + "\n".join(map(str, NUMBERS)) + "\n")
    with zipfile.ZipFile('paper_final.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for p in glob.glob(os.path.join(PAPER_DIR, '**', '*'), recursive=True):
            if os.path.isfile(p):
                z.write(p, os.path.relpath(p, os.path.dirname(PAPER_DIR) or '.'))
    print(f"\n{len(MADE['figures'])} figures, {len(MADE['tables'])} tables → {os.path.abspath(PAPER_DIR)} and paper_final.zip")
    print("figures: " + ", ".join(MADE['figures']))
    print("tables:  " + ", ".join(MADE['tables']))
    return os.path.abspath(PAPER_DIR)


PAPER_FINAL = paper_final_cell(LOCAL_ROOT, TUNED, DIRECT, DIAG, ECHO_PREDS, EN_COMMON, EXT_CLASSICAL, EXT_LYAPUNOV)
